# BAIXANDO ARQUIVOS

In [3]:
from getpass import getpass
from huggingface_hub import snapshot_download

TOKEN = getpass("Cole o token 'Luís Paulo Guedes': ")

snapshot_download(
    repo_id="alan-turing-institute/turing-synthetic-radar-dataset",
    repo_type="dataset",
    local_dir=r"D:\Fusion\turing_synthetic_radar_dataset",
    token=TOKEN,
    max_workers=2,
)

print("Download concluído.")

Fetching ... files: 0it [00:00, ?it/s]

Download concluído.


# LENDO ARQUIVOS

In [1]:
# ============================================================
# read_turing_synthetic_radar_dataset.py
# ============================================================
# Lê/inspeciona:
#   D:\Fusion\turing_synthetic_radar_dataset\archive
#   D:\Fusion\turing_synthetic_radar_dataset\scan
#
# Saídas:
#   - manifest_files.csv
#   - preview_by_extension.csv
#   - printed summary no terminal
# ============================================================

from pathlib import Path
import os
import sys
import json
import tarfile
import zipfile
import subprocess
import warnings
warnings.filterwarnings("ignore")


# ============================================================
# INSTALL / IMPORTS
# ============================================================
def ensure_package(import_name, pip_name=None):
    try:
        __import__(import_name)
    except Exception:
        pkg = pip_name if pip_name is not None else import_name
        print(f"[INFO] Installing missing package: {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


ensure_package("numpy")
ensure_package("pandas")
ensure_package("scipy")
ensure_package("h5py")
ensure_package("PIL", "pillow")

import numpy as np
import pandas as pd
import scipy.io as sio
import h5py
from PIL import Image


# ============================================================
# PATHS
# ============================================================
ROOT = Path(r"D:\Fusion\turing_synthetic_radar_dataset")

DATA_DIRS = {
    "archive": ROOT / "archive",
    "scan": ROOT / "scan",
}

OUT_DIR = ROOT / "read_report"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_EAGER_MB = 300  # evita carregar arquivos .mat/.csv enormes sem necessidade


# ============================================================
# HELPERS
# ============================================================
def sizeof_mb(path: Path):
    try:
        return path.stat().st_size / (1024 ** 2)
    except Exception:
        return np.nan


def safe_json(obj):
    try:
        return json.dumps(obj, ensure_ascii=False)
    except Exception:
        return str(obj)


def list_files(base_dir: Path, source_name: str):
    rows = []

    if not base_dir.exists():
        print(f"[WARNING] Missing folder: {base_dir}")
        return rows

    for p in base_dir.rglob("*"):
        if p.is_file():
            rows.append({
                "source": source_name,
                "path": str(p),
                "relative_path": str(p.relative_to(base_dir)),
                "extension": p.suffix.lower(),
                "size_mb": round(sizeof_mb(p), 4),
            })

    return rows


def summarize_npy(path: Path):
    arr = np.load(path, mmap_mode="r")
    return {
        "type": "npy",
        "shape": tuple(arr.shape),
        "dtype": str(arr.dtype),
        "min": float(np.nanmin(arr)) if arr.size > 0 and arr.size < 50_000_000 else None,
        "max": float(np.nanmax(arr)) if arr.size > 0 and arr.size < 50_000_000 else None,
    }


def summarize_npz(path: Path):
    data = np.load(path, allow_pickle=False)
    keys = list(data.keys())

    out = {}
    for k in keys:
        arr = data[k]
        out[k] = {
            "shape": tuple(arr.shape),
            "dtype": str(arr.dtype),
        }

    return {
        "type": "npz",
        "keys": keys,
        "arrays": out,
    }


def summarize_mat_scipy(path: Path):
    mat = sio.loadmat(path)
    out = {}

    for k, v in mat.items():
        if k.startswith("__"):
            continue

        if isinstance(v, np.ndarray):
            out[k] = {
                "shape": tuple(v.shape),
                "dtype": str(v.dtype),
            }
        else:
            out[k] = {
                "type": str(type(v)),
            }

    return {
        "type": "mat_scipy",
        "variables": out,
    }


def walk_h5_group(g, prefix=""):
    out = {}

    for key in g.keys():
        item = g[key]
        name = f"{prefix}/{key}" if prefix else key

        if isinstance(item, h5py.Dataset):
            out[name] = {
                "shape": tuple(item.shape),
                "dtype": str(item.dtype),
            }
        elif isinstance(item, h5py.Group):
            out.update(walk_h5_group(item, prefix=name))

    return out


def summarize_h5(path: Path):
    with h5py.File(path, "r") as f:
        datasets = walk_h5_group(f)

    return {
        "type": "hdf5",
        "datasets": datasets,
    }


def summarize_mat(path: Path):
    """
    Tenta ler .mat tradicional com scipy.
    Se for MATLAB v7.3/HDF5, tenta com h5py.
    """
    try:
        return summarize_mat_scipy(path)
    except NotImplementedError:
        return summarize_h5(path)
    except ValueError:
        return summarize_h5(path)


def summarize_csv(path: Path):
    df = pd.read_csv(path, nrows=10)

    return {
        "type": "csv",
        "shape_preview": tuple(df.shape),
        "columns": list(df.columns),
        "head": df.head(3).to_dict(orient="records"),
    }


def summarize_txt(path: Path):
    lines = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for _ in range(10):
            line = f.readline()
            if not line:
                break
            lines.append(line.rstrip("\n"))

    return {
        "type": "text",
        "first_lines": lines,
    }


def summarize_json(path: Path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        obj = json.load(f)

    if isinstance(obj, dict):
        info = {
            "object_type": "dict",
            "keys": list(obj.keys())[:30],
        }
    elif isinstance(obj, list):
        info = {
            "object_type": "list",
            "length": len(obj),
            "first_item_type": str(type(obj[0])) if len(obj) > 0 else None,
        }
    else:
        info = {
            "object_type": str(type(obj)),
        }

    return {
        "type": "json",
        **info,
    }


def summarize_image(path: Path):
    with Image.open(path) as img:
        return {
            "type": "image",
            "mode": img.mode,
            "size": img.size,
        }


def summarize_zip(path: Path):
    with zipfile.ZipFile(path, "r") as z:
        names = z.namelist()

    return {
        "type": "zip",
        "n_files": len(names),
        "first_files": names[:20],
    }


def summarize_tar(path: Path):
    with tarfile.open(path, "r:*") as t:
        names = t.getnames()

    return {
        "type": "tar",
        "n_files": len(names),
        "first_files": names[:20],
    }


def read_file_preview(path: Path):
    ext = path.suffix.lower()
    size_mb = sizeof_mb(path)

    if ext == ".npy":
        return summarize_npy(path)

    if ext == ".npz":
        return summarize_npz(path)

    if ext == ".mat":
        if size_mb > MAX_EAGER_MB:
            # scipy .mat pode carregar tudo em RAM; por segurança, só tenta h5py.
            try:
                return summarize_h5(path)
            except Exception:
                return {
                    "type": "mat",
                    "warning": f"File larger than {MAX_EAGER_MB} MB. Skipped eager scipy load.",
                    "size_mb": round(size_mb, 4),
                }
        return summarize_mat(path)

    if ext in [".h5", ".hdf5", ".he5"]:
        return summarize_h5(path)

    if ext in [".csv"]:
        if size_mb > MAX_EAGER_MB:
            return {
                "type": "csv",
                "warning": f"Large CSV. Reading only header.",
                "columns": list(pd.read_csv(path, nrows=0).columns),
            }
        return summarize_csv(path)

    if ext in [".txt", ".log", ".md"]:
        return summarize_txt(path)

    if ext in [".json"]:
        return summarize_json(path)

    if ext in [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]:
        return summarize_image(path)

    if ext == ".zip":
        return summarize_zip(path)

    if ext in [".tar", ".gz", ".tgz", ".tar.gz"]:
        return summarize_tar(path)

    if ext in [".pkl", ".pickle"]:
        return {
            "type": "pickle",
            "warning": "Pickle was not loaded for safety.",
        }

    return {
        "type": "unknown",
        "message": "No reader implemented for this extension.",
    }


# ============================================================
# MAIN
# ============================================================
def main():
    print("=" * 80)
    print("SCANNING DATASET FOLDERS")
    print("=" * 80)

    all_rows = []

    for source_name, folder in DATA_DIRS.items():
        print(f"[SCAN] {source_name}: {folder}")
        rows = list_files(folder, source_name)
        print(f"  files found: {len(rows)}")
        all_rows.extend(rows)

    manifest = pd.DataFrame(all_rows)

    if len(manifest) == 0:
        print("[ERROR] No files found.")
        return

    manifest_path = OUT_DIR / "manifest_files.csv"
    manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 80)
    print("FILE EXTENSION SUMMARY")
    print("=" * 80)

    ext_summary = (
        manifest
        .groupby(["source", "extension"], dropna=False)
        .agg(
            n_files=("path", "count"),
            total_mb=("size_mb", "sum"),
            mean_mb=("size_mb", "mean"),
            max_mb=("size_mb", "max"),
        )
        .reset_index()
        .sort_values(["source", "n_files"], ascending=[True, False])
    )

    print(ext_summary.to_string(index=False))

    ext_summary_path = OUT_DIR / "extension_summary.csv"
    ext_summary.to_csv(ext_summary_path, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 80)
    print("READING ONE SAMPLE PER EXTENSION")
    print("=" * 80)

    preview_rows = []

    for (source, ext), g in manifest.groupby(["source", "extension"], dropna=False):
        sample_path = Path(g.sort_values("size_mb").iloc[0]["path"])

        print(f"\n[SAMPLE] source={source} | ext={ext} | file={sample_path.name}")

        try:
            preview = read_file_preview(sample_path)
            print(safe_json(preview)[:3000])

            preview_rows.append({
                "source": source,
                "extension": ext,
                "sample_path": str(sample_path),
                "status": "ok",
                "preview": safe_json(preview),
            })

        except Exception as e:
            print(f"[ERROR] {e}")

            preview_rows.append({
                "source": source,
                "extension": ext,
                "sample_path": str(sample_path),
                "status": "error",
                "preview": str(e),
            })

    preview_df = pd.DataFrame(preview_rows)
    preview_path = OUT_DIR / "preview_by_extension.csv"
    preview_df.to_csv(preview_path, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 80)
    print("DONE")
    print("=" * 80)
    print(f"Manifest saved to : {manifest_path}")
    print(f"Summary saved to  : {ext_summary_path}")
    print(f"Preview saved to  : {preview_path}")


if __name__ == "__main__":
    main()

SCANNING DATASET FOLDERS
[SCAN] archive: D:\Fusion\turing_synthetic_radar_dataset\archive
  files found: 3000
[SCAN] scan: D:\Fusion\turing_synthetic_radar_dataset\scan
  files found: 3009

FILE EXTENSION SUMMARY
 source extension  n_files  total_mb  mean_mb  max_mb
archive       .h5     3000 4003.1029 1.334368 18.9849
   scan       .h5     3000 5713.3852 1.904462  8.1370
   scan      .csv        6    1.4446 0.240767  1.2428
   scan      .txt        3    0.0000 0.000000  0.0000

READING ONE SAMPLE PER EXTENSION

[SAMPLE] source=archive | ext=.h5 | file=test_163.h5
{"type": "hdf5", "datasets": {"data": {"shape": [23, 5], "dtype": "float32"}, "labels": {"shape": [23, 1], "dtype": "int8"}, "metadata/feature_names": {"shape": [6], "dtype": "|S10"}, "metadata/transmitters": {"shape": [1], "dtype": "|S980"}}}

[SAMPLE] source=scan | ext=.csv | file=data_stats_global.csv
{"type": "csv", "shape_preview": [1, 31], "columns": ["Unnamed: 0", "n_trains", "mean_pulses", "max_pulses", "min_pulses", 

In [2]:
# ============================================================
# eda_turing_synthetic_radar_dataset_FIXED.py
# ============================================================
# Exploratory Data Analysis for:
#   D:\Fusion\turing_synthetic_radar_dataset\archive
#   D:\Fusion\turing_synthetic_radar_dataset\scan
#
# Dataset: Turing Synthetic Radar Dataset - TSRD
# Data type: Pulse Descriptor Words (PDWs)
#
# Expected H5 structure:
#   data   : [n_pulses, 5]
#   labels : [n_pulses, 1]
#
# PDW features:
#   ToA, Frequency, PulseWidth, AoA, Amplitude
#
# This version fixes:
#   - NaN / +inf / -inf in statistics and plots
#   - empty H5 files
#   - large sampled tables
#   - non-finite histogram errors
# ============================================================

from pathlib import Path
import sys
import json
import subprocess
import warnings
warnings.filterwarnings("ignore")


# ============================================================
# INSTALL / IMPORTS
# ============================================================
def ensure_package(import_name, pip_name=None):
    try:
        __import__(import_name)
    except Exception:
        pkg = pip_name if pip_name is not None else import_name
        print(f"[INFO] Installing missing package: {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


ensure_package("numpy")
ensure_package("pandas")
ensure_package("matplotlib")
ensure_package("h5py")
ensure_package("tabulate")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py


# ============================================================
# PATHS
# ============================================================
ROOT = Path(r"D:\Fusion\turing_synthetic_radar_dataset")

DATA_DIRS = {
    "archive": ROOT / "archive",
    "scan": ROOT / "scan",
}

OUT_DIR = ROOT / "eda_outputs"
PLOTS_DIR = OUT_DIR / "plots"
TABLES_DIR = OUT_DIR / "tables"

OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# CONFIG
# ============================================================
SEED = 42
rng = np.random.default_rng(SEED)

# Use None to process all files.
MAX_FILES_PER_SOURCE = None

# Sampling avoids loading all pulses into one dataframe.
SAMPLE_PULSES_PER_FILE = 2000
GLOBAL_SAMPLE_LIMIT = 1_000_000

# Read all labels per file to estimate number of emitters and imbalance.
READ_FULL_LABELS_FOR_FILE_STATS = True

PDW_FEATURES = [
    "ToA_us",
    "Frequency_MHz",
    "PulseWidth_us",
    "AoA_deg",
    "Amplitude_dB",
]

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 180


# ============================================================
# HELPERS
# ============================================================
def safe_json(obj):
    try:
        return json.dumps(obj, ensure_ascii=False)
    except Exception:
        return str(obj)


def finite_series(series):
    """
    Converts a Series to numeric and removes NaN, +inf and -inf.
    """
    s = pd.to_numeric(series, errors="coerce")
    s = s.replace([np.inf, -np.inf], np.nan)
    s = s.dropna()
    return s


def clean_numeric_df(df, cols):
    """
    Converts selected columns to numeric and removes non-finite values.
    """
    out = df.copy()
    for col in cols:
        out[col] = pd.to_numeric(out[col], errors="coerce")
        out[col] = out[col].replace([np.inf, -np.inf], np.nan)
    return out


def decode_h5_array(arr):
    arr = np.asarray(arr)

    if arr.dtype.kind == "S":
        return [x.decode("utf-8", errors="ignore") for x in arr.reshape(-1)]

    if arr.dtype.kind == "O":
        out = []
        for x in arr.reshape(-1):
            if isinstance(x, bytes):
                out.append(x.decode("utf-8", errors="ignore"))
            else:
                out.append(str(x))
        return out

    return arr.reshape(-1).tolist()


def h5_has_dataset(f, name):
    try:
        return isinstance(f[name], h5py.Dataset)
    except Exception:
        return False


def read_feature_names(f):
    possible_paths = [
        "metadata/feature_names",
        "feature_names",
    ]

    for p in possible_paths:
        if h5_has_dataset(f, p):
            names = decode_h5_array(f[p][()])
            names = [str(x).strip() for x in names]
            return names

    return []


def normalized_entropy_from_counts(counts):
    counts = np.asarray(counts, dtype=np.float64)
    total = counts.sum()

    if total <= 0:
        return np.nan

    p = counts / total
    p = p[p > 0]

    if len(p) <= 1:
        return 0.0

    h = -np.sum(p * np.log(p))
    return float(h / np.log(len(p)))


def q05(x):
    return x.quantile(0.05)
q05.__name__ = "q05"


def q25(x):
    return x.quantile(0.25)
q25.__name__ = "q25"


def q75(x):
    return x.quantile(0.75)
q75.__name__ = "q75"


def q95(x):
    return x.quantile(0.95)
q95.__name__ = "q95"


def flatten_columns(df):
    df = df.copy()
    df.columns = [
        "_".join([str(x) for x in col if str(x) != ""])
        if isinstance(col, tuple)
        else str(col)
        for col in df.columns
    ]
    return df


def safe_nanmin(x):
    x = np.asarray(x)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    return float(np.min(x))


def safe_nanmax(x):
    x = np.asarray(x)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    return float(np.max(x))


# ============================================================
# H5 INSPECTION
# ============================================================
def summarize_h5_file(path: Path, source: str):
    row = {
        "source": source,
        "file_name": path.name,
        "file_id": path.stem,
        "path": str(path),
        "size_mb": round(path.stat().st_size / (1024 ** 2), 6),
        "status": "ok",
        "has_data": False,
        "has_labels": False,
        "data_shape": None,
        "labels_shape": None,
        "n_pulses": 0,
        "n_features": 0,
        "feature_names_raw": None,
        "n_emitters": np.nan,
        "label_min": np.nan,
        "label_max": np.nan,
        "dominant_label": np.nan,
        "dominant_label_count": np.nan,
        "dominant_label_share": np.nan,
        "label_entropy_norm": np.nan,
        "toa_min": np.nan,
        "toa_max": np.nan,
        "toa_span": np.nan,
        "freq_min": np.nan,
        "freq_max": np.nan,
        "pulsewidth_min": np.nan,
        "pulsewidth_max": np.nan,
        "aoa_min": np.nan,
        "aoa_max": np.nan,
        "amp_min": np.nan,
        "amp_max": np.nan,
        "n_nonfinite_data_values": 0,
        "error": None,
    }

    label_count_rows = []

    try:
        with h5py.File(path, "r") as f:
            row["has_data"] = h5_has_dataset(f, "data")
            row["has_labels"] = h5_has_dataset(f, "labels")

            names = read_feature_names(f)
            row["feature_names_raw"] = safe_json(names)

            if row["has_data"]:
                data = f["data"]
                data_shape = tuple(data.shape)
                row["data_shape"] = str(data_shape)

                if len(data_shape) >= 1:
                    row["n_pulses"] = int(data_shape[0])
                if len(data_shape) >= 2:
                    row["n_features"] = int(data_shape[1])

                if row["n_pulses"] > 0 and row["n_features"] >= 5:
                    x = np.asarray(data[:, :5], dtype=np.float32)

                    nonfinite = int(np.sum(~np.isfinite(x)))
                    row["n_nonfinite_data_values"] = nonfinite

                    # Keep non-finite values out of min/max.
                    row["toa_min"] = safe_nanmin(x[:, 0])
                    row["toa_max"] = safe_nanmax(x[:, 0])
                    row["toa_span"] = (
                        float(row["toa_max"] - row["toa_min"])
                        if np.isfinite(row["toa_min"]) and np.isfinite(row["toa_max"])
                        else np.nan
                    )

                    row["freq_min"] = safe_nanmin(x[:, 1])
                    row["freq_max"] = safe_nanmax(x[:, 1])

                    row["pulsewidth_min"] = safe_nanmin(x[:, 2])
                    row["pulsewidth_max"] = safe_nanmax(x[:, 2])

                    row["aoa_min"] = safe_nanmin(x[:, 3])
                    row["aoa_max"] = safe_nanmax(x[:, 3])

                    row["amp_min"] = safe_nanmin(x[:, 4])
                    row["amp_max"] = safe_nanmax(x[:, 4])

            if row["has_labels"] and row["n_pulses"] > 0 and READ_FULL_LABELS_FOR_FILE_STATS:
                labels = np.asarray(f["labels"][()]).reshape(-1)

                if labels.size > 0:
                    labels = labels.astype(int)
                    uniq, cnt = np.unique(labels, return_counts=True)

                    row["n_emitters"] = int(len(uniq))
                    row["label_min"] = int(uniq.min())
                    row["label_max"] = int(uniq.max())

                    j = int(np.argmax(cnt))
                    row["dominant_label"] = int(uniq[j])
                    row["dominant_label_count"] = int(cnt[j])
                    row["dominant_label_share"] = float(cnt[j] / max(cnt.sum(), 1))

                    row["label_entropy_norm"] = normalized_entropy_from_counts(cnt)

                    for u, c in zip(uniq, cnt):
                        label_count_rows.append({
                            "source": source,
                            "file_id": path.stem,
                            "file_name": path.name,
                            "path": str(path),
                            "label": int(u),
                            "count": int(c),
                            "share_in_file": float(c / max(cnt.sum(), 1)),
                        })

    except Exception as e:
        row["status"] = "error"
        row["error"] = str(e)

    return row, label_count_rows


def build_inventory():
    inventory_rows = []
    label_rows = []

    for source, folder in DATA_DIRS.items():
        print("=" * 80)
        print(f"Scanning source: {source}")
        print(f"Folder: {folder}")
        print("=" * 80)

        h5_files = sorted(folder.rglob("*.h5"))

        if MAX_FILES_PER_SOURCE is not None:
            h5_files = h5_files[:int(MAX_FILES_PER_SOURCE)]

        print(f"H5 files found: {len(h5_files)}")

        for i, path in enumerate(h5_files, 1):
            row, labs = summarize_h5_file(path, source)
            inventory_rows.append(row)
            label_rows.extend(labs)

            if i % 250 == 0:
                print(f"  processed {i}/{len(h5_files)}")

    inventory_df = pd.DataFrame(inventory_rows)
    label_df = pd.DataFrame(label_rows)

    return inventory_df, label_df


# ============================================================
# PDW SAMPLING
# ============================================================
def sample_pdws_from_file(path: Path, source: str, file_id: str, max_samples: int):
    rows = []

    try:
        with h5py.File(path, "r") as f:
            if not h5_has_dataset(f, "data"):
                return rows

            data = f["data"]
            n = int(data.shape[0])

            if n <= 0:
                return rows

            n_features = int(data.shape[1]) if len(data.shape) > 1 else 0

            if n_features < 5:
                return rows

            sample_n = min(int(max_samples), n)

            if sample_n <= 0:
                return rows

            if sample_n < n:
                idx = rng.choice(n, size=sample_n, replace=False)
                idx = np.sort(idx)
            else:
                idx = np.arange(n)

            x = np.asarray(data[idx, :5], dtype=np.float32)

            # Critical fix: replace non-finite values before storing.
            x[~np.isfinite(x)] = np.nan

            if h5_has_dataset(f, "labels"):
                y = np.asarray(f["labels"][idx]).reshape(-1).astype(int)
            else:
                y = np.full(sample_n, -1, dtype=int)

            for j in range(sample_n):
                rows.append({
                    "source": source,
                    "file_id": file_id,
                    "pulse_index": int(idx[j]),
                    "label": int(y[j]),
                    "ToA_us": float(x[j, 0]) if np.isfinite(x[j, 0]) else np.nan,
                    "Frequency_MHz": float(x[j, 1]) if np.isfinite(x[j, 1]) else np.nan,
                    "PulseWidth_us": float(x[j, 2]) if np.isfinite(x[j, 2]) else np.nan,
                    "AoA_deg": float(x[j, 3]) if np.isfinite(x[j, 3]) else np.nan,
                    "Amplitude_dB": float(x[j, 4]) if np.isfinite(x[j, 4]) else np.nan,
                })

    except Exception as e:
        print(f"[Warning] Failed sampling {path}: {e}")

    return rows


def build_sampled_pdw_table(valid_df):
    sampled_rows = []
    total = 0

    print("=" * 80)
    print("SAMPLING PDWs")
    print("=" * 80)

    for i, row in valid_df.reset_index(drop=True).iterrows():
        remaining = GLOBAL_SAMPLE_LIMIT - total
        if remaining <= 0:
            break

        max_samples = min(SAMPLE_PULSES_PER_FILE, remaining)

        rows = sample_pdws_from_file(
            path=Path(row["path"]),
            source=row["source"],
            file_id=row["file_id"],
            max_samples=max_samples,
        )

        sampled_rows.extend(rows)
        total += len(rows)

        if (i + 1) % 250 == 0:
            print(f"  files processed: {i + 1}/{len(valid_df)} | sampled pulses: {total}")

    sampled_df = pd.DataFrame(sampled_rows)

    print(f"Total sampled pulses: {len(sampled_df)}")

    return sampled_df


# ============================================================
# STATS
# ============================================================
def compute_inventory_stats(inventory_df):
    valid_df = inventory_df[
        (inventory_df["status"] == "ok") &
        (inventory_df["has_data"] == True) &
        (inventory_df["has_labels"] == True) &
        (inventory_df["n_pulses"] > 0) &
        (inventory_df["n_features"] >= 5)
    ].copy()

    empty_df = inventory_df[
        (inventory_df["status"] == "ok") &
        (
            (inventory_df["n_pulses"] <= 0) |
            (inventory_df["has_data"] == False) |
            (inventory_df["has_labels"] == False)
        )
    ].copy()

    error_df = inventory_df[inventory_df["status"] == "error"].copy()

    source_summary = (
        inventory_df
        .groupby("source", as_index=False)
        .agg(
            n_files=("path", "count"),
            n_valid=("n_pulses", lambda x: int(np.sum(np.asarray(x) > 0))),
            total_size_mb=("size_mb", "sum"),
            total_pulses=("n_pulses", "sum"),
            mean_pulses=("n_pulses", "mean"),
            median_pulses=("n_pulses", "median"),
            max_pulses=("n_pulses", "max"),
            mean_emitters=("n_emitters", "mean"),
            median_emitters=("n_emitters", "median"),
            max_emitters=("n_emitters", "max"),
            mean_dominant_share=("dominant_label_share", "mean"),
            median_dominant_share=("dominant_label_share", "median"),
            total_nonfinite_values=("n_nonfinite_data_values", "sum"),
        )
    )

    return valid_df, empty_df, error_df, source_summary


def compute_sampled_feature_stats(sampled_df):
    if len(sampled_df) == 0:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    df = clean_numeric_df(sampled_df, PDW_FEATURES)

    nonfinite_summary = []
    for col in PDW_FEATURES:
        raw = pd.to_numeric(sampled_df[col], errors="coerce")
        n_total = len(raw)
        n_nan = int(raw.isna().sum())
        n_posinf = int(np.isposinf(raw.replace([np.inf, -np.inf], np.nan).values).sum()) if False else 0
        n_finite = int(np.isfinite(raw.dropna().values).sum())
        n_nonfinite = int(n_total - n_finite)

        nonfinite_summary.append({
            "feature": col,
            "n_total": n_total,
            "n_finite": n_finite,
            "n_nonfinite_or_nan": n_nonfinite,
            "share_nonfinite_or_nan": float(n_nonfinite / max(n_total, 1)),
        })

    nonfinite_df = pd.DataFrame(nonfinite_summary)

    feature_stats_by_source = (
        df
        .groupby("source")[PDW_FEATURES]
        .agg(["count", "mean", "std", "min", q05, q25, "median", q75, q95, "max"])
    )
    feature_stats_by_source = flatten_columns(feature_stats_by_source).reset_index()

    feature_stats_global = (
        df[PDW_FEATURES]
        .agg(["count", "mean", "std", "min", q05, q25, "median", q75, q95, "max"])
        .T
        .reset_index()
        .rename(columns={"index": "feature"})
    )

    return feature_stats_by_source, feature_stats_global, nonfinite_df


def compute_label_stats(label_df):
    if len(label_df) == 0:
        return pd.DataFrame(), pd.DataFrame()

    global_label_counts = (
        label_df
        .groupby(["source", "label"], as_index=False)
        .agg(count=("count", "sum"))
        .sort_values(["source", "label"])
    )

    global_label_counts["share_in_source"] = (
        global_label_counts["count"] /
        global_label_counts.groupby("source")["count"].transform("sum")
    )

    label_file_presence = (
        label_df
        .groupby(["source", "label"], as_index=False)
        .agg(
            n_files=("file_id", "nunique"),
            total_count=("count", "sum"),
            mean_share_in_file=("share_in_file", "mean"),
            max_share_in_file=("share_in_file", "max"),
        )
        .sort_values(["source", "label"])
    )

    return global_label_counts, label_file_presence


# ============================================================
# PLOTS
# ============================================================
def save_bar(series, title, ylabel, path):
    series = series.dropna()

    if len(series) == 0:
        print(f"[Warning] Skipping empty bar plot: {title}")
        return

    plt.figure(figsize=(8, 5))
    plt.bar(series.index.astype(str), series.values)
    plt.title(title)
    plt.ylabel(ylabel)
    plt.xlabel("")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()


def save_hist(series, title, xlabel, path, bins=60, log_y=False):
    series = finite_series(series)

    if len(series) == 0:
        print(f"[Warning] Skipping empty/non-finite histogram: {title}")
        return

    if not np.isfinite(series.min()) or not np.isfinite(series.max()):
        print(f"[Warning] Skipping non-finite histogram: {title}")
        return

    if series.min() == series.max():
        print(f"[Warning] Skipping constant histogram: {title}")
        return

    plt.figure(figsize=(8, 5))
    plt.hist(series.values, bins=bins)

    if log_y:
        plt.yscale("log")

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()


def save_boxplot_by_source(df, feature, path):
    data = []
    labels = []

    for src, g in df.groupby("source"):
        vals = finite_series(g[feature]).values

        if len(vals) > 0:
            data.append(vals)
            labels.append(src)

    if len(data) == 0:
        print(f"[Warning] Skipping empty/non-finite boxplot: {feature}")
        return

    plt.figure(figsize=(8, 5))
    plt.boxplot(data, labels=labels, showfliers=False)
    plt.title(f"{feature} distribution by source")
    plt.ylabel(feature)
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()


def save_correlation_heatmap(df, source_name, path):
    g = df[df["source"] == source_name].copy()

    if len(g) < 5:
        return

    g = clean_numeric_df(g, PDW_FEATURES)
    g = g.dropna(subset=PDW_FEATURES)

    if len(g) < 5:
        print(f"[Warning] Skipping correlation heatmap for {source_name}: insufficient finite data.")
        return

    corr = g[PDW_FEATURES].corr()

    if corr.isna().all().all():
        print(f"[Warning] Skipping correlation heatmap for {source_name}: all correlations NaN.")
        return

    plt.figure(figsize=(7, 6))
    plt.imshow(corr.values, aspect="auto", vmin=-1, vmax=1)
    plt.colorbar(label="Correlation")
    plt.xticks(np.arange(len(PDW_FEATURES)), PDW_FEATURES, rotation=45, ha="right")
    plt.yticks(np.arange(len(PDW_FEATURES)), PDW_FEATURES)
    plt.title(f"Feature correlation - {source_name}")
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()


def save_scatter(df, source_name, xcol, ycol, path, max_points=50_000):
    g = df[df["source"] == source_name].copy()

    g[xcol] = pd.to_numeric(g[xcol], errors="coerce")
    g[ycol] = pd.to_numeric(g[ycol], errors="coerce")

    g[xcol] = g[xcol].replace([np.inf, -np.inf], np.nan)
    g[ycol] = g[ycol].replace([np.inf, -np.inf], np.nan)

    g = g.dropna(subset=[xcol, ycol])

    if len(g) == 0:
        print(f"[Warning] Skipping scatter {ycol} vs {xcol} for {source_name}: no finite data.")
        return

    if len(g) > max_points:
        g = g.sample(max_points, random_state=SEED)

    plt.figure(figsize=(7, 5))
    plt.scatter(g[xcol].values, g[ycol].values, s=3, alpha=0.25)
    plt.title(f"{ycol} vs {xcol} - {source_name}")
    plt.xlabel(xcol)
    plt.ylabel(ycol)
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()


def generate_plots(inventory_df, valid_df, sampled_df):
    print("=" * 80)
    print("GENERATING PLOTS")
    print("=" * 80)

    save_bar(
        inventory_df["source"].value_counts().sort_index(),
        "Number of H5 files by source",
        "Files",
        PLOTS_DIR / "files_by_source.png",
    )

    save_bar(
        valid_df["source"].value_counts().sort_index(),
        "Number of valid H5 files by source",
        "Valid files",
        PLOTS_DIR / "valid_files_by_source.png",
    )

    for src, g in valid_df.groupby("source"):
        save_hist(
            g["n_pulses"],
            f"Number of pulses per file - {src}",
            "n_pulses",
            PLOTS_DIR / f"hist_n_pulses_{src}.png",
            bins=60,
            log_y=True,
        )

        save_hist(
            g["n_emitters"],
            f"Number of emitters per file - {src}",
            "n_emitters",
            PLOTS_DIR / f"hist_n_emitters_{src}.png",
            bins=40,
            log_y=False,
        )

        save_hist(
            g["dominant_label_share"],
            f"Dominant emitter share per file - {src}",
            "dominant_label_share",
            PLOTS_DIR / f"hist_dominant_share_{src}.png",
            bins=50,
            log_y=False,
        )

        save_hist(
            g["label_entropy_norm"],
            f"Normalized label entropy per file - {src}",
            "label_entropy_norm",
            PLOTS_DIR / f"hist_label_entropy_{src}.png",
            bins=50,
            log_y=False,
        )

        save_hist(
            g["n_nonfinite_data_values"],
            f"Non-finite data values per file - {src}",
            "n_nonfinite_data_values",
            PLOTS_DIR / f"hist_nonfinite_values_{src}.png",
            bins=50,
            log_y=True,
        )

    if len(sampled_df) > 0:
        sampled_df = clean_numeric_df(sampled_df, PDW_FEATURES)

        for feature in PDW_FEATURES:
            for src, g in sampled_df.groupby("source"):
                save_hist(
                    g[feature],
                    f"{feature} sampled distribution - {src}",
                    feature,
                    PLOTS_DIR / f"hist_{feature}_{src}.png",
                    bins=80,
                    log_y=True,
                )

            save_boxplot_by_source(
                sampled_df,
                feature,
                PLOTS_DIR / f"boxplot_{feature}_by_source.png",
            )

        for src in sampled_df["source"].dropna().unique():
            save_correlation_heatmap(
                sampled_df,
                src,
                PLOTS_DIR / f"corr_features_{src}.png",
            )

            save_scatter(
                sampled_df,
                src,
                "Frequency_MHz",
                "PulseWidth_us",
                PLOTS_DIR / f"scatter_frequency_vs_pulsewidth_{src}.png",
            )

            save_scatter(
                sampled_df,
                src,
                "ToA_us",
                "Frequency_MHz",
                PLOTS_DIR / f"scatter_toa_vs_frequency_{src}.png",
            )

            save_scatter(
                sampled_df,
                src,
                "AoA_deg",
                "Amplitude_dB",
                PLOTS_DIR / f"scatter_aoa_vs_amplitude_{src}.png",
            )


# ============================================================
# CSV FILES IN DATASET
# ============================================================
def inspect_csv_files():
    rows = []

    for source, folder in DATA_DIRS.items():
        csv_files = sorted(folder.rglob("*.csv"))

        for p in csv_files:
            try:
                df = pd.read_csv(p, nrows=10)
                rows.append({
                    "source": source,
                    "file_name": p.name,
                    "path": str(p),
                    "size_mb": round(p.stat().st_size / (1024 ** 2), 6),
                    "n_preview_rows": len(df),
                    "n_columns": len(df.columns),
                    "columns": safe_json(list(df.columns)),
                })
            except Exception as e:
                rows.append({
                    "source": source,
                    "file_name": p.name,
                    "path": str(p),
                    "size_mb": round(p.stat().st_size / (1024 ** 2), 6),
                    "n_preview_rows": 0,
                    "n_columns": 0,
                    "columns": None,
                    "error": str(e),
                })

    return pd.DataFrame(rows)


# ============================================================
# REPORT
# ============================================================
def write_report(
    inventory_df,
    valid_df,
    empty_df,
    error_df,
    source_summary,
    sampled_df,
    feature_stats_global,
    global_label_counts,
    nonfinite_df,
):
    report_path = OUT_DIR / "eda_report.md"

    total_files = len(inventory_df)
    valid_files = len(valid_df)
    empty_files = len(empty_df)
    error_files = len(error_df)
    total_pulses = int(valid_df["n_pulses"].sum()) if len(valid_df) > 0 else 0
    total_nonfinite = int(valid_df["n_nonfinite_data_values"].sum()) if len(valid_df) > 0 else 0

    with open(report_path, "w", encoding="utf-8") as f:
        f.write("# Exploratory Data Analysis — Turing Synthetic Radar Dataset\n\n")

        f.write("## Dataset interpretation\n\n")
        f.write(
            "The dataset is composed of radar pulse trains represented as "
            "Pulse Descriptor Words (PDWs). Each pulse is described by five features: "
            "Time of Arrival, Centre Frequency, Pulse Width, Angle of Arrival, and Amplitude.\n\n"
        )

        f.write(
            "Important note: labels are interpreted as emitter assignments within each pulse train. "
            "They should not automatically be treated as global class IDs across all files.\n\n"
        )

        f.write("## File summary\n\n")
        f.write(f"- Total H5 files: {total_files}\n")
        f.write(f"- Valid H5 files: {valid_files}\n")
        f.write(f"- Empty/problematic H5 files: {empty_files}\n")
        f.write(f"- Error files: {error_files}\n")
        f.write(f"- Total pulses in valid files: {total_pulses:,}\n")
        f.write(f"- Total non-finite PDW values detected: {total_nonfinite:,}\n\n")

        f.write("## Source summary\n\n")
        if len(source_summary) > 0:
            f.write(source_summary.to_markdown(index=False))
            f.write("\n\n")

        f.write("## Sampled PDW summary\n\n")
        f.write(f"- Sampled pulses: {len(sampled_df):,}\n")
        f.write(f"- Sampling per file: up to {SAMPLE_PULSES_PER_FILE:,} pulses\n")
        f.write(f"- Global sample limit: {GLOBAL_SAMPLE_LIMIT:,} pulses\n\n")

        f.write("## Non-finite values in sampled table\n\n")
        if len(nonfinite_df) > 0:
            f.write(nonfinite_df.to_markdown(index=False))
            f.write("\n\n")

        f.write("## Global sampled feature statistics\n\n")
        if len(feature_stats_global) > 0:
            f.write(feature_stats_global.to_markdown(index=False))
            f.write("\n\n")

        f.write("## Generated plots\n\n")
        f.write("Plots are available in:\n\n")
        f.write(f"`{PLOTS_DIR}`\n\n")

        f.write("Main plots include:\n\n")
        f.write("- number of files by source\n")
        f.write("- number of valid files by source\n")
        f.write("- number of pulses per file\n")
        f.write("- number of emitters per file\n")
        f.write("- dominant emitter share per file\n")
        f.write("- normalized label entropy per file\n")
        f.write("- non-finite values per file\n")
        f.write("- feature histograms\n")
        f.write("- feature boxplots by source\n")
        f.write("- feature correlation heatmaps\n")
        f.write("- scatter plots between selected PDW features\n\n")

        f.write("## Methodological implication\n\n")
        f.write(
            "For this dataset, exploratory analysis and modeling should operate directly "
            "on PDW sequences. LOFAR, DEMON, spectrogram-based acoustic features, and RF waveform "
            "representations are not appropriate unless raw waveform data are available.\n\n"
        )

    return report_path


# ============================================================
# MAIN
# ============================================================
def main():
    print("=" * 80)
    print("EDA - TURING SYNTHETIC RADAR DATASET")
    print("=" * 80)

    # --------------------------------------------------------
    # 1. Inventory
    # --------------------------------------------------------
    inventory_df, label_df = build_inventory()

    inventory_path = TABLES_DIR / "h5_inventory_full.csv"
    label_by_file_path = TABLES_DIR / "label_counts_by_file.csv"

    inventory_df.to_csv(inventory_path, index=False, encoding="utf-8-sig")
    label_df.to_csv(label_by_file_path, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # 2. Inventory stats
    # --------------------------------------------------------
    valid_df, empty_df, error_df, source_summary = compute_inventory_stats(inventory_df)

    valid_path = TABLES_DIR / "h5_valid_files.csv"
    empty_path = TABLES_DIR / "h5_empty_files.csv"
    error_path = TABLES_DIR / "h5_error_files.csv"
    source_summary_path = TABLES_DIR / "source_summary.csv"

    valid_df.to_csv(valid_path, index=False, encoding="utf-8-sig")
    empty_df.to_csv(empty_path, index=False, encoding="utf-8-sig")
    error_df.to_csv(error_path, index=False, encoding="utf-8-sig")
    source_summary.to_csv(source_summary_path, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # 3. Label stats
    # --------------------------------------------------------
    global_label_counts, label_file_presence = compute_label_stats(label_df)

    global_label_counts_path = TABLES_DIR / "global_label_counts.csv"
    label_file_presence_path = TABLES_DIR / "label_file_presence.csv"

    global_label_counts.to_csv(global_label_counts_path, index=False, encoding="utf-8-sig")
    label_file_presence.to_csv(label_file_presence_path, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # 4. Sample PDWs
    # --------------------------------------------------------
    sampled_df = build_sampled_pdw_table(valid_df)

    sampled_path = TABLES_DIR / "sampled_pdws.csv.gz"
    sampled_df.to_csv(sampled_path, index=False, encoding="utf-8-sig", compression="gzip")

    # --------------------------------------------------------
    # 5. Sampled feature stats
    # --------------------------------------------------------
    feature_stats_by_source, feature_stats_global, nonfinite_df = compute_sampled_feature_stats(sampled_df)

    feature_stats_by_source_path = TABLES_DIR / "sampled_feature_stats_by_source.csv"
    feature_stats_global_path = TABLES_DIR / "sampled_feature_stats_global.csv"
    nonfinite_path = TABLES_DIR / "sampled_nonfinite_summary.csv"

    feature_stats_by_source.to_csv(feature_stats_by_source_path, index=False, encoding="utf-8-sig")
    feature_stats_global.to_csv(feature_stats_global_path, index=False, encoding="utf-8-sig")
    nonfinite_df.to_csv(nonfinite_path, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # 6. CSV inspection
    # --------------------------------------------------------
    csv_inventory = inspect_csv_files()
    csv_inventory_path = TABLES_DIR / "csv_inventory.csv"
    csv_inventory.to_csv(csv_inventory_path, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # 7. Plots
    # --------------------------------------------------------
    generate_plots(inventory_df, valid_df, sampled_df)

    # --------------------------------------------------------
    # 8. Report
    # --------------------------------------------------------
    report_path = write_report(
        inventory_df=inventory_df,
        valid_df=valid_df,
        empty_df=empty_df,
        error_df=error_df,
        source_summary=source_summary,
        sampled_df=sampled_df,
        feature_stats_global=feature_stats_global,
        global_label_counts=global_label_counts,
        nonfinite_df=nonfinite_df,
    )

    # --------------------------------------------------------
    # 9. Console summary
    # --------------------------------------------------------
    print("\n" + "=" * 80)
    print("EDA SUMMARY")
    print("=" * 80)

    print(f"Total H5 files : {len(inventory_df)}")
    print(f"Valid H5 files : {len(valid_df)}")
    print(f"Empty H5 files : {len(empty_df)}")
    print(f"Error H5 files : {len(error_df)}")

    if len(valid_df) > 0:
        print(f"Total pulses   : {int(valid_df['n_pulses'].sum()):,}")
        print(f"Non-finite vals: {int(valid_df['n_nonfinite_data_values'].sum()):,}")
        print("\nSource summary:")
        print(source_summary.to_string(index=False))

    if len(nonfinite_df) > 0:
        print("\nSampled non-finite summary:")
        print(nonfinite_df.to_string(index=False))

    if len(feature_stats_global) > 0:
        print("\nSampled feature statistics:")
        print(feature_stats_global.to_string(index=False))

    print("\n" + "=" * 80)
    print("OUTPUT FILES")
    print("=" * 80)

    print(f"Inventory full              : {inventory_path}")
    print(f"Valid files                 : {valid_path}")
    print(f"Empty files                 : {empty_path}")
    print(f"Error files                 : {error_path}")
    print(f"Source summary              : {source_summary_path}")
    print(f"Label counts by file        : {label_by_file_path}")
    print(f"Global label counts         : {global_label_counts_path}")
    print(f"Label file presence         : {label_file_presence_path}")
    print(f"Sampled PDWs                : {sampled_path}")
    print(f"Feature stats by source     : {feature_stats_by_source_path}")
    print(f"Feature stats global        : {feature_stats_global_path}")
    print(f"Non-finite summary          : {nonfinite_path}")
    print(f"CSV inventory               : {csv_inventory_path}")
    print(f"Plots directory             : {PLOTS_DIR}")
    print(f"Markdown report             : {report_path}")

    print("\nDone.")


if __name__ == "__main__":
    main()

EDA - TURING SYNTHETIC RADAR DATASET
Scanning source: archive
Folder: D:\Fusion\turing_synthetic_radar_dataset\archive
H5 files found: 3000
  processed 250/3000
  processed 500/3000
  processed 750/3000
  processed 1000/3000
  processed 1250/3000
  processed 1500/3000
  processed 1750/3000
  processed 2000/3000
  processed 2250/3000
  processed 2500/3000
  processed 2750/3000
  processed 3000/3000
Scanning source: scan
Folder: D:\Fusion\turing_synthetic_radar_dataset\scan
H5 files found: 3000
  processed 250/3000
  processed 500/3000
  processed 750/3000
  processed 1000/3000
  processed 1250/3000
  processed 1500/3000
  processed 1750/3000
  processed 2000/3000
  processed 2250/3000
  processed 2500/3000
  processed 2750/3000
  processed 3000/3000
SAMPLING PDWs
  files processed: 250/5992 | sampled pulses: 474933
  files processed: 500/5992 | sampled pulses: 954906
Total sampled pulses: 1000000
GENERATING PLOTS
[Warning] Skipping constant histogram: Non-finite data values per file - s

In [3]:
# ============================================================
# spectral_pulse_analysis_tsrd.py
# ============================================================
# Spectral / frequency-domain exploratory analysis for the
# Turing Synthetic Radar Dataset (TSRD)
#
# Input folders:
#   D:\Fusion\turing_synthetic_radar_dataset\archive
#   D:\Fusion\turing_synthetic_radar_dataset\scan
#
# Dataset format:
#   data   : [n_pulses, 5]
#   labels : [n_pulses, 1]
#
# PDW columns:
#   0 -> ToA_us
#   1 -> Frequency_MHz
#   2 -> PulseWidth_us
#   3 -> AoA_deg
#   4 -> Amplitude_dB
#
# Outputs:
#   spectral_counts_by_source.csv
#   spectral_counts_by_file.csv
#   spectral_counts_by_file_band.csv
#   time_frequency_heatmap_*.csv
#   summary.json
#   plots/*.png
# ============================================================

from pathlib import Path
import sys
import json
import math
import subprocess
import warnings
warnings.filterwarnings("ignore")


# ============================================================
# INSTALL / IMPORTS
# ============================================================
def ensure_package(import_name, pip_name=None):
    try:
        __import__(import_name)
    except Exception:
        pkg = pip_name if pip_name is not None else import_name
        print(f"[INFO] Installing missing package: {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


ensure_package("numpy")
ensure_package("pandas")
ensure_package("matplotlib")
ensure_package("h5py")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py


# ============================================================
# PATHS
# ============================================================
ROOT = Path(r"D:\Fusion\turing_synthetic_radar_dataset")

DATA_DIRS = {
    "archive": ROOT / "archive",
    "scan": ROOT / "scan",
}

OUT_DIR = ROOT / "spectral_pulse_analysis"
TABLES_DIR = OUT_DIR / "tables"
PLOTS_DIR = OUT_DIR / "plots"
HEATMAP_DIR = OUT_DIR / "heatmaps"

OUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
HEATMAP_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# CONFIG
# ============================================================
SEED = 42
rng = np.random.default_rng(SEED)

# Use None to process all files.
MAX_FILES_PER_SOURCE = None

# HDF5 chunk size.
CHUNK_SIZE = 1_000_000

# Frequency analysis.
# The paper describes scan bands up to 18 GHz, and your files use MHz.
FREQ_MIN_MHZ = 0.0
FREQ_MAX_MHZ = 18_000.0
FREQ_BIN_MHZ = 500.0

FREQ_EDGES = np.arange(
    FREQ_MIN_MHZ,
    FREQ_MAX_MHZ + FREQ_BIN_MHZ,
    FREQ_BIN_MHZ,
    dtype=np.float64,
)
N_FREQ_BINS = len(FREQ_EDGES) - 1
FREQ_CENTERS = 0.5 * (FREQ_EDGES[:-1] + FREQ_EDGES[1:])

# Time-frequency heatmap.
# Use a broad ToA range compatible with scan/stare values observed in the paper/EDA.
TOA_MIN_US = 0.0
TOA_MAX_US = 45_000_000.0
N_TOA_BINS = 300
TOA_EDGES = np.linspace(TOA_MIN_US, TOA_MAX_US, N_TOA_BINS + 1)
TOA_CENTERS = 0.5 * (TOA_EDGES[:-1] + TOA_EDGES[1:])

# Optional sample for scatter plots.
MAKE_SAMPLE_SCATTER = True
SAMPLE_PULSES_PER_SOURCE = 250_000
SAMPLE_PULSES_PER_FILE = 1000

PDW_FEATURES = [
    "ToA_us",
    "Frequency_MHz",
    "PulseWidth_us",
    "AoA_deg",
    "Amplitude_dB",
]

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 180


# ============================================================
# HELPERS
# ============================================================
def save_json(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def h5_has_dataset(f, name):
    try:
        return isinstance(f[name], h5py.Dataset)
    except Exception:
        return False


def finite_mask(x):
    return np.isfinite(x)


def safe_div(a, b):
    return np.divide(a, b, out=np.zeros_like(a, dtype=np.float64), where=(b != 0))


def entropy_from_counts(counts):
    counts = np.asarray(counts, dtype=np.float64)
    total = counts.sum()
    if total <= 0:
        return np.nan

    p = counts / total
    p = p[p > 0]

    if len(p) <= 1:
        return 0.0

    return float(-np.sum(p * np.log(p)) / np.log(len(p)))


def file_size_mb(path: Path):
    return float(path.stat().st_size / (1024 ** 2))


def band_labels():
    labels = []
    for i, (a, b) in enumerate(zip(FREQ_EDGES[:-1], FREQ_EDGES[1:])):
        labels.append(f"{int(a)}-{int(b)}")
    return labels


def get_freq_bin_indices(freq_mhz):
    """
    Returns bin index for each frequency.
    Valid bins are 0 ... N_FREQ_BINS-1.
    """
    idx = np.searchsorted(FREQ_EDGES, freq_mhz, side="right") - 1
    return idx.astype(np.int32)


def add_band_stats(acc, band_idx, values):
    """
    Updates count/sum/sumsq per frequency band for one feature.
    """
    mask = (band_idx >= 0) & (band_idx < N_FREQ_BINS) & np.isfinite(values)

    if mask.sum() == 0:
        return

    b = band_idx[mask]
    v = values[mask].astype(np.float64)

    acc["count"] += np.bincount(b, minlength=N_FREQ_BINS)
    acc["sum"] += np.bincount(b, weights=v, minlength=N_FREQ_BINS)
    acc["sumsq"] += np.bincount(b, weights=v * v, minlength=N_FREQ_BINS)


def finalize_mean_std(acc):
    count = acc["count"].astype(np.float64)
    mean = safe_div(acc["sum"], count)
    var = safe_div(acc["sumsq"], count) - mean * mean
    var = np.maximum(var, 0.0)
    std = np.sqrt(var)
    mean[count == 0] = np.nan
    std[count == 0] = np.nan
    return mean, std


def init_feature_accumulators():
    return {
        "ToA_us": {
            "count": np.zeros(N_FREQ_BINS, dtype=np.float64),
            "sum": np.zeros(N_FREQ_BINS, dtype=np.float64),
            "sumsq": np.zeros(N_FREQ_BINS, dtype=np.float64),
        },
        "PulseWidth_us": {
            "count": np.zeros(N_FREQ_BINS, dtype=np.float64),
            "sum": np.zeros(N_FREQ_BINS, dtype=np.float64),
            "sumsq": np.zeros(N_FREQ_BINS, dtype=np.float64),
        },
        "AoA_deg": {
            "count": np.zeros(N_FREQ_BINS, dtype=np.float64),
            "sum": np.zeros(N_FREQ_BINS, dtype=np.float64),
            "sumsq": np.zeros(N_FREQ_BINS, dtype=np.float64),
        },
        "Amplitude_dB": {
            "count": np.zeros(N_FREQ_BINS, dtype=np.float64),
            "sum": np.zeros(N_FREQ_BINS, dtype=np.float64),
            "sumsq": np.zeros(N_FREQ_BINS, dtype=np.float64),
        },
    }


# ============================================================
# PER-FILE PROCESSING
# ============================================================
def process_one_h5(path: Path, source: str, source_acc: dict, sample_rows_source: list):
    """
    Processes one H5 file in chunks.

    Returns:
      file_summary_row
      file_band_rows
    """
    file_id = path.stem

    row = {
        "source": source,
        "file_id": file_id,
        "file_name": path.name,
        "path": str(path),
        "size_mb": file_size_mb(path),
        "status": "ok",
        "n_pulses": 0,
        "n_valid_freq": 0,
        "n_freq_underflow": 0,
        "n_freq_overflow": 0,
        "n_freq_nonfinite": 0,
        "n_nonfinite_amplitude": 0,
        "n_emitters": np.nan,
        "n_active_freq_bins": 0,
        "dominant_freq_bin": np.nan,
        "dominant_freq_low_mhz": np.nan,
        "dominant_freq_high_mhz": np.nan,
        "dominant_freq_count": 0,
        "dominant_freq_share": np.nan,
        "spectral_entropy_norm": np.nan,
        "freq_min_mhz": np.nan,
        "freq_max_mhz": np.nan,
        "toa_min_us": np.nan,
        "toa_max_us": np.nan,
        "error": None,
    }

    file_band_counts = np.zeros(N_FREQ_BINS, dtype=np.int64)
    file_band_label_sets = [set() for _ in range(N_FREQ_BINS)]
    file_label_set = set()

    freq_min = np.inf
    freq_max = -np.inf
    toa_min = np.inf
    toa_max = -np.inf

    sample_budget = SAMPLE_PULSES_PER_FILE if MAKE_SAMPLE_SCATTER else 0
    sample_rows = []

    try:
        with h5py.File(path, "r") as f:
            if not h5_has_dataset(f, "data"):
                row["status"] = "no_data"
                return row, []

            data = f["data"]
            n = int(data.shape[0])
            row["n_pulses"] = n

            if n <= 0:
                row["status"] = "empty"
                return row, []

            has_labels = h5_has_dataset(f, "labels")

            for start in range(0, n, CHUNK_SIZE):
                end = min(start + CHUNK_SIZE, n)

                x = np.asarray(data[start:end, :5], dtype=np.float32)

                toa = x[:, 0]
                freq = x[:, 1]
                pw = x[:, 2]
                aoa = x[:, 3]
                amp = x[:, 4]

                freq_finite = np.isfinite(freq)
                toa_finite = np.isfinite(toa)
                amp_finite = np.isfinite(amp)

                row["n_freq_nonfinite"] += int((~freq_finite).sum())
                row["n_nonfinite_amplitude"] += int((~amp_finite).sum())

                if freq_finite.any():
                    freq_min = min(freq_min, float(np.nanmin(freq[freq_finite])))
                    freq_max = max(freq_max, float(np.nanmax(freq[freq_finite])))

                if toa_finite.any():
                    toa_min = min(toa_min, float(np.nanmin(toa[toa_finite])))
                    toa_max = max(toa_max, float(np.nanmax(toa[toa_finite])))

                valid_freq_range = (
                    freq_finite &
                    (freq >= FREQ_MIN_MHZ) &
                    (freq < FREQ_MAX_MHZ)
                )

                row["n_freq_underflow"] += int((freq_finite & (freq < FREQ_MIN_MHZ)).sum())
                row["n_freq_overflow"] += int((freq_finite & (freq >= FREQ_MAX_MHZ)).sum())
                row["n_valid_freq"] += int(valid_freq_range.sum())

                if valid_freq_range.sum() == 0:
                    continue

                band_idx = get_freq_bin_indices(freq)
                valid_band = valid_freq_range & (band_idx >= 0) & (band_idx < N_FREQ_BINS)

                b = band_idx[valid_band]
                counts = np.bincount(b, minlength=N_FREQ_BINS).astype(np.int64)

                file_band_counts += counts
                source_acc["freq_counts"] += counts
                source_acc["total_valid_freq"] += int(valid_band.sum())

                # Feature summaries by spectral band.
                add_band_stats(source_acc["features"]["ToA_us"], band_idx, toa)
                add_band_stats(source_acc["features"]["PulseWidth_us"], band_idx, pw)
                add_band_stats(source_acc["features"]["AoA_deg"], band_idx, aoa)
                add_band_stats(source_acc["features"]["Amplitude_dB"], band_idx, amp)

                # Time-frequency heatmap.
                heat_mask = (
                    valid_band &
                    np.isfinite(toa) &
                    (toa >= TOA_MIN_US) &
                    (toa < TOA_MAX_US)
                )

                if heat_mask.sum() > 0:
                    H, _, _ = np.histogram2d(
                        toa[heat_mask].astype(np.float64),
                        freq[heat_mask].astype(np.float64),
                        bins=[TOA_EDGES, FREQ_EDGES],
                    )
                    source_acc["time_freq_heatmap"] += H.astype(np.float64)

                # Labels / emitters by spectral band.
                if has_labels:
                    y = np.asarray(f["labels"][start:end]).reshape(-1).astype(int)
                    y_valid = y[valid_band]
                    b_valid = band_idx[valid_band]

                    if y.size > 0:
                        file_label_set.update(np.unique(y).astype(int).tolist())

                    for bb in np.unique(b_valid):
                        yy = y_valid[b_valid == bb]
                        if yy.size > 0:
                            file_band_label_sets[int(bb)].update(np.unique(yy).astype(int).tolist())

                # Sample for scatter plots.
                if MAKE_SAMPLE_SCATTER and sample_budget > 0:
                    chunk_valid_idx = np.where(valid_band)[0]
                    if len(chunk_valid_idx) > 0:
                        take = min(sample_budget, len(chunk_valid_idx))
                        chosen = rng.choice(chunk_valid_idx, size=take, replace=False)

                        if has_labels:
                            y = np.asarray(f["labels"][start:end]).reshape(-1).astype(int)
                            labels_chosen = y[chosen]
                        else:
                            labels_chosen = np.full(take, -1, dtype=int)

                        for jj, ii in enumerate(chosen):
                            sample_rows.append({
                                "source": source,
                                "file_id": file_id,
                                "pulse_index": int(start + ii),
                                "label": int(labels_chosen[jj]),
                                "ToA_us": float(toa[ii]) if np.isfinite(toa[ii]) else np.nan,
                                "Frequency_MHz": float(freq[ii]) if np.isfinite(freq[ii]) else np.nan,
                                "PulseWidth_us": float(pw[ii]) if np.isfinite(pw[ii]) else np.nan,
                                "AoA_deg": float(aoa[ii]) if np.isfinite(aoa[ii]) else np.nan,
                                "Amplitude_dB": float(amp[ii]) if np.isfinite(amp[ii]) else np.nan,
                            })

                        sample_budget -= take

    except Exception as e:
        row["status"] = "error"
        row["error"] = str(e)
        return row, []

    # Finalize file summary.
    active = np.where(file_band_counts > 0)[0]
    row["n_active_freq_bins"] = int(len(active))

    if len(file_label_set) > 0:
        row["n_emitters"] = int(len(file_label_set))

    if len(active) > 0:
        dom = int(np.argmax(file_band_counts))
        row["dominant_freq_bin"] = dom
        row["dominant_freq_low_mhz"] = float(FREQ_EDGES[dom])
        row["dominant_freq_high_mhz"] = float(FREQ_EDGES[dom + 1])
        row["dominant_freq_count"] = int(file_band_counts[dom])
        row["dominant_freq_share"] = float(file_band_counts[dom] / max(file_band_counts.sum(), 1))
        row["spectral_entropy_norm"] = entropy_from_counts(file_band_counts)

    if np.isfinite(freq_min):
        row["freq_min_mhz"] = float(freq_min)
    if np.isfinite(freq_max):
        row["freq_max_mhz"] = float(freq_max)
    if np.isfinite(toa_min):
        row["toa_min_us"] = float(toa_min)
    if np.isfinite(toa_max):
        row["toa_max_us"] = float(toa_max)

    # Per-file per-band rows.
    file_band_rows = []
    total_valid = max(int(file_band_counts.sum()), 1)

    for i in range(N_FREQ_BINS):
        c = int(file_band_counts[i])
        file_band_rows.append({
            "source": source,
            "file_id": file_id,
            "file_name": path.name,
            "freq_bin": int(i),
            "freq_low_mhz": float(FREQ_EDGES[i]),
            "freq_high_mhz": float(FREQ_EDGES[i + 1]),
            "freq_center_mhz": float(FREQ_CENTERS[i]),
            "pulse_count": c,
            "pulse_share_in_file": float(c / total_valid),
            "n_emitters_in_band": int(len(file_band_label_sets[i])),
        })

    # Add sampled pulses to source-level sample list with cap.
    if MAKE_SAMPLE_SCATTER and len(sample_rows) > 0:
        remaining = SAMPLE_PULSES_PER_SOURCE - len(sample_rows_source)
        if remaining > 0:
            sample_rows_source.extend(sample_rows[:remaining])

    return row, file_band_rows


def init_source_accumulator():
    return {
        "freq_counts": np.zeros(N_FREQ_BINS, dtype=np.int64),
        "total_valid_freq": 0,
        "features": init_feature_accumulators(),
        "time_freq_heatmap": np.zeros((N_TOA_BINS, N_FREQ_BINS), dtype=np.float64),
    }


# ============================================================
# BUILD SOURCE TABLES
# ============================================================
def finalize_source_spectral_table(source, source_acc):
    counts = source_acc["freq_counts"].astype(np.float64)
    total = counts.sum()

    rows = []

    feature_means = {}
    feature_stds = {}

    for feat_name, acc in source_acc["features"].items():
        mean, std = finalize_mean_std(acc)
        feature_means[feat_name] = mean
        feature_stds[feat_name] = std

    for i in range(N_FREQ_BINS):
        rows.append({
            "source": source,
            "freq_bin": int(i),
            "freq_low_mhz": float(FREQ_EDGES[i]),
            "freq_high_mhz": float(FREQ_EDGES[i + 1]),
            "freq_center_mhz": float(FREQ_CENTERS[i]),
            "pulse_count": int(counts[i]),
            "pulse_share_in_source": float(counts[i] / total) if total > 0 else np.nan,

            "mean_ToA_us": feature_means["ToA_us"][i],
            "std_ToA_us": feature_stds["ToA_us"][i],

            "mean_PulseWidth_us": feature_means["PulseWidth_us"][i],
            "std_PulseWidth_us": feature_stds["PulseWidth_us"][i],

            "mean_AoA_deg": feature_means["AoA_deg"][i],
            "std_AoA_deg": feature_stds["AoA_deg"][i],

            "mean_Amplitude_dB": feature_means["Amplitude_dB"][i],
            "std_Amplitude_dB": feature_stds["Amplitude_dB"][i],
        })

    return pd.DataFrame(rows)


# ============================================================
# PLOTS
# ============================================================
def plot_frequency_counts(source_table):
    for source, g in source_table.groupby("source"):
        g = g.sort_values("freq_center_mhz")

        plt.figure(figsize=(12, 5))
        plt.bar(g["freq_center_mhz"], g["pulse_count"], width=FREQ_BIN_MHZ * 0.85)
        plt.yscale("log")
        plt.title(f"Pulse count by frequency band - {source}")
        plt.xlabel("Frequency (MHz)")
        plt.ylabel("Pulse count, log scale")
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"pulse_count_by_frequency_{source}.png", bbox_inches="tight")
        plt.close()

        plt.figure(figsize=(12, 5))
        plt.bar(g["freq_center_mhz"], g["pulse_share_in_source"], width=FREQ_BIN_MHZ * 0.85)
        plt.title(f"Pulse share by frequency band - {source}")
        plt.xlabel("Frequency (MHz)")
        plt.ylabel("Share")
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"pulse_share_by_frequency_{source}.png", bbox_inches="tight")
        plt.close()


def plot_feature_by_frequency(source_table, feature_mean_col, ylabel, out_name):
    for source, g in source_table.groupby("source"):
        g = g.sort_values("freq_center_mhz")

        plt.figure(figsize=(12, 5))
        plt.plot(g["freq_center_mhz"], g[feature_mean_col], marker="o", linewidth=1.5)
        plt.title(f"{ylabel} by frequency band - {source}")
        plt.xlabel("Frequency (MHz)")
        plt.ylabel(ylabel)
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"{out_name}_{source}.png", bbox_inches="tight")
        plt.close()


def plot_file_level_distributions(file_df):
    for source, g in file_df.groupby("source"):
        g_valid = g[g["status"] == "ok"].copy()

        if len(g_valid) == 0:
            continue

        plt.figure(figsize=(8, 5))
        plt.hist(g_valid["n_pulses"].dropna(), bins=70)
        plt.yscale("log")
        plt.title(f"Pulses per file - {source}")
        plt.xlabel("n_pulses")
        plt.ylabel("Count, log scale")
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"hist_pulses_per_file_{source}.png", bbox_inches="tight")
        plt.close()

        plt.figure(figsize=(8, 5))
        plt.hist(g_valid["n_active_freq_bins"].dropna(), bins=np.arange(0, N_FREQ_BINS + 2) - 0.5)
        plt.title(f"Active frequency bins per file - {source}")
        plt.xlabel("Number of active 500 MHz frequency bins")
        plt.ylabel("Files")
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"hist_active_frequency_bins_{source}.png", bbox_inches="tight")
        plt.close()

        plt.figure(figsize=(8, 5))
        plt.hist(g_valid["dominant_freq_share"].dropna(), bins=50)
        plt.title(f"Dominant spectral band share per file - {source}")
        plt.xlabel("Dominant frequency-band share")
        plt.ylabel("Files")
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"hist_dominant_band_share_{source}.png", bbox_inches="tight")
        plt.close()

        if "n_emitters" in g_valid.columns:
            plt.figure(figsize=(8, 5))
            plt.hist(g_valid["n_emitters"].dropna(), bins=50)
            plt.title(f"Number of emitters per file - {source}")
            plt.xlabel("n_emitters")
            plt.ylabel("Files")
            plt.tight_layout()
            plt.savefig(PLOTS_DIR / f"hist_emitters_per_file_{source}.png", bbox_inches="tight")
            plt.close()


def plot_time_frequency_heatmaps(source_accs):
    for source, acc in source_accs.items():
        H = acc["time_freq_heatmap"]

        # Save raw matrix.
        heatmap_df = pd.DataFrame(
            H,
            index=[f"{a:.0f}-{b:.0f}" for a, b in zip(TOA_EDGES[:-1], TOA_EDGES[1:])],
            columns=[f"{int(a)}-{int(b)}" for a, b in zip(FREQ_EDGES[:-1], FREQ_EDGES[1:])],
        )
        heatmap_csv = HEATMAP_DIR / f"time_frequency_heatmap_{source}.csv"
        heatmap_df.to_csv(heatmap_csv, encoding="utf-8-sig")

        plt.figure(figsize=(13, 6))
        plt.imshow(
            np.log1p(H),
            aspect="auto",
            origin="lower",
            extent=[
                FREQ_EDGES[0],
                FREQ_EDGES[-1],
                TOA_EDGES[0] / 1e6,
                TOA_EDGES[-1] / 1e6,
            ],
        )
        plt.colorbar(label="log(1 + pulse count)")
        plt.title(f"Time-frequency pulse density - {source}")
        plt.xlabel("Frequency (MHz)")
        plt.ylabel("ToA (s)")
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"time_frequency_heatmap_{source}.png", bbox_inches="tight")
        plt.close()


def plot_sample_scatter(sample_df):
    if len(sample_df) == 0:
        return

    for source, g in sample_df.groupby("source"):
        if len(g) == 0:
            continue

        g = g.copy()
        for col in PDW_FEATURES:
            g[col] = pd.to_numeric(g[col], errors="coerce")
            g[col] = g[col].replace([np.inf, -np.inf], np.nan)

        g = g.dropna(subset=["ToA_us", "Frequency_MHz", "PulseWidth_us", "AoA_deg"])

        if len(g) == 0:
            continue

        max_points = 100_000
        if len(g) > max_points:
            g = g.sample(max_points, random_state=SEED)

        plt.figure(figsize=(12, 5))
        plt.scatter(g["ToA_us"] / 1e6, g["Frequency_MHz"], s=2, alpha=0.25)
        plt.title(f"Sampled pulses: Frequency vs ToA - {source}")
        plt.xlabel("ToA (s)")
        plt.ylabel("Frequency (MHz)")
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"scatter_frequency_vs_toa_{source}.png", bbox_inches="tight")
        plt.close()

        plt.figure(figsize=(8, 5))
        plt.scatter(g["Frequency_MHz"], g["PulseWidth_us"], s=2, alpha=0.25)
        plt.title(f"Sampled pulses: PulseWidth vs Frequency - {source}")
        plt.xlabel("Frequency (MHz)")
        plt.ylabel("PulseWidth (us)")
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"scatter_pulsewidth_vs_frequency_{source}.png", bbox_inches="tight")
        plt.close()

        plt.figure(figsize=(8, 5))
        plt.scatter(g["Frequency_MHz"], g["AoA_deg"], s=2, alpha=0.25)
        plt.title(f"Sampled pulses: AoA vs Frequency - {source}")
        plt.xlabel("Frequency (MHz)")
        plt.ylabel("AoA (deg)")
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"scatter_aoa_vs_frequency_{source}.png", bbox_inches="tight")
        plt.close()


def generate_all_plots(source_table, file_df, source_accs, sample_df):
    print("=" * 80)
    print("GENERATING PLOTS")
    print("=" * 80)

    plot_frequency_counts(source_table)

    plot_feature_by_frequency(
        source_table,
        "mean_PulseWidth_us",
        "Mean pulse width (us)",
        "mean_pulsewidth_by_frequency",
    )

    plot_feature_by_frequency(
        source_table,
        "mean_AoA_deg",
        "Mean AoA (deg)",
        "mean_aoa_by_frequency",
    )

    plot_feature_by_frequency(
        source_table,
        "mean_Amplitude_dB",
        "Mean amplitude (dB)",
        "mean_amplitude_by_frequency",
    )

    plot_file_level_distributions(file_df)
    plot_time_frequency_heatmaps(source_accs)

    if MAKE_SAMPLE_SCATTER:
        plot_sample_scatter(sample_df)


# ============================================================
# REPORT
# ============================================================
def write_markdown_report(source_table, file_df, source_summary, sample_df):
    report_path = OUT_DIR / "spectral_pulse_report.md"

    with open(report_path, "w", encoding="utf-8") as f:
        f.write("# Spectral Pulse Analysis — TSRD\n\n")

        f.write("## Objective\n\n")
        f.write(
            "This analysis counts and characterizes radar pulses by spectral band using the "
            "`Frequency_MHz` field of the Pulse Descriptor Word (PDW). It does not perform FFT "
            "because the TSRD stores PDW sequences, not raw IQ/waveform data.\n\n"
        )

        f.write("## Frequency binning\n\n")
        f.write(f"- Frequency range: {FREQ_MIN_MHZ:.0f} to {FREQ_MAX_MHZ:.0f} MHz\n")
        f.write(f"- Bin width: {FREQ_BIN_MHZ:.0f} MHz\n")
        f.write(f"- Number of frequency bins: {N_FREQ_BINS}\n\n")

        f.write("## Source summary\n\n")
        if len(source_summary) > 0:
            f.write(source_summary.to_markdown(index=False))
            f.write("\n\n")

        f.write("## Most occupied frequency bands by source\n\n")
        for source, g in source_table.groupby("source"):
            top = g.sort_values("pulse_count", ascending=False).head(10)
            f.write(f"### {source}\n\n")
            f.write(top[
                [
                    "freq_low_mhz",
                    "freq_high_mhz",
                    "pulse_count",
                    "pulse_share_in_source",
                    "mean_PulseWidth_us",
                    "mean_Amplitude_dB",
                ]
            ].to_markdown(index=False))
            f.write("\n\n")

        f.write("## Output tables\n\n")
        f.write("- `tables/spectral_counts_by_source.csv`\n")
        f.write("- `tables/spectral_counts_by_file.csv`\n")
        f.write("- `tables/spectral_counts_by_file_band.csv`\n")
        f.write("- `tables/sampled_pulses_for_scatter.csv.gz`\n")
        f.write("- `heatmaps/time_frequency_heatmap_<source>.csv`\n\n")

        f.write("## Output plots\n\n")
        f.write("- pulse count by frequency band\n")
        f.write("- pulse share by frequency band\n")
        f.write("- mean pulse width by frequency band\n")
        f.write("- mean amplitude by frequency band\n")
        f.write("- time-frequency heatmaps\n")
        f.write("- file-level pulse and frequency-bin distributions\n")
        f.write("- sampled scatter plots\n\n")

    return report_path


# ============================================================
# MAIN
# ============================================================
def main():
    print("=" * 80)
    print("SPECTRAL PULSE ANALYSIS - TSRD")
    print("=" * 80)

    source_accs = {}
    all_file_rows = []
    all_file_band_rows = []
    all_source_tables = []
    all_sample_rows = []

    for source, folder in DATA_DIRS.items():
        print("\n" + "=" * 80)
        print(f"PROCESSING SOURCE: {source}")
        print(f"FOLDER: {folder}")
        print("=" * 80)

        h5_files = sorted(folder.rglob("*.h5"))

        if MAX_FILES_PER_SOURCE is not None:
            h5_files = h5_files[:int(MAX_FILES_PER_SOURCE)]

        print(f"H5 files found: {len(h5_files)}")

        source_acc = init_source_accumulator()
        source_sample_rows = []

        for i, path in enumerate(h5_files, 1):
            file_row, band_rows = process_one_h5(
                path=path,
                source=source,
                source_acc=source_acc,
                sample_rows_source=source_sample_rows,
            )

            all_file_rows.append(file_row)
            all_file_band_rows.extend(band_rows)

            if i % 100 == 0 or i == len(h5_files):
                print(
                    f"  processed {i}/{len(h5_files)} | "
                    f"valid pulses counted in spectrum: {source_acc['total_valid_freq']:,}"
                )

        source_accs[source] = source_acc

        source_table = finalize_source_spectral_table(source, source_acc)
        all_source_tables.append(source_table)

        all_sample_rows.extend(source_sample_rows)

    # --------------------------------------------------------
    # Build dataframes
    # --------------------------------------------------------
    source_table = pd.concat(all_source_tables, axis=0).reset_index(drop=True)
    file_df = pd.DataFrame(all_file_rows)
    file_band_df = pd.DataFrame(all_file_band_rows)
    sample_df = pd.DataFrame(all_sample_rows)

    # --------------------------------------------------------
    # Source-level summary
    # --------------------------------------------------------
    source_summary = (
        file_df
        .groupby("source", as_index=False)
        .agg(
            n_files=("path", "count"),
            n_ok=("status", lambda x: int(np.sum(np.asarray(x) == "ok"))),
            n_empty=("status", lambda x: int(np.sum(np.asarray(x) == "empty"))),
            n_error=("status", lambda x: int(np.sum(np.asarray(x) == "error"))),
            total_pulses=("n_pulses", "sum"),
            total_valid_freq=("n_valid_freq", "sum"),
            total_freq_underflow=("n_freq_underflow", "sum"),
            total_freq_overflow=("n_freq_overflow", "sum"),
            total_freq_nonfinite=("n_freq_nonfinite", "sum"),
            total_nonfinite_amplitude=("n_nonfinite_amplitude", "sum"),
            mean_pulses_per_file=("n_pulses", "mean"),
            median_pulses_per_file=("n_pulses", "median"),
            max_pulses_per_file=("n_pulses", "max"),
            mean_active_freq_bins=("n_active_freq_bins", "mean"),
            median_active_freq_bins=("n_active_freq_bins", "median"),
            mean_dominant_freq_share=("dominant_freq_share", "mean"),
            median_dominant_freq_share=("dominant_freq_share", "median"),
        )
    )

    # --------------------------------------------------------
    # Save tables
    # --------------------------------------------------------
    source_table_path = TABLES_DIR / "spectral_counts_by_source.csv"
    file_df_path = TABLES_DIR / "spectral_counts_by_file.csv"
    file_band_path = TABLES_DIR / "spectral_counts_by_file_band.csv"
    source_summary_path = TABLES_DIR / "spectral_source_summary.csv"
    sample_path = TABLES_DIR / "sampled_pulses_for_scatter.csv.gz"

    source_table.to_csv(source_table_path, index=False, encoding="utf-8-sig")
    file_df.to_csv(file_df_path, index=False, encoding="utf-8-sig")
    file_band_df.to_csv(file_band_path, index=False, encoding="utf-8-sig")
    source_summary.to_csv(source_summary_path, index=False, encoding="utf-8-sig")

    if len(sample_df) > 0:
        sample_df.to_csv(sample_path, index=False, encoding="utf-8-sig", compression="gzip")

    # --------------------------------------------------------
    # Plots
    # --------------------------------------------------------
    generate_all_plots(source_table, file_df, source_accs, sample_df)

    # --------------------------------------------------------
    # Report
    # --------------------------------------------------------
    report_path = write_markdown_report(
        source_table=source_table,
        file_df=file_df,
        source_summary=source_summary,
        sample_df=sample_df,
    )

    # --------------------------------------------------------
    # Save JSON summary
    # --------------------------------------------------------
    summary = {
        "root": str(ROOT),
        "frequency_range_mhz": [FREQ_MIN_MHZ, FREQ_MAX_MHZ],
        "frequency_bin_mhz": FREQ_BIN_MHZ,
        "n_frequency_bins": N_FREQ_BINS,
        "toa_range_us": [TOA_MIN_US, TOA_MAX_US],
        "n_toa_bins": N_TOA_BINS,
        "chunk_size": CHUNK_SIZE,
        "max_files_per_source": MAX_FILES_PER_SOURCE,
        "outputs": {
            "source_table": str(source_table_path),
            "file_table": str(file_df_path),
            "file_band_table": str(file_band_path),
            "source_summary": str(source_summary_path),
            "sample": str(sample_path) if len(sample_df) > 0 else None,
            "plots": str(PLOTS_DIR),
            "heatmaps": str(HEATMAP_DIR),
            "report": str(report_path),
        },
    }

    save_json(summary, OUT_DIR / "summary.json")

    # --------------------------------------------------------
    # Console output
    # --------------------------------------------------------
    print("\n" + "=" * 80)
    print("SOURCE SUMMARY")
    print("=" * 80)
    print(source_summary.to_string(index=False))

    print("\n" + "=" * 80)
    print("TOP FREQUENCY BANDS BY SOURCE")
    print("=" * 80)

    for source, g in source_table.groupby("source"):
        print(f"\n[{source}]")
        top = g.sort_values("pulse_count", ascending=False).head(10)
        print(
            top[
                [
                    "freq_low_mhz",
                    "freq_high_mhz",
                    "pulse_count",
                    "pulse_share_in_source",
                    "mean_PulseWidth_us",
                    "mean_Amplitude_dB",
                ]
            ].to_string(index=False)
        )

    print("\n" + "=" * 80)
    print("OUTPUT FILES")
    print("=" * 80)
    print(f"Source spectral table : {source_table_path}")
    print(f"File spectral table   : {file_df_path}")
    print(f"File-band table       : {file_band_path}")
    print(f"Source summary        : {source_summary_path}")
    if len(sample_df) > 0:
        print(f"Sampled pulses        : {sample_path}")
    print(f"Plots directory       : {PLOTS_DIR}")
    print(f"Heatmaps directory    : {HEATMAP_DIR}")
    print(f"Markdown report       : {report_path}")
    print(f"JSON summary          : {OUT_DIR / 'summary.json'}")

    print("\nDone.")


if __name__ == "__main__":
    main()

SPECTRAL PULSE ANALYSIS - TSRD

PROCESSING SOURCE: archive
FOLDER: D:\Fusion\turing_synthetic_radar_dataset\archive
H5 files found: 3000
  processed 100/3000 | valid pulses counted in spectrum: 13,623,238
  processed 200/3000 | valid pulses counted in spectrum: 26,923,228
  processed 300/3000 | valid pulses counted in spectrum: 43,967,229
  processed 400/3000 | valid pulses counted in spectrum: 54,406,658
  processed 500/3000 | valid pulses counted in spectrum: 74,663,031
  processed 600/3000 | valid pulses counted in spectrum: 89,458,525
  processed 700/3000 | valid pulses counted in spectrum: 99,616,448
  processed 800/3000 | valid pulses counted in spectrum: 115,156,739
  processed 900/3000 | valid pulses counted in spectrum: 133,317,734
  processed 1000/3000 | valid pulses counted in spectrum: 151,979,572
  processed 1100/3000 | valid pulses counted in spectrum: 168,462,119
  processed 1200/3000 | valid pulses counted in spectrum: 182,606,589
  processed 1300/3000 | valid pulses co

In [4]:
# ============================================================
# count_sources_tsrd.py
# ============================================================
# Conta quantas fontes/emissores existem no TSRD usando labels.
#
# Entrada:
#   D:\Fusion\turing_synthetic_radar_dataset\archive
#   D:\Fusion\turing_synthetic_radar_dataset\scan
#
# Cada .h5:
#   data   : [n_pulses, 5]
#   labels : [n_pulses, 1]
#
# Saídas:
#   source_count_by_file.csv
#   source_count_by_window_1024.csv
#   emitter_counts_by_file.csv
#   source_summary_by_mode.csv
#   plots/*.png
# ============================================================

from pathlib import Path
import sys
import subprocess
import warnings
warnings.filterwarnings("ignore")


# ============================================================
# INSTALL / IMPORTS
# ============================================================
def ensure_package(import_name, pip_name=None):
    try:
        __import__(import_name)
    except Exception:
        pkg = pip_name if pip_name is not None else import_name
        print(f"[INFO] Installing missing package: {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


ensure_package("numpy")
ensure_package("pandas")
ensure_package("matplotlib")
ensure_package("h5py")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py


# ============================================================
# PATHS
# ============================================================
ROOT = Path(r"D:\Fusion\turing_synthetic_radar_dataset")

DATA_DIRS = {
    "archive": ROOT / "archive",
    "scan": ROOT / "scan",
}

OUT_DIR = ROOT / "source_count_analysis"
TABLES_DIR = OUT_DIR / "tables"
PLOTS_DIR = OUT_DIR / "plots"

OUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# CONFIG
# ============================================================
WINDOW_SIZE = 1024

# Use None para processar tudo.
MAX_FILES_PER_SOURCE = None

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 180


# ============================================================
# HELPERS
# ============================================================
def h5_has_dataset(f, name):
    try:
        return isinstance(f[name], h5py.Dataset)
    except Exception:
        return False


def normalized_entropy_from_counts(counts):
    counts = np.asarray(counts, dtype=np.float64)
    total = counts.sum()

    if total <= 0:
        return np.nan

    p = counts / total
    p = p[p > 0]

    if len(p) <= 1:
        return 0.0

    return float(-np.sum(p * np.log(p)) / np.log(len(p)))


def gini_from_counts(counts):
    """
    Mede desbalanceamento entre emissores.
    0 = perfeitamente balanceado.
    próximo de 1 = altamente concentrado em poucos emissores.
    """
    x = np.asarray(counts, dtype=np.float64)

    if x.size == 0 or x.sum() <= 0:
        return np.nan

    x = np.sort(x)
    n = len(x)
    cumx = np.cumsum(x)

    return float((n + 1 - 2 * np.sum(cumx) / cumx[-1]) / n)


# ============================================================
# FILE PROCESSING
# ============================================================
def analyze_one_file(path: Path, source: str):
    file_id = path.stem

    file_row = {
        "source": source,
        "file_id": file_id,
        "file_name": path.name,
        "path": str(path),
        "status": "ok",
        "n_pulses": 0,
        "n_sources": 0,
        "source_min_label": np.nan,
        "source_max_label": np.nan,
        "dominant_source": np.nan,
        "dominant_source_count": 0,
        "dominant_source_share": np.nan,
        "median_pulses_per_source": np.nan,
        "mean_pulses_per_source": np.nan,
        "min_pulses_per_source": np.nan,
        "max_pulses_per_source": np.nan,
        "source_entropy_norm": np.nan,
        "source_gini": np.nan,
        "error": None,
    }

    emitter_rows = []
    window_rows = []

    try:
        with h5py.File(path, "r") as f:
            if not h5_has_dataset(f, "labels"):
                file_row["status"] = "missing_labels"
                return file_row, emitter_rows, window_rows

            labels = np.asarray(f["labels"][()]).reshape(-1).astype(int)

            n = len(labels)
            file_row["n_pulses"] = int(n)

            if n == 0:
                file_row["status"] = "empty"
                return file_row, emitter_rows, window_rows

            uniq, counts = np.unique(labels, return_counts=True)

            n_sources = len(uniq)
            file_row["n_sources"] = int(n_sources)
            file_row["source_min_label"] = int(uniq.min())
            file_row["source_max_label"] = int(uniq.max())

            j = int(np.argmax(counts))
            file_row["dominant_source"] = int(uniq[j])
            file_row["dominant_source_count"] = int(counts[j])
            file_row["dominant_source_share"] = float(counts[j] / n)

            file_row["median_pulses_per_source"] = float(np.median(counts))
            file_row["mean_pulses_per_source"] = float(np.mean(counts))
            file_row["min_pulses_per_source"] = int(np.min(counts))
            file_row["max_pulses_per_source"] = int(np.max(counts))

            file_row["source_entropy_norm"] = normalized_entropy_from_counts(counts)
            file_row["source_gini"] = gini_from_counts(counts)

            for u, c in zip(uniq, counts):
                emitter_rows.append({
                    "source": source,
                    "file_id": file_id,
                    "file_name": path.name,
                    "emitter_label": int(u),
                    "pulse_count": int(c),
                    "pulse_share_in_file": float(c / n),
                })

            # ------------------------------------------------
            # Window-level source count
            # ------------------------------------------------
            n_windows = int(np.ceil(n / WINDOW_SIZE))

            for w in range(n_windows):
                start = w * WINDOW_SIZE
                end = min((w + 1) * WINDOW_SIZE, n)

                y_win = labels[start:end]

                if len(y_win) == 0:
                    continue

                u_win, c_win = np.unique(y_win, return_counts=True)

                jj = int(np.argmax(c_win))

                window_rows.append({
                    "source": source,
                    "file_id": file_id,
                    "file_name": path.name,
                    "window_id": f"{file_id}__w{w:06d}",
                    "window_index": int(w),
                    "start_pulse": int(start),
                    "end_pulse": int(end),
                    "window_size_real": int(end - start),
                    "n_sources_window": int(len(u_win)),
                    "dominant_source_window": int(u_win[jj]),
                    "dominant_source_count_window": int(c_win[jj]),
                    "dominant_source_share_window": float(c_win[jj] / len(y_win)),
                    "median_pulses_per_source_window": float(np.median(c_win)),
                    "mean_pulses_per_source_window": float(np.mean(c_win)),
                    "source_entropy_norm_window": normalized_entropy_from_counts(c_win),
                    "source_gini_window": gini_from_counts(c_win),
                })

    except Exception as e:
        file_row["status"] = "error"
        file_row["error"] = str(e)

    return file_row, emitter_rows, window_rows


# ============================================================
# PLOTS
# ============================================================
def save_hist(series, title, xlabel, path, bins=50, log_y=False):
    s = pd.to_numeric(series, errors="coerce").dropna()

    if len(s) == 0:
        return

    plt.figure(figsize=(8, 5))
    plt.hist(s.values, bins=bins)

    if log_y:
        plt.yscale("log")

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()


def generate_plots(file_df, window_df, emitter_df):
    print("=" * 80)
    print("GENERATING PLOTS")
    print("=" * 80)

    for src, g in file_df.groupby("source"):
        g_ok = g[g["status"] == "ok"].copy()

        save_hist(
            g_ok["n_sources"],
            f"Number of sources per pulse train - {src}",
            "Number of sources/emitter labels",
            PLOTS_DIR / f"hist_n_sources_per_file_{src}.png",
            bins=60,
            log_y=False,
        )

        save_hist(
            g_ok["n_pulses"],
            f"Number of pulses per pulse train - {src}",
            "Number of pulses",
            PLOTS_DIR / f"hist_n_pulses_per_file_{src}.png",
            bins=70,
            log_y=True,
        )

        save_hist(
            g_ok["dominant_source_share"],
            f"Dominant source share per pulse train - {src}",
            "Dominant source share",
            PLOTS_DIR / f"hist_dominant_source_share_file_{src}.png",
            bins=50,
            log_y=False,
        )

        save_hist(
            g_ok["source_entropy_norm"],
            f"Normalized source entropy per pulse train - {src}",
            "Normalized entropy",
            PLOTS_DIR / f"hist_source_entropy_file_{src}.png",
            bins=50,
            log_y=False,
        )

        save_hist(
            g_ok["source_gini"],
            f"Source-count Gini per pulse train - {src}",
            "Gini",
            PLOTS_DIR / f"hist_source_gini_file_{src}.png",
            bins=50,
            log_y=False,
        )

    for src, g in window_df.groupby("source"):
        save_hist(
            g["n_sources_window"],
            f"Number of sources per {WINDOW_SIZE}-pulse window - {src}",
            "Number of sources in window",
            PLOTS_DIR / f"hist_n_sources_per_window_{src}.png",
            bins=60,
            log_y=False,
        )

        save_hist(
            g["dominant_source_share_window"],
            f"Dominant source share per {WINDOW_SIZE}-pulse window - {src}",
            "Dominant source share in window",
            PLOTS_DIR / f"hist_dominant_source_share_window_{src}.png",
            bins=50,
            log_y=False,
        )

        save_hist(
            g["source_entropy_norm_window"],
            f"Normalized source entropy per {WINDOW_SIZE}-pulse window - {src}",
            "Normalized entropy in window",
            PLOTS_DIR / f"hist_source_entropy_window_{src}.png",
            bins=50,
            log_y=False,
        )

    for src, g in emitter_df.groupby("source"):
        save_hist(
            g["pulse_count"],
            f"Pulses per source/emitter - {src}",
            "Pulse count per source",
            PLOTS_DIR / f"hist_pulses_per_source_{src}.png",
            bins=80,
            log_y=True,
        )


# ============================================================
# MAIN
# ============================================================
def main():
    print("=" * 80)
    print("COUNTING SOURCES / EMITTERS IN TSRD")
    print("=" * 80)

    file_rows = []
    emitter_rows_all = []
    window_rows_all = []

    for source, folder in DATA_DIRS.items():
        print("\n" + "=" * 80)
        print(f"PROCESSING SOURCE: {source}")
        print(f"FOLDER: {folder}")
        print("=" * 80)

        h5_files = sorted(folder.rglob("*.h5"))

        if MAX_FILES_PER_SOURCE is not None:
            h5_files = h5_files[:int(MAX_FILES_PER_SOURCE)]

        print(f"H5 files found: {len(h5_files)}")

        for i, path in enumerate(h5_files, 1):
            file_row, emitter_rows, window_rows = analyze_one_file(path, source)

            file_rows.append(file_row)
            emitter_rows_all.extend(emitter_rows)
            window_rows_all.extend(window_rows)

            if i % 250 == 0 or i == len(h5_files):
                print(f"  processed {i}/{len(h5_files)}")

    file_df = pd.DataFrame(file_rows)
    emitter_df = pd.DataFrame(emitter_rows_all)
    window_df = pd.DataFrame(window_rows_all)

    # --------------------------------------------------------
    # Save main tables
    # --------------------------------------------------------
    file_path = TABLES_DIR / "source_count_by_file.csv"
    emitter_path = TABLES_DIR / "emitter_counts_by_file.csv"
    window_path = TABLES_DIR / f"source_count_by_window_{WINDOW_SIZE}.csv"

    file_df.to_csv(file_path, index=False, encoding="utf-8-sig")
    emitter_df.to_csv(emitter_path, index=False, encoding="utf-8-sig")
    window_df.to_csv(window_path, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # Summaries
    # --------------------------------------------------------
    ok_df = file_df[file_df["status"] == "ok"].copy()

    summary_by_source = (
        ok_df
        .groupby("source", as_index=False)
        .agg(
            n_files=("file_id", "count"),
            total_pulses=("n_pulses", "sum"),
            min_sources=("n_sources", "min"),
            mean_sources=("n_sources", "mean"),
            median_sources=("n_sources", "median"),
            max_sources=("n_sources", "max"),
            mean_dominant_source_share=("dominant_source_share", "mean"),
            median_dominant_source_share=("dominant_source_share", "median"),
            mean_source_entropy=("source_entropy_norm", "mean"),
            median_source_entropy=("source_entropy_norm", "median"),
            mean_source_gini=("source_gini", "mean"),
            median_source_gini=("source_gini", "median"),
        )
    )

    window_summary_by_source = (
        window_df
        .groupby("source", as_index=False)
        .agg(
            n_windows=("window_id", "count"),
            min_sources_window=("n_sources_window", "min"),
            mean_sources_window=("n_sources_window", "mean"),
            median_sources_window=("n_sources_window", "median"),
            max_sources_window=("n_sources_window", "max"),
            mean_dominant_source_share_window=("dominant_source_share_window", "mean"),
            median_dominant_source_share_window=("dominant_source_share_window", "median"),
            mean_source_entropy_window=("source_entropy_norm_window", "mean"),
            median_source_entropy_window=("source_entropy_norm_window", "median"),
        )
    )

    summary_path = TABLES_DIR / "source_summary_by_mode.csv"
    window_summary_path = TABLES_DIR / f"source_summary_by_window_{WINDOW_SIZE}.csv"

    summary_by_source.to_csv(summary_path, index=False, encoding="utf-8-sig")
    window_summary_by_source.to_csv(window_summary_path, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # Plots
    # --------------------------------------------------------
    generate_plots(file_df, window_df, emitter_df)

    # --------------------------------------------------------
    # Console output
    # --------------------------------------------------------
    print("\n" + "=" * 80)
    print("SOURCE COUNT SUMMARY BY FILE")
    print("=" * 80)
    print(summary_by_source.to_string(index=False))

    print("\n" + "=" * 80)
    print(f"SOURCE COUNT SUMMARY BY {WINDOW_SIZE}-PULSE WINDOW")
    print("=" * 80)
    print(window_summary_by_source.to_string(index=False))

    print("\n" + "=" * 80)
    print("TOP 10 FILES WITH MOST SOURCES")
    print("=" * 80)
    print(
        ok_df[
            [
                "source",
                "file_id",
                "n_pulses",
                "n_sources",
                "dominant_source_share",
                "source_entropy_norm",
                "source_gini",
            ]
        ]
        .sort_values("n_sources", ascending=False)
        .head(10)
        .to_string(index=False)
    )

    print("\n" + "=" * 80)
    print("OUTPUT FILES")
    print("=" * 80)
    print(f"Source count by file   : {file_path}")
    print(f"Emitter counts by file : {emitter_path}")
    print(f"Source count by window : {window_path}")
    print(f"Summary by source      : {summary_path}")
    print(f"Window summary         : {window_summary_path}")
    print(f"Plots directory        : {PLOTS_DIR}")

    print("\nDone.")


if __name__ == "__main__":
    main()

COUNTING SOURCES / EMITTERS IN TSRD

PROCESSING SOURCE: archive
FOLDER: D:\Fusion\turing_synthetic_radar_dataset\archive
H5 files found: 3000
  processed 250/3000
  processed 500/3000
  processed 750/3000
  processed 1000/3000
  processed 1250/3000
  processed 1500/3000
  processed 1750/3000
  processed 2000/3000
  processed 2250/3000
  processed 2500/3000
  processed 2750/3000
  processed 3000/3000

PROCESSING SOURCE: scan
FOLDER: D:\Fusion\turing_synthetic_radar_dataset\scan
H5 files found: 3000
  processed 250/3000
  processed 500/3000
  processed 750/3000
  processed 1000/3000
  processed 1250/3000
  processed 1500/3000
  processed 1750/3000
  processed 2000/3000
  processed 2250/3000
  processed 2500/3000
  processed 2750/3000
  processed 3000/3000
GENERATING PLOTS

SOURCE COUNT SUMMARY BY FILE
 source  n_files  total_pulses  min_sources  mean_sources  median_sources  max_sources  mean_dominant_source_share  median_dominant_source_share  mean_source_entropy  median_source_entropy 

In [5]:
# ============================================================
# baseline_hdbscan_deinterleaving_tsrd.py
# ============================================================
# Baseline de deinterleaving para o TSRD
#
# Tarefa:
#   Separar pulsos intercalados em clusters/emissores.
#
# Entrada:
#   data   : [n_pulses, 5]
#   labels : [n_pulses, 1]
#
# PDW features:
#   0 -> ToA_us
#   1 -> Frequency_MHz
#   2 -> PulseWidth_us
#   3 -> AoA_deg
#   4 -> Amplitude_dB
#
# Método:
#   - Divide cada pulse train em janelas não sobrepostas de 1024 pulsos.
#   - Normaliza os PDWs dentro da janela.
#   - Roda HDBSCAN sem usar o número real de fontes.
#   - Compara clusters previstos com labels reais.
#
# Métricas:
#   - V-measure
#   - ARI
#   - AMI
#   - Homogeneity
#   - Completeness
#   - Pairwise F1
#   - Pairwise MCC
#
# Pastas:
#   D:\Fusion\turing_synthetic_radar_dataset\archive
#   D:\Fusion\turing_synthetic_radar_dataset\scan
# ============================================================

from pathlib import Path
import sys
import json
import math
import time
import subprocess
import warnings
warnings.filterwarnings("ignore")


# ============================================================
# INSTALL / IMPORTS
# ============================================================
def ensure_package(import_name, pip_name=None):
    try:
        __import__(import_name)
    except Exception:
        pkg = pip_name if pip_name is not None else import_name
        print(f"[INFO] Installing missing package: {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


ensure_package("numpy")
ensure_package("pandas")
ensure_package("matplotlib")
ensure_package("h5py")
ensure_package("sklearn", "scikit-learn")
ensure_package("tqdm")

# HDBSCAN pode falhar em alguns Windows se não houver wheel.
# O código tenta usar hdbscan; se não conseguir, tenta sklearn.cluster.HDBSCAN.
try:
    ensure_package("hdbscan")
    import hdbscan
    HDBSCAN_BACKEND = "hdbscan"
except Exception:
    hdbscan = None
    HDBSCAN_BACKEND = "sklearn"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py

from tqdm import tqdm

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import (
    v_measure_score,
    adjusted_rand_score,
    adjusted_mutual_info_score,
    homogeneity_score,
    completeness_score,
    f1_score,
    matthews_corrcoef,
)

try:
    from sklearn.cluster import HDBSCAN as SKHDBSCAN
except Exception:
    SKHDBSCAN = None

from sklearn.cluster import KMeans


# ============================================================
# PATHS
# ============================================================
ROOT = Path(r"D:\Fusion\turing_synthetic_radar_dataset")

DATA_DIRS = {
    "archive": ROOT / "archive",
    "scan": ROOT / "scan",
}

OUT_DIR = ROOT / "deinterleaving_baseline_hdbscan"
TABLES_DIR = OUT_DIR / "tables"
PLOTS_DIR = OUT_DIR / "plots"

OUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# CONFIG
# ============================================================
SEED = 42
rng = np.random.default_rng(SEED)

WINDOW_SIZE = 1024
DROP_INCOMPLETE_WINDOWS = True

# Para teste rápido, use um limite.
# Para rodar tudo, coloque None.
MAX_FILES_PER_SOURCE = None

# Muito importante:
# Rodar todas as ~700 mil janelas pode demorar bastante.
# Para primeiro teste, use 1000, 5000 ou 10000.
# Para benchmark completo, coloque None.
MAX_WINDOWS_PER_SOURCE = 5000

# Se quiser avaliar apenas alguns splits pelo nome do arquivo:
#   None         -> usa todos os arquivos
#   ["test"]     -> usa apenas test_*.h5
#   ["train"]    -> usa apenas train_*.h5
#   ["validation"] -> usa apenas validation_*.h5
#
# Atenção: em algumas pastas, especialmente scan, os nomes podem ser config_*.h5.
# Nesse caso, deixe None.
SPLITS_TO_PROCESS = None

# Modos de pré-processamento:
#   "paper_raw"  -> usa os 5 PDWs com limpeza e StandardScaler
#   "engineered" -> usa features derivadas: ToA relativo, delta-ToA,
#                   frequência, log PW, AoA sin/cos, amplitude e máscara.
PREPROCESS_MODE = "engineered"

# HDBSCAN parameters.
HDBSCAN_MIN_CLUSTER_SIZE = 5
HDBSCAN_MIN_SAMPLES = 3
HDBSCAN_CLUSTER_SELECTION_EPSILON = 0.0
HDBSCAN_CLUSTER_SELECTION_METHOD = "eom"
HDBSCAN_ALLOW_SINGLE_CLUSTER = False

# Rodar KMeans com k real como diagnóstico oracular.
# Não use como resultado principal, pois usa o número real de fontes.
RUN_ORACLE_KMEANS = False

# Save incremental results.
SAVE_EVERY_WINDOWS = 500

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 180


# ============================================================
# BASIC HELPERS
# ============================================================
def save_json(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def h5_has_dataset(f, name):
    try:
        return isinstance(f[name], h5py.Dataset)
    except Exception:
        return False


def get_split_from_name(file_name: str):
    name = file_name.lower()

    if name.startswith("train"):
        return "train"
    if name.startswith("validation") or name.startswith("val"):
        return "validation"
    if name.startswith("test"):
        return "test"

    return "unknown"


def should_process_file(path: Path):
    if SPLITS_TO_PROCESS is None:
        return True

    split = get_split_from_name(path.name)
    return split in set(SPLITS_TO_PROCESS)


def comb2(x):
    x = np.asarray(x, dtype=np.int64)
    return x * (x - 1) // 2


def safe_float(x):
    try:
        if np.isfinite(x):
            return float(x)
        return float("nan")
    except Exception:
        return float("nan")


# ============================================================
# PREPROCESSING
# ============================================================
def clean_raw_pdw(x):
    """
    Limpa os 5 PDWs originais.
    """
    x = np.asarray(x, dtype=np.float32).copy()

    # Amplitude não finita aparece bastante no archive.
    # Usa -200 dB como valor físico baixo.
    amp = x[:, 4]
    amp_missing = ~np.isfinite(amp)
    amp[amp_missing] = -200.0
    x[:, 4] = amp

    # Demais não finitos.
    for j in range(x.shape[1]):
        col = x[:, j]
        finite = np.isfinite(col)

        if finite.any():
            med = np.nanmedian(col[finite])
        else:
            med = 0.0

        col[~finite] = med
        x[:, j] = col

    return x


def preprocess_window_paper_raw(x):
    """
    Modo próximo ao baseline do paper:
    usa os 5 PDWs, apenas limpa e normaliza.
    """
    x = clean_raw_pdw(x)

    scaler = StandardScaler()
    z = scaler.fit_transform(x)

    z = np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0)
    return z.astype(np.float32)


def preprocess_window_engineered(x):
    """
    Features derivadas para facilitar clustering.

    Entrada bruta:
      ToA_us, Frequency_MHz, PulseWidth_us, AoA_deg, Amplitude_dB

    Saída:
      toa_rel_norm
      delta_toa_log_norm
      frequency_norm
      pulsewidth_log
      aoa_sin
      aoa_cos
      amplitude_norm
      amplitude_missing
    """
    x = np.asarray(x, dtype=np.float32).copy()

    toa = x[:, 0]
    freq = x[:, 1]
    pw = x[:, 2]
    aoa = x[:, 3]
    amp = x[:, 4]

    # --------------------------------------------------------
    # Non-finite handling
    # --------------------------------------------------------
    amp_missing = (~np.isfinite(amp)).astype(np.float32)
    amp = amp.copy()
    amp[~np.isfinite(amp)] = -200.0

    # Other fields
    for arr_name in ["toa", "freq", "pw", "aoa"]:
        arr = locals()[arr_name]
        finite = np.isfinite(arr)

        if finite.any():
            med = np.nanmedian(arr[finite])
        else:
            med = 0.0

        arr[~finite] = med
        locals()[arr_name] = arr

    # --------------------------------------------------------
    # ToA relative within window
    # --------------------------------------------------------
    toa = np.asarray(toa, dtype=np.float32)
    toa_rel = toa - np.min(toa)

    toa_span = np.max(toa_rel) - np.min(toa_rel)
    if toa_span > 0:
        toa_rel_norm = toa_rel / (toa_span + 1e-9)
    else:
        toa_rel_norm = np.zeros_like(toa_rel, dtype=np.float32)

    # Delta-ToA
    dtoa = np.diff(toa, prepend=toa[0])
    dtoa = np.maximum(dtoa, 0.0)
    dtoa_log = np.log1p(dtoa)

    # Frequency normalized by 18 GHz = 18000 MHz
    freq = np.asarray(freq, dtype=np.float32)
    freq_norm = freq / 18000.0

    # Pulse width log transform
    pw = np.asarray(pw, dtype=np.float32)
    pw = np.maximum(pw, 0.0)
    pw_log = np.log1p(pw)

    # AoA circular encoding
    aoa_rad = np.deg2rad(aoa.astype(np.float32))
    aoa_sin = np.sin(aoa_rad)
    aoa_cos = np.cos(aoa_rad)

    # Amplitude approximate normalization
    # Typical range around -220 to +40 dB.
    amp = np.asarray(amp, dtype=np.float32)
    amp_norm = (amp + 220.0) / 260.0

    feats = np.stack(
        [
            toa_rel_norm,
            dtoa_log,
            freq_norm,
            pw_log,
            aoa_sin,
            aoa_cos,
            amp_norm,
            amp_missing,
        ],
        axis=1,
    ).astype(np.float32)

    # Robust scale continuous columns, leave missing mask mostly intact.
    continuous_cols = [0, 1, 2, 3, 4, 5, 6]
    z = feats.copy()

    scaler = RobustScaler()
    z[:, continuous_cols] = scaler.fit_transform(z[:, continuous_cols])

    z = np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0)
    return z.astype(np.float32)


def preprocess_window(x):
    if PREPROCESS_MODE == "paper_raw":
        return preprocess_window_paper_raw(x)

    if PREPROCESS_MODE == "engineered":
        return preprocess_window_engineered(x)

    raise ValueError(f"Unknown PREPROCESS_MODE: {PREPROCESS_MODE}")


# ============================================================
# CLUSTERING
# ============================================================
def run_hdbscan(X):
    """
    Roda HDBSCAN sem informar número de fontes.
    """
    X = np.asarray(X, dtype=np.float32)

    if HDBSCAN_BACKEND == "hdbscan" and hdbscan is not None:
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
            min_samples=HDBSCAN_MIN_SAMPLES,
            metric="euclidean",
            cluster_selection_epsilon=HDBSCAN_CLUSTER_SELECTION_EPSILON,
            cluster_selection_method=HDBSCAN_CLUSTER_SELECTION_METHOD,
            allow_single_cluster=HDBSCAN_ALLOW_SINGLE_CLUSTER,
        )
        y_pred = clusterer.fit_predict(X)
        return y_pred.astype(int)

    if SKHDBSCAN is not None:
        clusterer = SKHDBSCAN(
            min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
            min_samples=HDBSCAN_MIN_SAMPLES,
            metric="euclidean",
            cluster_selection_epsilon=HDBSCAN_CLUSTER_SELECTION_EPSILON,
            allow_single_cluster=HDBSCAN_ALLOW_SINGLE_CLUSTER,
        )
        y_pred = clusterer.fit_predict(X)
        return y_pred.astype(int)

    raise RuntimeError(
        "No HDBSCAN implementation available. Try: pip install hdbscan"
    )


def run_oracle_kmeans(X, n_clusters):
    """
    Diagnóstico oracular: usa o número real de fontes.
    Não deve ser usado como resultado principal.
    """
    if n_clusters <= 1:
        return np.zeros(X.shape[0], dtype=int)

    km = KMeans(
        n_clusters=int(n_clusters),
        random_state=SEED,
        n_init=10,
        max_iter=300,
    )
    return km.fit_predict(X).astype(int)


# ============================================================
# METRICS
# ============================================================
def efficient_pairwise_metrics(y_true, y_pred):
    """
    Calcula pairwise F1 e MCC sem gerar todos os pares explicitamente.

    Pares:
      positivo = dois pulsos pertencem ao mesmo emissor/cluster.
      negativo = dois pulsos pertencem a emissores/clusters diferentes.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    n = len(y_true)
    total_pairs = n * (n - 1) // 2

    if n < 2 or total_pairs <= 0:
        return {
            "pairwise_f1": np.nan,
            "pairwise_mcc": np.nan,
            "pairwise_tp": 0,
            "pairwise_fp": 0,
            "pairwise_fn": 0,
            "pairwise_tn": 0,
        }

    true_vals, true_inv = np.unique(y_true, return_inverse=True)
    pred_vals, pred_inv = np.unique(y_pred, return_inverse=True)

    n_true = len(true_vals)
    n_pred = len(pred_vals)

    contingency = np.zeros((n_true, n_pred), dtype=np.int64)
    np.add.at(contingency, (true_inv, pred_inv), 1)

    tp = int(comb2(contingency).sum())

    true_cluster_sizes = contingency.sum(axis=1)
    pred_cluster_sizes = contingency.sum(axis=0)

    true_pairs = int(comb2(true_cluster_sizes).sum())
    pred_pairs = int(comb2(pred_cluster_sizes).sum())

    fp = int(pred_pairs - tp)
    fn = int(true_pairs - tp)
    tn = int(total_pairs - tp - fp - fn)

    denom_f1 = 2 * tp + fp + fn
    if denom_f1 > 0:
        pairwise_f1 = float((2 * tp) / denom_f1)
    else:
        pairwise_f1 = 0.0

    denom_mcc = math.sqrt(
        max(tp + fp, 0)
        * max(tp + fn, 0)
        * max(tn + fp, 0)
        * max(tn + fn, 0)
    )

    if denom_mcc > 0:
        pairwise_mcc = float((tp * tn - fp * fn) / denom_mcc)
    else:
        pairwise_mcc = 0.0

    return {
        "pairwise_f1": pairwise_f1,
        "pairwise_mcc": pairwise_mcc,
        "pairwise_tp": tp,
        "pairwise_fp": fp,
        "pairwise_fn": fn,
        "pairwise_tn": tn,
    }


def deinterleaving_metrics(y_true, y_pred):
    """
    Compara labels reais com clusters previstos.
    O nome dos clusters não precisa coincidir com o nome dos labels reais.
    """
    y_true = np.asarray(y_true).reshape(-1).astype(int)
    y_pred = np.asarray(y_pred).reshape(-1).astype(int)

    out = {}

    out["v_measure"] = safe_float(v_measure_score(y_true, y_pred))
    out["ari"] = safe_float(adjusted_rand_score(y_true, y_pred))
    out["ami"] = safe_float(adjusted_mutual_info_score(y_true, y_pred))
    out["homogeneity"] = safe_float(homogeneity_score(y_true, y_pred))
    out["completeness"] = safe_float(completeness_score(y_true, y_pred))

    pair = efficient_pairwise_metrics(y_true, y_pred)
    out.update(pair)

    true_sources = np.unique(y_true)
    pred_clusters = np.unique(y_pred)

    out["n_pulses"] = int(len(y_true))
    out["n_true_sources"] = int(len(true_sources))
    out["n_pred_clusters_including_noise"] = int(len(pred_clusters))
    out["n_pred_clusters_excluding_noise"] = int(len([c for c in pred_clusters if c != -1]))
    out["has_noise_cluster"] = int(np.any(y_pred == -1))
    out["noise_share"] = float(np.mean(y_pred == -1))

    return out


def dominant_source_share(y_true):
    y_true = np.asarray(y_true).reshape(-1)
    if len(y_true) == 0:
        return np.nan

    _, counts = np.unique(y_true, return_counts=True)
    return float(counts.max() / counts.sum())


def source_entropy_norm(y_true):
    y_true = np.asarray(y_true).reshape(-1)
    if len(y_true) == 0:
        return np.nan

    _, counts = np.unique(y_true, return_counts=True)
    counts = counts.astype(np.float64)
    p = counts / counts.sum()
    p = p[p > 0]

    if len(p) <= 1:
        return 0.0

    return float(-np.sum(p * np.log(p)) / np.log(len(p)))


# ============================================================
# PROCESSING
# ============================================================
def process_window(
    x_window,
    y_window,
    source,
    split,
    file_id,
    file_name,
    window_index,
    start_pulse,
    end_pulse,
):
    """
    Processa uma janela:
      - normaliza PDWs
      - roda HDBSCAN
      - calcula métricas
    """
    t0 = time.time()

    x_window = np.asarray(x_window, dtype=np.float32)
    y_window = np.asarray(y_window).reshape(-1).astype(int)

    X = preprocess_window(x_window)

    y_pred = run_hdbscan(X)
    metrics = deinterleaving_metrics(y_window, y_pred)

    elapsed = time.time() - t0

    row = {
        "source": source,
        "split": split,
        "file_id": file_id,
        "file_name": file_name,
        "window_index": int(window_index),
        "start_pulse": int(start_pulse),
        "end_pulse": int(end_pulse),
        "window_size": int(end_pulse - start_pulse),
        "preprocess_mode": PREPROCESS_MODE,
        "cluster_method": "HDBSCAN",
        "hdbscan_backend": HDBSCAN_BACKEND,
        "hdbscan_min_cluster_size": HDBSCAN_MIN_CLUSTER_SIZE,
        "hdbscan_min_samples": HDBSCAN_MIN_SAMPLES,
        "hdbscan_epsilon": HDBSCAN_CLUSTER_SELECTION_EPSILON,
        "dominant_source_share": dominant_source_share(y_window),
        "source_entropy_norm": source_entropy_norm(y_window),
        "elapsed_sec": float(elapsed),
    }

    row.update(metrics)

    return row


def process_one_file(path: Path, source: str, max_windows_remaining=None):
    """
    Processa um arquivo .h5 em janelas não sobrepostas.
    """
    rows = []
    split = get_split_from_name(path.name)
    file_id = path.stem

    try:
        with h5py.File(path, "r") as f:
            if not h5_has_dataset(f, "data") or not h5_has_dataset(f, "labels"):
                return rows

            data = f["data"]
            labels = f["labels"]

            n = int(data.shape[0])

            if n <= 0:
                return rows

            n_windows = n // WINDOW_SIZE

            if not DROP_INCOMPLETE_WINDOWS and n % WINDOW_SIZE != 0:
                n_windows += 1

            if max_windows_remaining is not None:
                n_windows = min(n_windows, int(max_windows_remaining))

            for w in range(n_windows):
                start = w * WINDOW_SIZE
                end = min((w + 1) * WINDOW_SIZE, n)

                if DROP_INCOMPLETE_WINDOWS and (end - start) < WINDOW_SIZE:
                    continue

                x_win = np.asarray(data[start:end, :5], dtype=np.float32)
                y_win = np.asarray(labels[start:end]).reshape(-1).astype(int)

                row = process_window(
                    x_window=x_win,
                    y_window=y_win,
                    source=source,
                    split=split,
                    file_id=file_id,
                    file_name=path.name,
                    window_index=w,
                    start_pulse=start,
                    end_pulse=end,
                )

                rows.append(row)

    except Exception as e:
        rows.append({
            "source": source,
            "split": get_split_from_name(path.name),
            "file_id": path.stem,
            "file_name": path.name,
            "path": str(path),
            "cluster_method": "HDBSCAN",
            "status": "error",
            "error": str(e),
        })

    return rows


def append_rows_to_csv(rows, path: Path):
    if len(rows) == 0:
        return

    df = pd.DataFrame(rows)

    write_header = not path.exists()
    df.to_csv(
        path,
        mode="a",
        header=write_header,
        index=False,
        encoding="utf-8-sig",
    )


# ============================================================
# SUMMARIES AND PLOTS
# ============================================================
def summarize_results(results_df):
    metric_cols = [
        "v_measure",
        "ari",
        "ami",
        "homogeneity",
        "completeness",
        "pairwise_f1",
        "pairwise_mcc",
        "n_true_sources",
        "n_pred_clusters_excluding_noise",
        "noise_share",
        "dominant_source_share",
        "source_entropy_norm",
        "elapsed_sec",
    ]

    group_cols = ["source", "split", "cluster_method", "preprocess_mode"]

    summaries = []

    for keys, g in results_df.groupby(group_cols, dropna=False):
        row = dict(zip(group_cols, keys))
        row["n_windows"] = int(len(g))

        for c in metric_cols:
            if c not in g.columns:
                continue

            vals = pd.to_numeric(g[c], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()

            if len(vals) == 0:
                row[f"{c}_mean"] = np.nan
                row[f"{c}_median"] = np.nan
                row[f"{c}_std"] = np.nan
                continue

            row[f"{c}_mean"] = float(vals.mean())
            row[f"{c}_median"] = float(vals.median())
            row[f"{c}_std"] = float(vals.std())

        summaries.append(row)

    return pd.DataFrame(summaries)


def summarize_by_complexity(results_df):
    df = results_df.copy()

    df["n_true_sources_bin"] = pd.cut(
        df["n_true_sources"],
        bins=[0, 1, 2, 5, 10, 20, 40, 80, 200],
        labels=[
            "1",
            "2",
            "3-5",
            "6-10",
            "11-20",
            "21-40",
            "41-80",
            "81+",
        ],
        include_lowest=True,
    )

    rows = []

    for keys, g in df.groupby(["source", "n_true_sources_bin"], dropna=False):
        source, bin_name = keys

        row = {
            "source": source,
            "n_true_sources_bin": str(bin_name),
            "n_windows": int(len(g)),
        }

        for c in [
            "v_measure",
            "ari",
            "ami",
            "homogeneity",
            "completeness",
            "pairwise_f1",
            "pairwise_mcc",
            "noise_share",
        ]:
            vals = pd.to_numeric(g[c], errors="coerce").dropna()

            row[f"{c}_mean"] = float(vals.mean()) if len(vals) > 0 else np.nan
            row[f"{c}_median"] = float(vals.median()) if len(vals) > 0 else np.nan

        rows.append(row)

    return pd.DataFrame(rows)


def save_hist(series, title, xlabel, path, bins=50):
    s = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()

    if len(s) == 0:
        return

    plt.figure(figsize=(8, 5))
    plt.hist(s.values, bins=bins)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Windows")
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()


def save_scatter(df, xcol, ycol, title, path, max_points=100_000):
    g = df[[xcol, ycol, "source"]].copy()
    g[xcol] = pd.to_numeric(g[xcol], errors="coerce")
    g[ycol] = pd.to_numeric(g[ycol], errors="coerce")
    g = g.replace([np.inf, -np.inf], np.nan).dropna()

    if len(g) == 0:
        return

    if len(g) > max_points:
        g = g.sample(max_points, random_state=SEED)

    plt.figure(figsize=(8, 5))

    for source, gs in g.groupby("source"):
        plt.scatter(gs[xcol], gs[ycol], s=6, alpha=0.35, label=source)

    plt.title(title)
    plt.xlabel(xcol)
    plt.ylabel(ycol)
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()


def generate_plots(results_df):
    print("=" * 80)
    print("GENERATING PLOTS")
    print("=" * 80)

    for source, g in results_df.groupby("source"):
        save_hist(
            g["v_measure"],
            f"HDBSCAN V-measure - {source}",
            "V-measure",
            PLOTS_DIR / f"hist_v_measure_{source}.png",
            bins=50,
        )

        save_hist(
            g["ari"],
            f"HDBSCAN ARI - {source}",
            "ARI",
            PLOTS_DIR / f"hist_ari_{source}.png",
            bins=50,
        )

        save_hist(
            g["ami"],
            f"HDBSCAN AMI - {source}",
            "AMI",
            PLOTS_DIR / f"hist_ami_{source}.png",
            bins=50,
        )

        save_hist(
            g["pairwise_mcc"],
            f"HDBSCAN pairwise MCC - {source}",
            "Pairwise MCC",
            PLOTS_DIR / f"hist_pairwise_mcc_{source}.png",
            bins=50,
        )

        save_hist(
            g["n_true_sources"],
            f"True sources per window - {source}",
            "True sources",
            PLOTS_DIR / f"hist_n_true_sources_{source}.png",
            bins=60,
        )

        save_hist(
            g["n_pred_clusters_excluding_noise"],
            f"Predicted clusters per window - {source}",
            "Predicted clusters excluding noise",
            PLOTS_DIR / f"hist_n_pred_clusters_{source}.png",
            bins=60,
        )

        save_hist(
            g["noise_share"],
            f"HDBSCAN noise share - {source}",
            "Noise share",
            PLOTS_DIR / f"hist_noise_share_{source}.png",
            bins=50,
        )

    save_scatter(
        results_df,
        "n_true_sources",
        "v_measure",
        "V-measure vs number of true sources",
        PLOTS_DIR / "scatter_v_measure_vs_n_true_sources.png",
    )

    save_scatter(
        results_df,
        "n_true_sources",
        "n_pred_clusters_excluding_noise",
        "Predicted clusters vs true sources",
        PLOTS_DIR / "scatter_pred_clusters_vs_true_sources.png",
    )

    save_scatter(
        results_df,
        "dominant_source_share",
        "v_measure",
        "V-measure vs dominant source share",
        PLOTS_DIR / "scatter_v_measure_vs_dominant_share.png",
    )


def write_markdown_report(results_df, summary_df, complexity_df):
    report_path = OUT_DIR / "hdbscan_deinterleaving_report.md"

    with open(report_path, "w", encoding="utf-8") as f:
        f.write("# HDBSCAN Baseline for TSRD Pulse Deinterleaving\n\n")

        f.write("## Task\n\n")
        f.write(
            "The task is to separate interleaved radar pulses into clusters corresponding "
            "to their originating emitters. The algorithm receives only the PDW features; "
            "ground-truth labels are used only after clustering for evaluation.\n\n"
        )

        f.write("## Configuration\n\n")
        f.write(f"- Window size: {WINDOW_SIZE} pulses\n")
        f.write(f"- Preprocessing mode: `{PREPROCESS_MODE}`\n")
        f.write(f"- HDBSCAN backend: `{HDBSCAN_BACKEND}`\n")
        f.write(f"- min_cluster_size: {HDBSCAN_MIN_CLUSTER_SIZE}\n")
        f.write(f"- min_samples: {HDBSCAN_MIN_SAMPLES}\n")
        f.write(f"- cluster_selection_epsilon: {HDBSCAN_CLUSTER_SELECTION_EPSILON}\n")
        f.write(f"- max_windows_per_source: {MAX_WINDOWS_PER_SOURCE}\n\n")

        f.write("## Summary\n\n")
        if len(summary_df) > 0:
            f.write(summary_df.to_markdown(index=False))
            f.write("\n\n")

        f.write("## Complexity summary\n\n")
        if len(complexity_df) > 0:
            f.write(complexity_df.to_markdown(index=False))
            f.write("\n\n")

        f.write("## Output files\n\n")
        f.write("- `tables/window_metrics_hdbscan.csv`\n")
        f.write("- `tables/summary_by_source_split.csv`\n")
        f.write("- `tables/summary_by_n_true_sources.csv`\n")
        f.write("- `plots/*.png`\n\n")

    return report_path


# ============================================================
# MAIN
# ============================================================
def main():
    print("=" * 80)
    print("TSRD HDBSCAN DEINTERLEAVING BASELINE")
    print("=" * 80)

    print(f"HDBSCAN backend       : {HDBSCAN_BACKEND}")
    print(f"Preprocess mode       : {PREPROCESS_MODE}")
    print(f"Window size           : {WINDOW_SIZE}")
    print(f"Max windows per source: {MAX_WINDOWS_PER_SOURCE}")

    results_path = TABLES_DIR / "window_metrics_hdbscan.csv"

    # Remove previous results for a clean run.
    if results_path.exists():
        results_path.unlink()

    total_windows_done = 0

    for source, folder in DATA_DIRS.items():
        print("\n" + "=" * 80)
        print(f"PROCESSING SOURCE: {source}")
        print(f"FOLDER: {folder}")
        print("=" * 80)

        h5_files = sorted(folder.rglob("*.h5"))
        h5_files = [p for p in h5_files if should_process_file(p)]

        if MAX_FILES_PER_SOURCE is not None:
            h5_files = h5_files[:int(MAX_FILES_PER_SOURCE)]

        print(f"H5 files selected: {len(h5_files)}")

        source_windows_done = 0
        buffer_rows = []

        pbar = tqdm(h5_files, desc=f"{source}", unit="file")

        for path in pbar:
            if MAX_WINDOWS_PER_SOURCE is not None:
                remaining = int(MAX_WINDOWS_PER_SOURCE) - source_windows_done
                if remaining <= 0:
                    break
            else:
                remaining = None

            rows = process_one_file(
                path=path,
                source=source,
                max_windows_remaining=remaining,
            )

            if len(rows) > 0:
                buffer_rows.extend(rows)

                # Count only successful window rows.
                n_new_windows = sum(
                    1 for r in rows
                    if "v_measure" in r and r.get("cluster_method") == "HDBSCAN"
                )

                source_windows_done += n_new_windows
                total_windows_done += n_new_windows

            pbar.set_postfix({
                "source_windows": source_windows_done,
                "total_windows": total_windows_done,
            })

            if len(buffer_rows) >= SAVE_EVERY_WINDOWS:
                append_rows_to_csv(buffer_rows, results_path)
                buffer_rows = []

        if len(buffer_rows) > 0:
            append_rows_to_csv(buffer_rows, results_path)
            buffer_rows = []

        print(f"[DONE] {source}: {source_windows_done} windows processed")

    if not results_path.exists():
        print("[ERROR] No results were generated.")
        return

    # --------------------------------------------------------
    # Load results
    # --------------------------------------------------------
    results_df = pd.read_csv(results_path)

    # Keep only valid metric rows.
    results_df = results_df[results_df["cluster_method"] == "HDBSCAN"].copy()
    results_df = results_df.dropna(subset=["v_measure"])

    # --------------------------------------------------------
    # Summaries
    # --------------------------------------------------------
    summary_df = summarize_results(results_df)
    complexity_df = summarize_by_complexity(results_df)

    summary_path = TABLES_DIR / "summary_by_source_split.csv"
    complexity_path = TABLES_DIR / "summary_by_n_true_sources.csv"

    summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
    complexity_df.to_csv(complexity_path, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # Optional oracle KMeans diagnostic
    # --------------------------------------------------------
    if RUN_ORACLE_KMEANS:
        print(
            "[INFO] RUN_ORACLE_KMEANS=True, but oracle KMeans is not implemented "
            "in the main loop to keep the baseline runtime manageable."
        )

    # --------------------------------------------------------
    # Plots and report
    # --------------------------------------------------------
    generate_plots(results_df)

    report_path = write_markdown_report(
        results_df=results_df,
        summary_df=summary_df,
        complexity_df=complexity_df,
    )

    # --------------------------------------------------------
    # JSON config
    # --------------------------------------------------------
    config = {
        "root": str(ROOT),
        "window_size": WINDOW_SIZE,
        "drop_incomplete_windows": DROP_INCOMPLETE_WINDOWS,
        "max_files_per_source": MAX_FILES_PER_SOURCE,
        "max_windows_per_source": MAX_WINDOWS_PER_SOURCE,
        "splits_to_process": SPLITS_TO_PROCESS,
        "preprocess_mode": PREPROCESS_MODE,
        "hdbscan_backend": HDBSCAN_BACKEND,
        "hdbscan_min_cluster_size": HDBSCAN_MIN_CLUSTER_SIZE,
        "hdbscan_min_samples": HDBSCAN_MIN_SAMPLES,
        "hdbscan_cluster_selection_epsilon": HDBSCAN_CLUSTER_SELECTION_EPSILON,
        "hdbscan_cluster_selection_method": HDBSCAN_CLUSTER_SELECTION_METHOD,
        "hdbscan_allow_single_cluster": HDBSCAN_ALLOW_SINGLE_CLUSTER,
        "run_oracle_kmeans": RUN_ORACLE_KMEANS,
        "outputs": {
            "window_metrics": str(results_path),
            "summary": str(summary_path),
            "complexity_summary": str(complexity_path),
            "plots": str(PLOTS_DIR),
            "report": str(report_path),
        },
    }

    save_json(config, OUT_DIR / "config.json")

    # --------------------------------------------------------
    # Console summary
    # --------------------------------------------------------
    print("\n" + "=" * 80)
    print("FINAL SUMMARY")
    print("=" * 80)

    cols_to_show = [
        "source",
        "split",
        "cluster_method",
        "preprocess_mode",
        "n_windows",
        "v_measure_median",
        "ari_median",
        "ami_median",
        "homogeneity_median",
        "completeness_median",
        "pairwise_f1_median",
        "pairwise_mcc_median",
        "n_true_sources_median",
        "n_pred_clusters_excluding_noise_median",
        "noise_share_median",
    ]

    show_cols = [c for c in cols_to_show if c in summary_df.columns]

    print(summary_df[show_cols].to_string(index=False))

    print("\n" + "=" * 80)
    print("OUTPUT FILES")
    print("=" * 80)
    print(f"Window metrics     : {results_path}")
    print(f"Summary            : {summary_path}")
    print(f"Complexity summary : {complexity_path}")
    print(f"Plots directory    : {PLOTS_DIR}")
    print(f"Markdown report    : {report_path}")
    print(f"Config JSON        : {OUT_DIR / 'config.json'}")

    print("\nDone.")


if __name__ == "__main__":
    main()

[INFO] Installing missing package: hdbscan
TSRD HDBSCAN DEINTERLEAVING BASELINE
HDBSCAN backend       : hdbscan
Preprocess mode       : engineered
Window size           : 1024
Max windows per source: 5000

PROCESSING SOURCE: archive
FOLDER: D:\Fusion\turing_synthetic_radar_dataset\archive
H5 files selected: 3000


archive:   1%|          | 34/3000 [04:13<6:09:05,  7.47s/file, source_windows=5000, total_windows=5000] 


[DONE] archive: 5000 windows processed

PROCESSING SOURCE: scan
FOLDER: D:\Fusion\turing_synthetic_radar_dataset\scan
H5 files selected: 3000


scan:   1%|▏         | 43/3000 [04:38<5:18:40,  6.47s/file, source_windows=5000, total_windows=1e+4]


[DONE] scan: 5000 windows processed
GENERATING PLOTS

FINAL SUMMARY
 source   split cluster_method preprocess_mode  n_windows  v_measure_median  ari_median  ami_median  homogeneity_median  completeness_median  pairwise_f1_median  pairwise_mcc_median  n_true_sources_median  n_pred_clusters_excluding_noise_median  noise_share_median
archive    test        HDBSCAN      engineered       5000          0.541267    0.269792    0.511742            0.901181             0.388593            0.588027             0.378438                   13.0                                    11.0            0.041016
   scan unknown        HDBSCAN      engineered       5000          0.786754    0.743529    0.779946            0.949955             0.716885            0.834839             0.762698                    8.0                                    11.0            0.020996

OUTPUT FILES
Window metrics     : D:\Fusion\turing_synthetic_radar_dataset\deinterleaving_baseline_hdbscan\tables\window_metrics_hdbscan

In [6]:
# ============================================================
# baseline_hdbscan_deinterleaving_archive_splits.py
# ============================================================
# TSRD - HDBSCAN deinterleaving baseline
#
# Inputs:
#   D:\Fusion\turing_synthetic_radar_dataset\archive\train
#   D:\Fusion\turing_synthetic_radar_dataset\archive\validation
#   D:\Fusion\turing_synthetic_radar_dataset\archive\test
#
# Each .h5:
#   data   : [n_pulses, 5]
#   labels : [n_pulses, 1]
#
# PDW columns:
#   0 -> ToA_us
#   1 -> Frequency_MHz
#   2 -> PulseWidth_us
#   3 -> AoA_deg
#   4 -> Amplitude_dB
#
# Method:
#   - Non-overlapping windows of 1024 pulses
#   - PDW preprocessing
#   - HDBSCAN clustering without using true number of sources
#   - Evaluation with clustering and pairwise metrics
#
# Outputs:
#   D:\Fusion\turing_synthetic_radar_dataset\deinterleaving_baseline_hdbscan_archive_splits
# ============================================================

from pathlib import Path
import sys
import json
import math
import time
import subprocess
import warnings
warnings.filterwarnings("ignore")


# ============================================================
# INSTALL / IMPORTS
# ============================================================
def ensure_package(import_name, pip_name=None):
    try:
        __import__(import_name)
    except Exception:
        pkg = pip_name if pip_name is not None else import_name
        print(f"[INFO] Installing missing package: {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


ensure_package("numpy")
ensure_package("pandas")
ensure_package("matplotlib")
ensure_package("h5py")
ensure_package("sklearn", "scikit-learn")
ensure_package("tqdm")
ensure_package("tabulate")

try:
    ensure_package("hdbscan")
    import hdbscan
    HDBSCAN_BACKEND = "hdbscan"
except Exception:
    hdbscan = None
    HDBSCAN_BACKEND = "sklearn"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py

from tqdm import tqdm

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import (
    v_measure_score,
    adjusted_rand_score,
    adjusted_mutual_info_score,
    homogeneity_score,
    completeness_score,
)

try:
    from sklearn.cluster import HDBSCAN as SKHDBSCAN
except Exception:
    SKHDBSCAN = None


# ============================================================
# PATHS
# ============================================================
ROOT = Path(r"D:\Fusion\turing_synthetic_radar_dataset")

DATA_DIRS = {
    "train": ROOT / "archive" / "train",
    "validation": ROOT / "archive" / "validation",
    "test": ROOT / "archive" / "test",
}

SOURCE_NAME = "archive"

OUT_DIR = ROOT / "deinterleaving_baseline_hdbscan_archive_splits"
TABLES_DIR = OUT_DIR / "tables"
PLOTS_DIR = OUT_DIR / "plots"

OUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# CONFIG
# ============================================================
SEED = 42
rng = np.random.default_rng(SEED)

WINDOW_SIZE = 1024
DROP_INCOMPLETE_WINDOWS = True

# For fast tests, use e.g. 5000.
# For full split evaluation, set to None.
MAX_WINDOWS_PER_SPLIT = 5000

# Options:
#   "random"     -> uniformly samples windows across all files in each split
#   "sequential" -> reads first windows/files until limit
WINDOW_SELECTION_MODE = "random"

# Use None to process all files in each split.
MAX_FILES_PER_SPLIT = None

# Options:
#   "paper_raw"  -> clean + StandardScaler over original 5 PDWs
#   "engineered" -> derived features: relative ToA, delta-ToA, log PW,
#                   normalized frequency, AoA sin/cos, amplitude and missing mask
PREPROCESS_MODE = "engineered"

# HDBSCAN parameters
HDBSCAN_MIN_CLUSTER_SIZE = 5
HDBSCAN_MIN_SAMPLES = 3
HDBSCAN_CLUSTER_SELECTION_EPSILON = 0.0
HDBSCAN_CLUSTER_SELECTION_METHOD = "eom"
HDBSCAN_ALLOW_SINGLE_CLUSTER = False

SAVE_EVERY_WINDOWS = 500

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 180


# ============================================================
# HELPERS
# ============================================================
def save_json(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def h5_has_dataset(f, name):
    try:
        return isinstance(f[name], h5py.Dataset)
    except Exception:
        return False


def comb2(x):
    x = np.asarray(x, dtype=np.int64)
    return x * (x - 1) // 2


def safe_float(x):
    try:
        if np.isfinite(x):
            return float(x)
        return float("nan")
    except Exception:
        return float("nan")


def fill_nonfinite_with_median(v, default=0.0):
    v = np.asarray(v, dtype=np.float32).copy()
    finite = np.isfinite(v)

    if finite.any():
        med = float(np.nanmedian(v[finite]))
    else:
        med = float(default)

    v[~finite] = med
    return v


def append_rows_to_csv(rows, path: Path):
    if len(rows) == 0:
        return

    df = pd.DataFrame(rows)
    write_header = not path.exists()

    df.to_csv(
        path,
        mode="a",
        header=write_header,
        index=False,
        encoding="utf-8-sig",
    )


# ============================================================
# FILE / WINDOW INDEX
# ============================================================
def collect_file_index(split_name: str, folder: Path):
    rows = []

    if not folder.exists():
        print(f"[WARNING] Missing folder: {folder}")
        return pd.DataFrame(rows)

    h5_files = sorted(folder.rglob("*.h5"))

    if MAX_FILES_PER_SPLIT is not None:
        h5_files = h5_files[:int(MAX_FILES_PER_SPLIT)]

    print(f"[{split_name}] H5 files found: {len(h5_files)}")

    for p in tqdm(h5_files, desc=f"index_{split_name}", unit="file"):
        row = {
            "source": SOURCE_NAME,
            "split": split_name,
            "file_id": p.stem,
            "file_name": p.name,
            "path": str(p),
            "status": "ok",
            "n_pulses": 0,
            "n_windows": 0,
            "error": None,
        }

        try:
            with h5py.File(p, "r") as f:
                if not h5_has_dataset(f, "data") or not h5_has_dataset(f, "labels"):
                    row["status"] = "missing_data_or_labels"
                    rows.append(row)
                    continue

                n = int(f["data"].shape[0])
                row["n_pulses"] = n

                if n <= 0:
                    row["status"] = "empty"
                    rows.append(row)
                    continue

                if DROP_INCOMPLETE_WINDOWS:
                    n_windows = n // WINDOW_SIZE
                else:
                    n_windows = int(math.ceil(n / WINDOW_SIZE))

                row["n_windows"] = int(n_windows)

        except Exception as e:
            row["status"] = "error"
            row["error"] = str(e)

        rows.append(row)

    df = pd.DataFrame(rows)

    return df


def select_random_window_tasks(file_index_df: pd.DataFrame, max_windows):
    valid = file_index_df[
        (file_index_df["status"] == "ok") &
        (file_index_df["n_windows"] > 0)
    ].copy().reset_index(drop=True)

    if len(valid) == 0:
        return pd.DataFrame()

    total_windows = int(valid["n_windows"].sum())

    if max_windows is None:
        sample_n = total_windows
    else:
        sample_n = min(int(max_windows), total_windows)

    if sample_n <= 0:
        return pd.DataFrame()

    # Random without replacement over the global window index.
    global_indices = rng.choice(total_windows, size=sample_n, replace=False)
    global_indices = np.sort(global_indices)

    n_windows = valid["n_windows"].values.astype(np.int64)
    cum_end = np.cumsum(n_windows)
    cum_start = cum_end - n_windows

    file_pos = np.searchsorted(cum_end, global_indices, side="right")
    local_win = global_indices - cum_start[file_pos]

    tasks = valid.iloc[file_pos].copy()
    tasks["window_index"] = local_win.astype(int)
    tasks = tasks.sort_values(["path", "window_index"]).reset_index(drop=True)

    return tasks


# ============================================================
# PREPROCESSING
# ============================================================
def clean_raw_pdw(x):
    x = np.asarray(x, dtype=np.float32).copy()

    # Amplitude can contain inf in archive/stare.
    amp = x[:, 4].copy()
    amp[~np.isfinite(amp)] = -200.0
    x[:, 4] = amp

    for j in range(x.shape[1]):
        x[:, j] = fill_nonfinite_with_median(x[:, j], default=0.0)

    return x


def preprocess_window_paper_raw(x):
    x = clean_raw_pdw(x)

    scaler = StandardScaler()
    z = scaler.fit_transform(x)

    z = np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0)
    return z.astype(np.float32)


def preprocess_window_engineered(x):
    x = np.asarray(x, dtype=np.float32).copy()

    toa = fill_nonfinite_with_median(x[:, 0], default=0.0)
    freq = fill_nonfinite_with_median(x[:, 1], default=0.0)
    pw = fill_nonfinite_with_median(x[:, 2], default=0.0)
    aoa = fill_nonfinite_with_median(x[:, 3], default=0.0)

    amp = x[:, 4].copy()
    amp_missing = (~np.isfinite(amp)).astype(np.float32)
    amp[~np.isfinite(amp)] = -200.0
    amp = fill_nonfinite_with_median(amp, default=-200.0)

    # Relative ToA inside the window.
    toa_rel = toa - np.min(toa)
    span = float(np.max(toa_rel) - np.min(toa_rel))

    if span > 0:
        toa_rel_norm = toa_rel / (span + 1e-9)
    else:
        toa_rel_norm = np.zeros_like(toa_rel, dtype=np.float32)

    # Delta-ToA.
    dtoa = np.diff(toa, prepend=toa[0])
    dtoa = np.maximum(dtoa, 0.0)
    dtoa_log = np.log1p(dtoa)

    # Frequency: MHz normalized by 18 GHz.
    freq_norm = freq / 18000.0

    # Pulse width.
    pw = np.maximum(pw, 0.0)
    pw_log = np.log1p(pw)

    # AoA circular encoding.
    aoa_rad = np.deg2rad(aoa.astype(np.float32))
    aoa_sin = np.sin(aoa_rad)
    aoa_cos = np.cos(aoa_rad)

    # Amplitude rough normalization.
    amp_norm = (amp + 220.0) / 260.0

    feats = np.stack(
        [
            toa_rel_norm,
            dtoa_log,
            freq_norm,
            pw_log,
            aoa_sin,
            aoa_cos,
            amp_norm,
            amp_missing,
        ],
        axis=1,
    ).astype(np.float32)

    # Robust scale continuous columns.
    z = feats.copy()
    continuous_cols = [0, 1, 2, 3, 4, 5, 6]

    scaler = RobustScaler()
    z[:, continuous_cols] = scaler.fit_transform(z[:, continuous_cols])

    z = np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0)
    return z.astype(np.float32)


def preprocess_window(x):
    if PREPROCESS_MODE == "paper_raw":
        return preprocess_window_paper_raw(x)

    if PREPROCESS_MODE == "engineered":
        return preprocess_window_engineered(x)

    raise ValueError(f"Unknown PREPROCESS_MODE: {PREPROCESS_MODE}")


# ============================================================
# CLUSTERING
# ============================================================
def run_hdbscan(X):
    X = np.asarray(X, dtype=np.float32)

    if HDBSCAN_BACKEND == "hdbscan" and hdbscan is not None:
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
            min_samples=HDBSCAN_MIN_SAMPLES,
            metric="euclidean",
            cluster_selection_epsilon=HDBSCAN_CLUSTER_SELECTION_EPSILON,
            cluster_selection_method=HDBSCAN_CLUSTER_SELECTION_METHOD,
            allow_single_cluster=HDBSCAN_ALLOW_SINGLE_CLUSTER,
        )
        return clusterer.fit_predict(X).astype(int)

    if SKHDBSCAN is not None:
        clusterer = SKHDBSCAN(
            min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
            min_samples=HDBSCAN_MIN_SAMPLES,
            metric="euclidean",
            cluster_selection_epsilon=HDBSCAN_CLUSTER_SELECTION_EPSILON,
            allow_single_cluster=HDBSCAN_ALLOW_SINGLE_CLUSTER,
        )
        return clusterer.fit_predict(X).astype(int)

    raise RuntimeError("No HDBSCAN backend available. Try: pip install hdbscan")


# ============================================================
# METRICS
# ============================================================
def efficient_pairwise_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    n = len(y_true)
    total_pairs = n * (n - 1) // 2

    if n < 2 or total_pairs <= 0:
        return {
            "pairwise_f1": np.nan,
            "pairwise_mcc": np.nan,
            "pairwise_tp": 0,
            "pairwise_fp": 0,
            "pairwise_fn": 0,
            "pairwise_tn": 0,
        }

    true_vals, true_inv = np.unique(y_true, return_inverse=True)
    pred_vals, pred_inv = np.unique(y_pred, return_inverse=True)

    contingency = np.zeros((len(true_vals), len(pred_vals)), dtype=np.int64)
    np.add.at(contingency, (true_inv, pred_inv), 1)

    tp = int(comb2(contingency).sum())

    true_sizes = contingency.sum(axis=1)
    pred_sizes = contingency.sum(axis=0)

    true_pairs = int(comb2(true_sizes).sum())
    pred_pairs = int(comb2(pred_sizes).sum())

    fp = int(pred_pairs - tp)
    fn = int(true_pairs - tp)
    tn = int(total_pairs - tp - fp - fn)

    denom_f1 = 2 * tp + fp + fn

    if denom_f1 > 0:
        pairwise_f1 = float((2 * tp) / denom_f1)
    else:
        pairwise_f1 = 0.0

    denom_mcc = math.sqrt(
        max(tp + fp, 0)
        * max(tp + fn, 0)
        * max(tn + fp, 0)
        * max(tn + fn, 0)
    )

    if denom_mcc > 0:
        pairwise_mcc = float((tp * tn - fp * fn) / denom_mcc)
    else:
        pairwise_mcc = 0.0

    return {
        "pairwise_f1": pairwise_f1,
        "pairwise_mcc": pairwise_mcc,
        "pairwise_tp": tp,
        "pairwise_fp": fp,
        "pairwise_fn": fn,
        "pairwise_tn": tn,
    }


def dominant_source_share(y_true):
    y_true = np.asarray(y_true).reshape(-1)

    if len(y_true) == 0:
        return np.nan

    _, counts = np.unique(y_true, return_counts=True)
    return float(counts.max() / counts.sum())


def source_entropy_norm(y_true):
    y_true = np.asarray(y_true).reshape(-1)

    if len(y_true) == 0:
        return np.nan

    _, counts = np.unique(y_true, return_counts=True)

    counts = counts.astype(np.float64)
    p = counts / counts.sum()
    p = p[p > 0]

    if len(p) <= 1:
        return 0.0

    return float(-np.sum(p * np.log(p)) / np.log(len(p)))


def deinterleaving_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1).astype(int)
    y_pred = np.asarray(y_pred).reshape(-1).astype(int)

    out = {
        "v_measure": safe_float(v_measure_score(y_true, y_pred)),
        "ari": safe_float(adjusted_rand_score(y_true, y_pred)),
        "ami": safe_float(adjusted_mutual_info_score(y_true, y_pred)),
        "homogeneity": safe_float(homogeneity_score(y_true, y_pred)),
        "completeness": safe_float(completeness_score(y_true, y_pred)),
    }

    out.update(efficient_pairwise_metrics(y_true, y_pred))

    true_sources = np.unique(y_true)
    pred_clusters = np.unique(y_pred)

    out["n_pulses"] = int(len(y_true))
    out["n_true_sources"] = int(len(true_sources))
    out["n_pred_clusters_including_noise"] = int(len(pred_clusters))
    out["n_pred_clusters_excluding_noise"] = int(len([c for c in pred_clusters if c != -1]))
    out["has_noise_cluster"] = int(np.any(y_pred == -1))
    out["noise_share"] = float(np.mean(y_pred == -1))

    return out


# ============================================================
# WINDOW PROCESSING
# ============================================================
def process_window(
    x_window,
    y_window,
    split_name,
    file_id,
    file_name,
    window_index,
    start_pulse,
    end_pulse,
):
    t0 = time.time()

    x_window = np.asarray(x_window, dtype=np.float32)
    y_window = np.asarray(y_window).reshape(-1).astype(int)

    X = preprocess_window(x_window)
    y_pred = run_hdbscan(X)

    metrics = deinterleaving_metrics(y_window, y_pred)

    row = {
        "source": SOURCE_NAME,
        "split": split_name,
        "file_id": file_id,
        "file_name": file_name,
        "window_index": int(window_index),
        "start_pulse": int(start_pulse),
        "end_pulse": int(end_pulse),
        "window_size": int(end_pulse - start_pulse),
        "status": "ok",
        "preprocess_mode": PREPROCESS_MODE,
        "cluster_method": "HDBSCAN",
        "hdbscan_backend": HDBSCAN_BACKEND,
        "hdbscan_min_cluster_size": HDBSCAN_MIN_CLUSTER_SIZE,
        "hdbscan_min_samples": HDBSCAN_MIN_SAMPLES,
        "hdbscan_epsilon": HDBSCAN_CLUSTER_SELECTION_EPSILON,
        "dominant_source_share": dominant_source_share(y_window),
        "source_entropy_norm": source_entropy_norm(y_window),
        "elapsed_sec": float(time.time() - t0),
    }

    row.update(metrics)

    return row


def process_file_selected_windows(path: Path, split_name: str, window_indices):
    rows = []
    file_id = path.stem

    try:
        with h5py.File(path, "r") as f:
            if not h5_has_dataset(f, "data") or not h5_has_dataset(f, "labels"):
                return rows

            data = f["data"]
            labels = f["labels"]
            n = int(data.shape[0])

            for w in window_indices:
                w = int(w)
                start = w * WINDOW_SIZE
                end = min((w + 1) * WINDOW_SIZE, n)

                if DROP_INCOMPLETE_WINDOWS and (end - start) < WINDOW_SIZE:
                    continue

                x_win = np.asarray(data[start:end, :5], dtype=np.float32)
                y_win = np.asarray(labels[start:end]).reshape(-1).astype(int)

                row = process_window(
                    x_window=x_win,
                    y_window=y_win,
                    split_name=split_name,
                    file_id=file_id,
                    file_name=path.name,
                    window_index=w,
                    start_pulse=start,
                    end_pulse=end,
                )

                rows.append(row)

    except Exception as e:
        rows.append({
            "source": SOURCE_NAME,
            "split": split_name,
            "file_id": path.stem,
            "file_name": path.name,
            "path": str(path),
            "status": "error",
            "cluster_method": "HDBSCAN",
            "preprocess_mode": PREPROCESS_MODE,
            "error": str(e),
        })

    return rows


def process_split_random(split_name: str, file_index_df: pd.DataFrame, results_path: Path):
    tasks = select_random_window_tasks(file_index_df, MAX_WINDOWS_PER_SPLIT)

    print(f"[{split_name}] Random selected windows: {len(tasks)}")

    if len(tasks) == 0:
        return 0

    buffer_rows = []
    done = 0

    grouped = tasks.groupby("path", sort=False)

    pbar = tqdm(grouped, total=tasks["path"].nunique(), desc=f"{split_name}", unit="file")

    for path_str, g in pbar:
        path = Path(path_str)
        window_indices = g["window_index"].values

        rows = process_file_selected_windows(
            path=path,
            split_name=split_name,
            window_indices=window_indices,
        )

        buffer_rows.extend(rows)

        n_ok = sum(1 for r in rows if r.get("status") == "ok")
        done += n_ok

        pbar.set_postfix({"windows": done})

        if len(buffer_rows) >= SAVE_EVERY_WINDOWS:
            append_rows_to_csv(buffer_rows, results_path)
            buffer_rows = []

    if len(buffer_rows) > 0:
        append_rows_to_csv(buffer_rows, results_path)

    return done


def process_split_sequential(split_name: str, file_index_df: pd.DataFrame, results_path: Path):
    valid = file_index_df[
        (file_index_df["status"] == "ok") &
        (file_index_df["n_windows"] > 0)
    ].copy().reset_index(drop=True)

    buffer_rows = []
    done = 0

    pbar = tqdm(valid.itertuples(index=False), total=len(valid), desc=f"{split_name}", unit="file")

    for row in pbar:
        if MAX_WINDOWS_PER_SPLIT is not None:
            remaining = int(MAX_WINDOWS_PER_SPLIT) - done

            if remaining <= 0:
                break
        else:
            remaining = int(row.n_windows)

        n_to_process = min(int(row.n_windows), remaining)
        window_indices = list(range(n_to_process))

        rows = process_file_selected_windows(
            path=Path(row.path),
            split_name=split_name,
            window_indices=window_indices,
        )

        buffer_rows.extend(rows)

        n_ok = sum(1 for r in rows if r.get("status") == "ok")
        done += n_ok

        pbar.set_postfix({"windows": done})

        if len(buffer_rows) >= SAVE_EVERY_WINDOWS:
            append_rows_to_csv(buffer_rows, results_path)
            buffer_rows = []

    if len(buffer_rows) > 0:
        append_rows_to_csv(buffer_rows, results_path)

    return done


# ============================================================
# SUMMARIES
# ============================================================
def summarize_results(results_df):
    metric_cols = [
        "v_measure",
        "ari",
        "ami",
        "homogeneity",
        "completeness",
        "pairwise_f1",
        "pairwise_mcc",
        "n_true_sources",
        "n_pred_clusters_excluding_noise",
        "noise_share",
        "dominant_source_share",
        "source_entropy_norm",
        "elapsed_sec",
    ]

    group_cols = ["source", "split", "cluster_method", "preprocess_mode"]

    rows = []

    for keys, g in results_df.groupby(group_cols, dropna=False):
        row = dict(zip(group_cols, keys))
        row["n_windows"] = int(len(g))

        for c in metric_cols:
            if c not in g.columns:
                continue

            vals = (
                pd.to_numeric(g[c], errors="coerce")
                .replace([np.inf, -np.inf], np.nan)
                .dropna()
            )

            if len(vals) == 0:
                row[f"{c}_mean"] = np.nan
                row[f"{c}_median"] = np.nan
                row[f"{c}_std"] = np.nan
            else:
                row[f"{c}_mean"] = float(vals.mean())
                row[f"{c}_median"] = float(vals.median())
                row[f"{c}_std"] = float(vals.std())

        rows.append(row)

    return pd.DataFrame(rows)


def summarize_by_complexity(results_df):
    df = results_df.copy()

    df["n_true_sources_bin"] = pd.cut(
        df["n_true_sources"],
        bins=[0, 1, 2, 5, 10, 20, 40, 80, 200],
        labels=[
            "1",
            "2",
            "3-5",
            "6-10",
            "11-20",
            "21-40",
            "41-80",
            "81+",
        ],
        include_lowest=True,
    )

    rows = []

    for keys, g in df.groupby(["split", "n_true_sources_bin"], dropna=False):
        split_name, bin_name = keys

        row = {
            "source": SOURCE_NAME,
            "split": split_name,
            "n_true_sources_bin": str(bin_name),
            "n_windows": int(len(g)),
        }

        for c in [
            "v_measure",
            "ari",
            "ami",
            "homogeneity",
            "completeness",
            "pairwise_f1",
            "pairwise_mcc",
            "noise_share",
        ]:
            vals = pd.to_numeric(g[c], errors="coerce").dropna()

            row[f"{c}_mean"] = float(vals.mean()) if len(vals) > 0 else np.nan
            row[f"{c}_median"] = float(vals.median()) if len(vals) > 0 else np.nan

        rows.append(row)

    return pd.DataFrame(rows)


# ============================================================
# PLOTS
# ============================================================
def save_hist(series, title, xlabel, path, bins=50):
    s = (
        pd.to_numeric(series, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    if len(s) == 0:
        return

    plt.figure(figsize=(8, 5))
    plt.hist(s.values, bins=bins)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Windows")
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()


def save_scatter(df, xcol, ycol, title, path, max_points=100_000):
    g = df[[xcol, ycol, "split"]].copy()

    g[xcol] = pd.to_numeric(g[xcol], errors="coerce")
    g[ycol] = pd.to_numeric(g[ycol], errors="coerce")

    g = g.replace([np.inf, -np.inf], np.nan).dropna()

    if len(g) == 0:
        return

    if len(g) > max_points:
        g = g.sample(max_points, random_state=SEED)

    plt.figure(figsize=(8, 5))

    for split_name, gs in g.groupby("split"):
        plt.scatter(gs[xcol], gs[ycol], s=6, alpha=0.35, label=split_name)

    plt.title(title)
    plt.xlabel(xcol)
    plt.ylabel(ycol)
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()


def generate_plots(results_df):
    print("=" * 80)
    print("GENERATING PLOTS")
    print("=" * 80)

    for split_name, g in results_df.groupby("split"):
        save_hist(
            g["v_measure"],
            f"HDBSCAN V-measure - {split_name}",
            "V-measure",
            PLOTS_DIR / f"hist_v_measure_{split_name}.png",
            bins=50,
        )

        save_hist(
            g["ari"],
            f"HDBSCAN ARI - {split_name}",
            "ARI",
            PLOTS_DIR / f"hist_ari_{split_name}.png",
            bins=50,
        )

        save_hist(
            g["ami"],
            f"HDBSCAN AMI - {split_name}",
            "AMI",
            PLOTS_DIR / f"hist_ami_{split_name}.png",
            bins=50,
        )

        save_hist(
            g["pairwise_mcc"],
            f"HDBSCAN pairwise MCC - {split_name}",
            "Pairwise MCC",
            PLOTS_DIR / f"hist_pairwise_mcc_{split_name}.png",
            bins=50,
        )

        save_hist(
            g["n_true_sources"],
            f"True sources per window - {split_name}",
            "True sources",
            PLOTS_DIR / f"hist_n_true_sources_{split_name}.png",
            bins=60,
        )

        save_hist(
            g["n_pred_clusters_excluding_noise"],
            f"Predicted clusters per window - {split_name}",
            "Predicted clusters excluding noise",
            PLOTS_DIR / f"hist_n_pred_clusters_{split_name}.png",
            bins=60,
        )

        save_hist(
            g["noise_share"],
            f"HDBSCAN noise share - {split_name}",
            "Noise share",
            PLOTS_DIR / f"hist_noise_share_{split_name}.png",
            bins=50,
        )

    save_scatter(
        results_df,
        "n_true_sources",
        "v_measure",
        "V-measure vs number of true sources",
        PLOTS_DIR / "scatter_v_measure_vs_n_true_sources.png",
    )

    save_scatter(
        results_df,
        "n_true_sources",
        "n_pred_clusters_excluding_noise",
        "Predicted clusters vs true sources",
        PLOTS_DIR / "scatter_pred_clusters_vs_true_sources.png",
    )

    save_scatter(
        results_df,
        "dominant_source_share",
        "v_measure",
        "V-measure vs dominant source share",
        PLOTS_DIR / "scatter_v_measure_vs_dominant_share.png",
    )


# ============================================================
# REPORT
# ============================================================
def write_markdown_report(summary_df, complexity_df, file_index_df):
    report_path = OUT_DIR / "hdbscan_deinterleaving_archive_splits_report.md"

    with open(report_path, "w", encoding="utf-8") as f:
        f.write("# HDBSCAN Baseline for TSRD Archive Splits\n\n")

        f.write("## Inputs\n\n")
        for split_name, folder in DATA_DIRS.items():
            f.write(f"- {split_name}: `{folder}`\n")

        f.write("\n## Task\n\n")
        f.write(
            "The task is pulse deinterleaving: clustering interleaved radar PDWs "
            "according to their originating emitters. The algorithm receives only PDW "
            "features. Ground-truth labels are used only for evaluation.\n\n"
        )

        f.write("## Configuration\n\n")
        f.write(f"- Source: `{SOURCE_NAME}`\n")
        f.write(f"- Window size: {WINDOW_SIZE}\n")
        f.write(f"- Window selection mode: `{WINDOW_SELECTION_MODE}`\n")
        f.write(f"- Max windows per split: {MAX_WINDOWS_PER_SPLIT}\n")
        f.write(f"- Preprocessing mode: `{PREPROCESS_MODE}`\n")
        f.write(f"- HDBSCAN backend: `{HDBSCAN_BACKEND}`\n")
        f.write(f"- min_cluster_size: {HDBSCAN_MIN_CLUSTER_SIZE}\n")
        f.write(f"- min_samples: {HDBSCAN_MIN_SAMPLES}\n")
        f.write(f"- cluster_selection_epsilon: {HDBSCAN_CLUSTER_SELECTION_EPSILON}\n\n")

        f.write("## File index summary\n\n")
        if len(file_index_df) > 0:
            idx_summary = (
                file_index_df
                .groupby("split", as_index=False)
                .agg(
                    n_files=("path", "count"),
                    n_ok=("status", lambda x: int(np.sum(np.asarray(x) == "ok"))),
                    total_pulses=("n_pulses", "sum"),
                    total_windows=("n_windows", "sum"),
                    mean_pulses=("n_pulses", "mean"),
                    median_pulses=("n_pulses", "median"),
                )
            )
            f.write(idx_summary.to_markdown(index=False))
            f.write("\n\n")

        f.write("## Metric summary\n\n")
        if len(summary_df) > 0:
            f.write(summary_df.to_markdown(index=False))
            f.write("\n\n")

        f.write("## Complexity summary\n\n")
        if len(complexity_df) > 0:
            f.write(complexity_df.to_markdown(index=False))
            f.write("\n\n")

        f.write("## Output files\n\n")
        f.write("- `tables/window_metrics_hdbscan.csv`\n")
        f.write("- `tables/file_index_archive_splits.csv`\n")
        f.write("- `tables/summary_by_split.csv`\n")
        f.write("- `tables/summary_by_n_true_sources.csv`\n")
        f.write("- `plots/*.png`\n\n")

    return report_path


# ============================================================
# MAIN
# ============================================================
def main():
    print("=" * 80)
    print("TSRD HDBSCAN DEINTERLEAVING BASELINE - ARCHIVE SPLITS")
    print("=" * 80)

    print(f"Source                 : {SOURCE_NAME}")
    print(f"HDBSCAN backend         : {HDBSCAN_BACKEND}")
    print(f"Preprocess mode         : {PREPROCESS_MODE}")
    print(f"Window size             : {WINDOW_SIZE}")
    print(f"Window selection mode   : {WINDOW_SELECTION_MODE}")
    print(f"Max windows per split   : {MAX_WINDOWS_PER_SPLIT}")

    results_path = TABLES_DIR / "window_metrics_hdbscan.csv"
    file_index_path = TABLES_DIR / "file_index_archive_splits.csv"

    if results_path.exists():
        results_path.unlink()

    all_index = []
    split_done = {}

    # --------------------------------------------------------
    # Build file index and process each split
    # --------------------------------------------------------
    for split_name, folder in DATA_DIRS.items():
        print("\n" + "=" * 80)
        print(f"INDEXING SPLIT: {split_name}")
        print(f"FOLDER: {folder}")
        print("=" * 80)

        idx_df = collect_file_index(split_name, folder)
        all_index.append(idx_df)

        if len(idx_df) == 0:
            print(f"[WARNING] No files indexed for split: {split_name}")
            split_done[split_name] = 0
            continue

        print("\n" + "=" * 80)
        print(f"PROCESSING SPLIT: {split_name}")
        print("=" * 80)

        if WINDOW_SELECTION_MODE == "random" and MAX_WINDOWS_PER_SPLIT is not None:
            n_done = process_split_random(split_name, idx_df, results_path)
        else:
            n_done = process_split_sequential(split_name, idx_df, results_path)

        split_done[split_name] = int(n_done)
        print(f"[DONE] {split_name}: {n_done} windows processed")

    if len(all_index) > 0:
        file_index_df = pd.concat(all_index, axis=0).reset_index(drop=True)
    else:
        file_index_df = pd.DataFrame()

    file_index_df.to_csv(file_index_path, index=False, encoding="utf-8-sig")

    if not results_path.exists():
        print("[ERROR] No HDBSCAN results were generated.")
        return

    # --------------------------------------------------------
    # Load metrics
    # --------------------------------------------------------
    results_df = pd.read_csv(results_path)

    results_df = results_df[
        (results_df["status"] == "ok") &
        (results_df["cluster_method"] == "HDBSCAN")
    ].copy()

    results_df = results_df.dropna(subset=["v_measure"])

    if len(results_df) == 0:
        print("[ERROR] No valid metric rows found.")
        return

    # --------------------------------------------------------
    # Summaries
    # --------------------------------------------------------
    summary_df = summarize_results(results_df)
    complexity_df = summarize_by_complexity(results_df)

    summary_path = TABLES_DIR / "summary_by_split.csv"
    complexity_path = TABLES_DIR / "summary_by_n_true_sources.csv"

    summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
    complexity_df.to_csv(complexity_path, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # Plots and report
    # --------------------------------------------------------
    generate_plots(results_df)

    report_path = write_markdown_report(
        summary_df=summary_df,
        complexity_df=complexity_df,
        file_index_df=file_index_df,
    )

    # --------------------------------------------------------
    # Config
    # --------------------------------------------------------
    config = {
        "root": str(ROOT),
        "data_dirs": {k: str(v) for k, v in DATA_DIRS.items()},
        "source_name": SOURCE_NAME,
        "window_size": WINDOW_SIZE,
        "drop_incomplete_windows": DROP_INCOMPLETE_WINDOWS,
        "max_files_per_split": MAX_FILES_PER_SPLIT,
        "max_windows_per_split": MAX_WINDOWS_PER_SPLIT,
        "window_selection_mode": WINDOW_SELECTION_MODE,
        "preprocess_mode": PREPROCESS_MODE,
        "hdbscan_backend": HDBSCAN_BACKEND,
        "hdbscan_min_cluster_size": HDBSCAN_MIN_CLUSTER_SIZE,
        "hdbscan_min_samples": HDBSCAN_MIN_SAMPLES,
        "hdbscan_cluster_selection_epsilon": HDBSCAN_CLUSTER_SELECTION_EPSILON,
        "hdbscan_cluster_selection_method": HDBSCAN_CLUSTER_SELECTION_METHOD,
        "hdbscan_allow_single_cluster": HDBSCAN_ALLOW_SINGLE_CLUSTER,
        "split_windows_processed": split_done,
        "outputs": {
            "window_metrics": str(results_path),
            "file_index": str(file_index_path),
            "summary": str(summary_path),
            "complexity_summary": str(complexity_path),
            "plots": str(PLOTS_DIR),
            "report": str(report_path),
        },
    }

    config_path = OUT_DIR / "config.json"
    save_json(config, config_path)

    # --------------------------------------------------------
    # Console summary
    # --------------------------------------------------------
    print("\n" + "=" * 80)
    print("FINAL SUMMARY")
    print("=" * 80)

    cols_to_show = [
        "source",
        "split",
        "cluster_method",
        "preprocess_mode",
        "n_windows",
        "v_measure_median",
        "ari_median",
        "ami_median",
        "homogeneity_median",
        "completeness_median",
        "pairwise_f1_median",
        "pairwise_mcc_median",
        "n_true_sources_median",
        "n_pred_clusters_excluding_noise_median",
        "noise_share_median",
    ]

    show_cols = [c for c in cols_to_show if c in summary_df.columns]
    print(summary_df[show_cols].to_string(index=False))

    print("\n" + "=" * 80)
    print("OUTPUT FILES")
    print("=" * 80)
    print(f"Window metrics     : {results_path}")
    print(f"File index         : {file_index_path}")
    print(f"Summary            : {summary_path}")
    print(f"Complexity summary : {complexity_path}")
    print(f"Plots directory    : {PLOTS_DIR}")
    print(f"Markdown report    : {report_path}")
    print(f"Config JSON        : {config_path}")

    print("\nDone.")


if __name__ == "__main__":
    main()

TSRD HDBSCAN DEINTERLEAVING BASELINE - ARCHIVE SPLITS
Source                 : archive
HDBSCAN backend         : hdbscan
Preprocess mode         : engineered
Window size             : 1024
Window selection mode   : random
Max windows per split   : 5000

INDEXING SPLIT: train
FOLDER: D:\Fusion\turing_synthetic_radar_dataset\archive\train
[train] H5 files found: 2500


index_train: 100%|██████████| 2500/2500 [00:41<00:00, 59.94file/s]



PROCESSING SPLIT: train
[train] Random selected windows: 5000


train: 100%|██████████| 1165/1165 [05:08<00:00,  3.78file/s, windows=5000]


[DONE] train: 5000 windows processed

INDEXING SPLIT: validation
FOLDER: D:\Fusion\turing_synthetic_radar_dataset\archive\validation
[validation] H5 files found: 250


...

[DONE] validation: 5000 windows processed

INDEXING SPLIT: test
FOLDER: D:\Fusion\turing_synthetic_radar_dataset\archive\test
[test] H5 files found: 250


index_test: 100%|██████████| 250/250 [00:04<00:00, 57.97file/s]



PROCESSING SPLIT: test
[test] Random selected windows: 5000


test: 100%|██████████| 216/216 [04:16<00:00,  1.19s/file, windows=5000]


[DONE] test: 5000 windows processed
GENERATING PLOTS

FINAL SUMMARY
 source      split cluster_method preprocess_mode  n_windows  v_measure_median  ari_median  ami_median  homogeneity_median  completeness_median  pairwise_f1_median  pairwise_mcc_median  n_true_sources_median  n_pred_clusters_excluding_noise_median  noise_share_median
archive       test        HDBSCAN      engineered       5000          0.504271    0.223892    0.479213            0.894303             0.357303            0.563322             0.330252                   11.0                                    10.0            0.033203
archive      train        HDBSCAN      engineered       5000          0.526950    0.263072    0.505874            0.890674             0.379983            0.580026             0.363750                   12.0                                    10.0            0.030273
archive validation        HDBSCAN      engineered       5000          0.497615    0.233008    0.476702            0.896955      

# REPRODUÇÃO PAPER DE REFERÊNCIA: The Turing Synthetic Radar Dataset: A dataset for pulse deinterleaving.

In [7]:
# ============================================================
# baseline_hdbscan_reference_methodology.py
# ============================================================
# TSRD - HDBSCAN baseline following the reference methodology
#
# Objective:
#   Compare directly with the reference paper baseline:
#   raw PDWs + non-overlapping 1024-pulse windows + HDBSCAN.
#
# Inputs:
#   D:\Fusion\turing_synthetic_radar_dataset\archive\train
#   D:\Fusion\turing_synthetic_radar_dataset\archive\validation
#   D:\Fusion\turing_synthetic_radar_dataset\archive\test
#
# Main direct comparison:
#   archive/test  -> reference paper Stare test baseline
#
# Each .h5:
#   data   : [n_pulses, 5]
#   labels : [n_pulses, 1]
#
# PDW columns:
#   0 -> ToA_us
#   1 -> Frequency_MHz
#   2 -> PulseWidth_us
#   3 -> AoA_deg
#   4 -> Amplitude_dB
#
# Method:
#   1. Read raw PDWs.
#   2. Apply only minimal finite-value sanitation.
#   3. Split into non-overlapping windows of 1024 pulses.
#   4. Run HDBSCAN without using the true number of emitters.
#   5. Evaluate predicted clusters against labels.
#
# Outputs:
#   D:\Fusion\turing_synthetic_radar_dataset\reference_methodology_hdbscan
# ============================================================

from pathlib import Path
import sys
import json
import math
import time
import subprocess
import warnings
warnings.filterwarnings("ignore")


# ============================================================
# INSTALL / IMPORTS
# ============================================================
def ensure_package(import_name, pip_name=None):
    try:
        __import__(import_name)
    except Exception:
        pkg = pip_name if pip_name is not None else import_name
        print(f"[INFO] Installing missing package: {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


ensure_package("numpy")
ensure_package("pandas")
ensure_package("matplotlib")
ensure_package("h5py")
ensure_package("sklearn", "scikit-learn")
ensure_package("tqdm")
ensure_package("tabulate")

try:
    ensure_package("hdbscan")
    import hdbscan
    HDBSCAN_BACKEND = "hdbscan"
except Exception:
    hdbscan = None
    HDBSCAN_BACKEND = "sklearn"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py

from tqdm import tqdm

from sklearn.metrics import (
    v_measure_score,
    adjusted_rand_score,
    adjusted_mutual_info_score,
    homogeneity_score,
    completeness_score,
)

try:
    from sklearn.cluster import HDBSCAN as SKHDBSCAN
except Exception:
    SKHDBSCAN = None


# ============================================================
# PATHS
# ============================================================
ROOT = Path(r"D:\Fusion\turing_synthetic_radar_dataset")

DATA_DIRS = {
    "train": ROOT / "archive" / "train",
    "validation": ROOT / "archive" / "validation",
    "test": ROOT / "archive" / "test",
}

SOURCE_NAME = "archive"

OUT_DIR = ROOT / "reference_methodology_hdbscan"
TABLES_DIR = OUT_DIR / "tables"
PLOTS_DIR = OUT_DIR / "plots"

OUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# CONFIG
# ============================================================
SEED = 42
rng = np.random.default_rng(SEED)

WINDOW_SIZE = 1024
DROP_INCOMPLETE_WINDOWS = True

# Para comparação direta com o paper, o principal é "test".
# Use ["train", "validation", "test"] se quiser todos.
SPLITS_TO_PROCESS = ["test"]

# Para comparação direta, use todas as janelas do test.
# Para teste rápido, use 5000.
MAX_WINDOWS_PER_SPLIT = None

# "sequential" processa todas as janelas não sobrepostas em ordem.
# "random" amostra janelas aleatórias se MAX_WINDOWS_PER_SPLIT não for None.
WINDOW_SELECTION_MODE = "sequential"

MAX_FILES_PER_SPLIT = None

# Metodologia de referência:
# raw_pdw = usa os 5 PDWs originais sem engenharia de atributos e sem scaler.
PREPROCESS_MODE = "raw_pdw_reference"

# HDBSCAN próximo ao default:
# hdbscan default:
#   min_cluster_size=5
#   min_samples=None
#   metric='euclidean'
#   cluster_selection_method='eom'
HDBSCAN_MIN_CLUSTER_SIZE = 5
HDBSCAN_MIN_SAMPLES = None
HDBSCAN_CLUSTER_SELECTION_EPSILON = 0.0
HDBSCAN_CLUSTER_SELECTION_METHOD = "eom"
HDBSCAN_ALLOW_SINGLE_CLUSTER = False

# Como tratar ruído (-1) nas métricas.
# "as_cluster": deixa todos os ruídos como cluster -1.
# "unique_noise": cada ponto de ruído vira seu próprio cluster.
# Para comparação mais direta com sklearn metrics, use "as_cluster".
NOISE_POLICY_FOR_METRICS = "as_cluster"

SAVE_EVERY_WINDOWS = 500
OVERWRITE_PREVIOUS_RESULTS = True

# Métricas da referência para Stare/HDBSCAN no test.
# Ajuste se quiser usar exatamente outra tabela do paper.
REFERENCE_METRICS = {
    "test": {
        "reference_label": "Reference paper - Stare test - HDBSCAN raw PDWs",
        "v_measure": 0.54,
        "ari": 0.27,
        "ami": 0.50,
        "homogeneity": 0.64,
        "completeness": 0.50,
        "pairwise_mcc": 0.06,
        "pairwise_f1": 0.01,
    }
}

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 180


# ============================================================
# HELPERS
# ============================================================
def save_json(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def h5_has_dataset(f, name):
    try:
        return isinstance(f[name], h5py.Dataset)
    except Exception:
        return False


def comb2(x):
    x = np.asarray(x, dtype=np.int64)
    return x * (x - 1) // 2


def safe_float(x):
    try:
        if np.isfinite(x):
            return float(x)
        return float("nan")
    except Exception:
        return float("nan")


def append_rows_to_csv(rows, path: Path):
    if len(rows) == 0:
        return

    df = pd.DataFrame(rows)
    write_header = not path.exists()

    df.to_csv(
        path,
        mode="a",
        header=write_header,
        index=False,
        encoding="utf-8-sig",
    )


def finite_median(v, default=0.0):
    v = np.asarray(v, dtype=np.float32)
    mask = np.isfinite(v)

    if mask.any():
        return float(np.nanmedian(v[mask]))

    return float(default)


# ============================================================
# FILE INDEX
# ============================================================
def collect_file_index(split_name: str, folder: Path):
    rows = []

    if not folder.exists():
        print(f"[WARNING] Missing folder: {folder}")
        return pd.DataFrame(rows)

    h5_files = sorted(folder.rglob("*.h5"))

    if MAX_FILES_PER_SPLIT is not None:
        h5_files = h5_files[:int(MAX_FILES_PER_SPLIT)]

    print(f"[{split_name}] H5 files found: {len(h5_files)}")

    for p in tqdm(h5_files, desc=f"index_{split_name}", unit="file"):
        row = {
            "source": SOURCE_NAME,
            "split": split_name,
            "file_id": p.stem,
            "file_name": p.name,
            "path": str(p),
            "status": "ok",
            "n_pulses": 0,
            "n_windows": 0,
            "error": None,
        }

        try:
            with h5py.File(p, "r") as f:
                if not h5_has_dataset(f, "data") or not h5_has_dataset(f, "labels"):
                    row["status"] = "missing_data_or_labels"
                    rows.append(row)
                    continue

                n = int(f["data"].shape[0])
                row["n_pulses"] = n

                if n <= 0:
                    row["status"] = "empty"
                    rows.append(row)
                    continue

                if DROP_INCOMPLETE_WINDOWS:
                    n_windows = n // WINDOW_SIZE
                else:
                    n_windows = int(math.ceil(n / WINDOW_SIZE))

                row["n_windows"] = int(n_windows)

        except Exception as e:
            row["status"] = "error"
            row["error"] = str(e)

        rows.append(row)

    return pd.DataFrame(rows)


def select_random_window_tasks(file_index_df: pd.DataFrame, max_windows):
    valid = file_index_df[
        (file_index_df["status"] == "ok") &
        (file_index_df["n_windows"] > 0)
    ].copy().reset_index(drop=True)

    if len(valid) == 0:
        return pd.DataFrame()

    total_windows = int(valid["n_windows"].sum())

    if max_windows is None:
        sample_n = total_windows
    else:
        sample_n = min(int(max_windows), total_windows)

    global_indices = rng.choice(total_windows, size=sample_n, replace=False)
    global_indices = np.sort(global_indices)

    n_windows = valid["n_windows"].values.astype(np.int64)
    cum_end = np.cumsum(n_windows)
    cum_start = cum_end - n_windows

    file_pos = np.searchsorted(cum_end, global_indices, side="right")
    local_win = global_indices - cum_start[file_pos]

    tasks = valid.iloc[file_pos].copy()
    tasks["window_index"] = local_win.astype(int)
    tasks = tasks.sort_values(["path", "window_index"]).reset_index(drop=True)

    return tasks


# ============================================================
# REFERENCE PREPROCESSING
# ============================================================
def sanitize_raw_pdws_minimal(x):
    """
    Minimal sanitation required because HDBSCAN does not accept NaN/inf.

    Important:
      This does NOT standardize, scale, engineer, or transform the PDWs.
      It only makes the array finite.

    Handling:
      - Amplitude non-finite -> -200 dB sentinel.
      - Other non-finite values -> column median.
    """
    x = np.asarray(x, dtype=np.float32).copy()

    if x.ndim != 2 or x.shape[1] < 5:
        raise ValueError(f"Expected [n_pulses, >=5], got {x.shape}")

    x = x[:, :5]

    # Amplitude: archive/stare may contain non-finite values.
    amp = x[:, 4].copy()
    amp[~np.isfinite(amp)] = -200.0
    x[:, 4] = amp

    # Other columns: replace non-finite by median.
    for j in range(5):
        col = x[:, j].copy()

        if not np.isfinite(col).all():
            med = finite_median(col, default=0.0)
            col[~np.isfinite(col)] = med
            x[:, j] = col

    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    return x.astype(np.float32)


def preprocess_window_reference(x):
    if PREPROCESS_MODE != "raw_pdw_reference":
        raise ValueError("This script is intended for raw_pdw_reference only.")

    return sanitize_raw_pdws_minimal(x)


# ============================================================
# CLUSTERING
# ============================================================
def run_hdbscan(X):
    X = np.asarray(X, dtype=np.float32)

    if HDBSCAN_BACKEND == "hdbscan" and hdbscan is not None:
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
            min_samples=HDBSCAN_MIN_SAMPLES,
            metric="euclidean",
            cluster_selection_epsilon=HDBSCAN_CLUSTER_SELECTION_EPSILON,
            cluster_selection_method=HDBSCAN_CLUSTER_SELECTION_METHOD,
            allow_single_cluster=HDBSCAN_ALLOW_SINGLE_CLUSTER,
        )
        return clusterer.fit_predict(X).astype(int)

    if SKHDBSCAN is not None:
        kwargs = dict(
            min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
            metric="euclidean",
            cluster_selection_epsilon=HDBSCAN_CLUSTER_SELECTION_EPSILON,
            allow_single_cluster=HDBSCAN_ALLOW_SINGLE_CLUSTER,
        )

        if HDBSCAN_MIN_SAMPLES is not None:
            kwargs["min_samples"] = HDBSCAN_MIN_SAMPLES

        clusterer = SKHDBSCAN(**kwargs)
        return clusterer.fit_predict(X).astype(int)

    raise RuntimeError("No HDBSCAN backend available. Try: pip install hdbscan")


# ============================================================
# METRICS
# ============================================================
def apply_noise_policy(y_pred):
    y_pred = np.asarray(y_pred).reshape(-1).astype(int).copy()

    if NOISE_POLICY_FOR_METRICS == "as_cluster":
        return y_pred

    if NOISE_POLICY_FOR_METRICS == "unique_noise":
        noise_idx = np.where(y_pred == -1)[0]

        if len(noise_idx) == 0:
            return y_pred

        max_cluster = int(np.max(y_pred)) if len(y_pred) > 0 else 0
        next_label = max_cluster + 1

        for k, idx in enumerate(noise_idx):
            y_pred[idx] = next_label + k

        return y_pred

    raise ValueError(f"Unknown NOISE_POLICY_FOR_METRICS: {NOISE_POLICY_FOR_METRICS}")


def efficient_pairwise_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    n = len(y_true)
    total_pairs = n * (n - 1) // 2

    if n < 2 or total_pairs <= 0:
        return {
            "pairwise_f1": np.nan,
            "pairwise_mcc": np.nan,
            "pairwise_tp": 0,
            "pairwise_fp": 0,
            "pairwise_fn": 0,
            "pairwise_tn": 0,
        }

    true_vals, true_inv = np.unique(y_true, return_inverse=True)
    pred_vals, pred_inv = np.unique(y_pred, return_inverse=True)

    contingency = np.zeros((len(true_vals), len(pred_vals)), dtype=np.int64)
    np.add.at(contingency, (true_inv, pred_inv), 1)

    tp = int(comb2(contingency).sum())

    true_sizes = contingency.sum(axis=1)
    pred_sizes = contingency.sum(axis=0)

    true_pairs = int(comb2(true_sizes).sum())
    pred_pairs = int(comb2(pred_sizes).sum())

    fp = int(pred_pairs - tp)
    fn = int(true_pairs - tp)
    tn = int(total_pairs - tp - fp - fn)

    denom_f1 = 2 * tp + fp + fn
    pairwise_f1 = float((2 * tp) / denom_f1) if denom_f1 > 0 else 0.0

    denom_mcc = math.sqrt(
        max(tp + fp, 0)
        * max(tp + fn, 0)
        * max(tn + fp, 0)
        * max(tn + fn, 0)
    )

    if denom_mcc > 0:
        pairwise_mcc = float((tp * tn - fp * fn) / denom_mcc)
    else:
        pairwise_mcc = 0.0

    return {
        "pairwise_f1": pairwise_f1,
        "pairwise_mcc": pairwise_mcc,
        "pairwise_tp": tp,
        "pairwise_fp": fp,
        "pairwise_fn": fn,
        "pairwise_tn": tn,
    }


def dominant_source_share(y_true):
    y_true = np.asarray(y_true).reshape(-1)

    if len(y_true) == 0:
        return np.nan

    _, counts = np.unique(y_true, return_counts=True)
    return float(counts.max() / counts.sum())


def source_entropy_norm(y_true):
    y_true = np.asarray(y_true).reshape(-1)

    if len(y_true) == 0:
        return np.nan

    _, counts = np.unique(y_true, return_counts=True)

    counts = counts.astype(np.float64)
    p = counts / counts.sum()
    p = p[p > 0]

    if len(p) <= 1:
        return 0.0

    return float(-np.sum(p * np.log(p)) / np.log(len(p)))


def deinterleaving_metrics(y_true, y_pred_raw):
    y_true = np.asarray(y_true).reshape(-1).astype(int)
    y_pred_raw = np.asarray(y_pred_raw).reshape(-1).astype(int)

    y_pred_eval = apply_noise_policy(y_pred_raw)

    out = {
        "v_measure": safe_float(v_measure_score(y_true, y_pred_eval)),
        "ari": safe_float(adjusted_rand_score(y_true, y_pred_eval)),
        "ami": safe_float(adjusted_mutual_info_score(y_true, y_pred_eval)),
        "homogeneity": safe_float(homogeneity_score(y_true, y_pred_eval)),
        "completeness": safe_float(completeness_score(y_true, y_pred_eval)),
    }

    out.update(efficient_pairwise_metrics(y_true, y_pred_eval))

    true_sources = np.unique(y_true)
    pred_clusters_raw = np.unique(y_pred_raw)
    pred_clusters_eval = np.unique(y_pred_eval)

    out["n_pulses"] = int(len(y_true))
    out["n_true_sources"] = int(len(true_sources))

    out["n_pred_clusters_raw_including_noise"] = int(len(pred_clusters_raw))
    out["n_pred_clusters_raw_excluding_noise"] = int(len([c for c in pred_clusters_raw if c != -1]))

    out["n_pred_clusters_eval"] = int(len(pred_clusters_eval))
    out["has_noise_cluster"] = int(np.any(y_pred_raw == -1))
    out["noise_share"] = float(np.mean(y_pred_raw == -1))

    return out


# ============================================================
# WINDOW PROCESSING
# ============================================================
def process_window(
    x_window,
    y_window,
    split_name,
    file_id,
    file_name,
    window_index,
    start_pulse,
    end_pulse,
):
    t0 = time.time()

    x_window = np.asarray(x_window, dtype=np.float32)
    y_window = np.asarray(y_window).reshape(-1).astype(int)

    X = preprocess_window_reference(x_window)
    y_pred = run_hdbscan(X)

    metrics = deinterleaving_metrics(y_window, y_pred)

    row = {
        "source": SOURCE_NAME,
        "split": split_name,
        "file_id": file_id,
        "file_name": file_name,
        "window_index": int(window_index),
        "start_pulse": int(start_pulse),
        "end_pulse": int(end_pulse),
        "window_size": int(end_pulse - start_pulse),
        "status": "ok",
        "preprocess_mode": PREPROCESS_MODE,
        "cluster_method": "HDBSCAN",
        "hdbscan_backend": HDBSCAN_BACKEND,
        "hdbscan_min_cluster_size": HDBSCAN_MIN_CLUSTER_SIZE,
        "hdbscan_min_samples": HDBSCAN_MIN_SAMPLES if HDBSCAN_MIN_SAMPLES is not None else "default",
        "hdbscan_epsilon": HDBSCAN_CLUSTER_SELECTION_EPSILON,
        "noise_policy_for_metrics": NOISE_POLICY_FOR_METRICS,
        "dominant_source_share": dominant_source_share(y_window),
        "source_entropy_norm": source_entropy_norm(y_window),
        "elapsed_sec": float(time.time() - t0),
    }

    row.update(metrics)

    return row


def process_file_selected_windows(path: Path, split_name: str, window_indices):
    rows = []
    file_id = path.stem

    try:
        with h5py.File(path, "r") as f:
            if not h5_has_dataset(f, "data") or not h5_has_dataset(f, "labels"):
                return rows

            data = f["data"]
            labels = f["labels"]
            n = int(data.shape[0])

            for w in window_indices:
                w = int(w)
                start = w * WINDOW_SIZE
                end = min((w + 1) * WINDOW_SIZE, n)

                if DROP_INCOMPLETE_WINDOWS and (end - start) < WINDOW_SIZE:
                    continue

                x_win = np.asarray(data[start:end, :5], dtype=np.float32)
                y_win = np.asarray(labels[start:end]).reshape(-1).astype(int)

                row = process_window(
                    x_window=x_win,
                    y_window=y_win,
                    split_name=split_name,
                    file_id=file_id,
                    file_name=path.name,
                    window_index=w,
                    start_pulse=start,
                    end_pulse=end,
                )

                rows.append(row)

    except Exception as e:
        rows.append({
            "source": SOURCE_NAME,
            "split": split_name,
            "file_id": path.stem,
            "file_name": path.name,
            "path": str(path),
            "status": "error",
            "cluster_method": "HDBSCAN",
            "preprocess_mode": PREPROCESS_MODE,
            "error": str(e),
        })

    return rows


def process_split_random(split_name: str, file_index_df: pd.DataFrame, results_path: Path):
    tasks = select_random_window_tasks(file_index_df, MAX_WINDOWS_PER_SPLIT)

    print(f"[{split_name}] Random selected windows: {len(tasks)}")

    if len(tasks) == 0:
        return 0

    buffer_rows = []
    done = 0

    grouped = tasks.groupby("path", sort=False)
    pbar = tqdm(grouped, total=tasks["path"].nunique(), desc=f"{split_name}", unit="file")

    for path_str, g in pbar:
        rows = process_file_selected_windows(
            path=Path(path_str),
            split_name=split_name,
            window_indices=g["window_index"].values,
        )

        buffer_rows.extend(rows)

        n_ok = sum(1 for r in rows if r.get("status") == "ok")
        done += n_ok

        pbar.set_postfix({"windows": done})

        if len(buffer_rows) >= SAVE_EVERY_WINDOWS:
            append_rows_to_csv(buffer_rows, results_path)
            buffer_rows = []

    if len(buffer_rows) > 0:
        append_rows_to_csv(buffer_rows, results_path)

    return done


def process_split_sequential(split_name: str, file_index_df: pd.DataFrame, results_path: Path):
    valid = file_index_df[
        (file_index_df["status"] == "ok") &
        (file_index_df["n_windows"] > 0)
    ].copy().reset_index(drop=True)

    buffer_rows = []
    done = 0

    pbar = tqdm(valid.itertuples(index=False), total=len(valid), desc=f"{split_name}", unit="file")

    for row in pbar:
        if MAX_WINDOWS_PER_SPLIT is not None:
            remaining = int(MAX_WINDOWS_PER_SPLIT) - done
            if remaining <= 0:
                break
        else:
            remaining = int(row.n_windows)

        n_to_process = min(int(row.n_windows), remaining)
        window_indices = list(range(n_to_process))

        rows = process_file_selected_windows(
            path=Path(row.path),
            split_name=split_name,
            window_indices=window_indices,
        )

        buffer_rows.extend(rows)

        n_ok = sum(1 for r in rows if r.get("status") == "ok")
        done += n_ok

        pbar.set_postfix({"windows": done})

        if len(buffer_rows) >= SAVE_EVERY_WINDOWS:
            append_rows_to_csv(buffer_rows, results_path)
            buffer_rows = []

    if len(buffer_rows) > 0:
        append_rows_to_csv(buffer_rows, results_path)

    return done


# ============================================================
# SUMMARIES
# ============================================================
def summarize_results(results_df):
    metric_cols = [
        "v_measure",
        "ari",
        "ami",
        "homogeneity",
        "completeness",
        "pairwise_f1",
        "pairwise_mcc",
        "n_true_sources",
        "n_pred_clusters_raw_excluding_noise",
        "n_pred_clusters_eval",
        "noise_share",
        "dominant_source_share",
        "source_entropy_norm",
        "elapsed_sec",
    ]

    group_cols = [
        "source",
        "split",
        "cluster_method",
        "preprocess_mode",
        "noise_policy_for_metrics",
    ]

    rows = []

    for keys, g in results_df.groupby(group_cols, dropna=False):
        row = dict(zip(group_cols, keys))
        row["n_windows"] = int(len(g))

        for c in metric_cols:
            if c not in g.columns:
                continue

            vals = (
                pd.to_numeric(g[c], errors="coerce")
                .replace([np.inf, -np.inf], np.nan)
                .dropna()
            )

            if len(vals) == 0:
                row[f"{c}_mean"] = np.nan
                row[f"{c}_median"] = np.nan
                row[f"{c}_std"] = np.nan
            else:
                row[f"{c}_mean"] = float(vals.mean())
                row[f"{c}_median"] = float(vals.median())
                row[f"{c}_std"] = float(vals.std())

        rows.append(row)

    return pd.DataFrame(rows)


def build_reference_comparison(summary_df):
    rows = []

    metric_names = [
        "v_measure",
        "ari",
        "ami",
        "homogeneity",
        "completeness",
        "pairwise_mcc",
        "pairwise_f1",
    ]

    for _, row in summary_df.iterrows():
        split_name = row["split"]

        if split_name not in REFERENCE_METRICS:
            continue

        ref = REFERENCE_METRICS[split_name]

        out = {
            "source": row["source"],
            "split": split_name,
            "reference_label": ref["reference_label"],
            "local_method": f"{row['cluster_method']} + {row['preprocess_mode']}",
            "n_windows_local": int(row["n_windows"]),
            "noise_policy_for_metrics": row["noise_policy_for_metrics"],
        }

        for m in metric_names:
            local_col = f"{m}_median"

            if local_col in row:
                local_val = float(row[local_col])
            else:
                local_val = np.nan

            ref_val = float(ref[m]) if m in ref else np.nan

            out[f"{m}_reference"] = ref_val
            out[f"{m}_local_median"] = local_val
            out[f"{m}_delta_local_minus_reference"] = (
                local_val - ref_val if np.isfinite(local_val) and np.isfinite(ref_val) else np.nan
            )

        rows.append(out)

    return pd.DataFrame(rows)


def summarize_by_complexity(results_df):
    df = results_df.copy()

    df["n_true_sources_bin"] = pd.cut(
        df["n_true_sources"],
        bins=[0, 1, 2, 5, 10, 20, 40, 80, 200],
        labels=[
            "1",
            "2",
            "3-5",
            "6-10",
            "11-20",
            "21-40",
            "41-80",
            "81+",
        ],
        include_lowest=True,
    )

    rows = []

    for keys, g in df.groupby(["split", "n_true_sources_bin"], dropna=False):
        split_name, bin_name = keys

        out = {
            "source": SOURCE_NAME,
            "split": split_name,
            "n_true_sources_bin": str(bin_name),
            "n_windows": int(len(g)),
        }

        for c in [
            "v_measure",
            "ari",
            "ami",
            "homogeneity",
            "completeness",
            "pairwise_f1",
            "pairwise_mcc",
            "noise_share",
        ]:
            vals = pd.to_numeric(g[c], errors="coerce").dropna()
            out[f"{c}_mean"] = float(vals.mean()) if len(vals) > 0 else np.nan
            out[f"{c}_median"] = float(vals.median()) if len(vals) > 0 else np.nan

        rows.append(out)

    return pd.DataFrame(rows)


# ============================================================
# PLOTS
# ============================================================
def save_hist(series, title, xlabel, path, bins=50):
    s = (
        pd.to_numeric(series, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    if len(s) == 0:
        return

    plt.figure(figsize=(8, 5))
    plt.hist(s.values, bins=bins)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Windows")
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()


def save_scatter(df, xcol, ycol, title, path, max_points=100_000):
    g = df[[xcol, ycol, "split"]].copy()

    g[xcol] = pd.to_numeric(g[xcol], errors="coerce")
    g[ycol] = pd.to_numeric(g[ycol], errors="coerce")

    g = g.replace([np.inf, -np.inf], np.nan).dropna()

    if len(g) == 0:
        return

    if len(g) > max_points:
        g = g.sample(max_points, random_state=SEED)

    plt.figure(figsize=(8, 5))

    for split_name, gs in g.groupby("split"):
        plt.scatter(gs[xcol], gs[ycol], s=6, alpha=0.35, label=split_name)

    plt.title(title)
    plt.xlabel(xcol)
    plt.ylabel(ycol)
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()


def generate_plots(results_df):
    print("=" * 80)
    print("GENERATING PLOTS")
    print("=" * 80)

    for split_name, g in results_df.groupby("split"):
        save_hist(
            g["v_measure"],
            f"Reference-method HDBSCAN V-measure - {split_name}",
            "V-measure",
            PLOTS_DIR / f"hist_v_measure_{split_name}.png",
            bins=50,
        )

        save_hist(
            g["ari"],
            f"Reference-method HDBSCAN ARI - {split_name}",
            "ARI",
            PLOTS_DIR / f"hist_ari_{split_name}.png",
            bins=50,
        )

        save_hist(
            g["ami"],
            f"Reference-method HDBSCAN AMI - {split_name}",
            "AMI",
            PLOTS_DIR / f"hist_ami_{split_name}.png",
            bins=50,
        )

        save_hist(
            g["homogeneity"],
            f"Reference-method HDBSCAN Homogeneity - {split_name}",
            "Homogeneity",
            PLOTS_DIR / f"hist_homogeneity_{split_name}.png",
            bins=50,
        )

        save_hist(
            g["completeness"],
            f"Reference-method HDBSCAN Completeness - {split_name}",
            "Completeness",
            PLOTS_DIR / f"hist_completeness_{split_name}.png",
            bins=50,
        )

        save_hist(
            g["pairwise_mcc"],
            f"Reference-method HDBSCAN Pairwise MCC - {split_name}",
            "Pairwise MCC",
            PLOTS_DIR / f"hist_pairwise_mcc_{split_name}.png",
            bins=50,
        )

        save_hist(
            g["pairwise_f1"],
            f"Reference-method HDBSCAN Pairwise F1 - {split_name}",
            "Pairwise F1",
            PLOTS_DIR / f"hist_pairwise_f1_{split_name}.png",
            bins=50,
        )

        save_hist(
            g["n_true_sources"],
            f"True sources per window - {split_name}",
            "True sources",
            PLOTS_DIR / f"hist_n_true_sources_{split_name}.png",
            bins=60,
        )

        save_hist(
            g["n_pred_clusters_raw_excluding_noise"],
            f"Predicted clusters per window - {split_name}",
            "Predicted clusters excluding noise",
            PLOTS_DIR / f"hist_n_pred_clusters_{split_name}.png",
            bins=60,
        )

        save_hist(
            g["noise_share"],
            f"HDBSCAN noise share - {split_name}",
            "Noise share",
            PLOTS_DIR / f"hist_noise_share_{split_name}.png",
            bins=50,
        )

    save_scatter(
        results_df,
        "n_true_sources",
        "v_measure",
        "V-measure vs number of true sources",
        PLOTS_DIR / "scatter_v_measure_vs_n_true_sources.png",
    )

    save_scatter(
        results_df,
        "n_true_sources",
        "n_pred_clusters_raw_excluding_noise",
        "Predicted clusters vs true sources",
        PLOTS_DIR / "scatter_pred_clusters_vs_true_sources.png",
    )

    save_scatter(
        results_df,
        "dominant_source_share",
        "v_measure",
        "V-measure vs dominant source share",
        PLOTS_DIR / "scatter_v_measure_vs_dominant_share.png",
    )


# ============================================================
# REPORT
# ============================================================
def write_markdown_report(summary_df, comparison_df, complexity_df, file_index_df):
    report_path = OUT_DIR / "reference_methodology_hdbscan_report.md"

    with open(report_path, "w", encoding="utf-8") as f:
        f.write("# HDBSCAN Baseline Following Reference Methodology\n\n")

        f.write("## Inputs\n\n")
        for split_name in SPLITS_TO_PROCESS:
            f.write(f"- {split_name}: `{DATA_DIRS[split_name]}`\n")

        f.write("\n## Methodology\n\n")
        f.write(
            "This experiment follows the reference baseline methodology as closely as possible: "
            "raw PDWs are split into non-overlapping 1024-pulse windows and clustered with HDBSCAN. "
            "The true number of emitters is not given to the algorithm. Labels are used only after "
            "clustering to compute external metrics.\n\n"
        )

        f.write("## Configuration\n\n")
        f.write(f"- Source: `{SOURCE_NAME}`\n")
        f.write(f"- Window size: {WINDOW_SIZE}\n")
        f.write(f"- Split(s): {SPLITS_TO_PROCESS}\n")
        f.write(f"- Window selection mode: `{WINDOW_SELECTION_MODE}`\n")
        f.write(f"- Max windows per split: {MAX_WINDOWS_PER_SPLIT}\n")
        f.write(f"- Preprocessing mode: `{PREPROCESS_MODE}`\n")
        f.write(f"- HDBSCAN backend: `{HDBSCAN_BACKEND}`\n")
        f.write(f"- min_cluster_size: {HDBSCAN_MIN_CLUSTER_SIZE}\n")
        f.write(f"- min_samples: {HDBSCAN_MIN_SAMPLES if HDBSCAN_MIN_SAMPLES is not None else 'default'}\n")
        f.write(f"- cluster_selection_epsilon: {HDBSCAN_CLUSTER_SELECTION_EPSILON}\n")
        f.write(f"- noise policy for metrics: `{NOISE_POLICY_FOR_METRICS}`\n\n")

        f.write("## File index summary\n\n")
        if len(file_index_df) > 0:
            idx_summary = (
                file_index_df
                .groupby("split", as_index=False)
                .agg(
                    n_files=("path", "count"),
                    n_ok=("status", lambda x: int(np.sum(np.asarray(x) == "ok"))),
                    total_pulses=("n_pulses", "sum"),
                    total_windows=("n_windows", "sum"),
                    mean_pulses=("n_pulses", "mean"),
                    median_pulses=("n_pulses", "median"),
                )
            )
            f.write(idx_summary.to_markdown(index=False))
            f.write("\n\n")

        f.write("## Local metric summary\n\n")
        if len(summary_df) > 0:
            f.write(summary_df.to_markdown(index=False))
            f.write("\n\n")

        f.write("## Direct comparison with reference paper\n\n")
        if len(comparison_df) > 0:
            f.write(comparison_df.to_markdown(index=False))
            f.write("\n\n")

        f.write("## Complexity summary\n\n")
        if len(complexity_df) > 0:
            f.write(complexity_df.to_markdown(index=False))
            f.write("\n\n")

        f.write("## Output files\n\n")
        f.write("- `tables/window_metrics_reference_hdbscan.csv`\n")
        f.write("- `tables/file_index_reference_methodology.csv`\n")
        f.write("- `tables/summary_by_split_reference_methodology.csv`\n")
        f.write("- `tables/comparison_with_reference_paper.csv`\n")
        f.write("- `tables/summary_by_n_true_sources.csv`\n")
        f.write("- `plots/*.png`\n\n")

    return report_path


# ============================================================
# MAIN
# ============================================================
def main():
    print("=" * 80)
    print("TSRD HDBSCAN BASELINE - REFERENCE METHODOLOGY")
    print("=" * 80)

    print(f"Source                    : {SOURCE_NAME}")
    print(f"HDBSCAN backend            : {HDBSCAN_BACKEND}")
    print(f"Preprocess mode            : {PREPROCESS_MODE}")
    print(f"Window size                : {WINDOW_SIZE}")
    print(f"Splits                     : {SPLITS_TO_PROCESS}")
    print(f"Window selection mode      : {WINDOW_SELECTION_MODE}")
    print(f"Max windows per split      : {MAX_WINDOWS_PER_SPLIT}")
    print(f"Noise policy for metrics   : {NOISE_POLICY_FOR_METRICS}")
    print(f"HDBSCAN min_cluster_size   : {HDBSCAN_MIN_CLUSTER_SIZE}")
    print(f"HDBSCAN min_samples        : {HDBSCAN_MIN_SAMPLES if HDBSCAN_MIN_SAMPLES is not None else 'default'}")

    results_path = TABLES_DIR / "window_metrics_reference_hdbscan.csv"
    file_index_path = TABLES_DIR / "file_index_reference_methodology.csv"

    if OVERWRITE_PREVIOUS_RESULTS and results_path.exists():
        results_path.unlink()

    all_index = []
    split_done = {}

    for split_name in SPLITS_TO_PROCESS:
        folder = DATA_DIRS[split_name]

        print("\n" + "=" * 80)
        print(f"INDEXING SPLIT: {split_name}")
        print(f"FOLDER: {folder}")
        print("=" * 80)

        idx_df = collect_file_index(split_name, folder)
        all_index.append(idx_df)

        if len(idx_df) == 0:
            print(f"[WARNING] No files indexed for split: {split_name}")
            split_done[split_name] = 0
            continue

        print("\n" + "=" * 80)
        print(f"PROCESSING SPLIT: {split_name}")
        print("=" * 80)

        if WINDOW_SELECTION_MODE == "random" and MAX_WINDOWS_PER_SPLIT is not None:
            n_done = process_split_random(split_name, idx_df, results_path)
        else:
            n_done = process_split_sequential(split_name, idx_df, results_path)

        split_done[split_name] = int(n_done)
        print(f"[DONE] {split_name}: {n_done} windows processed")

    if len(all_index) > 0:
        file_index_df = pd.concat(all_index, axis=0).reset_index(drop=True)
    else:
        file_index_df = pd.DataFrame()

    file_index_df.to_csv(file_index_path, index=False, encoding="utf-8-sig")

    if not results_path.exists():
        print("[ERROR] No HDBSCAN results were generated.")
        return

    results_df = pd.read_csv(results_path)

    results_df = results_df[
        (results_df["status"] == "ok") &
        (results_df["cluster_method"] == "HDBSCAN")
    ].copy()

    results_df = results_df.dropna(subset=["v_measure"])

    if len(results_df) == 0:
        print("[ERROR] No valid metric rows found.")
        return

    # --------------------------------------------------------
    # Summaries
    # --------------------------------------------------------
    summary_df = summarize_results(results_df)
    complexity_df = summarize_by_complexity(results_df)
    comparison_df = build_reference_comparison(summary_df)

    summary_path = TABLES_DIR / "summary_by_split_reference_methodology.csv"
    complexity_path = TABLES_DIR / "summary_by_n_true_sources.csv"
    comparison_path = TABLES_DIR / "comparison_with_reference_paper.csv"

    summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
    complexity_df.to_csv(complexity_path, index=False, encoding="utf-8-sig")
    comparison_df.to_csv(comparison_path, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # Plots and report
    # --------------------------------------------------------
    generate_plots(results_df)

    report_path = write_markdown_report(
        summary_df=summary_df,
        comparison_df=comparison_df,
        complexity_df=complexity_df,
        file_index_df=file_index_df,
    )

    # --------------------------------------------------------
    # Config
    # --------------------------------------------------------
    config = {
        "root": str(ROOT),
        "data_dirs": {k: str(v) for k, v in DATA_DIRS.items()},
        "source_name": SOURCE_NAME,
        "splits_to_process": SPLITS_TO_PROCESS,
        "window_size": WINDOW_SIZE,
        "drop_incomplete_windows": DROP_INCOMPLETE_WINDOWS,
        "max_files_per_split": MAX_FILES_PER_SPLIT,
        "max_windows_per_split": MAX_WINDOWS_PER_SPLIT,
        "window_selection_mode": WINDOW_SELECTION_MODE,
        "preprocess_mode": PREPROCESS_MODE,
        "minimal_sanitation": {
            "amplitude_nonfinite": "-200.0 dB",
            "other_nonfinite": "column median",
        },
        "hdbscan_backend": HDBSCAN_BACKEND,
        "hdbscan_min_cluster_size": HDBSCAN_MIN_CLUSTER_SIZE,
        "hdbscan_min_samples": HDBSCAN_MIN_SAMPLES if HDBSCAN_MIN_SAMPLES is not None else "default",
        "hdbscan_cluster_selection_epsilon": HDBSCAN_CLUSTER_SELECTION_EPSILON,
        "hdbscan_cluster_selection_method": HDBSCAN_CLUSTER_SELECTION_METHOD,
        "hdbscan_allow_single_cluster": HDBSCAN_ALLOW_SINGLE_CLUSTER,
        "noise_policy_for_metrics": NOISE_POLICY_FOR_METRICS,
        "reference_metrics": REFERENCE_METRICS,
        "split_windows_processed": split_done,
        "outputs": {
            "window_metrics": str(results_path),
            "file_index": str(file_index_path),
            "summary": str(summary_path),
            "complexity_summary": str(complexity_path),
            "comparison_with_reference": str(comparison_path),
            "plots": str(PLOTS_DIR),
            "report": str(report_path),
        },
    }

    config_path = OUT_DIR / "config.json"
    save_json(config, config_path)

    # --------------------------------------------------------
    # Console summary
    # --------------------------------------------------------
    print("\n" + "=" * 80)
    print("LOCAL SUMMARY")
    print("=" * 80)

    cols_to_show = [
        "source",
        "split",
        "cluster_method",
        "preprocess_mode",
        "noise_policy_for_metrics",
        "n_windows",
        "v_measure_median",
        "ari_median",
        "ami_median",
        "homogeneity_median",
        "completeness_median",
        "pairwise_f1_median",
        "pairwise_mcc_median",
        "n_true_sources_median",
        "n_pred_clusters_raw_excluding_noise_median",
        "noise_share_median",
    ]

    show_cols = [c for c in cols_to_show if c in summary_df.columns]
    print(summary_df[show_cols].to_string(index=False))

    print("\n" + "=" * 80)
    print("DIRECT COMPARISON WITH REFERENCE")
    print("=" * 80)

    if len(comparison_df) > 0:
        compare_cols = [
            "split",
            "reference_label",
            "n_windows_local",
            "v_measure_reference",
            "v_measure_local_median",
            "v_measure_delta_local_minus_reference",
            "ari_reference",
            "ari_local_median",
            "ari_delta_local_minus_reference",
            "ami_reference",
            "ami_local_median",
            "ami_delta_local_minus_reference",
            "homogeneity_reference",
            "homogeneity_local_median",
            "completeness_reference",
            "completeness_local_median",
            "pairwise_mcc_reference",
            "pairwise_mcc_local_median",
            "pairwise_f1_reference",
            "pairwise_f1_local_median",
        ]

        compare_cols = [c for c in compare_cols if c in comparison_df.columns]
        print(comparison_df[compare_cols].to_string(index=False))
    else:
        print("No comparison rows generated. Check REFERENCE_METRICS and processed splits.")

    print("\n" + "=" * 80)
    print("OUTPUT FILES")
    print("=" * 80)
    print(f"Window metrics      : {results_path}")
    print(f"File index          : {file_index_path}")
    print(f"Summary             : {summary_path}")
    print(f"Complexity summary  : {complexity_path}")
    print(f"Reference comparison: {comparison_path}")
    print(f"Plots directory     : {PLOTS_DIR}")
    print(f"Markdown report     : {report_path}")
    print(f"Config JSON         : {config_path}")

    print("\nDone.")


if __name__ == "__main__":
    main()

TSRD HDBSCAN BASELINE - REFERENCE METHODOLOGY
Source                    : archive
HDBSCAN backend            : hdbscan
Preprocess mode            : raw_pdw_reference
Window size                : 1024
Splits                     : ['test']
Window selection mode      : sequential
Max windows per split      : None
Noise policy for metrics   : as_cluster
HDBSCAN min_cluster_size   : 5
HDBSCAN min_samples        : default

INDEXING SPLIT: test
FOLDER: D:\Fusion\turing_synthetic_radar_dataset\archive\test
[test] H5 files found: 250


index_test: 100%|██████████| 250/250 [00:04<00:00, 59.01file/s]



PROCESSING SPLIT: test


test: 100%|██████████| 237/237 [23:19<00:00,  5.90s/file, windows=36479]


[DONE] test: 36479 windows processed
GENERATING PLOTS

LOCAL SUMMARY
 source split cluster_method   preprocess_mode noise_policy_for_metrics  n_windows  v_measure_median  ari_median  ami_median  homogeneity_median  completeness_median  pairwise_f1_median  pairwise_mcc_median  n_true_sources_median  n_pred_clusters_raw_excluding_noise_median  noise_share_median
archive  test        HDBSCAN raw_pdw_reference               as_cluster      36479          0.522578    0.280134     0.49381            0.799444             0.416608            0.718088             0.383768                   12.0                                         6.0            0.007812

DIRECT COMPARISON WITH REFERENCE
split                                 reference_label  n_windows_local  v_measure_reference  v_measure_local_median  v_measure_delta_local_minus_reference  ari_reference  ari_local_median  ari_delta_local_minus_reference  ami_reference  ami_local_median  ami_delta_local_minus_reference  homogeneity_reference

# Reprodução do baseline HDBSCAN do trabalho de referência

A reprodução foi conduzida com o objetivo de verificar se o pipeline local é fiel à metodologia de referência do **Turing Synthetic Radar Dataset (TSRD)**. O trabalho define a tarefa como **radar pulse deinterleaving**, em que uma sequência de PDWs deve ser particionada em grupos correspondentes aos emissores de origem. Cada pulso é representado por cinco atributos: **ToA**, **frequência**, **largura de pulso**, **AoA** e **amplitude**. 

A metodologia reproduzida segue o baseline descrito no paper: aplicação do **HDBSCAN diretamente sobre os PDWs brutos não transformados**, usando **janelas não sobrepostas de 1024 pulsos** no conjunto de teste `stare`. O artigo reporta os resultados desse baseline na Tabela IV. 

## Configuração da reprodução

| Item                        | Configuração usada                                         |
| --------------------------- | ---------------------------------------------------------- |
| Dataset                     | TSRD                                                       |
| Modo avaliado               | `archive/test`, correspondente ao modo `stare/test`        |
| Tarefa                      | Deinterleaving de pulsos por clustering                    |
| Entrada do algoritmo        | PDWs brutos `[ToA, Frequency, PulseWidth, AoA, Amplitude]` |
| Tamanho da janela           | 1024 pulsos                                                |
| Sobreposição                | Nenhuma                                                    |
| Algoritmo                   | HDBSCAN                                                    |
| Número real de emissores    | Não utilizado pelo algoritmo                               |
| Uso dos labels              | Apenas para avaliação                                      |
| Número de janelas avaliadas | 36.479                                                     |
| Política de ruído           | `-1` mantido como cluster de ruído                         |

## Comparação com o trabalho de referência

| Métrica      | Referência — HDBSCAN raw PDWs / Stare test | Reprodução local — HDBSCAN raw PDWs / `archive/test` | Diferença |
| ------------ | -----------------------------------------: | ---------------------------------------------------: | --------: |
| V-measure    |                                      0.540 |                                                0.523 |    -0.017 |
| ARI          |                                      0.270 |                                                0.280 |    +0.010 |
| AMI          |                                      0.500 |                                                0.494 |    -0.006 |
| Homogeneity  |                                      0.640 |                                                0.799 |    +0.159 |
| Completeness |                                      0.500 |                                                0.417 |    -0.083 |
| MCC          |                                      0.060 |                                                0.384 |    +0.324 |
| F1           |                                      0.010 |                                                0.718 |    +0.708 |

## Análise da fidelidade da reprodução

A reprodução pode ser considerada **metodologicamente fiel** ao baseline do trabalho de referência, pois emprega o mesmo tipo de entrada, o mesmo algoritmo e a mesma segmentação em janelas de 1024 pulsos. Além disso, as principais métricas externas de clustering ficaram muito próximas às reportadas no paper:

| Métrica principal | Diferença absoluta |
| ----------------- | -----------------: |
| V-measure         |              0.017 |
| ARI               |              0.010 |
| AMI               |              0.006 |

Essas diferenças são pequenas e indicam que o pipeline local reproduz adequadamente o comportamento geral do baseline. Em particular, a **V-measure local de 0.523** está muito próxima da **V-measure de referência de 0.540**, diferença de apenas **0.017 ponto absoluto**.

## Interpretação dos desvios observados

A reprodução local apresentou **homogeneity maior** e **completeness menor** do que a referência. Isso indica que os clusters produzidos localmente tendem a ser mais puros, mas menos completos: ou seja, cada cluster contém majoritariamente pulsos de um mesmo emissor, mas alguns emissores reais podem ter sido fragmentados em múltiplos clusters.

As diferenças em **MCC** e **F1** são substancialmente maiores. Isso sugere que a implementação local das métricas pairwise não replica exatamente a rotina oficial do desafio, especialmente porque o paper descreve uma agregação pairwise específica baseada em pior caso médio entre rótulos verdadeiros e previstos. Portanto, para atestar fidelidade de reprodução, as métricas mais comparáveis são **V-measure, ARI, AMI, homogeneity e completeness**. 

## Conclusão

A reprodução é fiel ao trabalho de referência nas métricas principais de clustering. A diferença reduzida em **V-measure**, **ARI** e **AMI** demonstra que o pipeline implementado reproduz adequadamente o baseline HDBSCAN sobre PDWs brutos no conjunto `stare/test`.

Em síntese:

| Critério                     | Avaliação |
| ---------------------------- | --------- |
| Mesma tarefa                 | Sim       |
| Mesmo tipo de entrada        | Sim       |
| Mesma janela de avaliação    | Sim       |
| Mesmo algoritmo              | Sim       |
| Métricas principais próximas | Sim       |
| Reprodução considerada fiel  | Sim       |

Assim, o experimento local pode ser usado como **baseline validado** para comparação com métodos posteriores, como modelos supervisionados, embeddings aprendidos ou arquiteturas Mixture-of-Experts para deinterleaving de pulsos radar.


# REPRODUZIR USANDO ENGENHARIA DE ATRIBUTOS E ANALISAR DIFERENÇAS 

The Turing Synthetic Radar Dataset: A dataset for pulse deinterleaving.

In [10]:
# ============================================================
# baseline_hdbscan_raw_vs_engineered.py
# ============================================================
# TSRD - HDBSCAN baseline with direct comparison:
#   1) raw PDWs, following reference methodology
#   2) engineered PDW features
#
# Objective:
#   Compare directly:
#       Reference paper HDBSCAN raw PDWs
#       Local reproduction with raw PDWs
#       Local HDBSCAN with engineered PDW features
#
# Inputs:
#   D:\Fusion\turing_synthetic_radar_dataset\archive\train
#   D:\Fusion\turing_synthetic_radar_dataset\archive\validation
#   D:\Fusion\turing_synthetic_radar_dataset\archive\test
#
# Main comparison:
#   archive/test -> reference paper Stare test baseline
#
# Each .h5:
#   data   : [n_pulses, 5]
#   labels : [n_pulses, 1]
#
# Raw PDW columns:
#   0 -> ToA_us
#   1 -> Frequency_MHz
#   2 -> PulseWidth_us
#   3 -> AoA_deg
#   4 -> Amplitude_dB
#
# Method:
#   1. Read PDWs.
#   2. Split into non-overlapping windows of 1024 pulses.
#   3. For each window, run:
#        a) HDBSCAN on raw PDWs
#        b) HDBSCAN on engineered PDW features
#   4. Evaluate predicted clusters against labels.
#   5. Compare metrics.
#
# Outputs:
#   D:\Fusion\turing_synthetic_radar_dataset\hdbscan_raw_vs_engineered
# ============================================================

from pathlib import Path
import sys
import json
import math
import time
import re
import subprocess
import warnings
warnings.filterwarnings("ignore")


# ============================================================
# INSTALL / IMPORTS
# ============================================================
def ensure_package(import_name, pip_name=None):
    try:
        __import__(import_name)
    except Exception:
        pkg = pip_name if pip_name is not None else import_name
        print(f"[INFO] Installing missing package: {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


ensure_package("numpy")
ensure_package("pandas")
ensure_package("matplotlib")
ensure_package("h5py")
ensure_package("sklearn", "scikit-learn")
ensure_package("tqdm")
ensure_package("tabulate")

try:
    ensure_package("hdbscan")
    import hdbscan
    HDBSCAN_BACKEND = "hdbscan"
except Exception:
    hdbscan = None
    HDBSCAN_BACKEND = "sklearn"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py

from tqdm import tqdm

from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    v_measure_score,
    adjusted_rand_score,
    adjusted_mutual_info_score,
    homogeneity_score,
    completeness_score,
)

try:
    from sklearn.cluster import HDBSCAN as SKHDBSCAN
except Exception:
    SKHDBSCAN = None


# ============================================================
# PATHS
# ============================================================
ROOT = Path(r"D:\Fusion\turing_synthetic_radar_dataset")

DATA_DIRS = {
    "train": ROOT / "archive" / "train",
    "validation": ROOT / "archive" / "validation",
    "test": ROOT / "archive" / "test",
}

SOURCE_NAME = "archive"

OUT_DIR = ROOT / "hdbscan_raw_vs_engineered"
TABLES_DIR = OUT_DIR / "tables"
PLOTS_DIR = OUT_DIR / "plots"

OUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# CONFIG
# ============================================================
SEED = 42
rng = np.random.default_rng(SEED)

WINDOW_SIZE = 1024
DROP_INCOMPLETE_WINDOWS = True

# Para comparação direta com o paper, use ["test"].
# Para análise exploratória adicional, use ["train", "validation", "test"].
SPLITS_TO_PROCESS = ["test"]

# Para comparação final, use None.
# Para teste rápido, use 5000.
MAX_WINDOWS_PER_SPLIT = None

# "sequential": processa todas as janelas em ordem.
# "random": amostra janelas aleatórias se MAX_WINDOWS_PER_SPLIT não for None.
WINDOW_SELECTION_MODE = "sequential"

MAX_FILES_PER_SPLIT = None

# Modos a comparar.
PREPROCESS_MODES = [
    "raw_pdw_reference",
    "engineered_pdw",
]

# HDBSCAN próximo ao default da referência.
HDBSCAN_MIN_CLUSTER_SIZE = 5
HDBSCAN_MIN_SAMPLES = None
HDBSCAN_CLUSTER_SELECTION_EPSILON = 0.0
HDBSCAN_CLUSTER_SELECTION_METHOD = "eom"
HDBSCAN_ALLOW_SINGLE_CLUSTER = False

# Como tratar ruído (-1) nas métricas.
# "as_cluster": deixa -1 como cluster de ruído.
# "unique_noise": cada ponto de ruído vira cluster único.
NOISE_POLICY_FOR_METRICS = "as_cluster"

SAVE_EVERY_WINDOWS = 250
OVERWRITE_PREVIOUS_RESULTS = True

# Métricas da referência para Stare/HDBSCAN no test.
REFERENCE_METRICS = {
    "test": {
        "reference_label": "Reference paper - Stare test - HDBSCAN raw PDWs",
        "v_measure": 0.54,
        "ari": 0.27,
        "ami": 0.50,
        "homogeneity": 0.64,
        "completeness": 0.50,
        "pairwise_mcc": 0.06,
        "pairwise_f1": 0.01,
    }
}

RAW_FEATURE_NAMES = [
    "ToA_us",
    "Frequency_MHz",
    "PulseWidth_us",
    "AoA_deg",
    "Amplitude_dB",
]

ENGINEERED_FEATURE_NAMES = [
    "toa_rel_norm",
    "delta_toa_log",
    "frequency_norm",
    "delta_frequency_abs_norm",
    "pulsewidth_log",
    "aoa_sin",
    "aoa_cos",
    "delta_aoa_abs_norm",
    "amplitude_norm",
    "amplitude_missing",
]

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 180


# ============================================================
# HELPERS
# ============================================================
def safe_filename(s: str) -> str:
    s = str(s)
    s = re.sub(r"[^\w\-.]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s[:180] if len(s) > 180 else s


def save_json(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def h5_has_dataset(f, name):
    try:
        return isinstance(f[name], h5py.Dataset)
    except Exception:
        return False


def comb2(x):
    x = np.asarray(x, dtype=np.int64)
    return x * (x - 1) // 2


def safe_float(x):
    try:
        if np.isfinite(x):
            return float(x)
        return float("nan")
    except Exception:
        return float("nan")


def append_rows_to_csv(rows, path: Path):
    if len(rows) == 0:
        return

    df = pd.DataFrame(rows)
    write_header = not path.exists()

    df.to_csv(
        path,
        mode="a",
        header=write_header,
        index=False,
        encoding="utf-8-sig",
    )


def finite_median(v, default=0.0):
    v = np.asarray(v, dtype=np.float32)
    mask = np.isfinite(v)

    if mask.any():
        return float(np.nanmedian(v[mask]))

    return float(default)


def fill_nonfinite_with_median(v, default=0.0):
    v = np.asarray(v, dtype=np.float32).copy()
    med = finite_median(v, default=default)
    v[~np.isfinite(v)] = med
    return v


def angular_abs_diff_deg(a):
    """
    Absolute circular difference between consecutive AoA samples.
    Output in [0, 180].
    """
    a = np.asarray(a, dtype=np.float32)
    prev = np.concatenate([[a[0]], a[:-1]])
    diff = (a - prev + 180.0) % 360.0 - 180.0
    return np.abs(diff).astype(np.float32)


# ============================================================
# FILE INDEX
# ============================================================
def collect_file_index(split_name: str, folder: Path):
    rows = []

    if not folder.exists():
        print(f"[WARNING] Missing folder: {folder}")
        return pd.DataFrame(rows)

    h5_files = sorted(folder.rglob("*.h5"))

    if MAX_FILES_PER_SPLIT is not None:
        h5_files = h5_files[:int(MAX_FILES_PER_SPLIT)]

    print(f"[{split_name}] H5 files found: {len(h5_files)}")

    for p in tqdm(h5_files, desc=f"index_{split_name}", unit="file"):
        row = {
            "source": SOURCE_NAME,
            "split": split_name,
            "file_id": p.stem,
            "file_name": p.name,
            "path": str(p),
            "status": "ok",
            "n_pulses": 0,
            "n_windows": 0,
            "error": None,
        }

        try:
            with h5py.File(p, "r") as f:
                if not h5_has_dataset(f, "data") or not h5_has_dataset(f, "labels"):
                    row["status"] = "missing_data_or_labels"
                    rows.append(row)
                    continue

                n = int(f["data"].shape[0])
                row["n_pulses"] = n

                if n <= 0:
                    row["status"] = "empty"
                    rows.append(row)
                    continue

                if DROP_INCOMPLETE_WINDOWS:
                    n_windows = n // WINDOW_SIZE
                else:
                    n_windows = int(math.ceil(n / WINDOW_SIZE))

                row["n_windows"] = int(n_windows)

        except Exception as e:
            row["status"] = "error"
            row["error"] = str(e)

        rows.append(row)

    return pd.DataFrame(rows)


def select_random_window_tasks(file_index_df: pd.DataFrame, max_windows):
    valid = file_index_df[
        (file_index_df["status"] == "ok") &
        (file_index_df["n_windows"] > 0)
    ].copy().reset_index(drop=True)

    if len(valid) == 0:
        return pd.DataFrame()

    total_windows = int(valid["n_windows"].sum())

    if max_windows is None:
        sample_n = total_windows
    else:
        sample_n = min(int(max_windows), total_windows)

    global_indices = rng.choice(total_windows, size=sample_n, replace=False)
    global_indices = np.sort(global_indices)

    n_windows = valid["n_windows"].values.astype(np.int64)
    cum_end = np.cumsum(n_windows)
    cum_start = cum_end - n_windows

    file_pos = np.searchsorted(cum_end, global_indices, side="right")
    local_win = global_indices - cum_start[file_pos]

    tasks = valid.iloc[file_pos].copy()
    tasks["window_index"] = local_win.astype(int)
    tasks = tasks.sort_values(["path", "window_index"]).reset_index(drop=True)

    return tasks


# ============================================================
# PREPROCESSING
# ============================================================
def sanitize_raw_pdws_minimal(x):
    """
    Minimal sanitation required because HDBSCAN does not accept NaN/inf.

    Important:
      This does NOT standardize, scale, engineer, or transform the PDWs.
      It only makes the array finite.

    Handling:
      - Amplitude non-finite -> -200 dB sentinel.
      - Other non-finite values -> column median.
    """
    x = np.asarray(x, dtype=np.float32).copy()

    if x.ndim != 2 or x.shape[1] < 5:
        raise ValueError(f"Expected [n_pulses, >=5], got {x.shape}")

    x = x[:, :5]

    amp = x[:, 4].copy()
    amp[~np.isfinite(amp)] = -200.0
    x[:, 4] = amp

    for j in range(5):
        col = x[:, j].copy()

        if not np.isfinite(col).all():
            med = finite_median(col, default=0.0)
            col[~np.isfinite(col)] = med
            x[:, j] = col

    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    return x.astype(np.float32)


def engineer_pdw_features(x):
    """
    Engineering of PDW attributes for HDBSCAN.

    Raw input:
      0 ToA_us
      1 Frequency_MHz
      2 PulseWidth_us
      3 AoA_deg
      4 Amplitude_dB

    Engineered output:
      0 toa_rel_norm
      1 delta_toa_log
      2 frequency_norm
      3 delta_frequency_abs_norm
      4 pulsewidth_log
      5 aoa_sin
      6 aoa_cos
      7 delta_aoa_abs_norm
      8 amplitude_norm
      9 amplitude_missing

    Then robust scaling is applied inside the window.
    """
    x = np.asarray(x, dtype=np.float32)

    if x.ndim != 2 or x.shape[1] < 5:
        raise ValueError(f"Expected [n_pulses, >=5], got {x.shape}")

    toa = fill_nonfinite_with_median(x[:, 0], default=0.0)
    freq = fill_nonfinite_with_median(x[:, 1], default=0.0)
    pw = fill_nonfinite_with_median(x[:, 2], default=0.0)
    aoa = fill_nonfinite_with_median(x[:, 3], default=0.0)

    amp = x[:, 4].astype(np.float32).copy()
    amp_missing = (~np.isfinite(amp)).astype(np.float32)
    amp[~np.isfinite(amp)] = -200.0
    amp = fill_nonfinite_with_median(amp, default=-200.0)

    # 1) Relative ToA in the window
    toa_rel = toa - np.min(toa)
    span = float(np.max(toa_rel) - np.min(toa_rel))
    if span > 0:
        toa_rel_norm = toa_rel / (span + 1e-9)
    else:
        toa_rel_norm = np.zeros_like(toa_rel, dtype=np.float32)

    # 2) PRI-like delta ToA
    delta_toa = np.diff(toa, prepend=toa[0])
    delta_toa = np.maximum(delta_toa, 0.0)
    delta_toa_log = np.log1p(delta_toa)

    # 3) Frequency normalized by approximate receiver span
    frequency_norm = freq / 18000.0

    # 4) Frequency transition
    delta_frequency = np.abs(np.diff(freq, prepend=freq[0]))
    delta_frequency_abs_norm = delta_frequency / 18000.0

    # 5) Pulse width
    pw = np.maximum(pw, 0.0)
    pulsewidth_log = np.log1p(pw)

    # 6-7) AoA circular representation
    aoa_rad = np.deg2rad(aoa)
    aoa_sin = np.sin(aoa_rad)
    aoa_cos = np.cos(aoa_rad)

    # 8) AoA transition, circular
    delta_aoa_abs_norm = angular_abs_diff_deg(aoa) / 180.0

    # 9) Amplitude normalized
    amplitude_norm = (amp + 220.0) / 260.0

    feats = np.stack(
        [
            toa_rel_norm,
            delta_toa_log,
            frequency_norm,
            delta_frequency_abs_norm,
            pulsewidth_log,
            aoa_sin,
            aoa_cos,
            delta_aoa_abs_norm,
            amplitude_norm,
            amp_missing,
        ],
        axis=1,
    ).astype(np.float32)

    feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)

    # Robust scaling inside each window.
    # Do not scale the binary missingness flag.
    continuous_cols = list(range(9))
    binary_cols = [9]

    z = feats.copy()

    try:
        scaler = RobustScaler()
        z[:, continuous_cols] = scaler.fit_transform(z[:, continuous_cols])
    except Exception:
        # If a window is constant or numerically unstable, keep unscaled finite features.
        z[:, continuous_cols] = feats[:, continuous_cols]

    z[:, binary_cols] = feats[:, binary_cols]

    z = np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0)

    return z.astype(np.float32)


def preprocess_window(x, mode):
    if mode == "raw_pdw_reference":
        return sanitize_raw_pdws_minimal(x)

    if mode == "engineered_pdw":
        return engineer_pdw_features(x)

    raise ValueError(f"Unknown preprocess mode: {mode}")


def get_feature_names(mode):
    if mode == "raw_pdw_reference":
        return RAW_FEATURE_NAMES
    if mode == "engineered_pdw":
        return ENGINEERED_FEATURE_NAMES
    return []


# ============================================================
# CLUSTERING
# ============================================================
def run_hdbscan(X):
    X = np.asarray(X, dtype=np.float32)

    if HDBSCAN_BACKEND == "hdbscan" and hdbscan is not None:
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
            min_samples=HDBSCAN_MIN_SAMPLES,
            metric="euclidean",
            cluster_selection_epsilon=HDBSCAN_CLUSTER_SELECTION_EPSILON,
            cluster_selection_method=HDBSCAN_CLUSTER_SELECTION_METHOD,
            allow_single_cluster=HDBSCAN_ALLOW_SINGLE_CLUSTER,
        )
        return clusterer.fit_predict(X).astype(int)

    if SKHDBSCAN is not None:
        kwargs = dict(
            min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
            metric="euclidean",
            cluster_selection_epsilon=HDBSCAN_CLUSTER_SELECTION_EPSILON,
            allow_single_cluster=HDBSCAN_ALLOW_SINGLE_CLUSTER,
        )

        if HDBSCAN_MIN_SAMPLES is not None:
            kwargs["min_samples"] = HDBSCAN_MIN_SAMPLES

        clusterer = SKHDBSCAN(**kwargs)
        return clusterer.fit_predict(X).astype(int)

    raise RuntimeError("No HDBSCAN backend available. Try: pip install hdbscan")


# ============================================================
# METRICS
# ============================================================
def apply_noise_policy(y_pred):
    y_pred = np.asarray(y_pred).reshape(-1).astype(int).copy()

    if NOISE_POLICY_FOR_METRICS == "as_cluster":
        return y_pred

    if NOISE_POLICY_FOR_METRICS == "unique_noise":
        noise_idx = np.where(y_pred == -1)[0]

        if len(noise_idx) == 0:
            return y_pred

        max_cluster = int(np.max(y_pred)) if len(y_pred) > 0 else 0
        next_label = max_cluster + 1

        for k, idx in enumerate(noise_idx):
            y_pred[idx] = next_label + k

        return y_pred

    raise ValueError(f"Unknown NOISE_POLICY_FOR_METRICS: {NOISE_POLICY_FOR_METRICS}")


def efficient_pairwise_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    n = len(y_true)
    total_pairs = n * (n - 1) // 2

    if n < 2 or total_pairs <= 0:
        return {
            "pairwise_f1": np.nan,
            "pairwise_mcc": np.nan,
            "pairwise_tp": 0,
            "pairwise_fp": 0,
            "pairwise_fn": 0,
            "pairwise_tn": 0,
        }

    true_vals, true_inv = np.unique(y_true, return_inverse=True)
    pred_vals, pred_inv = np.unique(y_pred, return_inverse=True)

    contingency = np.zeros((len(true_vals), len(pred_vals)), dtype=np.int64)
    np.add.at(contingency, (true_inv, pred_inv), 1)

    tp = int(comb2(contingency).sum())

    true_sizes = contingency.sum(axis=1)
    pred_sizes = contingency.sum(axis=0)

    true_pairs = int(comb2(true_sizes).sum())
    pred_pairs = int(comb2(pred_sizes).sum())

    fp = int(pred_pairs - tp)
    fn = int(true_pairs - tp)
    tn = int(total_pairs - tp - fp - fn)

    denom_f1 = 2 * tp + fp + fn
    pairwise_f1 = float((2 * tp) / denom_f1) if denom_f1 > 0 else 0.0

    denom_mcc = math.sqrt(
        max(tp + fp, 0)
        * max(tp + fn, 0)
        * max(tn + fp, 0)
        * max(tn + fn, 0)
    )

    if denom_mcc > 0:
        pairwise_mcc = float((tp * tn - fp * fn) / denom_mcc)
    else:
        pairwise_mcc = 0.0

    return {
        "pairwise_f1": pairwise_f1,
        "pairwise_mcc": pairwise_mcc,
        "pairwise_tp": tp,
        "pairwise_fp": fp,
        "pairwise_fn": fn,
        "pairwise_tn": tn,
    }


def dominant_source_share(y_true):
    y_true = np.asarray(y_true).reshape(-1)

    if len(y_true) == 0:
        return np.nan

    _, counts = np.unique(y_true, return_counts=True)
    return float(counts.max() / counts.sum())


def source_entropy_norm(y_true):
    y_true = np.asarray(y_true).reshape(-1)

    if len(y_true) == 0:
        return np.nan

    _, counts = np.unique(y_true, return_counts=True)

    counts = counts.astype(np.float64)
    p = counts / counts.sum()
    p = p[p > 0]

    if len(p) <= 1:
        return 0.0

    return float(-np.sum(p * np.log(p)) / np.log(len(p)))


def deinterleaving_metrics(y_true, y_pred_raw):
    y_true = np.asarray(y_true).reshape(-1).astype(int)
    y_pred_raw = np.asarray(y_pred_raw).reshape(-1).astype(int)

    y_pred_eval = apply_noise_policy(y_pred_raw)

    out = {
        "v_measure": safe_float(v_measure_score(y_true, y_pred_eval)),
        "ari": safe_float(adjusted_rand_score(y_true, y_pred_eval)),
        "ami": safe_float(adjusted_mutual_info_score(y_true, y_pred_eval)),
        "homogeneity": safe_float(homogeneity_score(y_true, y_pred_eval)),
        "completeness": safe_float(completeness_score(y_true, y_pred_eval)),
    }

    out.update(efficient_pairwise_metrics(y_true, y_pred_eval))

    true_sources = np.unique(y_true)
    pred_clusters_raw = np.unique(y_pred_raw)
    pred_clusters_eval = np.unique(y_pred_eval)

    out["n_pulses"] = int(len(y_true))
    out["n_true_sources"] = int(len(true_sources))

    out["n_pred_clusters_raw_including_noise"] = int(len(pred_clusters_raw))
    out["n_pred_clusters_raw_excluding_noise"] = int(len([c for c in pred_clusters_raw if c != -1]))

    out["n_pred_clusters_eval"] = int(len(pred_clusters_eval))
    out["has_noise_cluster"] = int(np.any(y_pred_raw == -1))
    out["noise_share"] = float(np.mean(y_pred_raw == -1))

    return out


# ============================================================
# WINDOW PROCESSING
# ============================================================
def process_window_one_mode(
    x_window,
    y_window,
    split_name,
    file_id,
    file_name,
    window_index,
    start_pulse,
    end_pulse,
    preprocess_mode,
):
    t0 = time.time()

    x_window = np.asarray(x_window, dtype=np.float32)
    y_window = np.asarray(y_window).reshape(-1).astype(int)

    X = preprocess_window(x_window, preprocess_mode)
    y_pred = run_hdbscan(X)

    metrics = deinterleaving_metrics(y_window, y_pred)

    row = {
        "source": SOURCE_NAME,
        "split": split_name,
        "file_id": file_id,
        "file_name": file_name,
        "window_index": int(window_index),
        "start_pulse": int(start_pulse),
        "end_pulse": int(end_pulse),
        "window_size": int(end_pulse - start_pulse),
        "status": "ok",
        "preprocess_mode": preprocess_mode,
        "n_features": int(X.shape[1]),
        "feature_names": "|".join(get_feature_names(preprocess_mode)),
        "cluster_method": "HDBSCAN",
        "hdbscan_backend": HDBSCAN_BACKEND,
        "hdbscan_min_cluster_size": HDBSCAN_MIN_CLUSTER_SIZE,
        "hdbscan_min_samples": HDBSCAN_MIN_SAMPLES if HDBSCAN_MIN_SAMPLES is not None else "default",
        "hdbscan_epsilon": HDBSCAN_CLUSTER_SELECTION_EPSILON,
        "noise_policy_for_metrics": NOISE_POLICY_FOR_METRICS,
        "dominant_source_share": dominant_source_share(y_window),
        "source_entropy_norm": source_entropy_norm(y_window),
        "elapsed_sec": float(time.time() - t0),
    }

    row.update(metrics)

    return row


def process_window_all_modes(
    x_window,
    y_window,
    split_name,
    file_id,
    file_name,
    window_index,
    start_pulse,
    end_pulse,
):
    rows = []

    for mode in PREPROCESS_MODES:
        try:
            row = process_window_one_mode(
                x_window=x_window,
                y_window=y_window,
                split_name=split_name,
                file_id=file_id,
                file_name=file_name,
                window_index=window_index,
                start_pulse=start_pulse,
                end_pulse=end_pulse,
                preprocess_mode=mode,
            )
        except Exception as e:
            row = {
                "source": SOURCE_NAME,
                "split": split_name,
                "file_id": file_id,
                "file_name": file_name,
                "window_index": int(window_index),
                "start_pulse": int(start_pulse),
                "end_pulse": int(end_pulse),
                "window_size": int(end_pulse - start_pulse),
                "status": "error",
                "preprocess_mode": mode,
                "cluster_method": "HDBSCAN",
                "error": str(e),
            }

        rows.append(row)

    return rows


def process_file_selected_windows(path: Path, split_name: str, window_indices):
    rows = []
    file_id = path.stem
    n_windows_done = 0

    try:
        with h5py.File(path, "r") as f:
            if not h5_has_dataset(f, "data") or not h5_has_dataset(f, "labels"):
                return rows, n_windows_done

            data = f["data"]
            labels = f["labels"]
            n = int(data.shape[0])

            for w in window_indices:
                w = int(w)
                start = w * WINDOW_SIZE
                end = min((w + 1) * WINDOW_SIZE, n)

                if DROP_INCOMPLETE_WINDOWS and (end - start) < WINDOW_SIZE:
                    continue

                x_win = np.asarray(data[start:end, :5], dtype=np.float32)
                y_win = np.asarray(labels[start:end]).reshape(-1).astype(int)

                mode_rows = process_window_all_modes(
                    x_window=x_win,
                    y_window=y_win,
                    split_name=split_name,
                    file_id=file_id,
                    file_name=path.name,
                    window_index=w,
                    start_pulse=start,
                    end_pulse=end,
                )

                rows.extend(mode_rows)
                n_windows_done += 1

    except Exception as e:
        rows.append({
            "source": SOURCE_NAME,
            "split": split_name,
            "file_id": path.stem,
            "file_name": path.name,
            "path": str(path),
            "status": "error",
            "cluster_method": "HDBSCAN",
            "preprocess_mode": "__file_error__",
            "error": str(e),
        })

    return rows, n_windows_done


def process_split_random(split_name: str, file_index_df: pd.DataFrame, results_path: Path):
    tasks = select_random_window_tasks(file_index_df, MAX_WINDOWS_PER_SPLIT)

    print(f"[{split_name}] Random selected windows: {len(tasks)}")

    if len(tasks) == 0:
        return 0

    buffer_rows = []
    done = 0

    grouped = tasks.groupby("path", sort=False)
    pbar = tqdm(grouped, total=tasks["path"].nunique(), desc=f"{split_name}", unit="file")

    for path_str, g in pbar:
        rows, n_done_file = process_file_selected_windows(
            path=Path(path_str),
            split_name=split_name,
            window_indices=g["window_index"].values,
        )

        buffer_rows.extend(rows)
        done += n_done_file

        pbar.set_postfix({"windows": done})

        if done > 0 and done % SAVE_EVERY_WINDOWS == 0:
            append_rows_to_csv(buffer_rows, results_path)
            buffer_rows = []

    if len(buffer_rows) > 0:
        append_rows_to_csv(buffer_rows, results_path)

    return done


def process_split_sequential(split_name: str, file_index_df: pd.DataFrame, results_path: Path):
    valid = file_index_df[
        (file_index_df["status"] == "ok") &
        (file_index_df["n_windows"] > 0)
    ].copy().reset_index(drop=True)

    buffer_rows = []
    done = 0

    pbar = tqdm(valid.itertuples(index=False), total=len(valid), desc=f"{split_name}", unit="file")

    for row in pbar:
        if MAX_WINDOWS_PER_SPLIT is not None:
            remaining = int(MAX_WINDOWS_PER_SPLIT) - done
            if remaining <= 0:
                break
        else:
            remaining = int(row.n_windows)

        n_to_process = min(int(row.n_windows), remaining)
        window_indices = list(range(n_to_process))

        rows, n_done_file = process_file_selected_windows(
            path=Path(row.path),
            split_name=split_name,
            window_indices=window_indices,
        )

        buffer_rows.extend(rows)
        done += n_done_file

        pbar.set_postfix({"windows": done})

        if done > 0 and done % SAVE_EVERY_WINDOWS == 0:
            append_rows_to_csv(buffer_rows, results_path)
            buffer_rows = []

    if len(buffer_rows) > 0:
        append_rows_to_csv(buffer_rows, results_path)

    return done


# ============================================================
# SUMMARIES
# ============================================================
def summarize_results(results_df):
    metric_cols = [
        "v_measure",
        "ari",
        "ami",
        "homogeneity",
        "completeness",
        "pairwise_f1",
        "pairwise_mcc",
        "n_true_sources",
        "n_pred_clusters_raw_excluding_noise",
        "n_pred_clusters_eval",
        "noise_share",
        "dominant_source_share",
        "source_entropy_norm",
        "elapsed_sec",
    ]

    group_cols = [
        "source",
        "split",
        "cluster_method",
        "preprocess_mode",
        "noise_policy_for_metrics",
    ]

    rows = []

    for keys, g in results_df.groupby(group_cols, dropna=False):
        row = dict(zip(group_cols, keys))
        row["n_windows"] = int(len(g))

        for c in metric_cols:
            if c not in g.columns:
                continue

            vals = (
                pd.to_numeric(g[c], errors="coerce")
                .replace([np.inf, -np.inf], np.nan)
                .dropna()
            )

            if len(vals) == 0:
                row[f"{c}_mean"] = np.nan
                row[f"{c}_median"] = np.nan
                row[f"{c}_std"] = np.nan
            else:
                row[f"{c}_mean"] = float(vals.mean())
                row[f"{c}_median"] = float(vals.median())
                row[f"{c}_std"] = float(vals.std())

        rows.append(row)

    return pd.DataFrame(rows)


def build_reference_comparison(summary_df):
    rows = []

    metric_names = [
        "v_measure",
        "ari",
        "ami",
        "homogeneity",
        "completeness",
        "pairwise_mcc",
        "pairwise_f1",
    ]

    for _, row in summary_df.iterrows():
        split_name = row["split"]

        if split_name not in REFERENCE_METRICS:
            continue

        ref = REFERENCE_METRICS[split_name]

        out = {
            "source": row["source"],
            "split": split_name,
            "reference_label": ref["reference_label"],
            "local_method": f"{row['cluster_method']} + {row['preprocess_mode']}",
            "preprocess_mode": row["preprocess_mode"],
            "n_windows_local": int(row["n_windows"]),
            "noise_policy_for_metrics": row["noise_policy_for_metrics"],
        }

        for m in metric_names:
            local_col = f"{m}_median"

            if local_col in row:
                local_val = float(row[local_col])
            else:
                local_val = np.nan

            ref_val = float(ref[m]) if m in ref else np.nan

            out[f"{m}_reference"] = ref_val
            out[f"{m}_local_median"] = local_val
            out[f"{m}_delta_local_minus_reference"] = (
                local_val - ref_val if np.isfinite(local_val) and np.isfinite(ref_val) else np.nan
            )

        rows.append(out)

    return pd.DataFrame(rows)


def build_mode_comparison(summary_df, base_mode="raw_pdw_reference", new_mode="engineered_pdw"):
    metric_names = [
        "v_measure",
        "ari",
        "ami",
        "homogeneity",
        "completeness",
        "pairwise_mcc",
        "pairwise_f1",
        "n_true_sources",
        "n_pred_clusters_raw_excluding_noise",
        "noise_share",
        "elapsed_sec",
    ]

    rows = []

    for split_name, g in summary_df.groupby("split"):
        base = g[g["preprocess_mode"] == base_mode]
        new = g[g["preprocess_mode"] == new_mode]

        if len(base) == 0 or len(new) == 0:
            continue

        base = base.iloc[0]
        new = new.iloc[0]

        out = {
            "source": SOURCE_NAME,
            "split": split_name,
            "base_mode": base_mode,
            "new_mode": new_mode,
            "n_windows_base": int(base["n_windows"]),
            "n_windows_new": int(new["n_windows"]),
        }

        for m in metric_names:
            b_col = f"{m}_median"
            n_col = f"{m}_median"

            b_val = float(base[b_col]) if b_col in base and pd.notna(base[b_col]) else np.nan
            n_val = float(new[n_col]) if n_col in new and pd.notna(new[n_col]) else np.nan

            out[f"{m}_base_median"] = b_val
            out[f"{m}_new_median"] = n_val
            out[f"{m}_delta_new_minus_base"] = (
                n_val - b_val if np.isfinite(n_val) and np.isfinite(b_val) else np.nan
            )

        rows.append(out)

    return pd.DataFrame(rows)


def summarize_by_complexity(results_df):
    df = results_df.copy()

    df["n_true_sources_bin"] = pd.cut(
        df["n_true_sources"],
        bins=[0, 1, 2, 5, 10, 20, 40, 80, 200],
        labels=[
            "1",
            "2",
            "3-5",
            "6-10",
            "11-20",
            "21-40",
            "41-80",
            "81+",
        ],
        include_lowest=True,
    )

    rows = []

    for keys, g in df.groupby(["split", "preprocess_mode", "n_true_sources_bin"], dropna=False):
        split_name, mode_name, bin_name = keys

        out = {
            "source": SOURCE_NAME,
            "split": split_name,
            "preprocess_mode": mode_name,
            "n_true_sources_bin": str(bin_name),
            "n_windows": int(len(g)),
        }

        for c in [
            "v_measure",
            "ari",
            "ami",
            "homogeneity",
            "completeness",
            "pairwise_f1",
            "pairwise_mcc",
            "noise_share",
            "elapsed_sec",
        ]:
            vals = pd.to_numeric(g[c], errors="coerce").dropna()
            out[f"{c}_mean"] = float(vals.mean()) if len(vals) > 0 else np.nan
            out[f"{c}_median"] = float(vals.median()) if len(vals) > 0 else np.nan

        rows.append(out)

    return pd.DataFrame(rows)


# ============================================================
# PLOTS
# ============================================================
def save_hist(series, title, xlabel, path, bins=50):
    s = (
        pd.to_numeric(series, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    if len(s) == 0:
        return

    plt.figure(figsize=(8, 5))
    plt.hist(s.values, bins=bins)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Windows")
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()


def save_scatter(df, xcol, ycol, title, path, max_points=100_000):
    g = df[[xcol, ycol, "split", "preprocess_mode"]].copy()

    g[xcol] = pd.to_numeric(g[xcol], errors="coerce")
    g[ycol] = pd.to_numeric(g[ycol], errors="coerce")

    g = g.replace([np.inf, -np.inf], np.nan).dropna()

    if len(g) == 0:
        return

    if len(g) > max_points:
        g = g.sample(max_points, random_state=SEED)

    plt.figure(figsize=(8, 5))

    for mode_name, gm in g.groupby("preprocess_mode"):
        plt.scatter(gm[xcol], gm[ycol], s=6, alpha=0.35, label=mode_name)

    plt.title(title)
    plt.xlabel(xcol)
    plt.ylabel(ycol)
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()


def save_bar_summary(summary_df, metric, path):
    col = f"{metric}_median"

    if col not in summary_df.columns:
        return

    g = summary_df[["split", "preprocess_mode", col]].copy()
    g[col] = pd.to_numeric(g[col], errors="coerce")
    g = g.dropna()

    if len(g) == 0:
        return

    plt.figure(figsize=(8, 5))
    labels = [f"{r.split}\n{r.preprocess_mode}" for r in g.itertuples(index=False)]
    values = g[col].values

    plt.bar(labels, values)
    plt.ylabel(f"{metric} median")
    plt.title(f"Median {metric} by preprocessing mode")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()


def generate_plots(results_df, summary_df):
    print("=" * 80)
    print("GENERATING PLOTS")
    print("=" * 80)

    metrics = [
        "v_measure",
        "ari",
        "ami",
        "homogeneity",
        "completeness",
        "pairwise_mcc",
        "pairwise_f1",
        "n_true_sources",
        "n_pred_clusters_raw_excluding_noise",
        "noise_share",
        "elapsed_sec",
    ]

    for split_name, g_split in results_df.groupby("split"):
        for mode_name, g in g_split.groupby("preprocess_mode"):
            for metric in metrics:
                if metric not in g.columns:
                    continue

                save_hist(
                    g[metric],
                    f"{metric} - {split_name} - {mode_name}",
                    metric,
                    PLOTS_DIR / f"hist_{metric}_{safe_filename(split_name)}_{safe_filename(mode_name)}.png",
                    bins=60,
                )

    for metric in [
        "v_measure",
        "ari",
        "ami",
        "homogeneity",
        "completeness",
        "pairwise_mcc",
        "pairwise_f1",
        "noise_share",
        "elapsed_sec",
    ]:
        save_bar_summary(
            summary_df,
            metric,
            PLOTS_DIR / f"bar_median_{metric}_by_mode.png",
        )

    save_scatter(
        results_df,
        "n_true_sources",
        "v_measure",
        "V-measure vs number of true sources",
        PLOTS_DIR / "scatter_v_measure_vs_n_true_sources_by_mode.png",
    )

    save_scatter(
        results_df,
        "n_true_sources",
        "n_pred_clusters_raw_excluding_noise",
        "Predicted clusters vs true sources",
        PLOTS_DIR / "scatter_pred_clusters_vs_true_sources_by_mode.png",
    )

    save_scatter(
        results_df,
        "dominant_source_share",
        "v_measure",
        "V-measure vs dominant source share",
        PLOTS_DIR / "scatter_v_measure_vs_dominant_share_by_mode.png",
    )


# ============================================================
# REPORT
# ============================================================
def write_markdown_report(summary_df, comparison_ref_df, comparison_modes_df, complexity_df, file_index_df):
    report_path = OUT_DIR / "hdbscan_raw_vs_engineered_report.md"

    with open(report_path, "w", encoding="utf-8") as f:
        f.write("# HDBSCAN Raw PDWs vs Engineered PDW Features\n\n")

        f.write("## Inputs\n\n")
        for split_name in SPLITS_TO_PROCESS:
            f.write(f"- {split_name}: `{DATA_DIRS[split_name]}`\n")

        f.write("\n## Methodology\n\n")
        f.write(
            "This experiment compares two HDBSCAN inputs under the same windowing "
            "and evaluation protocol: raw PDWs and engineered PDW features. "
            "The data are split into non-overlapping 1024-pulse windows. "
            "The true number of emitters is not given to HDBSCAN. Labels are used "
            "only after clustering to compute external metrics.\n\n"
        )

        f.write("## Preprocessing modes\n\n")
        f.write("### raw_pdw_reference\n\n")
        f.write(
            "Uses the original five PDW attributes: ToA, frequency, pulse width, AoA, "
            "and amplitude. Only minimal finite-value sanitation is applied.\n\n"
        )

        f.write("### engineered_pdw\n\n")
        f.write("Uses the following engineered features:\n\n")
        for name in ENGINEERED_FEATURE_NAMES:
            f.write(f"- `{name}`\n")

        f.write("\n## Configuration\n\n")
        f.write(f"- Source: `{SOURCE_NAME}`\n")
        f.write(f"- Window size: {WINDOW_SIZE}\n")
        f.write(f"- Split(s): {SPLITS_TO_PROCESS}\n")
        f.write(f"- Window selection mode: `{WINDOW_SELECTION_MODE}`\n")
        f.write(f"- Max windows per split: {MAX_WINDOWS_PER_SPLIT}\n")
        f.write(f"- Preprocessing modes: `{PREPROCESS_MODES}`\n")
        f.write(f"- HDBSCAN backend: `{HDBSCAN_BACKEND}`\n")
        f.write(f"- min_cluster_size: {HDBSCAN_MIN_CLUSTER_SIZE}\n")
        f.write(f"- min_samples: {HDBSCAN_MIN_SAMPLES if HDBSCAN_MIN_SAMPLES is not None else 'default'}\n")
        f.write(f"- cluster_selection_epsilon: {HDBSCAN_CLUSTER_SELECTION_EPSILON}\n")
        f.write(f"- noise policy for metrics: `{NOISE_POLICY_FOR_METRICS}`\n\n")

        f.write("## File index summary\n\n")
        if len(file_index_df) > 0:
            idx_summary = (
                file_index_df
                .groupby("split", as_index=False)
                .agg(
                    n_files=("path", "count"),
                    n_ok=("status", lambda x: int(np.sum(np.asarray(x) == "ok"))),
                    total_pulses=("n_pulses", "sum"),
                    total_windows=("n_windows", "sum"),
                    mean_pulses=("n_pulses", "mean"),
                    median_pulses=("n_pulses", "median"),
                )
            )
            f.write(idx_summary.to_markdown(index=False))
            f.write("\n\n")

        f.write("## Local metric summary\n\n")
        if len(summary_df) > 0:
            f.write(summary_df.to_markdown(index=False))
            f.write("\n\n")

        f.write("## Direct comparison with reference paper\n\n")
        if len(comparison_ref_df) > 0:
            f.write(comparison_ref_df.to_markdown(index=False))
            f.write("\n\n")

        f.write("## Raw vs engineered comparison\n\n")
        if len(comparison_modes_df) > 0:
            f.write(comparison_modes_df.to_markdown(index=False))
            f.write("\n\n")

        f.write("## Complexity summary\n\n")
        if len(complexity_df) > 0:
            f.write(complexity_df.to_markdown(index=False))
            f.write("\n\n")

        f.write("## Output files\n\n")
        f.write("- `tables/window_metrics_raw_vs_engineered.csv`\n")
        f.write("- `tables/file_index_raw_vs_engineered.csv`\n")
        f.write("- `tables/summary_by_split_and_mode.csv`\n")
        f.write("- `tables/comparison_with_reference_paper.csv`\n")
        f.write("- `tables/comparison_modes_raw_vs_engineered.csv`\n")
        f.write("- `tables/summary_by_n_true_sources.csv`\n")
        f.write("- `plots/*.png`\n\n")

    return report_path


# ============================================================
# MAIN
# ============================================================
def main():
    print("=" * 80)
    print("TSRD HDBSCAN - RAW PDWs VS ENGINEERED FEATURES")
    print("=" * 80)

    print(f"Source                    : {SOURCE_NAME}")
    print(f"HDBSCAN backend            : {HDBSCAN_BACKEND}")
    print(f"Preprocess modes           : {PREPROCESS_MODES}")
    print(f"Window size                : {WINDOW_SIZE}")
    print(f"Splits                     : {SPLITS_TO_PROCESS}")
    print(f"Window selection mode      : {WINDOW_SELECTION_MODE}")
    print(f"Max windows per split      : {MAX_WINDOWS_PER_SPLIT}")
    print(f"Noise policy for metrics   : {NOISE_POLICY_FOR_METRICS}")
    print(f"HDBSCAN min_cluster_size   : {HDBSCAN_MIN_CLUSTER_SIZE}")
    print(f"HDBSCAN min_samples        : {HDBSCAN_MIN_SAMPLES if HDBSCAN_MIN_SAMPLES is not None else 'default'}")

    results_path = TABLES_DIR / "window_metrics_raw_vs_engineered.csv"
    file_index_path = TABLES_DIR / "file_index_raw_vs_engineered.csv"

    if OVERWRITE_PREVIOUS_RESULTS and results_path.exists():
        results_path.unlink()

    all_index = []
    split_done = {}

    for split_name in SPLITS_TO_PROCESS:
        folder = DATA_DIRS[split_name]

        print("\n" + "=" * 80)
        print(f"INDEXING SPLIT: {split_name}")
        print(f"FOLDER: {folder}")
        print("=" * 80)

        idx_df = collect_file_index(split_name, folder)
        all_index.append(idx_df)

        if len(idx_df) == 0:
            print(f"[WARNING] No files indexed for split: {split_name}")
            split_done[split_name] = 0
            continue

        print("\n" + "=" * 80)
        print(f"PROCESSING SPLIT: {split_name}")
        print("=" * 80)

        if WINDOW_SELECTION_MODE == "random" and MAX_WINDOWS_PER_SPLIT is not None:
            n_done = process_split_random(split_name, idx_df, results_path)
        else:
            n_done = process_split_sequential(split_name, idx_df, results_path)

        split_done[split_name] = int(n_done)
        print(f"[DONE] {split_name}: {n_done} windows processed")

    if len(all_index) > 0:
        file_index_df = pd.concat(all_index, axis=0).reset_index(drop=True)
    else:
        file_index_df = pd.DataFrame()

    file_index_df.to_csv(file_index_path, index=False, encoding="utf-8-sig")

    if not results_path.exists():
        print("[ERROR] No HDBSCAN results were generated.")
        return

    results_df = pd.read_csv(results_path)

    results_df = results_df[
        (results_df["status"] == "ok") &
        (results_df["cluster_method"] == "HDBSCAN")
    ].copy()

    results_df = results_df.dropna(subset=["v_measure"])

    if len(results_df) == 0:
        print("[ERROR] No valid metric rows found.")
        return

    # --------------------------------------------------------
    # Summaries
    # --------------------------------------------------------
    summary_df = summarize_results(results_df)
    complexity_df = summarize_by_complexity(results_df)
    comparison_ref_df = build_reference_comparison(summary_df)
    comparison_modes_df = build_mode_comparison(
        summary_df,
        base_mode="raw_pdw_reference",
        new_mode="engineered_pdw",
    )

    summary_path = TABLES_DIR / "summary_by_split_and_mode.csv"
    complexity_path = TABLES_DIR / "summary_by_n_true_sources.csv"
    comparison_ref_path = TABLES_DIR / "comparison_with_reference_paper.csv"
    comparison_modes_path = TABLES_DIR / "comparison_modes_raw_vs_engineered.csv"

    summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
    complexity_df.to_csv(complexity_path, index=False, encoding="utf-8-sig")
    comparison_ref_df.to_csv(comparison_ref_path, index=False, encoding="utf-8-sig")
    comparison_modes_df.to_csv(comparison_modes_path, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # Plots and report
    # --------------------------------------------------------
    generate_plots(results_df, summary_df)

    report_path = write_markdown_report(
        summary_df=summary_df,
        comparison_ref_df=comparison_ref_df,
        comparison_modes_df=comparison_modes_df,
        complexity_df=complexity_df,
        file_index_df=file_index_df,
    )

    # --------------------------------------------------------
    # Config
    # --------------------------------------------------------
    config = {
        "root": str(ROOT),
        "data_dirs": {k: str(v) for k, v in DATA_DIRS.items()},
        "source_name": SOURCE_NAME,
        "splits_to_process": SPLITS_TO_PROCESS,
        "window_size": WINDOW_SIZE,
        "drop_incomplete_windows": DROP_INCOMPLETE_WINDOWS,
        "max_files_per_split": MAX_FILES_PER_SPLIT,
        "max_windows_per_split": MAX_WINDOWS_PER_SPLIT,
        "window_selection_mode": WINDOW_SELECTION_MODE,
        "preprocess_modes": PREPROCESS_MODES,
        "raw_feature_names": RAW_FEATURE_NAMES,
        "engineered_feature_names": ENGINEERED_FEATURE_NAMES,
        "minimal_sanitation": {
            "amplitude_nonfinite": "-200.0 dB",
            "other_nonfinite": "column median",
        },
        "engineering": {
            "features": ENGINEERED_FEATURE_NAMES,
            "scaler": "RobustScaler fitted per 1024-pulse window",
            "binary_features_not_scaled": ["amplitude_missing"],
        },
        "hdbscan_backend": HDBSCAN_BACKEND,
        "hdbscan_min_cluster_size": HDBSCAN_MIN_CLUSTER_SIZE,
        "hdbscan_min_samples": HDBSCAN_MIN_SAMPLES if HDBSCAN_MIN_SAMPLES is not None else "default",
        "hdbscan_cluster_selection_epsilon": HDBSCAN_CLUSTER_SELECTION_EPSILON,
        "hdbscan_cluster_selection_method": HDBSCAN_CLUSTER_SELECTION_METHOD,
        "hdbscan_allow_single_cluster": HDBSCAN_ALLOW_SINGLE_CLUSTER,
        "noise_policy_for_metrics": NOISE_POLICY_FOR_METRICS,
        "reference_metrics": REFERENCE_METRICS,
        "split_windows_processed": split_done,
        "outputs": {
            "window_metrics": str(results_path),
            "file_index": str(file_index_path),
            "summary": str(summary_path),
            "complexity_summary": str(complexity_path),
            "comparison_with_reference": str(comparison_ref_path),
            "comparison_modes": str(comparison_modes_path),
            "plots": str(PLOTS_DIR),
            "report": str(report_path),
        },
    }

    config_path = OUT_DIR / "config.json"
    save_json(config, config_path)

    # --------------------------------------------------------
    # Console summary
    # --------------------------------------------------------
    print("\n" + "=" * 80)
    print("LOCAL SUMMARY BY PREPROCESSING MODE")
    print("=" * 80)

    cols_to_show = [
        "source",
        "split",
        "cluster_method",
        "preprocess_mode",
        "noise_policy_for_metrics",
        "n_windows",
        "v_measure_median",
        "ari_median",
        "ami_median",
        "homogeneity_median",
        "completeness_median",
        "pairwise_f1_median",
        "pairwise_mcc_median",
        "n_true_sources_median",
        "n_pred_clusters_raw_excluding_noise_median",
        "noise_share_median",
        "elapsed_sec_median",
    ]

    show_cols = [c for c in cols_to_show if c in summary_df.columns]
    print(summary_df[show_cols].to_string(index=False))

    print("\n" + "=" * 80)
    print("DIRECT COMPARISON WITH REFERENCE")
    print("=" * 80)

    if len(comparison_ref_df) > 0:
        compare_cols = [
            "split",
            "preprocess_mode",
            "reference_label",
            "n_windows_local",
            "v_measure_reference",
            "v_measure_local_median",
            "v_measure_delta_local_minus_reference",
            "ari_reference",
            "ari_local_median",
            "ari_delta_local_minus_reference",
            "ami_reference",
            "ami_local_median",
            "ami_delta_local_minus_reference",
            "homogeneity_reference",
            "homogeneity_local_median",
            "completeness_reference",
            "completeness_local_median",
            "pairwise_mcc_reference",
            "pairwise_mcc_local_median",
            "pairwise_f1_reference",
            "pairwise_f1_local_median",
        ]

        compare_cols = [c for c in compare_cols if c in comparison_ref_df.columns]
        print(comparison_ref_df[compare_cols].to_string(index=False))
    else:
        print("No comparison rows generated. Check REFERENCE_METRICS and processed splits.")

    print("\n" + "=" * 80)
    print("RAW VS ENGINEERED COMPARISON")
    print("=" * 80)

    if len(comparison_modes_df) > 0:
        mode_cols = [
            "split",
            "n_windows_base",
            "n_windows_new",
            "v_measure_base_median",
            "v_measure_new_median",
            "v_measure_delta_new_minus_base",
            "ari_base_median",
            "ari_new_median",
            "ari_delta_new_minus_base",
            "ami_base_median",
            "ami_new_median",
            "ami_delta_new_minus_base",
            "homogeneity_base_median",
            "homogeneity_new_median",
            "homogeneity_delta_new_minus_base",
            "completeness_base_median",
            "completeness_new_median",
            "completeness_delta_new_minus_base",
            "pairwise_mcc_base_median",
            "pairwise_mcc_new_median",
            "pairwise_mcc_delta_new_minus_base",
            "noise_share_base_median",
            "noise_share_new_median",
            "noise_share_delta_new_minus_base",
            "elapsed_sec_base_median",
            "elapsed_sec_new_median",
            "elapsed_sec_delta_new_minus_base",
        ]

        mode_cols = [c for c in mode_cols if c in comparison_modes_df.columns]
        print(comparison_modes_df[mode_cols].to_string(index=False))
    else:
        print("No raw-vs-engineered comparison generated.")

    print("\n" + "=" * 80)
    print("OUTPUT FILES")
    print("=" * 80)
    print(f"Window metrics       : {results_path}")
    print(f"File index           : {file_index_path}")
    print(f"Summary              : {summary_path}")
    print(f"Complexity summary   : {complexity_path}")
    print(f"Reference comparison : {comparison_ref_path}")
    print(f"Mode comparison      : {comparison_modes_path}")
    print(f"Plots directory      : {PLOTS_DIR}")
    print(f"Markdown report      : {report_path}")
    print(f"Config JSON          : {config_path}")

    print("\nDone.")


if __name__ == "__main__":
    main()

TSRD HDBSCAN - RAW PDWs VS ENGINEERED FEATURES
Source                    : archive
HDBSCAN backend            : hdbscan
Preprocess modes           : ['raw_pdw_reference', 'engineered_pdw']
Window size                : 1024
Splits                     : ['test']
Window selection mode      : sequential
Max windows per split      : None
Noise policy for metrics   : as_cluster
HDBSCAN min_cluster_size   : 5
HDBSCAN min_samples        : default

INDEXING SPLIT: test
FOLDER: D:\Fusion\turing_synthetic_radar_dataset\archive\test
[test] H5 files found: 250


index_test: 100%|██████████| 250/250 [00:05<00:00, 48.20file/s]



PROCESSING SPLIT: test


test: 100%|██████████| 237/237 [1:10:20<00:00, 17.81s/file, windows=36479]


[DONE] test: 36479 windows processed
GENERATING PLOTS

LOCAL SUMMARY BY PREPROCESSING MODE
 source split cluster_method   preprocess_mode noise_policy_for_metrics  n_windows  v_measure_median  ari_median  ami_median  homogeneity_median  completeness_median  pairwise_f1_median  pairwise_mcc_median  n_true_sources_median  n_pred_clusters_raw_excluding_noise_median  noise_share_median  elapsed_sec_median
archive  test        HDBSCAN    engineered_pdw               as_cluster      36479          0.573791    0.351553    0.549712            0.854200             0.455564            0.688395             0.450307                   12.0                                         8.0            0.035156            0.063035
archive  test        HDBSCAN raw_pdw_reference               as_cluster      36479          0.522578    0.280134    0.493810            0.799444             0.416608            0.718088             0.383768                   12.0                                         6.0        

# ANÁLISE

# Análise comparativa: HDBSCAN com PDWs brutos vs. engenharia de atributos

O experimento comparou duas formas de entrada para o HDBSCAN no conjunto `archive/test` do TSRD: **PDWs brutos** e **PDWs com engenharia de atributos**. Em ambos os casos, foi usada a mesma metodologia de avaliação: janelas não sobrepostas de **1024 pulsos**, processamento sequencial de todas as janelas disponíveis, política de ruído `as_cluster` e o mesmo backend `hdbscan`. Foram avaliadas **36.479 janelas** em cada configuração. 

## Tabela comparativa

| Métrica                  |  PDWs brutos | PDWs com engenharia de atributos |   Diferença |
| ------------------------ | -----------: | -------------------------------: | ----------: |
| V-measure                |       0.5226 |                       **0.5738** | **+0.0512** |
| ARI                      |       0.2801 |                       **0.3516** | **+0.0714** |
| AMI                      |       0.4938 |                       **0.5497** | **+0.0559** |
| Homogeneity              |       0.7994 |                       **0.8542** | **+0.0548** |
| Completeness             |       0.4166 |                       **0.4556** | **+0.0390** |
| Pairwise MCC             |       0.3838 |                       **0.4503** | **+0.0665** |
| Pairwise F1              |   **0.7181** |                           0.6884 |     -0.0297 |
| Fontes reais por janela  |         12.0 |                             12.0 |         0.0 |
| Clusters previstos       |          6.0 |                          **8.0** |        +2.0 |
| Fração de ruído          |   **0.0078** |                           0.0352 |     +0.0273 |
| Tempo mediano por janela | **0.0462 s** |                         0.0630 s |   +0.0168 s |

## Comparação com o baseline de referência

| Método                               |  V-measure |        ARI |        AMI | Homogeneity | Completeness | Pairwise MCC | Pairwise F1 |
| ------------------------------------ | ---------: | ---------: | ---------: | ----------: | -----------: | -----------: | ----------: |
| Referência — HDBSCAN com PDWs brutos |     0.5400 |     0.2700 |     0.5000 |      0.6400 |       0.5000 |       0.0600 |      0.0100 |
| Reprodução local — PDWs brutos       |     0.5226 |     0.2801 |     0.4938 |      0.7994 |       0.4166 |       0.3838 |  **0.7181** |
| Proposta — engenharia de atributos   | **0.5738** | **0.3516** | **0.5497** |  **0.8542** |       0.4556 |   **0.4503** |      0.6884 |

## Análise

A engenharia de atributos melhorou o desempenho do HDBSCAN na maioria das métricas principais de deinterleaving. A **V-measure** aumentou de **0.5226** para **0.5738**, representando um ganho absoluto de **+0.0512**. Esse resultado é relevante porque a V-measure combina homogeneity e completeness, ou seja, avalia simultaneamente se os clusters são puros e se recuperam adequadamente os emissores reais.

O ganho em **ARI** foi ainda mais expressivo, passando de **0.2801** para **0.3516**. Isso indica que a estrutura dos agrupamentos obtidos com os atributos derivados ficou mais próxima da partição real dos emissores. O **AMI** também aumentou de **0.4938** para **0.5497**, reforçando que a engenharia de atributos tornou a representação mais informativa para a separação dos pulsos.

A melhora em **homogeneity**, de **0.7994** para **0.8542**, mostra que os clusters estimados passaram a ser mais puros, isto é, cada cluster tende a conter pulsos de um mesmo emissor. A **completeness** também melhorou, de **0.4166** para **0.4556**, embora permaneça inferior à homogeneity. Isso sugere que o método com engenharia de atributos ainda fragmenta alguns emissores em múltiplos clusters, mas recupera melhor a estrutura dos emissores do que os PDWs brutos.

O **Pairwise MCC** aumentou de **0.3838** para **0.4503**, indicando uma melhora importante na separação par-a-par dos pulsos. O único indicador que diminuiu foi o **Pairwise F1**, de **0.7181** para **0.6884**. Essa queda deve ser interpretada com cautela, pois o F1 pairwise pode favorecer soluções que agrupam muitos pares como pertencentes ao mesmo cluster. Já o MCC tende a ser mais equilibrado por considerar simultaneamente verdadeiros positivos, falsos positivos, falsos negativos e verdadeiros negativos.

A engenharia de atributos também aumentou o número mediano de clusters previstos, de **6** para **8**, aproximando-se do número mediano de fontes reais, que foi **12** em ambos os casos. Entretanto, esse aumento veio acompanhado de maior fração de ruído: de **0.0078** para **0.0352**. Isso indica que a representação engenheirada tornou o HDBSCAN mais seletivo, separando melhor alguns padrões, mas também rejeitando mais pontos como ruído.

Em termos computacionais, o tempo mediano por janela aumentou de **0.0462 s** para **0.0630 s**, um acréscimo de aproximadamente **0.0168 s** por janela. Esse custo é moderado diante dos ganhos observados em V-measure, ARI, AMI e Pairwise MCC.

## Conclusão

A engenharia de atributos aplicada aos PDWs melhorou o baseline HDBSCAN de forma consistente. O método com atributos derivados superou tanto a reprodução local com PDWs brutos quanto o valor de referência do paper em **V-measure**, **ARI**, **AMI**, **homogeneity** e **Pairwise MCC**. Assim, os resultados indicam que atributos como `delta_toa_log`, variação de frequência, representação circular de AoA e máscara de amplitude ausente fornecem uma representação mais discriminativa para a tarefa de deinterleaving de pulsos radar.

Em síntese, a abordagem com engenharia de atributos é superior ao uso direto dos PDWs brutos para o HDBSCAN, com ganho claro de desempenho e custo computacional moderado.


# IMPLEMENTANDO MOE TAL COMO O TRABALHO DE ACÚSTICA

In [8]:
# ============================================================
# proposed_moe_pdw_deinterleaving_sweep.py
# ============================================================
# TSRD - Feature-Specialized Mixture-of-Experts for
# radar pulse deinterleaving.
#
# Task:
#   Separate interleaved radar pulses into emitter clusters.
#
# Inputs:
#   D:\Fusion\turing_synthetic_radar_dataset\archive\train
#   D:\Fusion\turing_synthetic_radar_dataset\archive\validation
#   D:\Fusion\turing_synthetic_radar_dataset\archive\test
#
# Each .h5:
#   data   : [n_pulses, 5]
#   labels : [n_pulses, 1]
#
# PDW columns:
#   0 -> ToA_us
#   1 -> Frequency_MHz
#   2 -> PulseWidth_us
#   3 -> AoA_deg
#   4 -> Amplitude_dB
#
# Method:
#   train:
#       PDW windows -> Feature-Specialized MoE -> pulse embeddings
#       supervised contrastive loss within each window
#
#   validation/test:
#       PDW windows -> embeddings -> HDBSCAN -> clusters
#       compare clusters with true labels
#
# Important:
#   Labels are used for training and evaluation, but the clustering step
#   does NOT receive the true number of emitters.
#
# Sweep:
#   N_EXPERTS in {2, 3, 4, 5}
#
# Outputs:
#   D:\Fusion\turing_synthetic_radar_dataset\benchmark_moe_pdw_deinterleaving_sweep
# ============================================================

from pathlib import Path
import sys
import json
import math
import time
import random
import subprocess
import warnings
warnings.filterwarnings("ignore")


# ============================================================
# INSTALL / IMPORTS
# ============================================================
def ensure_package(import_name, pip_name=None):
    try:
        __import__(import_name)
    except Exception:
        pkg = pip_name if pip_name is not None else import_name
        print(f"[INFO] Installing missing package: {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


ensure_package("numpy")
ensure_package("pandas")
ensure_package("matplotlib")
ensure_package("h5py")
ensure_package("sklearn", "scikit-learn")
ensure_package("torch")
ensure_package("tqdm")
ensure_package("tabulate")

try:
    ensure_package("hdbscan")
    import hdbscan
    HDBSCAN_BACKEND = "hdbscan"
except Exception:
    hdbscan = None
    HDBSCAN_BACKEND = "sklearn"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    v_measure_score,
    adjusted_rand_score,
    adjusted_mutual_info_score,
    homogeneity_score,
    completeness_score,
)

try:
    from sklearn.cluster import HDBSCAN as SKHDBSCAN
except Exception:
    SKHDBSCAN = None


# ============================================================
# PATHS
# ============================================================
ROOT = Path(r"D:\Fusion\turing_synthetic_radar_dataset")

DATA_DIRS = {
    "train": ROOT / "archive" / "train",
    "validation": ROOT / "archive" / "validation",
    "test": ROOT / "archive" / "test",
}

SOURCE_NAME = "archive"

OUT_DIR = ROOT / "benchmark_moe_pdw_deinterleaving_sweep"
TABLES_DIR = OUT_DIR / "tables"
PLOTS_DIR = OUT_DIR / "plots"

OUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# CONFIG
# ============================================================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE_TYPE = "cuda" if DEVICE == "cuda" else "cpu"
USE_AMP = DEVICE == "cuda"

WINDOW_SIZE = 1024
DROP_INCOMPLETE_WINDOWS = True

# Para teste rápido:
MAX_TRAIN_WINDOWS = 8000
MAX_VAL_WINDOWS = 2000
MAX_TEST_WINDOWS = 5000

# Para avaliação final completa no teste, use:
# MAX_TEST_WINDOWS = None
# TEST_WINDOW_SELECTION_MODE = "sequential"

TRAIN_WINDOW_SELECTION_MODE = "random"
VAL_WINDOW_SELECTION_MODE = "random"
TEST_WINDOW_SELECTION_MODE = "random"

MAX_FILES_PER_SPLIT = None

# PDW engineered features:
#   0 toa_rel_norm
#   1 delta_toa_log
#   2 frequency_norm
#   3 pulsewidth_log
#   4 aoa_sin
#   5 aoa_cos
#   6 amplitude_norm
#   7 amplitude_missing
N_INPUT_FEATURES = 8

# Sweep
EXPERTS_LIST = [2, 3, 4, 5]

# Model
EMB_DIM = 64
HIDDEN_DIM = 128
DROP = 0.15

# Training
BATCH_SIZE = 4
EPOCHS = 20
LR = 2.0e-4
WEIGHT_DECAY = 1.0e-4
PATIENCE = 5
GRAD_CLIP = 2.0
NUM_WORKERS = 0

# Contrastive loss
SUPCON_TEMPERATURE = 0.10

# HDBSCAN over learned embeddings
HDBSCAN_MIN_CLUSTER_SIZE = 5
HDBSCAN_MIN_SAMPLES = 3
HDBSCAN_CLUSTER_SELECTION_EPSILON = 0.0
HDBSCAN_CLUSTER_SELECTION_METHOD = "eom"
HDBSCAN_ALLOW_SINGLE_CLUSTER = False

# Metrics policy
NOISE_POLICY_FOR_METRICS = "as_cluster"  # or "unique_noise"

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 180


# ============================================================
# REPRODUCIBILITY
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
rng = np.random.default_rng(SEED)


# ============================================================
# HELPERS
# ============================================================
def save_json(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def h5_has_dataset(f, name):
    try:
        return isinstance(f[name], h5py.Dataset)
    except Exception:
        return False


def comb2(x):
    x = np.asarray(x, dtype=np.int64)
    return x * (x - 1) // 2


def safe_float(x):
    try:
        if np.isfinite(x):
            return float(x)
        return float("nan")
    except Exception:
        return float("nan")


def finite_median(v, default=0.0):
    v = np.asarray(v, dtype=np.float32)
    mask = np.isfinite(v)
    if mask.any():
        return float(np.nanmedian(v[mask]))
    return float(default)


def fill_nonfinite_with_median(v, default=0.0):
    v = np.asarray(v, dtype=np.float32).copy()
    mask = np.isfinite(v)
    med = finite_median(v, default=default)
    v[~mask] = med
    return v


# ============================================================
# FILE INDEX AND WINDOW TASKS
# ============================================================
def collect_file_index(split_name: str, folder: Path):
    rows = []

    if not folder.exists():
        raise FileNotFoundError(f"Missing folder: {folder}")

    h5_files = sorted(folder.rglob("*.h5"))

    if MAX_FILES_PER_SPLIT is not None:
        h5_files = h5_files[:int(MAX_FILES_PER_SPLIT)]

    print(f"[{split_name}] H5 files found: {len(h5_files)}")

    for p in tqdm(h5_files, desc=f"index_{split_name}", unit="file"):
        row = {
            "source": SOURCE_NAME,
            "split": split_name,
            "file_id": p.stem,
            "file_name": p.name,
            "path": str(p),
            "status": "ok",
            "n_pulses": 0,
            "n_windows": 0,
            "error": None,
        }

        try:
            with h5py.File(p, "r") as f:
                if not h5_has_dataset(f, "data") or not h5_has_dataset(f, "labels"):
                    row["status"] = "missing_data_or_labels"
                    rows.append(row)
                    continue

                n = int(f["data"].shape[0])
                row["n_pulses"] = n

                if n <= 0:
                    row["status"] = "empty"
                    rows.append(row)
                    continue

                if DROP_INCOMPLETE_WINDOWS:
                    n_windows = n // WINDOW_SIZE
                else:
                    n_windows = int(math.ceil(n / WINDOW_SIZE))

                row["n_windows"] = int(n_windows)

        except Exception as e:
            row["status"] = "error"
            row["error"] = str(e)

        rows.append(row)

    return pd.DataFrame(rows)


def select_window_tasks(file_index_df: pd.DataFrame, max_windows=None, mode="random"):
    valid = file_index_df[
        (file_index_df["status"] == "ok") &
        (file_index_df["n_windows"] > 0)
    ].copy().reset_index(drop=True)

    if len(valid) == 0:
        return pd.DataFrame()

    if mode == "sequential":
        rows = []
        done = 0

        for _, r in valid.iterrows():
            n_w = int(r["n_windows"])

            for w in range(n_w):
                if max_windows is not None and done >= int(max_windows):
                    break

                rr = r.to_dict()
                rr["window_index"] = int(w)
                rows.append(rr)
                done += 1

            if max_windows is not None and done >= int(max_windows):
                break

        return pd.DataFrame(rows)

    if mode != "random":
        raise ValueError(f"Unknown window selection mode: {mode}")

    total_windows = int(valid["n_windows"].sum())

    if max_windows is None:
        sample_n = total_windows
    else:
        sample_n = min(int(max_windows), total_windows)

    if sample_n <= 0:
        return pd.DataFrame()

    global_indices = rng.choice(total_windows, size=sample_n, replace=False)
    global_indices = np.sort(global_indices)

    n_windows = valid["n_windows"].values.astype(np.int64)
    cum_end = np.cumsum(n_windows)
    cum_start = cum_end - n_windows

    file_pos = np.searchsorted(cum_end, global_indices, side="right")
    local_win = global_indices - cum_start[file_pos]

    tasks = valid.iloc[file_pos].copy()
    tasks["window_index"] = local_win.astype(int)
    tasks = tasks.sort_values(["path", "window_index"]).reset_index(drop=True)

    return tasks


# ============================================================
# PDW PREPROCESSING
# ============================================================
def preprocess_pdw_window(x):
    """
    Converts raw PDWs [ToA, Frequency, PulseWidth, AoA, Amplitude]
    into engineered feature vectors.

    Output features:
      0 toa_rel_norm
      1 delta_toa_log
      2 frequency_norm
      3 pulsewidth_log
      4 aoa_sin
      5 aoa_cos
      6 amplitude_norm
      7 amplitude_missing
    """
    x = np.asarray(x, dtype=np.float32)

    toa = fill_nonfinite_with_median(x[:, 0], default=0.0)
    freq = fill_nonfinite_with_median(x[:, 1], default=0.0)
    pw = fill_nonfinite_with_median(x[:, 2], default=0.0)
    aoa = fill_nonfinite_with_median(x[:, 3], default=0.0)

    amp = x[:, 4].astype(np.float32).copy()
    amp_missing = (~np.isfinite(amp)).astype(np.float32)
    amp[~np.isfinite(amp)] = -200.0
    amp = fill_nonfinite_with_median(amp, default=-200.0)

    # Relative ToA
    toa_rel = toa - np.min(toa)
    span = float(np.max(toa_rel) - np.min(toa_rel))
    if span > 0:
        toa_rel_norm = toa_rel / (span + 1e-9)
    else:
        toa_rel_norm = np.zeros_like(toa_rel, dtype=np.float32)

    # Delta ToA
    dtoa = np.diff(toa, prepend=toa[0])
    dtoa = np.maximum(dtoa, 0.0)
    dtoa_log = np.log1p(dtoa)

    # Frequency
    freq_norm = freq / 18000.0

    # Pulse width
    pw = np.maximum(pw, 0.0)
    pw_log = np.log1p(pw)

    # AoA circular representation
    aoa_rad = np.deg2rad(aoa)
    aoa_sin = np.sin(aoa_rad)
    aoa_cos = np.cos(aoa_rad)

    # Amplitude
    amp_norm = (amp + 220.0) / 260.0

    feats = np.stack(
        [
            toa_rel_norm,
            dtoa_log,
            freq_norm,
            pw_log,
            aoa_sin,
            aoa_cos,
            amp_norm,
            amp_missing,
        ],
        axis=1,
    ).astype(np.float32)

    # Robust scaling for continuous dimensions only.
    z = feats.copy()
    continuous_cols = [0, 1, 2, 3, 4, 5, 6]

    try:
        scaler = RobustScaler()
        z[:, continuous_cols] = scaler.fit_transform(z[:, continuous_cols])
    except Exception:
        pass

    z = np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0)
    return z.astype(np.float32)


# ============================================================
# DATASET
# ============================================================
class PDWWindowDataset(Dataset):
    def __init__(self, tasks_df: pd.DataFrame, train_mode=False):
        self.tasks = tasks_df.reset_index(drop=True)
        self.train_mode = bool(train_mode)

    def __len__(self):
        return len(self.tasks)

    def __getitem__(self, idx):
        r = self.tasks.iloc[idx]
        path = Path(r["path"])
        w = int(r["window_index"])

        start = w * WINDOW_SIZE
        end = start + WINDOW_SIZE

        with h5py.File(path, "r") as f:
            x_raw = np.asarray(f["data"][start:end, :5], dtype=np.float32)
            y = np.asarray(f["labels"][start:end]).reshape(-1).astype(np.int64)

        x = preprocess_pdw_window(x_raw)

        return {
            "x": torch.tensor(x, dtype=torch.float32),          # [1024, 8]
            "y": torch.tensor(y, dtype=torch.long),             # [1024]
            "file_id": str(r["file_id"]),
            "split": str(r["split"]),
            "window_index": int(w),
        }


# ============================================================
# FEATURE GROUPS FOR EXPERT SWEEP
# ============================================================
FEATURE_NAMES = [
    "toa_rel_norm",
    "delta_toa_log",
    "frequency_norm",
    "pulsewidth_log",
    "aoa_sin",
    "aoa_cos",
    "amplitude_norm",
    "amplitude_missing",
]


def get_feature_groups(n_experts):
    """
    Feature-specialized experts for PDWs.

    Engineered feature indices:
      0 toa_rel_norm
      1 delta_toa_log
      2 frequency_norm
      3 pulsewidth_log
      4 aoa_sin
      5 aoa_cos
      6 amplitude_norm
      7 amplitude_missing
    """
    if n_experts == 2:
        groups = [
            [0, 1, 3],          # time + PRI-like + pulse width
            [2, 4, 5, 6, 7],    # frequency + angle + amplitude
        ]

    elif n_experts == 3:
        groups = [
            [0, 1],             # time
            [2, 3],             # frequency + pulse width
            [4, 5, 6, 7],       # AoA + amplitude
        ]

    elif n_experts == 4:
        groups = [
            [0, 1],             # time
            [2],                # frequency
            [3],                # pulse width
            [4, 5, 6, 7],       # AoA + amplitude
        ]

    elif n_experts == 5:
        groups = [
            [0, 1],             # time
            [2],                # frequency
            [3],                # pulse width
            [4, 5],             # AoA
            [6, 7],             # amplitude + missing mask
        ]

    else:
        raise ValueError(f"Unsupported n_experts: {n_experts}")

    return groups


def describe_feature_groups(groups):
    return [[FEATURE_NAMES[i] for i in g] for g in groups]


# ============================================================
# MODEL
# ============================================================
class FeatureExpert(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, emb_dim=64, drop=0.15):
        super().__init__()

        self.point = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
        )

        self.temporal = nn.Sequential(
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2, groups=1),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2, groups=1),
            nn.GELU(),
        )

        self.norm = nn.LayerNorm(hidden_dim)

        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, emb_dim),
            nn.GELU(),
        )

    def forward(self, x):
        # x: [B, N, Fin]
        h = self.point(x)  # [B, N, H]

        ht = h.transpose(1, 2)        # [B, H, N]
        ht = self.temporal(ht)        # [B, H, N]
        ht = ht.transpose(1, 2)       # [B, N, H]

        h = self.norm(h + ht)

        z = self.proj(h)              # [B, N, D]
        return z


class PDWFeatureMoE(nn.Module):
    def __init__(self, feature_groups, input_dim=8, hidden_dim=128, emb_dim=64, drop=0.15):
        super().__init__()

        self.feature_groups = [list(g) for g in feature_groups]
        self.n_experts = len(self.feature_groups)
        self.emb_dim = emb_dim

        self.experts = nn.ModuleList([
            FeatureExpert(
                in_dim=len(g),
                hidden_dim=hidden_dim,
                emb_dim=emb_dim,
                drop=drop,
            )
            for g in self.feature_groups
        ])

        self.gate = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(hidden_dim, self.n_experts),
        )

        self.out_norm = nn.LayerNorm(emb_dim)

    def forward(self, x):
        # x: [B, N, F]
        gate_logits = self.gate(x)                    # [B, N, E]
        gate_w = torch.softmax(gate_logits, dim=-1)   # [B, N, E]

        expert_outs = []
        for expert, group in zip(self.experts, self.feature_groups):
            xg = x[:, :, group]
            zg = expert(xg)
            expert_outs.append(zg)

        E = torch.stack(expert_outs, dim=2)           # [B, N, E, D]
        z = (E * gate_w.unsqueeze(-1)).sum(dim=2)     # [B, N, D]
        z = self.out_norm(z)

        return {
            "emb": z,
            "gate_w": gate_w,
        }


# ============================================================
# SUPERVISED CONTRASTIVE LOSS WITHIN EACH WINDOW
# ============================================================
def supervised_contrastive_window_loss(emb, labels, temperature=0.10):
    """
    emb:    [B, N, D]
    labels: [B, N]

    Labels are meaningful only within a window. Therefore, contrastive
    positives are computed independently for each window.
    """
    B, N, D = emb.shape
    emb = F.normalize(emb, dim=-1)

    losses = []

    for b in range(B):
        z = emb[b]        # [N, D]
        y = labels[b]     # [N]

        sim = torch.matmul(z, z.T) / temperature  # [N, N]

        self_mask = torch.eye(N, device=emb.device, dtype=torch.bool)
        pos_mask = (y[:, None] == y[None, :]) & (~self_mask)
        valid_anchor = pos_mask.sum(dim=1) > 0

        if valid_anchor.sum() == 0:
            continue

        sim = sim.masked_fill(self_mask, -1e9)

        log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True)

        pos_log_prob = (log_prob * pos_mask.float()).sum(dim=1) / (
            pos_mask.float().sum(dim=1) + 1e-9
        )

        loss_b = -pos_log_prob[valid_anchor].mean()
        losses.append(loss_b)

    if len(losses) == 0:
        return emb.sum() * 0.0

    return torch.stack(losses).mean()


def gate_balance_regularizer(gate_w):
    # gate_w: [B, N, E]
    avg = gate_w.mean(dim=(0, 1))
    target = torch.full_like(avg, 1.0 / avg.numel())
    return ((avg - target) ** 2).mean()


# ============================================================
# HDBSCAN AND METRICS
# ============================================================
def run_hdbscan(X):
    X = np.asarray(X, dtype=np.float32)

    if HDBSCAN_BACKEND == "hdbscan" and hdbscan is not None:
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
            min_samples=HDBSCAN_MIN_SAMPLES,
            metric="euclidean",
            cluster_selection_epsilon=HDBSCAN_CLUSTER_SELECTION_EPSILON,
            cluster_selection_method=HDBSCAN_CLUSTER_SELECTION_METHOD,
            allow_single_cluster=HDBSCAN_ALLOW_SINGLE_CLUSTER,
        )
        return clusterer.fit_predict(X).astype(int)

    if SKHDBSCAN is not None:
        clusterer = SKHDBSCAN(
            min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
            min_samples=HDBSCAN_MIN_SAMPLES,
            metric="euclidean",
            cluster_selection_epsilon=HDBSCAN_CLUSTER_SELECTION_EPSILON,
            allow_single_cluster=HDBSCAN_ALLOW_SINGLE_CLUSTER,
        )
        return clusterer.fit_predict(X).astype(int)

    raise RuntimeError("No HDBSCAN backend available. Try: pip install hdbscan")


def apply_noise_policy(y_pred):
    y_pred = np.asarray(y_pred).reshape(-1).astype(int).copy()

    if NOISE_POLICY_FOR_METRICS == "as_cluster":
        return y_pred

    if NOISE_POLICY_FOR_METRICS == "unique_noise":
        noise_idx = np.where(y_pred == -1)[0]
        if len(noise_idx) == 0:
            return y_pred

        max_cluster = int(np.max(y_pred)) if len(y_pred) > 0 else 0
        next_label = max_cluster + 1

        for k, idx in enumerate(noise_idx):
            y_pred[idx] = next_label + k

        return y_pred

    raise ValueError(f"Unknown NOISE_POLICY_FOR_METRICS: {NOISE_POLICY_FOR_METRICS}")


def efficient_pairwise_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    n = len(y_true)
    total_pairs = n * (n - 1) // 2

    if n < 2 or total_pairs <= 0:
        return {
            "pairwise_f1": np.nan,
            "pairwise_mcc": np.nan,
            "pairwise_tp": 0,
            "pairwise_fp": 0,
            "pairwise_fn": 0,
            "pairwise_tn": 0,
        }

    true_vals, true_inv = np.unique(y_true, return_inverse=True)
    pred_vals, pred_inv = np.unique(y_pred, return_inverse=True)

    contingency = np.zeros((len(true_vals), len(pred_vals)), dtype=np.int64)
    np.add.at(contingency, (true_inv, pred_inv), 1)

    tp = int(comb2(contingency).sum())

    true_sizes = contingency.sum(axis=1)
    pred_sizes = contingency.sum(axis=0)

    true_pairs = int(comb2(true_sizes).sum())
    pred_pairs = int(comb2(pred_sizes).sum())

    fp = int(pred_pairs - tp)
    fn = int(true_pairs - tp)
    tn = int(total_pairs - tp - fp - fn)

    denom_f1 = 2 * tp + fp + fn
    pairwise_f1 = float((2 * tp) / denom_f1) if denom_f1 > 0 else 0.0

    denom_mcc = math.sqrt(
        max(tp + fp, 0)
        * max(tp + fn, 0)
        * max(tn + fp, 0)
        * max(tn + fn, 0)
    )

    if denom_mcc > 0:
        pairwise_mcc = float((tp * tn - fp * fn) / denom_mcc)
    else:
        pairwise_mcc = 0.0

    return {
        "pairwise_f1": pairwise_f1,
        "pairwise_mcc": pairwise_mcc,
        "pairwise_tp": tp,
        "pairwise_fp": fp,
        "pairwise_fn": fn,
        "pairwise_tn": tn,
    }


def dominant_source_share(y_true):
    y_true = np.asarray(y_true).reshape(-1)
    _, counts = np.unique(y_true, return_counts=True)
    return float(counts.max() / counts.sum())


def source_entropy_norm(y_true):
    y_true = np.asarray(y_true).reshape(-1)
    _, counts = np.unique(y_true, return_counts=True)

    counts = counts.astype(np.float64)
    p = counts / counts.sum()
    p = p[p > 0]

    if len(p) <= 1:
        return 0.0

    return float(-np.sum(p * np.log(p)) / np.log(len(p)))


def deinterleaving_metrics(y_true, y_pred_raw):
    y_true = np.asarray(y_true).reshape(-1).astype(int)
    y_pred_raw = np.asarray(y_pred_raw).reshape(-1).astype(int)
    y_pred_eval = apply_noise_policy(y_pred_raw)

    out = {
        "v_measure": safe_float(v_measure_score(y_true, y_pred_eval)),
        "ari": safe_float(adjusted_rand_score(y_true, y_pred_eval)),
        "ami": safe_float(adjusted_mutual_info_score(y_true, y_pred_eval)),
        "homogeneity": safe_float(homogeneity_score(y_true, y_pred_eval)),
        "completeness": safe_float(completeness_score(y_true, y_pred_eval)),
    }

    out.update(efficient_pairwise_metrics(y_true, y_pred_eval))

    true_sources = np.unique(y_true)
    pred_clusters_raw = np.unique(y_pred_raw)
    pred_clusters_eval = np.unique(y_pred_eval)

    out["n_pulses"] = int(len(y_true))
    out["n_true_sources"] = int(len(true_sources))
    out["n_pred_clusters_raw_excluding_noise"] = int(len([c for c in pred_clusters_raw if c != -1]))
    out["n_pred_clusters_eval"] = int(len(pred_clusters_eval))
    out["has_noise_cluster"] = int(np.any(y_pred_raw == -1))
    out["noise_share"] = float(np.mean(y_pred_raw == -1))
    out["dominant_source_share"] = dominant_source_share(y_true)
    out["source_entropy_norm"] = source_entropy_norm(y_true)

    return out


# ============================================================
# TRAIN / EVAL
# ============================================================
def run_train_epoch(model, loader, optimizer):
    model.train()

    total_loss = 0.0
    total_n = 0

    for batch in tqdm(loader, desc="train", leave=False):
        x = batch["x"].to(DEVICE)  # [B, 1024, 8]
        y = batch["y"].to(DEVICE)  # [B, 1024]

        with torch.autocast(device_type=DEVICE_TYPE, enabled=USE_AMP):
            out = model(x)
            emb = out["emb"]
            gate_w = out["gate_w"]

            loss_con = supervised_contrastive_window_loss(
                emb,
                y,
                temperature=SUPCON_TEMPERATURE,
            )
            loss_gate = gate_balance_regularizer(gate_w)

            loss = loss_con + 0.02 * loss_gate

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        bs = x.size(0)
        total_loss += float(loss.item()) * bs
        total_n += bs

    return total_loss / max(total_n, 1)


@torch.no_grad()
def infer_embeddings(model, x):
    model.eval()
    x = x.to(DEVICE)

    with torch.autocast(device_type=DEVICE_TYPE, enabled=USE_AMP):
        out = model(x)
        emb = out["emb"]

    emb = F.normalize(emb.float(), dim=-1)
    return emb.cpu().numpy()


def evaluate_model_clustering(model, loader, split_name, max_batches=None):
    rows = []

    model.eval()

    for bidx, batch in enumerate(tqdm(loader, desc=f"eval_{split_name}", leave=False)):
        if max_batches is not None and bidx >= int(max_batches):
            break

        x = batch["x"]
        y = batch["y"].numpy()
        file_ids = list(batch["file_id"])
        window_indices = batch["window_index"].numpy()

        emb = infer_embeddings(model, x)  # [B, 1024, D]

        for i in range(emb.shape[0]):
            t0 = time.time()

            y_true = y[i]
            z = emb[i]

            y_pred = run_hdbscan(z)
            metrics = deinterleaving_metrics(y_true, y_pred)

            row = {
                "source": SOURCE_NAME,
                "split": split_name,
                "file_id": file_ids[i],
                "window_index": int(window_indices[i]),
                "cluster_method": "HDBSCAN_on_MoE_embeddings",
                "hdbscan_backend": HDBSCAN_BACKEND,
                "elapsed_sec": float(time.time() - t0),
            }

            row.update(metrics)
            rows.append(row)

    return pd.DataFrame(rows)


def summarize_metric_df(df, prefix=""):
    metrics = [
        "v_measure",
        "ari",
        "ami",
        "homogeneity",
        "completeness",
        "pairwise_f1",
        "pairwise_mcc",
        "n_true_sources",
        "n_pred_clusters_raw_excluding_noise",
        "noise_share",
        "dominant_source_share",
        "source_entropy_norm",
    ]

    out = {}

    for m in metrics:
        if m not in df.columns:
            continue

        vals = pd.to_numeric(df[m], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()

        if len(vals) == 0:
            out[f"{prefix}{m}_mean"] = np.nan
            out[f"{prefix}{m}_median"] = np.nan
            out[f"{prefix}{m}_std"] = np.nan
        else:
            out[f"{prefix}{m}_mean"] = float(vals.mean())
            out[f"{prefix}{m}_median"] = float(vals.median())
            out[f"{prefix}{m}_std"] = float(vals.std())

    return out


def train_one_model(n_experts, train_loader, val_loader, run_dir):
    groups = get_feature_groups(n_experts)

    model = PDWFeatureMoE(
        feature_groups=groups,
        input_dim=N_INPUT_FEATURES,
        hidden_dim=HIDDEN_DIM,
        emb_dim=EMB_DIM,
        drop=DROP,
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    best_score = -np.inf
    best_epoch = -1
    best_state = None
    bad_epochs = 0
    history = []

    for epoch in range(1, EPOCHS + 1):
        print(f"\nEpoch {epoch:03d}/{EPOCHS}")

        train_loss = run_train_epoch(model, train_loader, optimizer)

        val_df = evaluate_model_clustering(
            model,
            val_loader,
            split_name="validation",
        )

        val_summary = summarize_metric_df(val_df, prefix="val_")
        val_score = val_summary.get("val_v_measure_median", np.nan)

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_score_v_measure_median": val_score,
        }
        row.update(val_summary)
        history.append(row)

        print(
            f"train_loss={train_loss:.4f} | "
            f"val_v_measure_median={val_score:.4f} | "
            f"val_ari_median={val_summary.get('val_ari_median', np.nan):.4f} | "
            f"val_pairwise_mcc_median={val_summary.get('val_pairwise_mcc_median', np.nan):.4f}"
        )

        if np.isfinite(val_score) and val_score > best_score:
            best_score = float(val_score)
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            torch.save(best_state, run_dir / "best_model.pt")
            val_df.to_csv(run_dir / "best_val_window_metrics.csv", index=False, encoding="utf-8-sig")
            bad_epochs = 0
            print(f"[BEST] epoch={epoch} val_v_measure_median={best_score:.4f}")
        else:
            bad_epochs += 1
            print(f"[NO IMPROVEMENT] bad_epochs={bad_epochs}/{PATIENCE}")

        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False, encoding="utf-8-sig")

        if bad_epochs >= PATIENCE:
            print("Early stopping triggered.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_epoch, best_score, pd.DataFrame(history)


# ============================================================
# PLOTS
# ============================================================
def plot_comparison(results_df):
    metrics = [
        "test_v_measure_median",
        "test_ari_median",
        "test_ami_median",
        "test_pairwise_mcc_median",
    ]

    for metric in metrics:
        if metric not in results_df.columns:
            continue

        plt.figure(figsize=(7, 5))
        plt.bar(results_df["n_experts"].astype(str), results_df[metric].values)
        plt.xlabel("N experts")
        plt.ylabel(metric)
        plt.title(metric)
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"bar_{metric}.png", bbox_inches="tight")
        plt.close()


# ============================================================
# SINGLE EXPERIMENT
# ============================================================
def run_single_experiment(
    n_experts,
    train_tasks,
    val_tasks,
    test_tasks,
):
    run_name = f"moe_pdw_deinterleaving_N{n_experts}"
    run_dir = OUT_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    groups = get_feature_groups(n_experts)
    group_desc = describe_feature_groups(groups)

    print("\n" + "=" * 80)
    print(f"RUNNING MoE PDW DEINTERLEAVING | N_EXPERTS={n_experts}")
    print("=" * 80)
    print(f"Feature groups: {group_desc}")

    save_json(
        {
            "n_experts": n_experts,
            "feature_groups_indices": groups,
            "feature_groups_names": group_desc,
            "feature_names": FEATURE_NAMES,
        },
        run_dir / "feature_groups.json",
    )

    train_ds = PDWWindowDataset(train_tasks, train_mode=True)
    val_ds = PDWWindowDataset(val_tasks, train_mode=False)
    test_ds = PDWWindowDataset(test_tasks, train_mode=False)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        drop_last=False,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        drop_last=False,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        drop_last=False,
    )

    model, best_epoch, best_val_score, history_df = train_one_model(
        n_experts=n_experts,
        train_loader=train_loader,
        val_loader=val_loader,
        run_dir=run_dir,
    )

    print("\nEvaluating best model on validation...")
    val_df = evaluate_model_clustering(model, val_loader, split_name="validation")
    val_df.to_csv(run_dir / "final_val_window_metrics.csv", index=False, encoding="utf-8-sig")

    print("\nEvaluating best model on test...")
    test_df = evaluate_model_clustering(model, test_loader, split_name="test")
    test_df.to_csv(run_dir / "test_window_metrics.csv", index=False, encoding="utf-8-sig")

    val_summary = summarize_metric_df(val_df, prefix="val_")
    test_summary = summarize_metric_df(test_df, prefix="test_")

    result = {
        "model_name": "PDWFeatureMoE",
        "n_experts": n_experts,
        "feature_groups": json.dumps(group_desc, ensure_ascii=False),
        "best_epoch": best_epoch,
        "best_val_v_measure_median": best_val_score,
        "n_train_windows": int(len(train_tasks)),
        "n_val_windows": int(len(val_tasks)),
        "n_test_windows": int(len(test_tasks)),
        "emb_dim": EMB_DIM,
        "hidden_dim": HIDDEN_DIM,
        "drop": DROP,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "supcon_temperature": SUPCON_TEMPERATURE,
        "hdbscan_min_cluster_size": HDBSCAN_MIN_CLUSTER_SIZE,
        "hdbscan_min_samples": HDBSCAN_MIN_SAMPLES,
    }

    result.update(val_summary)
    result.update(test_summary)

    save_json(result, run_dir / "results.json")
    pd.DataFrame([result]).to_csv(run_dir / "summary_result.csv", index=False, encoding="utf-8-sig")

    return result


# ============================================================
# MAIN
# ============================================================
def main():
    print("=" * 80)
    print("TSRD FEATURE-SPECIALIZED MoE FOR PDW DEINTERLEAVING")
    print("=" * 80)
    print(f"Device: {DEVICE}")
    print(f"HDBSCAN backend: {HDBSCAN_BACKEND}")

    # --------------------------------------------------------
    # Index files
    # --------------------------------------------------------
    print("\n" + "=" * 80)
    print("INDEXING FILES")
    print("=" * 80)

    train_index = collect_file_index("train", DATA_DIRS["train"])
    val_index = collect_file_index("validation", DATA_DIRS["validation"])
    test_index = collect_file_index("test", DATA_DIRS["test"])

    train_index.to_csv(TABLES_DIR / "file_index_train.csv", index=False, encoding="utf-8-sig")
    val_index.to_csv(TABLES_DIR / "file_index_validation.csv", index=False, encoding="utf-8-sig")
    test_index.to_csv(TABLES_DIR / "file_index_test.csv", index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # Select windows
    # --------------------------------------------------------
    print("\n" + "=" * 80)
    print("SELECTING WINDOW TASKS")
    print("=" * 80)

    train_tasks = select_window_tasks(
        train_index,
        max_windows=MAX_TRAIN_WINDOWS,
        mode=TRAIN_WINDOW_SELECTION_MODE,
    )

    val_tasks = select_window_tasks(
        val_index,
        max_windows=MAX_VAL_WINDOWS,
        mode=VAL_WINDOW_SELECTION_MODE,
    )

    test_tasks = select_window_tasks(
        test_index,
        max_windows=MAX_TEST_WINDOWS,
        mode=TEST_WINDOW_SELECTION_MODE,
    )

    print(f"Train windows     : {len(train_tasks)}")
    print(f"Validation windows: {len(val_tasks)}")
    print(f"Test windows      : {len(test_tasks)}")

    train_tasks.to_csv(TABLES_DIR / "tasks_train.csv", index=False, encoding="utf-8-sig")
    val_tasks.to_csv(TABLES_DIR / "tasks_validation.csv", index=False, encoding="utf-8-sig")
    test_tasks.to_csv(TABLES_DIR / "tasks_test.csv", index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # Run sweep
    # --------------------------------------------------------
    all_results = []

    for n_experts in EXPERTS_LIST:
        set_seed(SEED + int(n_experts))

        res = run_single_experiment(
            n_experts=n_experts,
            train_tasks=train_tasks,
            val_tasks=val_tasks,
            test_tasks=test_tasks,
        )

        all_results.append(res)

        results_df_partial = pd.DataFrame(all_results)
        results_df_partial.to_csv(
            OUT_DIR / "comparison_all_results_partial.csv",
            index=False,
            encoding="utf-8-sig",
        )

    results_df = pd.DataFrame(all_results)

    comparison_path = OUT_DIR / "comparison_all_results.csv"
    results_df.to_csv(comparison_path, index=False, encoding="utf-8-sig")

    results_df.sort_values("test_v_measure_median", ascending=False).to_csv(
        OUT_DIR / "comparison_sorted_by_test_v_measure.csv",
        index=False,
        encoding="utf-8-sig",
    )

    results_df.sort_values("test_pairwise_mcc_median", ascending=False).to_csv(
        OUT_DIR / "comparison_sorted_by_test_pairwise_mcc.csv",
        index=False,
        encoding="utf-8-sig",
    )

    plot_comparison(results_df)

    # --------------------------------------------------------
    # Config
    # --------------------------------------------------------
    config = {
        "root": str(ROOT),
        "data_dirs": {k: str(v) for k, v in DATA_DIRS.items()},
        "source_name": SOURCE_NAME,
        "window_size": WINDOW_SIZE,
        "max_train_windows": MAX_TRAIN_WINDOWS,
        "max_val_windows": MAX_VAL_WINDOWS,
        "max_test_windows": MAX_TEST_WINDOWS,
        "train_window_selection_mode": TRAIN_WINDOW_SELECTION_MODE,
        "val_window_selection_mode": VAL_WINDOW_SELECTION_MODE,
        "test_window_selection_mode": TEST_WINDOW_SELECTION_MODE,
        "experts_list": EXPERTS_LIST,
        "feature_names": FEATURE_NAMES,
        "emb_dim": EMB_DIM,
        "hidden_dim": HIDDEN_DIM,
        "drop": DROP,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "supcon_temperature": SUPCON_TEMPERATURE,
        "hdbscan_backend": HDBSCAN_BACKEND,
        "hdbscan_min_cluster_size": HDBSCAN_MIN_CLUSTER_SIZE,
        "hdbscan_min_samples": HDBSCAN_MIN_SAMPLES,
        "noise_policy_for_metrics": NOISE_POLICY_FOR_METRICS,
    }

    save_json(config, OUT_DIR / "config.json")

    # --------------------------------------------------------
    # Console output
    # --------------------------------------------------------
    print("\n" + "=" * 80)
    print("FINAL COMPARISON - SORTED BY TEST V-MEASURE")
    print("=" * 80)

    show_cols = [
        "n_experts",
        "best_epoch",
        "best_val_v_measure_median",
        "test_v_measure_median",
        "test_ari_median",
        "test_ami_median",
        "test_homogeneity_median",
        "test_completeness_median",
        "test_pairwise_f1_median",
        "test_pairwise_mcc_median",
        "test_n_true_sources_median",
        "test_n_pred_clusters_raw_excluding_noise_median",
        "test_noise_share_median",
    ]

    show_cols = [c for c in show_cols if c in results_df.columns]

    print(
        results_df[show_cols]
        .sort_values("test_v_measure_median", ascending=False)
        .to_string(index=False)
    )

    print("\n" + "=" * 80)
    print("OUTPUT FILES")
    print("=" * 80)
    print(f"Comparison all results : {comparison_path}")
    print(f"Sorted by V-measure    : {OUT_DIR / 'comparison_sorted_by_test_v_measure.csv'}")
    print(f"Sorted by MCC          : {OUT_DIR / 'comparison_sorted_by_test_pairwise_mcc.csv'}")
    print(f"Tables directory       : {TABLES_DIR}")
    print(f"Plots directory        : {PLOTS_DIR}")
    print(f"Output directory       : {OUT_DIR}")

    print("\nDone.")


if __name__ == "__main__":
    main()

TSRD FEATURE-SPECIALIZED MoE FOR PDW DEINTERLEAVING
Device: cpu
HDBSCAN backend: hdbscan

INDEXING FILES
[train] H5 files found: 2500


index_train: 100%|██████████| 2500/2500 [02:22<00:00, 17.50file/s]


[validation] H5 files found: 250


index_validation: 100%|██████████| 250/250 [00:13<00:00, 19.00file/s]


[test] H5 files found: 250


...

train_loss=6.2902 | val_v_measure_median=0.9597 | val_ari_median=0.9985 | val_pairwise_mcc_median=0.9985
[NO IMPROVEMENT] bad_epochs=1/5

Epoch 019/20


train_loss=6.2896 | val_v_measure_median=0.9603 | val_ari_median=0.9985 | val_pairwise_mcc_median=0.9986
[BEST] epoch=19 val_v_measure_median=0.9603

Epoch 020/20


train_loss=6.2892 | val_v_measure_median=0.9601 | val_ari_median=0.9985 | val_pairwise_mcc_median=0.9985
[NO IMPROVEMENT] bad_epochs=1/5

Evaluating best model on validation...



Evaluating best model on test...



FINAL COMPARISON - SORTED BY TEST V-MEASURE
 n_experts  best_epoch  best_val_v_measure_median  test_v_measure_median  test_ari_median  test_ami_median  test_homogeneity_median  test_completeness_median  test_pairwise_f1_median  test_pairwise_mcc_median  test_n_true_sources_median  test_n_pred_clusters_raw_excluding_noise_median  test_noise_share_median
         5          19                   0.960337               0.957959         0.998570         0.956337                 0.946902                  1.000000                 0.999730                  0.998571                        12.0                                              5.0                 0.006836
         3          19                   0.957249               0.956053         0.998439         0.954321                 0.943365                  1.000000                 0.999685                  0.998440                        12.0                                              5.0                 0.005859
         4          20

# MOE COM ENGENHARIA DE ATRIBUTOS

In [12]:
# ============================================================
# proposed_moe_pdw_raw_vs_engineered_sweep.py
# ============================================================
# TSRD - Feature-Specialized Mixture-of-Experts for radar
# pulse deinterleaving: RAW PDWs vs ENGINEERED PDW features.
#
# Task:
#   Separate interleaved radar pulses into emitter clusters.
#
# Inputs:
#   D:\Fusion\turing_synthetic_radar_dataset\archive\train
#   D:\Fusion\turing_synthetic_radar_dataset\archive\validation
#   D:\Fusion\turing_synthetic_radar_dataset\archive\test
#
# Each .h5:
#   data   : [n_pulses, 5]
#   labels : [n_pulses, 1]
#
# Raw PDW columns:
#   0 -> ToA_us
#   1 -> Frequency_MHz
#   2 -> PulseWidth_us
#   3 -> AoA_deg
#   4 -> Amplitude_dB
#
# Comparison:
#   1) raw_pdw_moe:
#        Minimal finite sanitation + per-window RobustScaler
#
#   2) engineered_pdw_moe:
#        Derived PDW features:
#           toa_rel_norm
#           delta_toa_log
#           frequency_norm
#           delta_frequency_abs_norm
#           pulsewidth_log
#           aoa_sin
#           aoa_cos
#           delta_aoa_abs_norm
#           amplitude_norm
#           amplitude_missing
#
# Method:
#   train:
#       PDW windows -> MoE -> pulse embeddings
#       supervised contrastive loss within each window
#
#   validation/test:
#       PDW windows -> embeddings -> HDBSCAN -> clusters
#       compare clusters with true labels
#
# Important:
#   Labels are used for training and evaluation, but HDBSCAN does
#   not receive the true number of emitters.
#
# Sweep:
#   N_EXPERTS in {2, 3, 4, 5}
#   PREPROCESS_MODES in {raw_pdw_moe, engineered_pdw_moe}
#
# Outputs:
#   D:\Fusion\turing_synthetic_radar_dataset\benchmark_moe_pdw_raw_vs_engineered_sweep
# ============================================================

from pathlib import Path
import sys
import json
import math
import time
import random
import subprocess
import warnings
warnings.filterwarnings("ignore")


# ============================================================
# INSTALL / IMPORTS
# ============================================================
def ensure_package(import_name, pip_name=None):
    try:
        __import__(import_name)
    except Exception:
        pkg = pip_name if pip_name is not None else import_name
        print(f"[INFO] Installing missing package: {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


ensure_package("numpy")
ensure_package("pandas")
ensure_package("matplotlib")
ensure_package("h5py")
ensure_package("sklearn", "scikit-learn")
ensure_package("torch")
ensure_package("tqdm")
ensure_package("tabulate")

try:
    ensure_package("hdbscan")
    import hdbscan
    HDBSCAN_BACKEND = "hdbscan"
except Exception:
    hdbscan = None
    HDBSCAN_BACKEND = "sklearn"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    v_measure_score,
    adjusted_rand_score,
    adjusted_mutual_info_score,
    homogeneity_score,
    completeness_score,
)

try:
    from sklearn.cluster import HDBSCAN as SKHDBSCAN
except Exception:
    SKHDBSCAN = None


# ============================================================
# PATHS
# ============================================================
ROOT = Path(r"D:\Fusion\turing_synthetic_radar_dataset")

DATA_DIRS = {
    "train": ROOT / "archive" / "train",
    "validation": ROOT / "archive" / "validation",
    "test": ROOT / "archive" / "test",
}

SOURCE_NAME = "archive"

OUT_DIR = ROOT / "benchmark_moe_pdw_raw_vs_engineered_sweep"
TABLES_DIR = OUT_DIR / "tables"
PLOTS_DIR = OUT_DIR / "plots"

OUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# CONFIG
# ============================================================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE_TYPE = "cuda" if DEVICE == "cuda" else "cpu"
USE_AMP = DEVICE == "cuda"

WINDOW_SIZE = 1024
DROP_INCOMPLETE_WINDOWS = True

# Para teste rápido:
MAX_TRAIN_WINDOWS = 8000
MAX_VAL_WINDOWS = 2000
MAX_TEST_WINDOWS = 5000

# Para avaliação final completa no teste, use:
# MAX_TEST_WINDOWS = None
# TEST_WINDOW_SELECTION_MODE = "sequential"

TRAIN_WINDOW_SELECTION_MODE = "random"
VAL_WINDOW_SELECTION_MODE = "random"
TEST_WINDOW_SELECTION_MODE = "random"

MAX_FILES_PER_SPLIT = None

# Comparação principal
PREPROCESS_MODES = [
    "raw_pdw_moe",
    "engineered_pdw_moe",
]

# Sweep
EXPERTS_LIST = [2, 3, 4, 5]

# Model
EMB_DIM = 64
HIDDEN_DIM = 128
DROP = 0.15

# Training
BATCH_SIZE = 4
EPOCHS = 20
LR = 2.0e-4
WEIGHT_DECAY = 1.0e-4
PATIENCE = 5
GRAD_CLIP = 2.0
NUM_WORKERS = 0

# Contrastive loss
SUPCON_TEMPERATURE = 0.10
GATE_BALANCE_WEIGHT = 0.02

# HDBSCAN over learned embeddings
HDBSCAN_MIN_CLUSTER_SIZE = 5
HDBSCAN_MIN_SAMPLES = 3
HDBSCAN_CLUSTER_SELECTION_EPSILON = 0.0
HDBSCAN_CLUSTER_SELECTION_METHOD = "eom"
HDBSCAN_ALLOW_SINGLE_CLUSTER = False

# Metrics policy
NOISE_POLICY_FOR_METRICS = "as_cluster"  # or "unique_noise"

RAW_FEATURE_NAMES = [
    "ToA_us",
    "Frequency_MHz",
    "PulseWidth_us",
    "AoA_deg",
    "Amplitude_dB",
]

ENGINEERED_FEATURE_NAMES = [
    "toa_rel_norm",
    "delta_toa_log",
    "frequency_norm",
    "delta_frequency_abs_norm",
    "pulsewidth_log",
    "aoa_sin",
    "aoa_cos",
    "delta_aoa_abs_norm",
    "amplitude_norm",
    "amplitude_missing",
]

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 180


# ============================================================
# REPRODUCIBILITY
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
rng = np.random.default_rng(SEED)


# ============================================================
# HELPERS
# ============================================================
def save_json(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def h5_has_dataset(f, name):
    try:
        return isinstance(f[name], h5py.Dataset)
    except Exception:
        return False


def comb2(x):
    x = np.asarray(x, dtype=np.int64)
    return x * (x - 1) // 2


def safe_float(x):
    try:
        if np.isfinite(x):
            return float(x)
        return float("nan")
    except Exception:
        return float("nan")


def finite_median(v, default=0.0):
    v = np.asarray(v, dtype=np.float32)
    mask = np.isfinite(v)
    if mask.any():
        return float(np.nanmedian(v[mask]))
    return float(default)


def fill_nonfinite_with_median(v, default=0.0):
    v = np.asarray(v, dtype=np.float32).copy()
    med = finite_median(v, default=default)
    v[~np.isfinite(v)] = med
    return v


def angular_abs_diff_deg(a):
    a = np.asarray(a, dtype=np.float32)
    prev = np.concatenate([[a[0]], a[:-1]])
    diff = (a - prev + 180.0) % 360.0 - 180.0
    return np.abs(diff).astype(np.float32)


def safe_mode_name(mode):
    return str(mode).replace("/", "_").replace("\\", "_").replace(" ", "_")


# ============================================================
# FILE INDEX AND WINDOW TASKS
# ============================================================
def collect_file_index(split_name: str, folder: Path):
    rows = []

    if not folder.exists():
        raise FileNotFoundError(f"Missing folder: {folder}")

    h5_files = sorted(folder.rglob("*.h5"))

    if MAX_FILES_PER_SPLIT is not None:
        h5_files = h5_files[:int(MAX_FILES_PER_SPLIT)]

    print(f"[{split_name}] H5 files found: {len(h5_files)}")

    for p in tqdm(h5_files, desc=f"index_{split_name}", unit="file"):
        row = {
            "source": SOURCE_NAME,
            "split": split_name,
            "file_id": p.stem,
            "file_name": p.name,
            "path": str(p),
            "status": "ok",
            "n_pulses": 0,
            "n_windows": 0,
            "error": None,
        }

        try:
            with h5py.File(p, "r") as f:
                if not h5_has_dataset(f, "data") or not h5_has_dataset(f, "labels"):
                    row["status"] = "missing_data_or_labels"
                    rows.append(row)
                    continue

                n = int(f["data"].shape[0])
                row["n_pulses"] = n

                if n <= 0:
                    row["status"] = "empty"
                    rows.append(row)
                    continue

                if DROP_INCOMPLETE_WINDOWS:
                    n_windows = n // WINDOW_SIZE
                else:
                    n_windows = int(math.ceil(n / WINDOW_SIZE))

                row["n_windows"] = int(n_windows)

        except Exception as e:
            row["status"] = "error"
            row["error"] = str(e)

        rows.append(row)

    return pd.DataFrame(rows)


def select_window_tasks(file_index_df: pd.DataFrame, max_windows=None, mode="random"):
    valid = file_index_df[
        (file_index_df["status"] == "ok") &
        (file_index_df["n_windows"] > 0)
    ].copy().reset_index(drop=True)

    if len(valid) == 0:
        return pd.DataFrame()

    if mode == "sequential":
        rows = []
        done = 0

        for _, r in valid.iterrows():
            n_w = int(r["n_windows"])

            for w in range(n_w):
                if max_windows is not None and done >= int(max_windows):
                    break

                rr = r.to_dict()
                rr["window_index"] = int(w)
                rows.append(rr)
                done += 1

            if max_windows is not None and done >= int(max_windows):
                break

        return pd.DataFrame(rows)

    if mode != "random":
        raise ValueError(f"Unknown window selection mode: {mode}")

    total_windows = int(valid["n_windows"].sum())

    if max_windows is None:
        sample_n = total_windows
    else:
        sample_n = min(int(max_windows), total_windows)

    if sample_n <= 0:
        return pd.DataFrame()

    global_indices = rng.choice(total_windows, size=sample_n, replace=False)
    global_indices = np.sort(global_indices)

    n_windows = valid["n_windows"].values.astype(np.int64)
    cum_end = np.cumsum(n_windows)
    cum_start = cum_end - n_windows

    file_pos = np.searchsorted(cum_end, global_indices, side="right")
    local_win = global_indices - cum_start[file_pos]

    tasks = valid.iloc[file_pos].copy()
    tasks["window_index"] = local_win.astype(int)
    tasks = tasks.sort_values(["path", "window_index"]).reset_index(drop=True)

    return tasks


# ============================================================
# PDW PREPROCESSING
# ============================================================
def sanitize_raw_pdws_for_moe(x):
    """
    RAW MoE input:
      - original 5 PDW columns
      - minimal finite sanitation
      - per-window RobustScaler

    Output shape:
      [N, 5]
    """
    x = np.asarray(x, dtype=np.float32).copy()

    if x.ndim != 2 or x.shape[1] < 5:
        raise ValueError(f"Expected [n_pulses, >=5], got {x.shape}")

    x = x[:, :5]

    # Amplitude may contain inf/non-finite in archive/stare.
    amp = x[:, 4].copy()
    amp[~np.isfinite(amp)] = -200.0
    x[:, 4] = amp

    for j in range(5):
        col = x[:, j].copy()
        if not np.isfinite(col).all():
            med = finite_median(col, default=0.0)
            col[~np.isfinite(col)] = med
            x[:, j] = col

    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    # Robust scaling per window is important for neural training.
    try:
        scaler = RobustScaler()
        z = scaler.fit_transform(x).astype(np.float32)
    except Exception:
        z = x.astype(np.float32)

    z = np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0)
    return z.astype(np.float32)


def engineer_pdw_features_for_moe(x):
    """
    ENGINEERED MoE input.

    Raw:
      0 ToA_us
      1 Frequency_MHz
      2 PulseWidth_us
      3 AoA_deg
      4 Amplitude_dB

    Engineered:
      0 toa_rel_norm
      1 delta_toa_log
      2 frequency_norm
      3 delta_frequency_abs_norm
      4 pulsewidth_log
      5 aoa_sin
      6 aoa_cos
      7 delta_aoa_abs_norm
      8 amplitude_norm
      9 amplitude_missing

    Output shape:
      [N, 10]
    """
    x = np.asarray(x, dtype=np.float32)

    if x.ndim != 2 or x.shape[1] < 5:
        raise ValueError(f"Expected [n_pulses, >=5], got {x.shape}")

    toa = fill_nonfinite_with_median(x[:, 0], default=0.0)
    freq = fill_nonfinite_with_median(x[:, 1], default=0.0)
    pw = fill_nonfinite_with_median(x[:, 2], default=0.0)
    aoa = fill_nonfinite_with_median(x[:, 3], default=0.0)

    amp = x[:, 4].astype(np.float32).copy()
    amp_missing = (~np.isfinite(amp)).astype(np.float32)
    amp[~np.isfinite(amp)] = -200.0
    amp = fill_nonfinite_with_median(amp, default=-200.0)

    # Relative ToA
    toa_rel = toa - np.min(toa)
    span = float(np.max(toa_rel) - np.min(toa_rel))
    if span > 0:
        toa_rel_norm = toa_rel / (span + 1e-9)
    else:
        toa_rel_norm = np.zeros_like(toa_rel, dtype=np.float32)

    # PRI-like delta ToA
    dtoa = np.diff(toa, prepend=toa[0])
    dtoa = np.maximum(dtoa, 0.0)
    dtoa_log = np.log1p(dtoa)

    # Frequency
    frequency_norm = freq / 18000.0

    # Frequency transition
    delta_frequency = np.abs(np.diff(freq, prepend=freq[0]))
    delta_frequency_abs_norm = delta_frequency / 18000.0

    # Pulse width
    pw = np.maximum(pw, 0.0)
    pulsewidth_log = np.log1p(pw)

    # AoA circular representation
    aoa_rad = np.deg2rad(aoa)
    aoa_sin = np.sin(aoa_rad)
    aoa_cos = np.cos(aoa_rad)

    # AoA transition
    delta_aoa_abs_norm = angular_abs_diff_deg(aoa) / 180.0

    # Amplitude
    amplitude_norm = (amp + 220.0) / 260.0

    feats = np.stack(
        [
            toa_rel_norm,
            dtoa_log,
            frequency_norm,
            delta_frequency_abs_norm,
            pulsewidth_log,
            aoa_sin,
            aoa_cos,
            delta_aoa_abs_norm,
            amplitude_norm,
            amp_missing,
        ],
        axis=1,
    ).astype(np.float32)

    feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)

    z = feats.copy()

    # Scale continuous features only. Keep amplitude_missing binary.
    continuous_cols = list(range(9))
    binary_cols = [9]

    try:
        scaler = RobustScaler()
        z[:, continuous_cols] = scaler.fit_transform(z[:, continuous_cols])
    except Exception:
        z[:, continuous_cols] = feats[:, continuous_cols]

    z[:, binary_cols] = feats[:, binary_cols]

    z = np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0)
    return z.astype(np.float32)


def preprocess_pdw_window(x_raw, preprocess_mode):
    if preprocess_mode == "raw_pdw_moe":
        return sanitize_raw_pdws_for_moe(x_raw)

    if preprocess_mode == "engineered_pdw_moe":
        return engineer_pdw_features_for_moe(x_raw)

    raise ValueError(f"Unknown preprocess_mode: {preprocess_mode}")


def get_feature_names(preprocess_mode):
    if preprocess_mode == "raw_pdw_moe":
        return RAW_FEATURE_NAMES

    if preprocess_mode == "engineered_pdw_moe":
        return ENGINEERED_FEATURE_NAMES

    raise ValueError(f"Unknown preprocess_mode: {preprocess_mode}")


# ============================================================
# FEATURE GROUPS FOR EXPERT SWEEP
# ============================================================
def get_feature_groups(preprocess_mode, n_experts):
    """
    Returns feature-specialized expert groups for each preprocessing mode.
    """

    if preprocess_mode == "raw_pdw_moe":
        # Raw indices:
        # 0 ToA_us
        # 1 Frequency_MHz
        # 2 PulseWidth_us
        # 3 AoA_deg
        # 4 Amplitude_dB
        if n_experts == 2:
            return [
                [0, 2],        # time + width
                [1, 3, 4],     # frequency + angle + amplitude
            ]

        if n_experts == 3:
            return [
                [0],           # time
                [1, 2],        # frequency + width
                [3, 4],        # angle + amplitude
            ]

        if n_experts == 4:
            return [
                [0],           # time
                [1],           # frequency
                [2],           # pulse width
                [3, 4],        # angle + amplitude
            ]

        if n_experts == 5:
            return [
                [0],           # time
                [1],           # frequency
                [2],           # pulse width
                [3],           # angle
                [4],           # amplitude
            ]

    if preprocess_mode == "engineered_pdw_moe":
        # Engineered indices:
        # 0 toa_rel_norm
        # 1 delta_toa_log
        # 2 frequency_norm
        # 3 delta_frequency_abs_norm
        # 4 pulsewidth_log
        # 5 aoa_sin
        # 6 aoa_cos
        # 7 delta_aoa_abs_norm
        # 8 amplitude_norm
        # 9 amplitude_missing
        if n_experts == 2:
            return [
                [0, 1, 4],             # timing + PRI + pulse width
                [2, 3, 5, 6, 7, 8, 9], # frequency + AoA + amplitude
            ]

        if n_experts == 3:
            return [
                [0, 1],                # timing
                [2, 3, 4],             # frequency + pulse width
                [5, 6, 7, 8, 9],       # AoA + amplitude
            ]

        if n_experts == 4:
            return [
                [0, 1],                # timing
                [2, 3],                # frequency
                [4],                   # pulse width
                [5, 6, 7, 8, 9],       # AoA + amplitude
            ]

        if n_experts == 5:
            return [
                [0, 1],                # timing
                [2, 3],                # frequency and frequency transition
                [4],                   # pulse width
                [5, 6, 7],             # AoA
                [8, 9],                # amplitude + missingness
            ]

    raise ValueError(f"Unsupported preprocess_mode={preprocess_mode}, n_experts={n_experts}")


def describe_feature_groups(preprocess_mode, groups):
    names = get_feature_names(preprocess_mode)
    return [[names[i] for i in g] for g in groups]


# ============================================================
# DATASET
# ============================================================
class PDWWindowDataset(Dataset):
    def __init__(self, tasks_df: pd.DataFrame, preprocess_mode: str, train_mode=False):
        self.tasks = tasks_df.reset_index(drop=True)
        self.preprocess_mode = preprocess_mode
        self.train_mode = bool(train_mode)

    def __len__(self):
        return len(self.tasks)

    def __getitem__(self, idx):
        r = self.tasks.iloc[idx]
        path = Path(r["path"])
        w = int(r["window_index"])

        start = w * WINDOW_SIZE
        end = start + WINDOW_SIZE

        with h5py.File(path, "r") as f:
            x_raw = np.asarray(f["data"][start:end, :5], dtype=np.float32)
            y = np.asarray(f["labels"][start:end]).reshape(-1).astype(np.int64)

        x = preprocess_pdw_window(x_raw, self.preprocess_mode)

        return {
            "x": torch.tensor(x, dtype=torch.float32),
            "y": torch.tensor(y, dtype=torch.long),
            "file_id": str(r["file_id"]),
            "split": str(r["split"]),
            "window_index": int(w),
        }


# ============================================================
# MODEL
# ============================================================
class FeatureExpert(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, emb_dim=64, drop=0.15):
        super().__init__()

        self.point = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
        )

        self.temporal = nn.Sequential(
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2, groups=1),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2, groups=1),
            nn.GELU(),
        )

        self.norm = nn.LayerNorm(hidden_dim)

        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, emb_dim),
            nn.GELU(),
        )

    def forward(self, x):
        # x: [B, N, Fin]
        h = self.point(x)             # [B, N, H]
        ht = h.transpose(1, 2)        # [B, H, N]
        ht = self.temporal(ht)        # [B, H, N]
        ht = ht.transpose(1, 2)       # [B, N, H]
        h = self.norm(h + ht)
        z = self.proj(h)              # [B, N, D]
        return z


class PDWFeatureMoE(nn.Module):
    def __init__(self, feature_groups, input_dim, hidden_dim=128, emb_dim=64, drop=0.15):
        super().__init__()

        self.feature_groups = [list(g) for g in feature_groups]
        self.n_experts = len(self.feature_groups)
        self.input_dim = int(input_dim)
        self.emb_dim = emb_dim

        self.experts = nn.ModuleList([
            FeatureExpert(
                in_dim=len(g),
                hidden_dim=hidden_dim,
                emb_dim=emb_dim,
                drop=drop,
            )
            for g in self.feature_groups
        ])

        self.gate = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(hidden_dim, self.n_experts),
        )

        self.out_norm = nn.LayerNorm(emb_dim)

    def forward(self, x):
        # x: [B, N, F]
        gate_logits = self.gate(x)                    # [B, N, E]
        gate_w = torch.softmax(gate_logits, dim=-1)   # [B, N, E]

        expert_outs = []
        for expert, group in zip(self.experts, self.feature_groups):
            xg = x[:, :, group]
            zg = expert(xg)
            expert_outs.append(zg)

        E = torch.stack(expert_outs, dim=2)           # [B, N, E, D]
        z = (E * gate_w.unsqueeze(-1)).sum(dim=2)     # [B, N, D]
        z = self.out_norm(z)

        return {
            "emb": z,
            "gate_w": gate_w,
        }


# ============================================================
# SUPERVISED CONTRASTIVE LOSS
# ============================================================
def supervised_contrastive_window_loss(emb, labels, temperature=0.10):
    """
    emb:    [B, N, D]
    labels: [B, N]

    Labels are meaningful only within each window.
    Therefore, positives are computed independently per window.
    """
    B, N, D = emb.shape
    emb = F.normalize(emb, dim=-1)

    losses = []

    for b in range(B):
        z = emb[b]        # [N, D]
        y = labels[b]     # [N]

        sim = torch.matmul(z, z.T) / temperature  # [N, N]

        self_mask = torch.eye(N, device=emb.device, dtype=torch.bool)
        pos_mask = (y[:, None] == y[None, :]) & (~self_mask)
        valid_anchor = pos_mask.sum(dim=1) > 0

        if valid_anchor.sum() == 0:
            continue

        sim = sim.masked_fill(self_mask, -1e9)
        log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True)

        pos_log_prob = (log_prob * pos_mask.float()).sum(dim=1) / (
            pos_mask.float().sum(dim=1) + 1e-9
        )

        loss_b = -pos_log_prob[valid_anchor].mean()
        losses.append(loss_b)

    if len(losses) == 0:
        return emb.sum() * 0.0

    return torch.stack(losses).mean()


def gate_balance_regularizer(gate_w):
    # gate_w: [B, N, E]
    avg = gate_w.mean(dim=(0, 1))
    target = torch.full_like(avg, 1.0 / avg.numel())
    return ((avg - target) ** 2).mean()


# ============================================================
# HDBSCAN AND METRICS
# ============================================================
def run_hdbscan(X):
    X = np.asarray(X, dtype=np.float32)

    if HDBSCAN_BACKEND == "hdbscan" and hdbscan is not None:
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
            min_samples=HDBSCAN_MIN_SAMPLES,
            metric="euclidean",
            cluster_selection_epsilon=HDBSCAN_CLUSTER_SELECTION_EPSILON,
            cluster_selection_method=HDBSCAN_CLUSTER_SELECTION_METHOD,
            allow_single_cluster=HDBSCAN_ALLOW_SINGLE_CLUSTER,
        )
        return clusterer.fit_predict(X).astype(int)

    if SKHDBSCAN is not None:
        kwargs = dict(
            min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
            metric="euclidean",
            cluster_selection_epsilon=HDBSCAN_CLUSTER_SELECTION_EPSILON,
            allow_single_cluster=HDBSCAN_ALLOW_SINGLE_CLUSTER,
        )

        if HDBSCAN_MIN_SAMPLES is not None:
            kwargs["min_samples"] = HDBSCAN_MIN_SAMPLES

        clusterer = SKHDBSCAN(**kwargs)
        return clusterer.fit_predict(X).astype(int)

    raise RuntimeError("No HDBSCAN backend available. Try: pip install hdbscan")


def apply_noise_policy(y_pred):
    y_pred = np.asarray(y_pred).reshape(-1).astype(int).copy()

    if NOISE_POLICY_FOR_METRICS == "as_cluster":
        return y_pred

    if NOISE_POLICY_FOR_METRICS == "unique_noise":
        noise_idx = np.where(y_pred == -1)[0]
        if len(noise_idx) == 0:
            return y_pred

        max_cluster = int(np.max(y_pred)) if len(y_pred) > 0 else 0
        next_label = max_cluster + 1

        for k, idx in enumerate(noise_idx):
            y_pred[idx] = next_label + k

        return y_pred

    raise ValueError(f"Unknown NOISE_POLICY_FOR_METRICS: {NOISE_POLICY_FOR_METRICS}")


def efficient_pairwise_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    n = len(y_true)
    total_pairs = n * (n - 1) // 2

    if n < 2 or total_pairs <= 0:
        return {
            "pairwise_f1": np.nan,
            "pairwise_mcc": np.nan,
            "pairwise_tp": 0,
            "pairwise_fp": 0,
            "pairwise_fn": 0,
            "pairwise_tn": 0,
        }

    true_vals, true_inv = np.unique(y_true, return_inverse=True)
    pred_vals, pred_inv = np.unique(y_pred, return_inverse=True)

    contingency = np.zeros((len(true_vals), len(pred_vals)), dtype=np.int64)
    np.add.at(contingency, (true_inv, pred_inv), 1)

    tp = int(comb2(contingency).sum())

    true_sizes = contingency.sum(axis=1)
    pred_sizes = contingency.sum(axis=0)

    true_pairs = int(comb2(true_sizes).sum())
    pred_pairs = int(comb2(pred_sizes).sum())

    fp = int(pred_pairs - tp)
    fn = int(true_pairs - tp)
    tn = int(total_pairs - tp - fp - fn)

    denom_f1 = 2 * tp + fp + fn
    pairwise_f1 = float((2 * tp) / denom_f1) if denom_f1 > 0 else 0.0

    denom_mcc = math.sqrt(
        max(tp + fp, 0)
        * max(tp + fn, 0)
        * max(tn + fp, 0)
        * max(tn + fn, 0)
    )

    if denom_mcc > 0:
        pairwise_mcc = float((tp * tn - fp * fn) / denom_mcc)
    else:
        pairwise_mcc = 0.0

    return {
        "pairwise_f1": pairwise_f1,
        "pairwise_mcc": pairwise_mcc,
        "pairwise_tp": tp,
        "pairwise_fp": fp,
        "pairwise_fn": fn,
        "pairwise_tn": tn,
    }


def dominant_source_share(y_true):
    y_true = np.asarray(y_true).reshape(-1)
    if len(y_true) == 0:
        return np.nan
    _, counts = np.unique(y_true, return_counts=True)
    return float(counts.max() / counts.sum())


def source_entropy_norm(y_true):
    y_true = np.asarray(y_true).reshape(-1)
    if len(y_true) == 0:
        return np.nan

    _, counts = np.unique(y_true, return_counts=True)

    counts = counts.astype(np.float64)
    p = counts / counts.sum()
    p = p[p > 0]

    if len(p) <= 1:
        return 0.0

    return float(-np.sum(p * np.log(p)) / np.log(len(p)))


def deinterleaving_metrics(y_true, y_pred_raw):
    y_true = np.asarray(y_true).reshape(-1).astype(int)
    y_pred_raw = np.asarray(y_pred_raw).reshape(-1).astype(int)
    y_pred_eval = apply_noise_policy(y_pred_raw)

    out = {
        "v_measure": safe_float(v_measure_score(y_true, y_pred_eval)),
        "ari": safe_float(adjusted_rand_score(y_true, y_pred_eval)),
        "ami": safe_float(adjusted_mutual_info_score(y_true, y_pred_eval)),
        "homogeneity": safe_float(homogeneity_score(y_true, y_pred_eval)),
        "completeness": safe_float(completeness_score(y_true, y_pred_eval)),
    }

    out.update(efficient_pairwise_metrics(y_true, y_pred_eval))

    true_sources = np.unique(y_true)
    pred_clusters_raw = np.unique(y_pred_raw)
    pred_clusters_eval = np.unique(y_pred_eval)

    out["n_pulses"] = int(len(y_true))
    out["n_true_sources"] = int(len(true_sources))
    out["n_pred_clusters_raw_excluding_noise"] = int(len([c for c in pred_clusters_raw if c != -1]))
    out["n_pred_clusters_eval"] = int(len(pred_clusters_eval))
    out["has_noise_cluster"] = int(np.any(y_pred_raw == -1))
    out["noise_share"] = float(np.mean(y_pred_raw == -1))
    out["dominant_source_share"] = dominant_source_share(y_true)
    out["source_entropy_norm"] = source_entropy_norm(y_true)

    return out


# ============================================================
# TRAIN / EVAL
# ============================================================
def run_train_epoch(model, loader, optimizer):
    model.train()

    total_loss = 0.0
    total_n = 0

    for batch in tqdm(loader, desc="train", leave=False):
        x = batch["x"].to(DEVICE)
        y = batch["y"].to(DEVICE)

        with torch.autocast(device_type=DEVICE_TYPE, enabled=USE_AMP):
            out = model(x)
            emb = out["emb"]
            gate_w = out["gate_w"]

            loss_con = supervised_contrastive_window_loss(
                emb,
                y,
                temperature=SUPCON_TEMPERATURE,
            )
            loss_gate = gate_balance_regularizer(gate_w)

            loss = loss_con + GATE_BALANCE_WEIGHT * loss_gate

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        bs = x.size(0)
        total_loss += float(loss.item()) * bs
        total_n += bs

    return total_loss / max(total_n, 1)


@torch.no_grad()
def infer_embeddings(model, x):
    model.eval()
    x = x.to(DEVICE)

    with torch.autocast(device_type=DEVICE_TYPE, enabled=USE_AMP):
        out = model(x)
        emb = out["emb"]

    emb = F.normalize(emb.float(), dim=-1)
    return emb.cpu().numpy()


def evaluate_model_clustering(model, loader, split_name, preprocess_mode, n_experts, max_batches=None):
    rows = []

    model.eval()

    for bidx, batch in enumerate(tqdm(loader, desc=f"eval_{split_name}", leave=False)):
        if max_batches is not None and bidx >= int(max_batches):
            break

        x = batch["x"]
        y = batch["y"].numpy()
        file_ids = list(batch["file_id"])
        window_indices = batch["window_index"].numpy()

        emb = infer_embeddings(model, x)  # [B, 1024, D]

        for i in range(emb.shape[0]):
            t0 = time.time()

            y_true = y[i]
            z = emb[i]

            y_pred = run_hdbscan(z)
            metrics = deinterleaving_metrics(y_true, y_pred)

            row = {
                "source": SOURCE_NAME,
                "split": split_name,
                "file_id": file_ids[i],
                "window_index": int(window_indices[i]),
                "preprocess_mode": preprocess_mode,
                "n_experts": int(n_experts),
                "cluster_method": "HDBSCAN_on_MoE_embeddings",
                "hdbscan_backend": HDBSCAN_BACKEND,
                "elapsed_sec": float(time.time() - t0),
            }

            row.update(metrics)
            rows.append(row)

    return pd.DataFrame(rows)


def summarize_metric_df(df, prefix=""):
    metrics = [
        "v_measure",
        "ari",
        "ami",
        "homogeneity",
        "completeness",
        "pairwise_f1",
        "pairwise_mcc",
        "n_true_sources",
        "n_pred_clusters_raw_excluding_noise",
        "noise_share",
        "dominant_source_share",
        "source_entropy_norm",
        "elapsed_sec",
    ]

    out = {}

    for m in metrics:
        if m not in df.columns:
            continue

        vals = pd.to_numeric(df[m], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()

        if len(vals) == 0:
            out[f"{prefix}{m}_mean"] = np.nan
            out[f"{prefix}{m}_median"] = np.nan
            out[f"{prefix}{m}_std"] = np.nan
        else:
            out[f"{prefix}{m}_mean"] = float(vals.mean())
            out[f"{prefix}{m}_median"] = float(vals.median())
            out[f"{prefix}{m}_std"] = float(vals.std())

    return out


def train_one_model(preprocess_mode, n_experts, train_loader, val_loader, run_dir):
    groups = get_feature_groups(preprocess_mode, n_experts)
    input_dim = len(get_feature_names(preprocess_mode))

    model = PDWFeatureMoE(
        feature_groups=groups,
        input_dim=input_dim,
        hidden_dim=HIDDEN_DIM,
        emb_dim=EMB_DIM,
        drop=DROP,
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    best_score = -np.inf
    best_epoch = -1
    best_state = None
    bad_epochs = 0
    history = []

    for epoch in range(1, EPOCHS + 1):
        print(f"\nEpoch {epoch:03d}/{EPOCHS}")

        train_loss = run_train_epoch(model, train_loader, optimizer)

        val_df = evaluate_model_clustering(
            model,
            val_loader,
            split_name="validation",
            preprocess_mode=preprocess_mode,
            n_experts=n_experts,
        )

        val_summary = summarize_metric_df(val_df, prefix="val_")
        val_score = val_summary.get("val_v_measure_median", np.nan)

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_score_v_measure_median": val_score,
        }
        row.update(val_summary)
        history.append(row)

        print(
            f"train_loss={train_loss:.4f} | "
            f"val_v_measure_median={val_score:.4f} | "
            f"val_ari_median={val_summary.get('val_ari_median', np.nan):.4f} | "
            f"val_pairwise_mcc_median={val_summary.get('val_pairwise_mcc_median', np.nan):.4f}"
        )

        if np.isfinite(val_score) and val_score > best_score:
            best_score = float(val_score)
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            torch.save(best_state, run_dir / "best_model.pt")
            val_df.to_csv(run_dir / "best_val_window_metrics.csv", index=False, encoding="utf-8-sig")
            bad_epochs = 0
            print(f"[BEST] epoch={epoch} val_v_measure_median={best_score:.4f}")
        else:
            bad_epochs += 1
            print(f"[NO IMPROVEMENT] bad_epochs={bad_epochs}/{PATIENCE}")

        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False, encoding="utf-8-sig")

        if bad_epochs >= PATIENCE:
            print("Early stopping triggered.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_epoch, best_score, pd.DataFrame(history)


# ============================================================
# PLOTS
# ============================================================
def plot_comparison(results_df):
    metrics = [
        "test_v_measure_median",
        "test_ari_median",
        "test_ami_median",
        "test_pairwise_mcc_median",
        "test_pairwise_f1_median",
        "test_noise_share_median",
    ]

    for metric in metrics:
        if metric not in results_df.columns:
            continue

        plt.figure(figsize=(9, 5))

        for mode in PREPROCESS_MODES:
            g = results_df[results_df["preprocess_mode"] == mode].sort_values("n_experts")
            if len(g) == 0:
                continue
            plt.plot(
                g["n_experts"].values,
                g[metric].values,
                marker="o",
                label=mode,
            )

        plt.xlabel("N experts")
        plt.ylabel(metric)
        plt.title(metric)
        plt.legend()
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"line_{metric}_by_mode.png", bbox_inches="tight")
        plt.close()

    for metric in metrics:
        if metric not in results_df.columns:
            continue

        plt.figure(figsize=(10, 5))
        labels = []
        values = []

        for _, r in results_df.sort_values(["n_experts", "preprocess_mode"]).iterrows():
            labels.append(f"N{int(r['n_experts'])}\n{r['preprocess_mode'].replace('_pdw_moe', '')}")
            values.append(float(r[metric]))

        plt.bar(labels, values)
        plt.ylabel(metric)
        plt.title(metric)
        plt.xticks(rotation=25, ha="right")
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"bar_{metric}_all_runs.png", bbox_inches="tight")
        plt.close()


def build_raw_vs_engineered_comparison(results_df):
    """
    Compares raw vs engineered for each N_EXPERTS.
    """
    metric_names = [
        "test_v_measure_median",
        "test_ari_median",
        "test_ami_median",
        "test_homogeneity_median",
        "test_completeness_median",
        "test_pairwise_f1_median",
        "test_pairwise_mcc_median",
        "test_n_pred_clusters_raw_excluding_noise_median",
        "test_noise_share_median",
        "test_elapsed_sec_median",
        "best_val_v_measure_median",
    ]

    rows = []

    for n_experts, g in results_df.groupby("n_experts"):
        raw = g[g["preprocess_mode"] == "raw_pdw_moe"]
        eng = g[g["preprocess_mode"] == "engineered_pdw_moe"]

        if len(raw) == 0 or len(eng) == 0:
            continue

        raw = raw.iloc[0]
        eng = eng.iloc[0]

        out = {
            "source": SOURCE_NAME,
            "n_experts": int(n_experts),
            "base_mode": "raw_pdw_moe",
            "new_mode": "engineered_pdw_moe",
            "n_train_windows_raw": int(raw["n_train_windows"]),
            "n_train_windows_engineered": int(eng["n_train_windows"]),
            "n_val_windows_raw": int(raw["n_val_windows"]),
            "n_val_windows_engineered": int(eng["n_val_windows"]),
            "n_test_windows_raw": int(raw["n_test_windows"]),
            "n_test_windows_engineered": int(eng["n_test_windows"]),
        }

        for m in metric_names:
            raw_val = float(raw[m]) if m in raw and pd.notna(raw[m]) else np.nan
            eng_val = float(eng[m]) if m in eng and pd.notna(eng[m]) else np.nan

            clean_m = m.replace("test_", "").replace("_median", "")
            out[f"{clean_m}_raw"] = raw_val
            out[f"{clean_m}_engineered"] = eng_val
            out[f"{clean_m}_delta_engineered_minus_raw"] = (
                eng_val - raw_val if np.isfinite(eng_val) and np.isfinite(raw_val) else np.nan
            )

        rows.append(out)

    return pd.DataFrame(rows)


# ============================================================
# SINGLE EXPERIMENT
# ============================================================
def run_single_experiment(
    preprocess_mode,
    n_experts,
    train_tasks,
    val_tasks,
    test_tasks,
):
    run_name = f"{safe_mode_name(preprocess_mode)}_N{n_experts}"
    run_dir = OUT_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    groups = get_feature_groups(preprocess_mode, n_experts)
    group_desc = describe_feature_groups(preprocess_mode, groups)
    feature_names = get_feature_names(preprocess_mode)

    print("\n" + "=" * 80)
    print(f"RUNNING MoE PDW DEINTERLEAVING | MODE={preprocess_mode} | N_EXPERTS={n_experts}")
    print("=" * 80)
    print(f"Input features : {feature_names}")
    print(f"Feature groups : {group_desc}")

    save_json(
        {
            "preprocess_mode": preprocess_mode,
            "n_experts": n_experts,
            "input_dim": len(feature_names),
            "feature_groups_indices": groups,
            "feature_groups_names": group_desc,
            "feature_names": feature_names,
        },
        run_dir / "feature_groups.json",
    )

    train_ds = PDWWindowDataset(train_tasks, preprocess_mode=preprocess_mode, train_mode=True)
    val_ds = PDWWindowDataset(val_tasks, preprocess_mode=preprocess_mode, train_mode=False)
    test_ds = PDWWindowDataset(test_tasks, preprocess_mode=preprocess_mode, train_mode=False)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        drop_last=False,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        drop_last=False,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        drop_last=False,
    )

    model, best_epoch, best_val_score, history_df = train_one_model(
        preprocess_mode=preprocess_mode,
        n_experts=n_experts,
        train_loader=train_loader,
        val_loader=val_loader,
        run_dir=run_dir,
    )

    print("\nEvaluating best model on validation...")
    val_df = evaluate_model_clustering(
        model,
        val_loader,
        split_name="validation",
        preprocess_mode=preprocess_mode,
        n_experts=n_experts,
    )
    val_df.to_csv(run_dir / "final_val_window_metrics.csv", index=False, encoding="utf-8-sig")

    print("\nEvaluating best model on test...")
    test_df = evaluate_model_clustering(
        model,
        test_loader,
        split_name="test",
        preprocess_mode=preprocess_mode,
        n_experts=n_experts,
    )
    test_df.to_csv(run_dir / "test_window_metrics.csv", index=False, encoding="utf-8-sig")

    val_summary = summarize_metric_df(val_df, prefix="val_")
    test_summary = summarize_metric_df(test_df, prefix="test_")

    result = {
        "model_name": "PDWFeatureMoE",
        "preprocess_mode": preprocess_mode,
        "n_experts": n_experts,
        "input_dim": len(feature_names),
        "feature_names": json.dumps(feature_names, ensure_ascii=False),
        "feature_groups": json.dumps(group_desc, ensure_ascii=False),
        "best_epoch": best_epoch,
        "best_val_v_measure_median": best_val_score,
        "n_train_windows": int(len(train_tasks)),
        "n_val_windows": int(len(val_tasks)),
        "n_test_windows": int(len(test_tasks)),
        "emb_dim": EMB_DIM,
        "hidden_dim": HIDDEN_DIM,
        "drop": DROP,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "supcon_temperature": SUPCON_TEMPERATURE,
        "gate_balance_weight": GATE_BALANCE_WEIGHT,
        "hdbscan_min_cluster_size": HDBSCAN_MIN_CLUSTER_SIZE,
        "hdbscan_min_samples": HDBSCAN_MIN_SAMPLES,
        "noise_policy_for_metrics": NOISE_POLICY_FOR_METRICS,
    }

    result.update(val_summary)
    result.update(test_summary)

    save_json(result, run_dir / "results.json")
    pd.DataFrame([result]).to_csv(run_dir / "summary_result.csv", index=False, encoding="utf-8-sig")

    return result


# ============================================================
# MAIN
# ============================================================
def main():
    print("=" * 80)
    print("TSRD MoE FOR PDW DEINTERLEAVING - RAW VS ENGINEERED")
    print("=" * 80)
    print(f"Device          : {DEVICE}")
    print(f"HDBSCAN backend : {HDBSCAN_BACKEND}")
    print(f"Preprocess modes: {PREPROCESS_MODES}")
    print(f"Experts list    : {EXPERTS_LIST}")

    # --------------------------------------------------------
    # Index files
    # --------------------------------------------------------
    print("\n" + "=" * 80)
    print("INDEXING FILES")
    print("=" * 80)

    train_index = collect_file_index("train", DATA_DIRS["train"])
    val_index = collect_file_index("validation", DATA_DIRS["validation"])
    test_index = collect_file_index("test", DATA_DIRS["test"])

    train_index.to_csv(TABLES_DIR / "file_index_train.csv", index=False, encoding="utf-8-sig")
    val_index.to_csv(TABLES_DIR / "file_index_validation.csv", index=False, encoding="utf-8-sig")
    test_index.to_csv(TABLES_DIR / "file_index_test.csv", index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # Select windows once.
    # Same tasks are reused by raw and engineered modes.
    # --------------------------------------------------------
    print("\n" + "=" * 80)
    print("SELECTING WINDOW TASKS")
    print("=" * 80)

    train_tasks = select_window_tasks(
        train_index,
        max_windows=MAX_TRAIN_WINDOWS,
        mode=TRAIN_WINDOW_SELECTION_MODE,
    )

    val_tasks = select_window_tasks(
        val_index,
        max_windows=MAX_VAL_WINDOWS,
        mode=VAL_WINDOW_SELECTION_MODE,
    )

    test_tasks = select_window_tasks(
        test_index,
        max_windows=MAX_TEST_WINDOWS,
        mode=TEST_WINDOW_SELECTION_MODE,
    )

    print(f"Train windows     : {len(train_tasks)}")
    print(f"Validation windows: {len(val_tasks)}")
    print(f"Test windows      : {len(test_tasks)}")

    train_tasks.to_csv(TABLES_DIR / "tasks_train.csv", index=False, encoding="utf-8-sig")
    val_tasks.to_csv(TABLES_DIR / "tasks_validation.csv", index=False, encoding="utf-8-sig")
    test_tasks.to_csv(TABLES_DIR / "tasks_test.csv", index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # Run sweep
    # --------------------------------------------------------
    all_results = []

    for preprocess_mode in PREPROCESS_MODES:
        for n_experts in EXPERTS_LIST:
            set_seed(SEED + int(n_experts) + (0 if preprocess_mode == "raw_pdw_moe" else 1000))

            res = run_single_experiment(
                preprocess_mode=preprocess_mode,
                n_experts=n_experts,
                train_tasks=train_tasks,
                val_tasks=val_tasks,
                test_tasks=test_tasks,
            )

            all_results.append(res)

            results_df_partial = pd.DataFrame(all_results)
            results_df_partial.to_csv(
                OUT_DIR / "comparison_all_results_partial.csv",
                index=False,
                encoding="utf-8-sig",
            )

    results_df = pd.DataFrame(all_results)

    comparison_path = OUT_DIR / "comparison_all_results.csv"
    results_df.to_csv(comparison_path, index=False, encoding="utf-8-sig")

    results_df.sort_values("test_v_measure_median", ascending=False).to_csv(
        OUT_DIR / "comparison_sorted_by_test_v_measure.csv",
        index=False,
        encoding="utf-8-sig",
    )

    results_df.sort_values("test_pairwise_mcc_median", ascending=False).to_csv(
        OUT_DIR / "comparison_sorted_by_test_pairwise_mcc.csv",
        index=False,
        encoding="utf-8-sig",
    )

    # Direct raw vs engineered comparison by N_EXPERTS
    mode_comparison_df = build_raw_vs_engineered_comparison(results_df)
    mode_comparison_path = OUT_DIR / "comparison_raw_vs_engineered_by_n_experts.csv"
    mode_comparison_df.to_csv(mode_comparison_path, index=False, encoding="utf-8-sig")

    # Best raw vs best engineered
    best_by_mode = (
        results_df.sort_values("test_v_measure_median", ascending=False)
        .groupby("preprocess_mode", as_index=False)
        .head(1)
        .sort_values("preprocess_mode")
    )
    best_by_mode_path = OUT_DIR / "comparison_best_raw_vs_best_engineered.csv"
    best_by_mode.to_csv(best_by_mode_path, index=False, encoding="utf-8-sig")

    plot_comparison(results_df)

    # --------------------------------------------------------
    # Config
    # --------------------------------------------------------
    config = {
        "root": str(ROOT),
        "data_dirs": {k: str(v) for k, v in DATA_DIRS.items()},
        "source_name": SOURCE_NAME,
        "window_size": WINDOW_SIZE,
        "max_train_windows": MAX_TRAIN_WINDOWS,
        "max_val_windows": MAX_VAL_WINDOWS,
        "max_test_windows": MAX_TEST_WINDOWS,
        "train_window_selection_mode": TRAIN_WINDOW_SELECTION_MODE,
        "val_window_selection_mode": VAL_WINDOW_SELECTION_MODE,
        "test_window_selection_mode": TEST_WINDOW_SELECTION_MODE,
        "preprocess_modes": PREPROCESS_MODES,
        "experts_list": EXPERTS_LIST,
        "raw_feature_names": RAW_FEATURE_NAMES,
        "engineered_feature_names": ENGINEERED_FEATURE_NAMES,
        "emb_dim": EMB_DIM,
        "hidden_dim": HIDDEN_DIM,
        "drop": DROP,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "supcon_temperature": SUPCON_TEMPERATURE,
        "gate_balance_weight": GATE_BALANCE_WEIGHT,
        "hdbscan_backend": HDBSCAN_BACKEND,
        "hdbscan_min_cluster_size": HDBSCAN_MIN_CLUSTER_SIZE,
        "hdbscan_min_samples": HDBSCAN_MIN_SAMPLES,
        "noise_policy_for_metrics": NOISE_POLICY_FOR_METRICS,
        "outputs": {
            "comparison_all_results": str(comparison_path),
            "comparison_raw_vs_engineered_by_n_experts": str(mode_comparison_path),
            "comparison_best_raw_vs_best_engineered": str(best_by_mode_path),
            "tables_dir": str(TABLES_DIR),
            "plots_dir": str(PLOTS_DIR),
            "out_dir": str(OUT_DIR),
        },
    }

    save_json(config, OUT_DIR / "config.json")

    # --------------------------------------------------------
    # Console output
    # --------------------------------------------------------
    print("\n" + "=" * 80)
    print("FINAL COMPARISON - SORTED BY TEST V-MEASURE")
    print("=" * 80)

    show_cols = [
        "preprocess_mode",
        "n_experts",
        "best_epoch",
        "best_val_v_measure_median",
        "test_v_measure_median",
        "test_ari_median",
        "test_ami_median",
        "test_homogeneity_median",
        "test_completeness_median",
        "test_pairwise_f1_median",
        "test_pairwise_mcc_median",
        "test_n_true_sources_median",
        "test_n_pred_clusters_raw_excluding_noise_median",
        "test_noise_share_median",
        "test_elapsed_sec_median",
    ]

    show_cols = [c for c in show_cols if c in results_df.columns]

    print(
        results_df[show_cols]
        .sort_values("test_v_measure_median", ascending=False)
        .to_string(index=False)
    )

    print("\n" + "=" * 80)
    print("RAW VS ENGINEERED BY N_EXPERTS")
    print("=" * 80)

    if len(mode_comparison_df) > 0:
        mode_show_cols = [
            "n_experts",
            "v_measure_raw",
            "v_measure_engineered",
            "v_measure_delta_engineered_minus_raw",
            "ari_raw",
            "ari_engineered",
            "ari_delta_engineered_minus_raw",
            "ami_raw",
            "ami_engineered",
            "ami_delta_engineered_minus_raw",
            "pairwise_mcc_raw",
            "pairwise_mcc_engineered",
            "pairwise_mcc_delta_engineered_minus_raw",
            "noise_share_raw",
            "noise_share_engineered",
            "noise_share_delta_engineered_minus_raw",
        ]
        mode_show_cols = [c for c in mode_show_cols if c in mode_comparison_df.columns]
        print(mode_comparison_df[mode_show_cols].to_string(index=False))
    else:
        print("No raw-vs-engineered comparison generated.")

    print("\n" + "=" * 80)
    print("BEST RAW VS BEST ENGINEERED")
    print("=" * 80)

    best_show_cols = [
        "preprocess_mode",
        "n_experts",
        "test_v_measure_median",
        "test_ari_median",
        "test_ami_median",
        "test_pairwise_mcc_median",
        "test_noise_share_median",
    ]
    best_show_cols = [c for c in best_show_cols if c in best_by_mode.columns]
    print(best_by_mode[best_show_cols].to_string(index=False))

    print("\n" + "=" * 80)
    print("OUTPUT FILES")
    print("=" * 80)
    print(f"Comparison all results       : {comparison_path}")
    print(f"Sorted by V-measure          : {OUT_DIR / 'comparison_sorted_by_test_v_measure.csv'}")
    print(f"Sorted by MCC                : {OUT_DIR / 'comparison_sorted_by_test_pairwise_mcc.csv'}")
    print(f"Raw vs engineered by experts : {mode_comparison_path}")
    print(f"Best raw vs best engineered  : {best_by_mode_path}")
    print(f"Tables directory             : {TABLES_DIR}")
    print(f"Plots directory              : {PLOTS_DIR}")
    print(f"Output directory             : {OUT_DIR}")

    print("\nDone.")


if __name__ == "__main__":
    main()

TSRD MoE FOR PDW DEINTERLEAVING - RAW VS ENGINEERED
Device          : cpu
HDBSCAN backend : hdbscan
Preprocess modes: ['raw_pdw_moe', 'engineered_pdw_moe']
Experts list    : [2, 3, 4, 5]

INDEXING FILES
[train] H5 files found: 2500


index_train: 100%|██████████| 2500/2500 [01:09<00:00, 35.72file/s]


[validation] H5 files found: 250


index_validation: 100%|██████████| 250/250 [00:08<00:00, 31.24file/s]


[test] H5 files found: 250


...

train_loss=6.2906 | val_v_measure_median=0.9565 | val_ari_median=0.9982 | val_pairwise_mcc_median=0.9982
[NO IMPROVEMENT] bad_epochs=1/5

Epoch 019/20


train_loss=6.2902 | val_v_measure_median=0.9540 | val_ari_median=0.9979 | val_pairwise_mcc_median=0.9979
[NO IMPROVEMENT] bad_epochs=2/5

Epoch 020/20


train_loss=6.2895 | val_v_measure_median=0.9577 | val_ari_median=0.9982 | val_pairwise_mcc_median=0.9982
[BEST] epoch=20 val_v_measure_median=0.9577

Evaluating best model on validation...



Evaluating best model on test...



FINAL COMPARISON - SORTED BY TEST V-MEASURE
   preprocess_mode  n_experts  best_epoch  best_val_v_measure_median  test_v_measure_median  test_ari_median  test_ami_median  test_homogeneity_median  test_completeness_median  test_pairwise_f1_median  test_pairwise_mcc_median  test_n_true_sources_median  test_n_pred_clusters_raw_excluding_noise_median  test_noise_share_median  test_elapsed_sec_median
engineered_pdw_moe          4          20                   0.957730               0.958403         0.998547         0.956684                 0.945996                  1.000000                 0.999713                  0.998548                        12.0                                              5.0                 0.005859                 0.141854
engineered_pdw_moe          3          18                   0.956480               0.956142         0.998350         0.954341                 0.943682                  1.000000                 0.999673                  0.998352                  

# PROXIMO PASSO: FAZER XAI IGUAL CODIGO ACUSTICA

In [9]:
# ============================================================
# xai_video_moe_pdw_deinterleaving.py
# ============================================================
# TSRD - XAI + videos for MoE-based PDW deinterleaving
#
# This script loads a trained PDWFeatureMoE model and generates
# explanation videos for radar pulse deinterleaving.
#
# Input model expected from:
#   D:\Fusion\turing_synthetic_radar_dataset\
#       benchmark_moe_pdw_deinterleaving_sweep\
#       moe_pdw_deinterleaving_N2\best_model.pt
#
# Data:
#   D:\Fusion\turing_synthetic_radar_dataset\archive\test
#
# Video frame:
#   - PDW feature matrix: features x pulses
#   - XAI saliency overlay: gradient x input
#   - HDBSCAN predicted clusters
#   - True emitter labels
#   - MoE gate weights
#
# XAI target:
#   Cluster compactness in learned embedding space.
#
# Important:
#   This is not class Grad-CAM because the deinterleaving model
#   does not output classes. It learns pulse embeddings, and HDBSCAN
#   separates pulses into emitter clusters.
# ============================================================

from pathlib import Path
import sys
import json
import math
import random
import re
import subprocess
import warnings
warnings.filterwarnings("ignore")


# ============================================================
# INSTALL / IMPORTS
# ============================================================
def ensure_package(import_name, pip_name=None):
    try:
        __import__(import_name)
    except Exception:
        pkg = pip_name if pip_name is not None else import_name
        print(f"[INFO] Installing missing package: {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


ensure_package("numpy")
ensure_package("pandas")
ensure_package("matplotlib")
ensure_package("h5py")
ensure_package("torch")
ensure_package("sklearn", "scikit-learn")
ensure_package("scipy")
ensure_package("imageio")
ensure_package("imageio_ffmpeg", "imageio-ffmpeg")
ensure_package("tqdm")

try:
    ensure_package("hdbscan")
    import hdbscan
    HDBSCAN_BACKEND = "hdbscan"
except Exception:
    hdbscan = None
    HDBSCAN_BACKEND = "sklearn"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
import imageio.v2 as imageio

from tqdm import tqdm
from scipy import ndimage
from matplotlib.patches import Rectangle

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    v_measure_score,
    adjusted_rand_score,
    adjusted_mutual_info_score,
    homogeneity_score,
    completeness_score,
)

try:
    from sklearn.cluster import HDBSCAN as SKHDBSCAN
except Exception:
    SKHDBSCAN = None


# ============================================================
# PATHS
# ============================================================
ROOT = Path(r"D:\Fusion\turing_synthetic_radar_dataset")

TEST_DIR = ROOT / "archive" / "test"

BASE_RUN_DIR = ROOT / "benchmark_moe_pdw_deinterleaving_sweep"

# Escolha o modelo treinado que deseja explicar.
N_EXPERTS_XAI = 2
RUN_DIR = BASE_RUN_DIR / f"moe_pdw_deinterleaving_N{N_EXPERTS_XAI}"

MODEL_PATH = RUN_DIR / "best_model.pt"
FEATURE_GROUPS_PATH = RUN_DIR / "feature_groups.json"
RESULTS_JSON_PATH = RUN_DIR / "results.json"

XAI_OUT_DIR = RUN_DIR / "xai_video_exports"
VIDEOS_DIR = XAI_OUT_DIR / "videos"
TABLES_DIR = XAI_OUT_DIR / "tables"
PLOTS_DIR = XAI_OUT_DIR / "plots"

XAI_OUT_DIR.mkdir(parents=True, exist_ok=True)
VIDEOS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# CONFIG
# ============================================================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

WINDOW_SIZE = 1024
DROP_INCOMPLETE_WINDOWS = True

# Quantos arquivos do test gerar em vídeo.
MAX_FILES_FOR_VIDEO = 10

# Quantas janelas por arquivo.
MAX_WINDOWS_PER_FILE_VIDEO = 24

# Seleção das janelas dentro do arquivo:
#   "first"  -> primeiras janelas
#   "even"   -> espalhadas pelo arquivo
WINDOW_SELECTION_MODE = "even"

# Vídeo
VIDEO_FPS = 2
VIDEO_CODEC = "libx264"
VIDEO_QUALITY = 7
FRAME_FIGSIZE = (13.5, 7.5)
FRAME_DPI = 130

# Modelo — será sobrescrito por results.json se existir.
N_INPUT_FEATURES = 8
EMB_DIM = 64
HIDDEN_DIM = 128
DROP = 0.15

# HDBSCAN sobre embeddings
HDBSCAN_MIN_CLUSTER_SIZE = 5
HDBSCAN_MIN_SAMPLES = 3
HDBSCAN_CLUSTER_SELECTION_EPSILON = 0.0
HDBSCAN_CLUSTER_SELECTION_METHOD = "eom"
HDBSCAN_ALLOW_SINGLE_CLUSTER = False

# XAI
XAI_METHOD = "cluster_compactness_grad_x_input"
SALIENCY_PERCENTILE = 95.0
REGION_MIN_AREA = 4
REGION_TOP_K = 8
REGION_STRUCT_SIZE = 2
REGION_OPEN_ITERS = 0
REGION_CLOSE_ITERS = 1

# Métricas
NOISE_POLICY_FOR_METRICS = "as_cluster"

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 180


# ============================================================
# REPRODUCIBILITY
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)


# ============================================================
# HELPERS
# ============================================================
FEATURE_NAMES = [
    "toa_rel",
    "delta_toa",
    "frequency",
    "pulse_width",
    "aoa_sin",
    "aoa_cos",
    "amplitude",
    "amp_missing",
]


def safe_filename(s: str) -> str:
    s = str(s)
    s = re.sub(r"[^\w\-.]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s[:180] if len(s) > 180 else s


def save_json(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def load_json(path: Path, default=None):
    if not path.exists():
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def normalize_map(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    if x.size == 0:
        return x
    mn = float(np.nanmin(x))
    mx = float(np.nanmax(x))
    if mx - mn < 1e-9:
        return np.zeros_like(x, dtype=np.float32)
    return ((x - mn) / (mx - mn + 1e-9)).astype(np.float32)


def robust_display_matrix(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    out = np.zeros_like(x, dtype=np.float32)

    for j in range(x.shape[1]):
        col = x[:, j]
        lo = np.percentile(col, 2)
        hi = np.percentile(col, 98)
        if hi - lo < 1e-9:
            out[:, j] = 0.0
        else:
            out[:, j] = np.clip((col - lo) / (hi - lo + 1e-9), 0, 1)

    return out


def h5_has_dataset(f, name):
    try:
        return isinstance(f[name], h5py.Dataset)
    except Exception:
        return False


def comb2(x):
    x = np.asarray(x, dtype=np.int64)
    return x * (x - 1) // 2


def safe_float(x):
    try:
        if np.isfinite(x):
            return float(x)
        return float("nan")
    except Exception:
        return float("nan")


def finite_median(v, default=0.0):
    v = np.asarray(v, dtype=np.float32)
    mask = np.isfinite(v)
    if mask.any():
        return float(np.nanmedian(v[mask]))
    return float(default)


def fill_nonfinite_with_median(v, default=0.0):
    v = np.asarray(v, dtype=np.float32).copy()
    med = finite_median(v, default=default)
    v[~np.isfinite(v)] = med
    return v


def compact_labels(labels):
    labels = np.asarray(labels).reshape(-1)
    uniques = sorted(np.unique(labels).tolist())
    mapping = {u: i for i, u in enumerate(uniques)}
    compact = np.array([mapping[v] for v in labels], dtype=np.int32)
    return compact, mapping


def fig_to_rgb_array(fig):
    fig.canvas.draw()
    w, h = fig.canvas.get_width_height()
    buf = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    buf = buf.reshape(h, w, 4)
    return buf[:, :, :3].copy()


def ensure_video_compatible_frame(im: np.ndarray) -> np.ndarray:
    if im.dtype != np.uint8:
        im = np.clip(im, 0, 255).astype(np.uint8)

    if im.ndim == 2:
        im = np.stack([im, im, im], axis=-1)

    if im.shape[2] == 4:
        im = im[:, :, :3]

    h, w = im.shape[:2]
    pad_h = h % 2
    pad_w = w % 2

    if pad_h != 0 or pad_w != 0:
        new_im = np.zeros((h + pad_h, w + pad_w, 3), dtype=np.uint8)
        new_im[:h, :w, :] = im
        im = new_im

    return im


# ============================================================
# PDW PREPROCESSING
# ============================================================
def preprocess_pdw_window(x):
    """
    Converts raw PDWs [ToA, Frequency, PulseWidth, AoA, Amplitude]
    into engineered feature vectors.

    Output:
      [N, 8]
    """
    x = np.asarray(x, dtype=np.float32)

    toa = fill_nonfinite_with_median(x[:, 0], default=0.0)
    freq = fill_nonfinite_with_median(x[:, 1], default=0.0)
    pw = fill_nonfinite_with_median(x[:, 2], default=0.0)
    aoa = fill_nonfinite_with_median(x[:, 3], default=0.0)

    amp = x[:, 4].astype(np.float32).copy()
    amp_missing = (~np.isfinite(amp)).astype(np.float32)
    amp[~np.isfinite(amp)] = -200.0
    amp = fill_nonfinite_with_median(amp, default=-200.0)

    toa_rel = toa - np.min(toa)
    span = float(np.max(toa_rel) - np.min(toa_rel))
    if span > 0:
        toa_rel_norm = toa_rel / (span + 1e-9)
    else:
        toa_rel_norm = np.zeros_like(toa_rel, dtype=np.float32)

    dtoa = np.diff(toa, prepend=toa[0])
    dtoa = np.maximum(dtoa, 0.0)
    dtoa_log = np.log1p(dtoa)

    freq_norm = freq / 18000.0

    pw = np.maximum(pw, 0.0)
    pw_log = np.log1p(pw)

    aoa_rad = np.deg2rad(aoa)
    aoa_sin = np.sin(aoa_rad)
    aoa_cos = np.cos(aoa_rad)

    amp_norm = (amp + 220.0) / 260.0

    feats = np.stack(
        [
            toa_rel_norm,
            dtoa_log,
            freq_norm,
            pw_log,
            aoa_sin,
            aoa_cos,
            amp_norm,
            amp_missing,
        ],
        axis=1,
    ).astype(np.float32)

    z = feats.copy()
    continuous_cols = [0, 1, 2, 3, 4, 5, 6]

    try:
        scaler = RobustScaler()
        z[:, continuous_cols] = scaler.fit_transform(z[:, continuous_cols])
    except Exception:
        pass

    z = np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0)
    return z.astype(np.float32)


# ============================================================
# MODEL DEFINITIONS
# ============================================================
class FeatureExpert(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, emb_dim=64, drop=0.15):
        super().__init__()

        self.point = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
        )

        self.temporal = nn.Sequential(
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2),
            nn.GELU(),
        )

        self.norm = nn.LayerNorm(hidden_dim)

        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, emb_dim),
            nn.GELU(),
        )

    def forward(self, x):
        # x: [B, N, Fin]
        h = self.point(x)
        ht = h.transpose(1, 2)
        ht = self.temporal(ht)
        ht = ht.transpose(1, 2)
        h = self.norm(h + ht)
        z = self.proj(h)
        return z


class PDWFeatureMoE(nn.Module):
    def __init__(self, feature_groups, input_dim=8, hidden_dim=128, emb_dim=64, drop=0.15):
        super().__init__()

        self.feature_groups = [list(g) for g in feature_groups]
        self.n_experts = len(self.feature_groups)
        self.emb_dim = emb_dim

        self.experts = nn.ModuleList([
            FeatureExpert(
                in_dim=len(g),
                hidden_dim=hidden_dim,
                emb_dim=emb_dim,
                drop=drop,
            )
            for g in self.feature_groups
        ])

        self.gate = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(hidden_dim, self.n_experts),
        )

        self.out_norm = nn.LayerNorm(emb_dim)

    def forward(self, x):
        # x: [B, N, F]
        gate_logits = self.gate(x)
        gate_w = torch.softmax(gate_logits, dim=-1)

        expert_outs = []
        for expert, group in zip(self.experts, self.feature_groups):
            xg = x[:, :, group]
            zg = expert(xg)
            expert_outs.append(zg)

        E = torch.stack(expert_outs, dim=2)
        z = (E * gate_w.unsqueeze(-1)).sum(dim=2)
        z = self.out_norm(z)

        return {
            "emb": z,
            "gate_w": gate_w,
        }


def default_feature_groups(n_experts):
    if n_experts == 2:
        return [
            [0, 1, 3],
            [2, 4, 5, 6, 7],
        ]
    if n_experts == 3:
        return [
            [0, 1],
            [2, 3],
            [4, 5, 6, 7],
        ]
    if n_experts == 4:
        return [
            [0, 1],
            [2],
            [3],
            [4, 5, 6, 7],
        ]
    if n_experts == 5:
        return [
            [0, 1],
            [2],
            [3],
            [4, 5],
            [6, 7],
        ]
    raise ValueError(f"Unsupported n_experts: {n_experts}")


def describe_feature_groups(groups):
    return [[FEATURE_NAMES[i] for i in g] for g in groups]


def load_trained_model():
    if not MODEL_PATH.exists():
        raise FileNotFoundError(
            f"Modelo não encontrado: {MODEL_PATH}\n"
            f"Rode primeiro o treinamento do MoE."
        )

    result_cfg = load_json(RESULTS_JSON_PATH, default={}) or {}
    emb_dim = int(result_cfg.get("emb_dim", EMB_DIM))
    hidden_dim = int(result_cfg.get("hidden_dim", HIDDEN_DIM))
    drop = float(result_cfg.get("drop", DROP))

    fg_cfg = load_json(FEATURE_GROUPS_PATH, default={}) or {}
    feature_groups = fg_cfg.get("feature_groups_indices", None)

    if feature_groups is None:
        feature_groups = default_feature_groups(N_EXPERTS_XAI)

    model = PDWFeatureMoE(
        feature_groups=feature_groups,
        input_dim=N_INPUT_FEATURES,
        hidden_dim=hidden_dim,
        emb_dim=emb_dim,
        drop=drop,
    ).to(DEVICE)

    state = torch.load(MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(state)
    model.eval()

    print("=" * 80)
    print("LOADED MODEL")
    print("=" * 80)
    print(f"Model path     : {MODEL_PATH}")
    print(f"N experts      : {len(feature_groups)}")
    print(f"Feature groups : {describe_feature_groups(feature_groups)}")
    print(f"emb_dim        : {emb_dim}")
    print(f"hidden_dim     : {hidden_dim}")
    print(f"drop           : {drop}")

    return model, feature_groups


# ============================================================
# HDBSCAN + METRICS
# ============================================================
def run_hdbscan(X):
    X = np.asarray(X, dtype=np.float32)

    if HDBSCAN_BACKEND == "hdbscan" and hdbscan is not None:
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
            min_samples=HDBSCAN_MIN_SAMPLES,
            metric="euclidean",
            cluster_selection_epsilon=HDBSCAN_CLUSTER_SELECTION_EPSILON,
            cluster_selection_method=HDBSCAN_CLUSTER_SELECTION_METHOD,
            allow_single_cluster=HDBSCAN_ALLOW_SINGLE_CLUSTER,
        )
        return clusterer.fit_predict(X).astype(int)

    if SKHDBSCAN is not None:
        clusterer = SKHDBSCAN(
            min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
            min_samples=HDBSCAN_MIN_SAMPLES,
            metric="euclidean",
            cluster_selection_epsilon=HDBSCAN_CLUSTER_SELECTION_EPSILON,
            allow_single_cluster=HDBSCAN_ALLOW_SINGLE_CLUSTER,
        )
        return clusterer.fit_predict(X).astype(int)

    raise RuntimeError("No HDBSCAN backend available. Try: pip install hdbscan")


def apply_noise_policy(y_pred):
    y_pred = np.asarray(y_pred).reshape(-1).astype(int).copy()

    if NOISE_POLICY_FOR_METRICS == "as_cluster":
        return y_pred

    if NOISE_POLICY_FOR_METRICS == "unique_noise":
        noise_idx = np.where(y_pred == -1)[0]
        if len(noise_idx) == 0:
            return y_pred

        max_cluster = int(np.max(y_pred)) if len(y_pred) > 0 else 0
        next_label = max_cluster + 1

        for k, idx in enumerate(noise_idx):
            y_pred[idx] = next_label + k

        return y_pred

    raise ValueError(f"Unknown NOISE_POLICY_FOR_METRICS: {NOISE_POLICY_FOR_METRICS}")


def efficient_pairwise_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    n = len(y_true)
    total_pairs = n * (n - 1) // 2

    if n < 2 or total_pairs <= 0:
        return {
            "pairwise_f1": np.nan,
            "pairwise_mcc": np.nan,
        }

    true_vals, true_inv = np.unique(y_true, return_inverse=True)
    pred_vals, pred_inv = np.unique(y_pred, return_inverse=True)

    contingency = np.zeros((len(true_vals), len(pred_vals)), dtype=np.int64)
    np.add.at(contingency, (true_inv, pred_inv), 1)

    tp = int(comb2(contingency).sum())
    true_sizes = contingency.sum(axis=1)
    pred_sizes = contingency.sum(axis=0)

    true_pairs = int(comb2(true_sizes).sum())
    pred_pairs = int(comb2(pred_sizes).sum())

    fp = int(pred_pairs - tp)
    fn = int(true_pairs - tp)
    tn = int(total_pairs - tp - fp - fn)

    denom_f1 = 2 * tp + fp + fn
    pairwise_f1 = float((2 * tp) / denom_f1) if denom_f1 > 0 else 0.0

    denom_mcc = math.sqrt(
        max(tp + fp, 0)
        * max(tp + fn, 0)
        * max(tn + fp, 0)
        * max(tn + fn, 0)
    )

    pairwise_mcc = float((tp * tn - fp * fn) / denom_mcc) if denom_mcc > 0 else 0.0

    return {
        "pairwise_f1": pairwise_f1,
        "pairwise_mcc": pairwise_mcc,
    }


def deinterleaving_metrics(y_true, y_pred_raw):
    y_true = np.asarray(y_true).reshape(-1).astype(int)
    y_pred_raw = np.asarray(y_pred_raw).reshape(-1).astype(int)
    y_pred_eval = apply_noise_policy(y_pred_raw)

    out = {
        "v_measure": safe_float(v_measure_score(y_true, y_pred_eval)),
        "ari": safe_float(adjusted_rand_score(y_true, y_pred_eval)),
        "ami": safe_float(adjusted_mutual_info_score(y_true, y_pred_eval)),
        "homogeneity": safe_float(homogeneity_score(y_true, y_pred_eval)),
        "completeness": safe_float(completeness_score(y_true, y_pred_eval)),
    }

    out.update(efficient_pairwise_metrics(y_true, y_pred_eval))

    pred_clusters_raw = np.unique(y_pred_raw)

    out["n_true_sources"] = int(len(np.unique(y_true)))
    out["n_pred_clusters_excluding_noise"] = int(len([c for c in pred_clusters_raw if c != -1]))
    out["noise_share"] = float(np.mean(y_pred_raw == -1))

    return out


# ============================================================
# FILE SELECTION
# ============================================================
def collect_test_files():
    rows = []

    h5_files = sorted(TEST_DIR.rglob("*.h5"))

    print("=" * 80)
    print("SCANNING TEST FILES")
    print("=" * 80)
    print(f"Folder: {TEST_DIR}")
    print(f"H5 files found: {len(h5_files)}")

    for p in tqdm(h5_files, desc="index_test", unit="file"):
        try:
            with h5py.File(p, "r") as f:
                if not h5_has_dataset(f, "data") or not h5_has_dataset(f, "labels"):
                    continue

                n = int(f["data"].shape[0])
                if DROP_INCOMPLETE_WINDOWS:
                    n_windows = n // WINDOW_SIZE
                else:
                    n_windows = int(math.ceil(n / WINDOW_SIZE))

                if n_windows <= 0:
                    continue

                rows.append({
                    "file_id": p.stem,
                    "file_name": p.name,
                    "path": str(p),
                    "n_pulses": n,
                    "n_windows": n_windows,
                })

        except Exception as e:
            print(f"[Warning] Could not read {p}: {e}")

    df = pd.DataFrame(rows)

    if len(df) == 0:
        raise RuntimeError("No valid test files found.")

    df = df.sort_values("n_windows", ascending=False).reset_index(drop=True)
    df.to_csv(TABLES_DIR / "xai_test_file_index.csv", index=False, encoding="utf-8-sig")

    return df


def select_window_indices(n_windows, max_windows):
    n_windows = int(n_windows)

    if max_windows is None or n_windows <= max_windows:
        return list(range(n_windows))

    if WINDOW_SELECTION_MODE == "first":
        return list(range(int(max_windows)))

    if WINDOW_SELECTION_MODE == "even":
        idx = np.linspace(0, n_windows - 1, int(max_windows)).round().astype(int)
        return sorted(set(idx.tolist()))

    raise ValueError(f"Unknown WINDOW_SELECTION_MODE: {WINDOW_SELECTION_MODE}")


def load_window(path: Path, window_index: int):
    start = int(window_index) * WINDOW_SIZE
    end = start + WINDOW_SIZE

    with h5py.File(path, "r") as f:
        x_raw = np.asarray(f["data"][start:end, :5], dtype=np.float32)
        y_true = np.asarray(f["labels"][start:end]).reshape(-1).astype(np.int64)

    x_proc = preprocess_pdw_window(x_raw)

    return x_raw, x_proc, y_true, start, end


# ============================================================
# XAI
# ============================================================
@torch.no_grad()
def infer_embedding_and_gate(model, x_proc):
    x = torch.tensor(x_proc, dtype=torch.float32, device=DEVICE).unsqueeze(0)

    out = model(x)
    emb = F.normalize(out["emb"].float(), dim=-1)[0].detach().cpu().numpy()
    gate = out["gate_w"][0].detach().cpu().numpy()

    return emb, gate


def compute_cluster_compactness_saliency(model, x_proc, y_pred):
    """
    XAI objective:
      maximize compactness of predicted clusters in embedding space.

    Saliency:
      abs(gradient * input) over [pulse, feature].
    """
    model.eval()

    x = torch.tensor(x_proc, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    x.requires_grad_(True)

    out = model(x)
    emb = F.normalize(out["emb"].float(), dim=-1)[0]  # [N, D]
    gate = out["gate_w"][0]                           # [N, E]

    y_pred = np.asarray(y_pred).astype(int)
    clusters = [c for c in np.unique(y_pred).tolist() if c != -1]

    scores = []

    for c in clusters:
        idx_np = np.where(y_pred == c)[0]
        if len(idx_np) < 2:
            continue

        idx = torch.tensor(idx_np, dtype=torch.long, device=DEVICE)
        zc = emb[idx]
        centroid = F.normalize(zc.mean(dim=0, keepdim=True), dim=-1)
        sim = (zc * centroid).sum(dim=-1).mean()
        scores.append(sim)

    if len(scores) == 0:
        score = emb.pow(2).mean()
    else:
        score = torch.stack(scores).mean()

    model.zero_grad(set_to_none=True)
    score.backward()

    grad = x.grad.detach()[0]       # [N, F]
    inp = x.detach()[0]             # [N, F]

    sal = torch.abs(grad * inp).cpu().numpy().astype(np.float32)
    sal = normalize_map(sal)

    gate_np = gate.detach().cpu().numpy().astype(np.float32)

    return {
        "saliency": sal,
        "gate": gate_np,
        "xai_score": float(score.detach().cpu().item()),
    }


def detect_salient_regions(saliency):
    """
    saliency: [N, F]
    returns bboxes in display coordinates:
        x1, y1, x2, y2
    where x = pulse index, y = feature index.
    """
    sal = normalize_map(saliency)

    if sal.size == 0:
        return [], np.zeros_like(sal, dtype=np.uint8), 0.0

    thr = float(np.percentile(sal, SALIENCY_PERCENTILE))
    thr = min(max(thr, 0.05), 0.98)

    mask = sal >= thr

    # connected components over [N, F]
    structure = np.ones((REGION_STRUCT_SIZE, REGION_STRUCT_SIZE), dtype=bool)

    if REGION_OPEN_ITERS > 0:
        mask = ndimage.binary_opening(mask, structure=structure, iterations=REGION_OPEN_ITERS)

    if REGION_CLOSE_ITERS > 0:
        mask = ndimage.binary_closing(mask, structure=structure, iterations=REGION_CLOSE_ITERS)

    labeled, n_comp = ndimage.label(mask)

    regions = []

    for lab in range(1, n_comp + 1):
        xx, yy = np.where(labeled == lab)  # xx=pulse, yy=feature
        if xx.size == 0:
            continue

        area = int(xx.size)
        if area < REGION_MIN_AREA:
            continue

        x1 = int(xx.min())
        x2 = int(xx.max())
        y1 = int(yy.min())
        y2 = int(yy.max())

        vals = sal[xx, yy]

        regions.append({
            "bbox": (x1, y1, x2, y2),
            "area": area,
            "mean_score": float(vals.mean()),
            "max_score": float(vals.max()),
            "sum_score": float(vals.sum()),
        })

    regions = sorted(
        regions,
        key=lambda r: (r["max_score"], r["mean_score"], r["area"]),
        reverse=True,
    )[:REGION_TOP_K]

    return regions, mask.astype(np.uint8), thr


# ============================================================
# FRAME RENDERING
# ============================================================
def render_xai_frame(
    x_proc,
    y_true,
    y_pred,
    saliency,
    gate,
    regions,
    metrics,
    file_id,
    window_index,
    pulse_start,
    pulse_end,
    xai_score,
):
    base = robust_display_matrix(x_proc)      # [N, F]
    sal = normalize_map(saliency)             # [N, F]

    y_true_compact, _ = compact_labels(y_true)
    y_pred_compact, _ = compact_labels(y_pred)

    mean_gate = gate.mean(axis=0)

    fig = plt.figure(figsize=FRAME_FIGSIZE, dpi=FRAME_DPI)
    gs = fig.add_gridspec(
        4,
        2,
        width_ratios=[4.7, 1.25],
        height_ratios=[3.0, 0.55, 0.55, 0.9],
    )

    ax0 = fig.add_subplot(gs[0, 0])
    ax_gate = fig.add_subplot(gs[0, 1])
    ax_pred = fig.add_subplot(gs[1, 0])
    ax_true = fig.add_subplot(gs[2, 0])
    ax_text = fig.add_subplot(gs[1:, 1])
    ax_sal = fig.add_subplot(gs[3, 0])

    # Main PDW-feature matrix
    ax0.imshow(base.T, aspect="auto", origin="lower", cmap="viridis")
    ax0.imshow(sal.T, aspect="auto", origin="lower", cmap="hot", alpha=0.45)

    for r in regions:
        x1, y1, x2, y2 = r["bbox"]
        rect = Rectangle(
            (x1, y1),
            max(1, x2 - x1 + 1),
            max(1, y2 - y1 + 1),
            linewidth=1.4,
            edgecolor="cyan",
            facecolor="none",
        )
        ax0.add_patch(rect)
        ax0.text(
            x1,
            min(y2 + 0.2, len(FEATURE_NAMES) - 1),
            f"R",
            color="white",
            fontsize=7,
            bbox=dict(boxstyle="round,pad=0.12", facecolor="black", alpha=0.6),
        )

    ax0.set_yticks(np.arange(len(FEATURE_NAMES)))
    ax0.set_yticklabels(FEATURE_NAMES, fontsize=7)
    ax0.set_ylabel("PDW features")
    ax0.set_xlabel("Pulse index inside window")
    ax0.set_title(
        f"{file_id} | window={window_index} | pulses={pulse_start}-{pulse_end}"
    )

    # Gate
    ax_gate.bar([f"E{i}" for i in range(len(mean_gate))], mean_gate)
    ax_gate.set_ylim(0, 1)
    ax_gate.set_title("Mean gate")
    ax_gate.tick_params(axis="x", labelsize=8)

    # Predicted clusters strip
    ax_pred.imshow(y_pred_compact.reshape(1, -1), aspect="auto", cmap="tab20")
    ax_pred.set_yticks([])
    ax_pred.set_ylabel("Pred", rotation=0, labelpad=24)
    ax_pred.set_xticks([])

    # True labels strip
    ax_true.imshow(y_true_compact.reshape(1, -1), aspect="auto", cmap="tab20")
    ax_true.set_yticks([])
    ax_true.set_ylabel("True", rotation=0, labelpad=24)
    ax_true.set_xticks([])

    # Saliency by pulse
    sal_pulse = sal.mean(axis=1)
    ax_sal.plot(np.arange(len(sal_pulse)), sal_pulse, linewidth=1.0)
    ax_sal.set_ylim(0, 1.05)
    ax_sal.set_ylabel("Saliency")
    ax_sal.set_xlabel("Pulse index")

    # Text panel
    ax_text.axis("off")

    txt = [
        f"XAI: {XAI_METHOD}",
        f"xai_score: {xai_score:.4f}",
        "",
        f"V-measure: {metrics['v_measure']:.3f}",
        f"ARI: {metrics['ari']:.3f}",
        f"AMI: {metrics['ami']:.3f}",
        f"Homogeneity: {metrics['homogeneity']:.3f}",
        f"Completeness: {metrics['completeness']:.3f}",
        f"Pairwise MCC: {metrics['pairwise_mcc']:.3f}",
        "",
        f"True sources: {metrics['n_true_sources']}",
        f"Pred clusters: {metrics['n_pred_clusters_excluding_noise']}",
        f"Noise share: {metrics['noise_share']:.3f}",
        f"Regions: {len(regions)}",
    ]

    ax_text.text(
        0.02,
        0.98,
        "\n".join(txt),
        transform=ax_text.transAxes,
        va="top",
        ha="left",
        fontsize=9,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.85),
    )

    plt.tight_layout()

    frame = fig_to_rgb_array(fig)
    plt.close(fig)

    return ensure_video_compatible_frame(frame)


# ============================================================
# VIDEO GENERATION
# ============================================================
def make_xai_video_for_file(model, file_row):
    file_path = Path(file_row["path"])
    file_id = str(file_row["file_id"])
    n_windows = int(file_row["n_windows"])

    win_indices = select_window_indices(
        n_windows=n_windows,
        max_windows=MAX_WINDOWS_PER_FILE_VIDEO,
    )

    out_video = VIDEOS_DIR / f"xai_{safe_filename(file_id)}.mp4"

    print(f"[VIDEO] {file_id} | windows={len(win_indices)} -> {out_video}")

    writer = None
    window_rows = []
    region_rows = []

    try:
        writer = imageio.get_writer(
            str(out_video),
            fps=VIDEO_FPS,
            codec=VIDEO_CODEC,
            quality=VIDEO_QUALITY,
            macro_block_size=None,
            pixelformat="yuv420p",
        )

        for w in tqdm(win_indices, desc=f"video_{file_id}", leave=False):
            x_raw, x_proc, y_true, pulse_start, pulse_end = load_window(file_path, w)

            emb, gate_initial = infer_embedding_and_gate(model, x_proc)
            y_pred = run_hdbscan(emb)

            metrics = deinterleaving_metrics(y_true, y_pred)

            xai = compute_cluster_compactness_saliency(model, x_proc, y_pred)
            saliency = xai["saliency"]
            gate = xai["gate"]
            xai_score = xai["xai_score"]

            regions, mask, thr = detect_salient_regions(saliency)

            for ridx, r in enumerate(regions):
                x1, y1, x2, y2 = r["bbox"]
                region_rows.append({
                    "file_id": file_id,
                    "window_index": int(w),
                    "region_id": int(ridx),
                    "x1_pulse": int(x1),
                    "x2_pulse": int(x2),
                    "y1_feature": int(y1),
                    "y2_feature": int(y2),
                    "feature_start": FEATURE_NAMES[y1] if 0 <= y1 < len(FEATURE_NAMES) else "",
                    "feature_end": FEATURE_NAMES[y2] if 0 <= y2 < len(FEATURE_NAMES) else "",
                    "area": int(r["area"]),
                    "mean_score": float(r["mean_score"]),
                    "max_score": float(r["max_score"]),
                    "sum_score": float(r["sum_score"]),
                    "threshold": float(thr),
                })

            row = {
                "file_id": file_id,
                "file_path": str(file_path),
                "window_index": int(w),
                "pulse_start": int(pulse_start),
                "pulse_end": int(pulse_end),
                "xai_score": float(xai_score),
                "n_regions": int(len(regions)),
                "mean_gate_json": json.dumps(gate.mean(axis=0).tolist()),
            }
            row.update(metrics)
            window_rows.append(row)

            frame = render_xai_frame(
                x_proc=x_proc,
                y_true=y_true,
                y_pred=y_pred,
                saliency=saliency,
                gate=gate,
                regions=regions,
                metrics=metrics,
                file_id=file_id,
                window_index=w,
                pulse_start=pulse_start,
                pulse_end=pulse_end,
                xai_score=xai_score,
            )

            writer.append_data(frame)

    finally:
        if writer is not None:
            writer.close()

    return out_video, window_rows, region_rows


# ============================================================
# SUMMARY PLOTS
# ============================================================
def plot_summary(window_df):
    if len(window_df) == 0:
        return

    metrics = [
        "v_measure",
        "ari",
        "ami",
        "homogeneity",
        "completeness",
        "pairwise_mcc",
        "noise_share",
    ]

    for m in metrics:
        if m not in window_df.columns:
            continue

        vals = pd.to_numeric(window_df[m], errors="coerce").dropna()

        if len(vals) == 0:
            continue

        plt.figure(figsize=(8, 5))
        plt.hist(vals.values, bins=40)
        plt.title(f"XAI video windows - {m}")
        plt.xlabel(m)
        plt.ylabel("Windows")
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"hist_{m}.png", bbox_inches="tight")
        plt.close()


# ============================================================
# MAIN
# ============================================================
def main():
    print("=" * 80)
    print("TSRD MoE DEINTERLEAVING - XAI VIDEO EXPORT")
    print("=" * 80)
    print(f"Device          : {DEVICE}")
    print(f"HDBSCAN backend : {HDBSCAN_BACKEND}")
    print(f"Run dir         : {RUN_DIR}")
    print(f"Model path      : {MODEL_PATH}")
    print(f"Test folder     : {TEST_DIR}")

    model, feature_groups = load_trained_model()

    test_files = collect_test_files()

    selected_files = test_files.head(MAX_FILES_FOR_VIDEO).copy().reset_index(drop=True)

    print("=" * 80)
    print("SELECTED FILES FOR VIDEO")
    print("=" * 80)
    print(selected_files[["file_id", "n_pulses", "n_windows"]].to_string(index=False))

    all_window_rows = []
    all_region_rows = []
    video_rows = []

    for _, row in selected_files.iterrows():
        video_path, window_rows, region_rows = make_xai_video_for_file(model, row)

        video_rows.append({
            "file_id": row["file_id"],
            "file_path": row["path"],
            "video_path": str(video_path),
            "n_windows_video": len(window_rows),
            "n_regions": len(region_rows),
        })

        all_window_rows.extend(window_rows)
        all_region_rows.extend(region_rows)

    video_df = pd.DataFrame(video_rows)
    window_df = pd.DataFrame(all_window_rows)
    region_df = pd.DataFrame(all_region_rows)

    video_path_csv = TABLES_DIR / "xai_video_files.csv"
    window_metrics_path = TABLES_DIR / "xai_window_metrics.csv"
    regions_path = TABLES_DIR / "xai_detected_regions.csv"

    video_df.to_csv(video_path_csv, index=False, encoding="utf-8-sig")
    window_df.to_csv(window_metrics_path, index=False, encoding="utf-8-sig")
    region_df.to_csv(regions_path, index=False, encoding="utf-8-sig")

    plot_summary(window_df)

    summary = {
        "root": str(ROOT),
        "run_dir": str(RUN_DIR),
        "model_path": str(MODEL_PATH),
        "test_dir": str(TEST_DIR),
        "n_experts": int(N_EXPERTS_XAI),
        "feature_groups": describe_feature_groups(feature_groups),
        "xai_method": XAI_METHOD,
        "max_files_for_video": MAX_FILES_FOR_VIDEO,
        "max_windows_per_file_video": MAX_WINDOWS_PER_FILE_VIDEO,
        "video_fps": VIDEO_FPS,
        "hdbscan_backend": HDBSCAN_BACKEND,
        "hdbscan_min_cluster_size": HDBSCAN_MIN_CLUSTER_SIZE,
        "hdbscan_min_samples": HDBSCAN_MIN_SAMPLES,
        "outputs": {
            "videos_dir": str(VIDEOS_DIR),
            "video_index": str(video_path_csv),
            "window_metrics": str(window_metrics_path),
            "regions": str(regions_path),
            "plots_dir": str(PLOTS_DIR),
        },
    }

    save_json(summary, XAI_OUT_DIR / "xai_video_config.json")

    print("\n" + "=" * 80)
    print("XAI VIDEO SUMMARY")
    print("=" * 80)

    if len(window_df) > 0:
        cols = [
            "v_measure",
            "ari",
            "ami",
            "homogeneity",
            "completeness",
            "pairwise_mcc",
            "n_true_sources",
            "n_pred_clusters_excluding_noise",
            "noise_share",
            "n_regions",
        ]

        rows = []
        for c in cols:
            vals = pd.to_numeric(window_df[c], errors="coerce").dropna()
            if len(vals) == 0:
                continue
            rows.append({
                "metric": c,
                "mean": float(vals.mean()),
                "median": float(vals.median()),
                "std": float(vals.std()),
            })

        summary_df = pd.DataFrame(rows)
        print(summary_df.to_string(index=False))
        summary_df.to_csv(TABLES_DIR / "xai_metric_summary.csv", index=False, encoding="utf-8-sig")

    print("\n" + "=" * 80)
    print("OUTPUT FILES")
    print("=" * 80)
    print(f"Videos dir      : {VIDEOS_DIR}")
    print(f"Video index     : {video_path_csv}")
    print(f"Window metrics  : {window_metrics_path}")
    print(f"Regions CSV     : {regions_path}")
    print(f"Plots dir       : {PLOTS_DIR}")
    print(f"Config JSON     : {XAI_OUT_DIR / 'xai_video_config.json'}")

    print("\nDone.")


if __name__ == "__main__":
    main()

TSRD MoE DEINTERLEAVING - XAI VIDEO EXPORT
Device          : cpu
HDBSCAN backend : hdbscan
Run dir         : D:\Fusion\turing_synthetic_radar_dataset\benchmark_moe_pdw_deinterleaving_sweep\moe_pdw_deinterleaving_N2
Model path      : D:\Fusion\turing_synthetic_radar_dataset\benchmark_moe_pdw_deinterleaving_sweep\moe_pdw_deinterleaving_N2\best_model.pt
Test folder     : D:\Fusion\turing_synthetic_radar_dataset\archive\test
LOADED MODEL
Model path     : D:\Fusion\turing_synthetic_radar_dataset\benchmark_moe_pdw_deinterleaving_sweep\moe_pdw_deinterleaving_N2\best_model.pt
N experts      : 2
Feature groups : [['toa_rel', 'delta_toa', 'pulse_width'], ['frequency', 'aoa_sin', 'aoa_cos', 'amplitude', 'amp_missing']]
emb_dim        : 64
hidden_dim     : 128
drop           : 0.15
SCANNING TEST FILES
Folder: D:\Fusion\turing_synthetic_radar_dataset\archive\test
H5 files found: 250


index_test: 100%|██████████| 250/250 [00:05<00:00, 45.19file/s]


SELECTED FILES FOR VIDEO
 file_id  n_pulses  n_windows
test_160   1324601       1293
test_248   1033601       1009
 test_97    945573        923
test_192    942433        920
test_126    912355        890
test_118    840930        821
test_174    820813        801
test_206    816434        797
 test_90    812943        793
 test_91    807539        788
[VIDEO] test_160 | windows=24 -> D:\Fusion\turing_synthetic_radar_dataset\benchmark_moe_pdw_deinterleaving_sweep\moe_pdw_deinterleaving_N2\xai_video_exports\videos\xai_test_160.mp4


[VIDEO] test_248 | windows=24 -> D:\Fusion\turing_synthetic_radar_dataset\benchmark_moe_pdw_deinterleaving_sweep\moe_pdw_deinterleaving_N2\xai_video_exports\videos\xai_test_248.mp4


[VIDEO] test_97 | windows=24 -> D:\Fusion\turing_synthetic_radar_dataset\benchmark_moe_pdw_deinterleaving_sweep\moe_pdw_deinterleaving_N2\xai_video_exports\videos\xai_test_97.mp4


...

[VIDEO] test_174 | windows=24 -> D:\Fusion\turing_synthetic_radar_dataset\benchmark_moe_pdw_deinterleaving_sweep\moe_pdw_deinterleaving_N2\xai_video_exports\videos\xai_test_174.mp4


[VIDEO] test_206 | windows=24 -> D:\Fusion\turing_synthetic_radar_dataset\benchmark_moe_pdw_deinterleaving_sweep\moe_pdw_deinterleaving_N2\xai_video_exports\videos\xai_test_206.mp4


[VIDEO] test_90 | windows=24 -> D:\Fusion\turing_synthetic_radar_dataset\benchmark_moe_pdw_deinterleaving_sweep\moe_pdw_deinterleaving_N2\xai_video_exports\videos\xai_test_90.mp4


[VIDEO] test_91 | windows=24 -> D:\Fusion\turing_synthetic_radar_dataset\benchmark_moe_pdw_deinterleaving_sweep\moe_pdw_deinterleaving_N2\xai_video_exports\videos\xai_test_91.mp4



XAI VIDEO SUMMARY
                         metric     mean   median      std
                      v_measure 0.684210 0.952836 0.435851
                            ari 0.697623 0.998666 0.447671
                            ami 0.683300 0.952006 0.435831
                    homogeneity 0.924666 0.989485 0.204151
                   completeness 0.686175 0.996544 0.443081
                   pairwise_mcc 0.699095 0.998667 0.447665
                 n_true_sources 6.904167 4.000000 7.204948
n_pred_clusters_excluding_noise 4.850000 3.000000 3.929611
                    noise_share 0.202946 0.005859 0.328239
                      n_regions 5.583333 8.000000 2.989755

OUTPUT FILES
Videos dir      : D:\Fusion\turing_synthetic_radar_dataset\benchmark_moe_pdw_deinterleaving_sweep\moe_pdw_deinterleaving_N2\xai_video_exports\videos
Video index     : D:\Fusion\turing_synthetic_radar_dataset\benchmark_moe_pdw_deinterleaving_sweep\moe_pdw_deinterleaving_N2\xai_video_exports\tables\xai_video_files.csv


# MOE/XAI COM ENG DE ATRIBUTOS

In [13]:
# ============================================================
# xai_video_moe_pdw_raw_vs_engineered.py
# ============================================================
# TSRD - XAI + videos for MoE-based PDW deinterleaving
# Comparison:
#   1) raw_pdw_moe
#   2) engineered_pdw_moe
#
# This script loads trained PDWFeatureMoE models from:
#   D:\Fusion\turing_synthetic_radar_dataset\
#       benchmark_moe_pdw_raw_vs_engineered_sweep\
#       raw_pdw_moe_N2\best_model.pt
#
#   D:\Fusion\turing_synthetic_radar_dataset\
#       benchmark_moe_pdw_raw_vs_engineered_sweep\
#       engineered_pdw_moe_N2\best_model.pt
#
# Data:
#   D:\Fusion\turing_synthetic_radar_dataset\archive\test
#
# Video frame:
#   - PDW feature matrix: features x pulses
#   - XAI saliency overlay: gradient x input
#   - HDBSCAN predicted clusters
#   - True emitter labels
#   - MoE gate weights
#
# XAI target:
#   Cluster compactness in learned embedding space.
#
# Important:
#   This is not class Grad-CAM because the model does not output
#   classes. It learns pulse embeddings, and HDBSCAN separates
#   pulses into emitter clusters.
#
# Outputs:
#   D:\Fusion\turing_synthetic_radar_dataset\
#       benchmark_moe_pdw_raw_vs_engineered_sweep\
#       xai_video_raw_vs_engineered_N2
# ============================================================

from pathlib import Path
import sys
import json
import math
import random
import re
import subprocess
import warnings
warnings.filterwarnings("ignore")


# ============================================================
# INSTALL / IMPORTS
# ============================================================
def ensure_package(import_name, pip_name=None):
    try:
        __import__(import_name)
    except Exception:
        pkg = pip_name if pip_name is not None else import_name
        print(f"[INFO] Installing missing package: {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


ensure_package("numpy")
ensure_package("pandas")
ensure_package("matplotlib")
ensure_package("h5py")
ensure_package("torch")
ensure_package("sklearn", "scikit-learn")
ensure_package("scipy")
ensure_package("imageio")
ensure_package("imageio_ffmpeg", "imageio-ffmpeg")
ensure_package("tqdm")
ensure_package("tabulate")

try:
    ensure_package("hdbscan")
    import hdbscan
    HDBSCAN_BACKEND = "hdbscan"
except Exception:
    hdbscan = None
    HDBSCAN_BACKEND = "sklearn"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
import imageio.v2 as imageio

from tqdm import tqdm
from scipy import ndimage
from matplotlib.patches import Rectangle

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    v_measure_score,
    adjusted_rand_score,
    adjusted_mutual_info_score,
    homogeneity_score,
    completeness_score,
)

try:
    from sklearn.cluster import HDBSCAN as SKHDBSCAN
except Exception:
    SKHDBSCAN = None


# ============================================================
# PATHS
# ============================================================
ROOT = Path(r"D:\Fusion\turing_synthetic_radar_dataset")

TEST_DIR = ROOT / "archive" / "test"

BASE_RUN_DIR = ROOT / "benchmark_moe_pdw_raw_vs_engineered_sweep"

N_EXPERTS_XAI = 2

PREPROCESS_MODES = [
    "raw_pdw_moe",
    "engineered_pdw_moe",
]

XAI_COMPARE_OUT_DIR = BASE_RUN_DIR / f"xai_video_raw_vs_engineered_N{N_EXPERTS_XAI}"
VIDEOS_DIR = XAI_COMPARE_OUT_DIR / "videos"
TABLES_DIR = XAI_COMPARE_OUT_DIR / "tables"
PLOTS_DIR = XAI_COMPARE_OUT_DIR / "plots"

XAI_COMPARE_OUT_DIR.mkdir(parents=True, exist_ok=True)
VIDEOS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# CONFIG
# ============================================================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

WINDOW_SIZE = 1024
DROP_INCOMPLETE_WINDOWS = True

# Number of test files for video generation.
MAX_FILES_FOR_VIDEO = 10

# Number of windows per selected file.
MAX_WINDOWS_PER_FILE_VIDEO = 24

# Window selection inside each file:
#   "first" -> first windows
#   "even"  -> evenly spread windows
WINDOW_SELECTION_MODE = "even"

# Video
VIDEO_FPS = 2
VIDEO_CODEC = "libx264"
VIDEO_QUALITY = 7
FRAME_FIGSIZE = (13.5, 7.5)
FRAME_DPI = 130

# Model defaults; overwritten by results.json when available.
EMB_DIM = 64
HIDDEN_DIM = 128
DROP = 0.15

# HDBSCAN over embeddings
HDBSCAN_MIN_CLUSTER_SIZE = 5
HDBSCAN_MIN_SAMPLES = 3
HDBSCAN_CLUSTER_SELECTION_EPSILON = 0.0
HDBSCAN_CLUSTER_SELECTION_METHOD = "eom"
HDBSCAN_ALLOW_SINGLE_CLUSTER = False

# XAI
XAI_METHOD = "cluster_compactness_grad_x_input"
SALIENCY_PERCENTILE = 95.0
REGION_MIN_AREA = 4
REGION_TOP_K = 8
REGION_STRUCT_SIZE = 2
REGION_OPEN_ITERS = 0
REGION_CLOSE_ITERS = 1

# Metrics
NOISE_POLICY_FOR_METRICS = "as_cluster"

RAW_FEATURE_NAMES = [
    "ToA_us",
    "Frequency_MHz",
    "PulseWidth_us",
    "AoA_deg",
    "Amplitude_dB",
]

ENGINEERED_FEATURE_NAMES = [
    "toa_rel_norm",
    "delta_toa_log",
    "frequency_norm",
    "delta_frequency_abs_norm",
    "pulsewidth_log",
    "aoa_sin",
    "aoa_cos",
    "delta_aoa_abs_norm",
    "amplitude_norm",
    "amplitude_missing",
]

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 180


# ============================================================
# REPRODUCIBILITY
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)


# ============================================================
# HELPERS
# ============================================================
def safe_filename(s: str) -> str:
    s = str(s)
    s = re.sub(r"[^\w\-.]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s[:180] if len(s) > 180 else s


def save_json(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def load_json(path: Path, default=None):
    if not path.exists():
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def normalize_map(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    if x.size == 0:
        return x
    mn = float(np.nanmin(x))
    mx = float(np.nanmax(x))
    if mx - mn < 1e-9:
        return np.zeros_like(x, dtype=np.float32)
    return ((x - mn) / (mx - mn + 1e-9)).astype(np.float32)


def robust_display_matrix(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    out = np.zeros_like(x, dtype=np.float32)

    for j in range(x.shape[1]):
        col = x[:, j]
        lo = np.percentile(col, 2)
        hi = np.percentile(col, 98)
        if hi - lo < 1e-9:
            out[:, j] = 0.0
        else:
            out[:, j] = np.clip((col - lo) / (hi - lo + 1e-9), 0, 1)

    return out


def h5_has_dataset(f, name):
    try:
        return isinstance(f[name], h5py.Dataset)
    except Exception:
        return False


def comb2(x):
    x = np.asarray(x, dtype=np.int64)
    return x * (x - 1) // 2


def safe_float(x):
    try:
        if np.isfinite(x):
            return float(x)
        return float("nan")
    except Exception:
        return float("nan")


def finite_median(v, default=0.0):
    v = np.asarray(v, dtype=np.float32)
    mask = np.isfinite(v)
    if mask.any():
        return float(np.nanmedian(v[mask]))
    return float(default)


def fill_nonfinite_with_median(v, default=0.0):
    v = np.asarray(v, dtype=np.float32).copy()
    med = finite_median(v, default=default)
    v[~np.isfinite(v)] = med
    return v


def compact_labels(labels):
    labels = np.asarray(labels).reshape(-1)
    uniques = sorted(np.unique(labels).tolist())
    mapping = {u: i for i, u in enumerate(uniques)}
    compact = np.array([mapping[v] for v in labels], dtype=np.int32)
    return compact, mapping


def fig_to_rgb_array(fig):
    fig.canvas.draw()
    w, h = fig.canvas.get_width_height()
    buf = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    buf = buf.reshape(h, w, 4)
    return buf[:, :, :3].copy()


def ensure_video_compatible_frame(im: np.ndarray) -> np.ndarray:
    if im.dtype != np.uint8:
        im = np.clip(im, 0, 255).astype(np.uint8)

    if im.ndim == 2:
        im = np.stack([im, im, im], axis=-1)

    if im.shape[2] == 4:
        im = im[:, :, :3]

    h, w = im.shape[:2]
    pad_h = h % 2
    pad_w = w % 2

    if pad_h != 0 or pad_w != 0:
        new_im = np.zeros((h + pad_h, w + pad_w, 3), dtype=np.uint8)
        new_im[:h, :w, :] = im
        im = new_im

    return im


def angular_abs_diff_deg(a):
    a = np.asarray(a, dtype=np.float32)
    prev = np.concatenate([[a[0]], a[:-1]])
    diff = (a - prev + 180.0) % 360.0 - 180.0
    return np.abs(diff).astype(np.float32)


def get_feature_names(preprocess_mode):
    if preprocess_mode == "raw_pdw_moe":
        return RAW_FEATURE_NAMES
    if preprocess_mode == "engineered_pdw_moe":
        return ENGINEERED_FEATURE_NAMES
    raise ValueError(f"Unknown preprocess_mode: {preprocess_mode}")


# ============================================================
# PDW PREPROCESSING
# ============================================================
def sanitize_raw_pdws_for_moe(x):
    """
    RAW MoE input:
      - original 5 PDW columns
      - minimal finite sanitation
      - per-window RobustScaler

    Output:
      [N, 5]
    """
    x = np.asarray(x, dtype=np.float32).copy()

    if x.ndim != 2 or x.shape[1] < 5:
        raise ValueError(f"Expected [n_pulses, >=5], got {x.shape}")

    x = x[:, :5]

    amp = x[:, 4].copy()
    amp[~np.isfinite(amp)] = -200.0
    x[:, 4] = amp

    for j in range(5):
        col = x[:, j].copy()
        if not np.isfinite(col).all():
            med = finite_median(col, default=0.0)
            col[~np.isfinite(col)] = med
            x[:, j] = col

    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    try:
        scaler = RobustScaler()
        z = scaler.fit_transform(x).astype(np.float32)
    except Exception:
        z = x.astype(np.float32)

    z = np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0)
    return z.astype(np.float32)


def engineer_pdw_features_for_moe(x):
    """
    ENGINEERED MoE input.

    Raw:
      0 ToA_us
      1 Frequency_MHz
      2 PulseWidth_us
      3 AoA_deg
      4 Amplitude_dB

    Engineered:
      0 toa_rel_norm
      1 delta_toa_log
      2 frequency_norm
      3 delta_frequency_abs_norm
      4 pulsewidth_log
      5 aoa_sin
      6 aoa_cos
      7 delta_aoa_abs_norm
      8 amplitude_norm
      9 amplitude_missing

    Output:
      [N, 10]
    """
    x = np.asarray(x, dtype=np.float32)

    if x.ndim != 2 or x.shape[1] < 5:
        raise ValueError(f"Expected [n_pulses, >=5], got {x.shape}")

    toa = fill_nonfinite_with_median(x[:, 0], default=0.0)
    freq = fill_nonfinite_with_median(x[:, 1], default=0.0)
    pw = fill_nonfinite_with_median(x[:, 2], default=0.0)
    aoa = fill_nonfinite_with_median(x[:, 3], default=0.0)

    amp = x[:, 4].astype(np.float32).copy()
    amp_missing = (~np.isfinite(amp)).astype(np.float32)
    amp[~np.isfinite(amp)] = -200.0
    amp = fill_nonfinite_with_median(amp, default=-200.0)

    toa_rel = toa - np.min(toa)
    span = float(np.max(toa_rel) - np.min(toa_rel))
    if span > 0:
        toa_rel_norm = toa_rel / (span + 1e-9)
    else:
        toa_rel_norm = np.zeros_like(toa_rel, dtype=np.float32)

    dtoa = np.diff(toa, prepend=toa[0])
    dtoa = np.maximum(dtoa, 0.0)
    dtoa_log = np.log1p(dtoa)

    frequency_norm = freq / 18000.0

    delta_frequency = np.abs(np.diff(freq, prepend=freq[0]))
    delta_frequency_abs_norm = delta_frequency / 18000.0

    pw = np.maximum(pw, 0.0)
    pulsewidth_log = np.log1p(pw)

    aoa_rad = np.deg2rad(aoa)
    aoa_sin = np.sin(aoa_rad)
    aoa_cos = np.cos(aoa_rad)

    delta_aoa_abs_norm = angular_abs_diff_deg(aoa) / 180.0

    amplitude_norm = (amp + 220.0) / 260.0

    feats = np.stack(
        [
            toa_rel_norm,
            dtoa_log,
            frequency_norm,
            delta_frequency_abs_norm,
            pulsewidth_log,
            aoa_sin,
            aoa_cos,
            delta_aoa_abs_norm,
            amplitude_norm,
            amp_missing,
        ],
        axis=1,
    ).astype(np.float32)

    feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)

    z = feats.copy()

    continuous_cols = list(range(9))
    binary_cols = [9]

    try:
        scaler = RobustScaler()
        z[:, continuous_cols] = scaler.fit_transform(z[:, continuous_cols])
    except Exception:
        z[:, continuous_cols] = feats[:, continuous_cols]

    z[:, binary_cols] = feats[:, binary_cols]

    z = np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0)
    return z.astype(np.float32)


def preprocess_pdw_window(x_raw, preprocess_mode):
    if preprocess_mode == "raw_pdw_moe":
        return sanitize_raw_pdws_for_moe(x_raw)

    if preprocess_mode == "engineered_pdw_moe":
        return engineer_pdw_features_for_moe(x_raw)

    raise ValueError(f"Unknown preprocess_mode: {preprocess_mode}")


# ============================================================
# MODEL DEFINITIONS
# ============================================================
class FeatureExpert(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, emb_dim=64, drop=0.15):
        super().__init__()

        self.point = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
        )

        self.temporal = nn.Sequential(
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2),
            nn.GELU(),
        )

        self.norm = nn.LayerNorm(hidden_dim)

        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, emb_dim),
            nn.GELU(),
        )

    def forward(self, x):
        # x: [B, N, Fin]
        h = self.point(x)
        ht = h.transpose(1, 2)
        ht = self.temporal(ht)
        ht = ht.transpose(1, 2)
        h = self.norm(h + ht)
        z = self.proj(h)
        return z


class PDWFeatureMoE(nn.Module):
    def __init__(self, feature_groups, input_dim, hidden_dim=128, emb_dim=64, drop=0.15):
        super().__init__()

        self.feature_groups = [list(g) for g in feature_groups]
        self.n_experts = len(self.feature_groups)
        self.emb_dim = emb_dim

        self.experts = nn.ModuleList([
            FeatureExpert(
                in_dim=len(g),
                hidden_dim=hidden_dim,
                emb_dim=emb_dim,
                drop=drop,
            )
            for g in self.feature_groups
        ])

        self.gate = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(hidden_dim, self.n_experts),
        )

        self.out_norm = nn.LayerNorm(emb_dim)

    def forward(self, x):
        # x: [B, N, F]
        gate_logits = self.gate(x)
        gate_w = torch.softmax(gate_logits, dim=-1)

        expert_outs = []
        for expert, group in zip(self.experts, self.feature_groups):
            xg = x[:, :, group]
            zg = expert(xg)
            expert_outs.append(zg)

        E = torch.stack(expert_outs, dim=2)
        z = (E * gate_w.unsqueeze(-1)).sum(dim=2)
        z = self.out_norm(z)

        return {
            "emb": z,
            "gate_w": gate_w,
        }


def default_feature_groups(preprocess_mode, n_experts):
    if preprocess_mode == "raw_pdw_moe":
        if n_experts == 2:
            return [[0, 2], [1, 3, 4]]
        if n_experts == 3:
            return [[0], [1, 2], [3, 4]]
        if n_experts == 4:
            return [[0], [1], [2], [3, 4]]
        if n_experts == 5:
            return [[0], [1], [2], [3], [4]]

    if preprocess_mode == "engineered_pdw_moe":
        if n_experts == 2:
            return [[0, 1, 4], [2, 3, 5, 6, 7, 8, 9]]
        if n_experts == 3:
            return [[0, 1], [2, 3, 4], [5, 6, 7, 8, 9]]
        if n_experts == 4:
            return [[0, 1], [2, 3], [4], [5, 6, 7, 8, 9]]
        if n_experts == 5:
            return [[0, 1], [2, 3], [4], [5, 6, 7], [8, 9]]

    raise ValueError(f"Unsupported preprocess_mode={preprocess_mode}, n_experts={n_experts}")


def describe_feature_groups(preprocess_mode, groups):
    names = get_feature_names(preprocess_mode)
    return [[names[i] for i in g] for g in groups]


def load_trained_model(preprocess_mode):
    run_dir = BASE_RUN_DIR / f"{preprocess_mode}_N{N_EXPERTS_XAI}"

    model_path = run_dir / "best_model.pt"
    feature_groups_path = run_dir / "feature_groups.json"
    results_json_path = run_dir / "results.json"

    if not model_path.exists():
        raise FileNotFoundError(
            f"Modelo não encontrado: {model_path}\n"
            f"Rode primeiro o treinamento raw vs engineered."
        )

    result_cfg = load_json(results_json_path, default={}) or {}
    emb_dim = int(result_cfg.get("emb_dim", EMB_DIM))
    hidden_dim = int(result_cfg.get("hidden_dim", HIDDEN_DIM))
    drop = float(result_cfg.get("drop", DROP))

    fg_cfg = load_json(feature_groups_path, default={}) or {}
    feature_groups = fg_cfg.get("feature_groups_indices", None)

    if feature_groups is None:
        feature_groups = default_feature_groups(preprocess_mode, N_EXPERTS_XAI)

    input_dim = len(get_feature_names(preprocess_mode))

    model = PDWFeatureMoE(
        feature_groups=feature_groups,
        input_dim=input_dim,
        hidden_dim=hidden_dim,
        emb_dim=emb_dim,
        drop=drop,
    ).to(DEVICE)

    state = torch.load(model_path, map_location=DEVICE)
    model.load_state_dict(state)
    model.eval()

    print("=" * 80)
    print("LOADED MODEL")
    print("=" * 80)
    print(f"Mode           : {preprocess_mode}")
    print(f"Run dir        : {run_dir}")
    print(f"Model path     : {model_path}")
    print(f"N experts      : {len(feature_groups)}")
    print(f"Input dim      : {input_dim}")
    print(f"Feature groups : {describe_feature_groups(preprocess_mode, feature_groups)}")
    print(f"emb_dim        : {emb_dim}")
    print(f"hidden_dim     : {hidden_dim}")
    print(f"drop           : {drop}")

    return model, feature_groups, run_dir


# ============================================================
# HDBSCAN + METRICS
# ============================================================
def run_hdbscan(X):
    X = np.asarray(X, dtype=np.float32)

    if HDBSCAN_BACKEND == "hdbscan" and hdbscan is not None:
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
            min_samples=HDBSCAN_MIN_SAMPLES,
            metric="euclidean",
            cluster_selection_epsilon=HDBSCAN_CLUSTER_SELECTION_EPSILON,
            cluster_selection_method=HDBSCAN_CLUSTER_SELECTION_METHOD,
            allow_single_cluster=HDBSCAN_ALLOW_SINGLE_CLUSTER,
        )
        return clusterer.fit_predict(X).astype(int)

    if SKHDBSCAN is not None:
        kwargs = dict(
            min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
            metric="euclidean",
            cluster_selection_epsilon=HDBSCAN_CLUSTER_SELECTION_EPSILON,
            allow_single_cluster=HDBSCAN_ALLOW_SINGLE_CLUSTER,
        )

        if HDBSCAN_MIN_SAMPLES is not None:
            kwargs["min_samples"] = HDBSCAN_MIN_SAMPLES

        clusterer = SKHDBSCAN(**kwargs)
        return clusterer.fit_predict(X).astype(int)

    raise RuntimeError("No HDBSCAN backend available. Try: pip install hdbscan")


def apply_noise_policy(y_pred):
    y_pred = np.asarray(y_pred).reshape(-1).astype(int).copy()

    if NOISE_POLICY_FOR_METRICS == "as_cluster":
        return y_pred

    if NOISE_POLICY_FOR_METRICS == "unique_noise":
        noise_idx = np.where(y_pred == -1)[0]
        if len(noise_idx) == 0:
            return y_pred

        max_cluster = int(np.max(y_pred)) if len(y_pred) > 0 else 0
        next_label = max_cluster + 1

        for k, idx in enumerate(noise_idx):
            y_pred[idx] = next_label + k

        return y_pred

    raise ValueError(f"Unknown NOISE_POLICY_FOR_METRICS: {NOISE_POLICY_FOR_METRICS}")


def efficient_pairwise_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    n = len(y_true)
    total_pairs = n * (n - 1) // 2

    if n < 2 or total_pairs <= 0:
        return {
            "pairwise_f1": np.nan,
            "pairwise_mcc": np.nan,
        }

    true_vals, true_inv = np.unique(y_true, return_inverse=True)
    pred_vals, pred_inv = np.unique(y_pred, return_inverse=True)

    contingency = np.zeros((len(true_vals), len(pred_vals)), dtype=np.int64)
    np.add.at(contingency, (true_inv, pred_inv), 1)

    tp = int(comb2(contingency).sum())
    true_sizes = contingency.sum(axis=1)
    pred_sizes = contingency.sum(axis=0)

    true_pairs = int(comb2(true_sizes).sum())
    pred_pairs = int(comb2(pred_sizes).sum())

    fp = int(pred_pairs - tp)
    fn = int(true_pairs - tp)
    tn = int(total_pairs - tp - fp - fn)

    denom_f1 = 2 * tp + fp + fn
    pairwise_f1 = float((2 * tp) / denom_f1) if denom_f1 > 0 else 0.0

    denom_mcc = math.sqrt(
        max(tp + fp, 0)
        * max(tp + fn, 0)
        * max(tn + fp, 0)
        * max(tn + fn, 0)
    )

    pairwise_mcc = float((tp * tn - fp * fn) / denom_mcc) if denom_mcc > 0 else 0.0

    return {
        "pairwise_f1": pairwise_f1,
        "pairwise_mcc": pairwise_mcc,
    }


def deinterleaving_metrics(y_true, y_pred_raw):
    y_true = np.asarray(y_true).reshape(-1).astype(int)
    y_pred_raw = np.asarray(y_pred_raw).reshape(-1).astype(int)
    y_pred_eval = apply_noise_policy(y_pred_raw)

    out = {
        "v_measure": safe_float(v_measure_score(y_true, y_pred_eval)),
        "ari": safe_float(adjusted_rand_score(y_true, y_pred_eval)),
        "ami": safe_float(adjusted_mutual_info_score(y_true, y_pred_eval)),
        "homogeneity": safe_float(homogeneity_score(y_true, y_pred_eval)),
        "completeness": safe_float(completeness_score(y_true, y_pred_eval)),
    }

    out.update(efficient_pairwise_metrics(y_true, y_pred_eval))

    pred_clusters_raw = np.unique(y_pred_raw)

    out["n_true_sources"] = int(len(np.unique(y_true)))
    out["n_pred_clusters_excluding_noise"] = int(len([c for c in pred_clusters_raw if c != -1]))
    out["noise_share"] = float(np.mean(y_pred_raw == -1))

    return out


# ============================================================
# FILE SELECTION
# ============================================================
def collect_test_files():
    rows = []

    h5_files = sorted(TEST_DIR.rglob("*.h5"))

    print("=" * 80)
    print("SCANNING TEST FILES")
    print("=" * 80)
    print(f"Folder: {TEST_DIR}")
    print(f"H5 files found: {len(h5_files)}")

    for p in tqdm(h5_files, desc="index_test", unit="file"):
        try:
            with h5py.File(p, "r") as f:
                if not h5_has_dataset(f, "data") or not h5_has_dataset(f, "labels"):
                    continue

                n = int(f["data"].shape[0])
                if DROP_INCOMPLETE_WINDOWS:
                    n_windows = n // WINDOW_SIZE
                else:
                    n_windows = int(math.ceil(n / WINDOW_SIZE))

                if n_windows <= 0:
                    continue

                rows.append({
                    "file_id": p.stem,
                    "file_name": p.name,
                    "path": str(p),
                    "n_pulses": n,
                    "n_windows": n_windows,
                })

        except Exception as e:
            print(f"[Warning] Could not read {p}: {e}")

    df = pd.DataFrame(rows)

    if len(df) == 0:
        raise RuntimeError("No valid test files found.")

    df = df.sort_values("n_windows", ascending=False).reset_index(drop=True)
    df.to_csv(TABLES_DIR / "xai_test_file_index.csv", index=False, encoding="utf-8-sig")

    return df


def select_window_indices(n_windows, max_windows):
    n_windows = int(n_windows)

    if max_windows is None or n_windows <= max_windows:
        return list(range(n_windows))

    if WINDOW_SELECTION_MODE == "first":
        return list(range(int(max_windows)))

    if WINDOW_SELECTION_MODE == "even":
        idx = np.linspace(0, n_windows - 1, int(max_windows)).round().astype(int)
        return sorted(set(idx.tolist()))

    raise ValueError(f"Unknown WINDOW_SELECTION_MODE: {WINDOW_SELECTION_MODE}")


def load_window(path: Path, window_index: int, preprocess_mode: str):
    start = int(window_index) * WINDOW_SIZE
    end = start + WINDOW_SIZE

    with h5py.File(path, "r") as f:
        x_raw = np.asarray(f["data"][start:end, :5], dtype=np.float32)
        y_true = np.asarray(f["labels"][start:end]).reshape(-1).astype(np.int64)

    x_proc = preprocess_pdw_window(x_raw, preprocess_mode)

    return x_raw, x_proc, y_true, start, end


# ============================================================
# XAI
# ============================================================
@torch.no_grad()
def infer_embedding_and_gate(model, x_proc):
    x = torch.tensor(x_proc, dtype=torch.float32, device=DEVICE).unsqueeze(0)

    out = model(x)
    emb = F.normalize(out["emb"].float(), dim=-1)[0].detach().cpu().numpy()
    gate = out["gate_w"][0].detach().cpu().numpy()

    return emb, gate


def compute_cluster_compactness_saliency(model, x_proc, y_pred):
    """
    XAI objective:
      maximize compactness of predicted clusters in embedding space.

    Saliency:
      abs(gradient * input) over [pulse, feature].
    """
    model.eval()

    x = torch.tensor(x_proc, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    x.requires_grad_(True)

    out = model(x)
    emb = F.normalize(out["emb"].float(), dim=-1)[0]
    gate = out["gate_w"][0]

    y_pred = np.asarray(y_pred).astype(int)
    clusters = [c for c in np.unique(y_pred).tolist() if c != -1]

    scores = []

    for c in clusters:
        idx_np = np.where(y_pred == c)[0]
        if len(idx_np) < 2:
            continue

        idx = torch.tensor(idx_np, dtype=torch.long, device=DEVICE)
        zc = emb[idx]
        centroid = F.normalize(zc.mean(dim=0, keepdim=True), dim=-1)
        sim = (zc * centroid).sum(dim=-1).mean()
        scores.append(sim)

    if len(scores) == 0:
        score = emb.pow(2).mean()
    else:
        score = torch.stack(scores).mean()

    model.zero_grad(set_to_none=True)
    score.backward()

    grad = x.grad.detach()[0]
    inp = x.detach()[0]

    sal = torch.abs(grad * inp).cpu().numpy().astype(np.float32)
    sal = normalize_map(sal)

    gate_np = gate.detach().cpu().numpy().astype(np.float32)

    return {
        "saliency": sal,
        "gate": gate_np,
        "xai_score": float(score.detach().cpu().item()),
    }


def detect_salient_regions(saliency):
    """
    saliency: [N, F]
    returns bboxes in display coordinates:
        x1, y1, x2, y2
    where x = pulse index, y = feature index.
    """
    sal = normalize_map(saliency)

    if sal.size == 0:
        return [], np.zeros_like(sal, dtype=np.uint8), 0.0

    thr = float(np.percentile(sal, SALIENCY_PERCENTILE))
    thr = min(max(thr, 0.05), 0.98)

    mask = sal >= thr

    structure = np.ones((REGION_STRUCT_SIZE, REGION_STRUCT_SIZE), dtype=bool)

    if REGION_OPEN_ITERS > 0:
        mask = ndimage.binary_opening(mask, structure=structure, iterations=REGION_OPEN_ITERS)

    if REGION_CLOSE_ITERS > 0:
        mask = ndimage.binary_closing(mask, structure=structure, iterations=REGION_CLOSE_ITERS)

    labeled, n_comp = ndimage.label(mask)

    regions = []

    for lab in range(1, n_comp + 1):
        xx, yy = np.where(labeled == lab)
        if xx.size == 0:
            continue

        area = int(xx.size)
        if area < REGION_MIN_AREA:
            continue

        x1 = int(xx.min())
        x2 = int(xx.max())
        y1 = int(yy.min())
        y2 = int(yy.max())

        vals = sal[xx, yy]

        regions.append({
            "bbox": (x1, y1, x2, y2),
            "area": area,
            "mean_score": float(vals.mean()),
            "max_score": float(vals.max()),
            "sum_score": float(vals.sum()),
        })

    regions = sorted(
        regions,
        key=lambda r: (r["max_score"], r["mean_score"], r["area"]),
        reverse=True,
    )[:REGION_TOP_K]

    return regions, mask.astype(np.uint8), thr


# ============================================================
# FRAME RENDERING
# ============================================================
def render_xai_frame(
    x_proc,
    y_true,
    y_pred,
    saliency,
    gate,
    regions,
    metrics,
    file_id,
    window_index,
    pulse_start,
    pulse_end,
    xai_score,
    preprocess_mode,
):
    feature_names = get_feature_names(preprocess_mode)

    base = robust_display_matrix(x_proc)
    sal = normalize_map(saliency)

    y_true_compact, _ = compact_labels(y_true)
    y_pred_compact, _ = compact_labels(y_pred)

    mean_gate = gate.mean(axis=0)

    fig = plt.figure(figsize=FRAME_FIGSIZE, dpi=FRAME_DPI)
    gs = fig.add_gridspec(
        4,
        2,
        width_ratios=[4.7, 1.25],
        height_ratios=[3.0, 0.55, 0.55, 0.9],
    )

    ax0 = fig.add_subplot(gs[0, 0])
    ax_gate = fig.add_subplot(gs[0, 1])
    ax_pred = fig.add_subplot(gs[1, 0])
    ax_true = fig.add_subplot(gs[2, 0])
    ax_text = fig.add_subplot(gs[1:, 1])
    ax_sal = fig.add_subplot(gs[3, 0])

    ax0.imshow(base.T, aspect="auto", origin="lower", cmap="viridis")
    ax0.imshow(sal.T, aspect="auto", origin="lower", cmap="hot", alpha=0.45)

    for r in regions:
        x1, y1, x2, y2 = r["bbox"]
        rect = Rectangle(
            (x1, y1),
            max(1, x2 - x1 + 1),
            max(1, y2 - y1 + 1),
            linewidth=1.4,
            edgecolor="cyan",
            facecolor="none",
        )
        ax0.add_patch(rect)
        ax0.text(
            x1,
            min(y2 + 0.2, len(feature_names) - 1),
            "R",
            color="white",
            fontsize=7,
            bbox=dict(boxstyle="round,pad=0.12", facecolor="black", alpha=0.6),
        )

    ax0.set_yticks(np.arange(len(feature_names)))
    ax0.set_yticklabels(feature_names, fontsize=7)
    ax0.set_ylabel("PDW features")
    ax0.set_xlabel("Pulse index inside window")
    ax0.set_title(
        f"{preprocess_mode} | {file_id} | window={window_index} | pulses={pulse_start}-{pulse_end}"
    )

    ax_gate.bar([f"E{i}" for i in range(len(mean_gate))], mean_gate)
    ax_gate.set_ylim(0, 1)
    ax_gate.set_title("Mean gate")
    ax_gate.tick_params(axis="x", labelsize=8)

    ax_pred.imshow(y_pred_compact.reshape(1, -1), aspect="auto", cmap="tab20")
    ax_pred.set_yticks([])
    ax_pred.set_ylabel("Pred", rotation=0, labelpad=24)
    ax_pred.set_xticks([])

    ax_true.imshow(y_true_compact.reshape(1, -1), aspect="auto", cmap="tab20")
    ax_true.set_yticks([])
    ax_true.set_ylabel("True", rotation=0, labelpad=24)
    ax_true.set_xticks([])

    sal_pulse = sal.mean(axis=1)
    ax_sal.plot(np.arange(len(sal_pulse)), sal_pulse, linewidth=1.0)
    ax_sal.set_ylim(0, 1.05)
    ax_sal.set_ylabel("Saliency")
    ax_sal.set_xlabel("Pulse index")

    ax_text.axis("off")

    txt = [
        f"Mode: {preprocess_mode}",
        f"XAI: {XAI_METHOD}",
        f"xai_score: {xai_score:.4f}",
        "",
        f"V-measure: {metrics['v_measure']:.3f}",
        f"ARI: {metrics['ari']:.3f}",
        f"AMI: {metrics['ami']:.3f}",
        f"Homogeneity: {metrics['homogeneity']:.3f}",
        f"Completeness: {metrics['completeness']:.3f}",
        f"Pairwise MCC: {metrics['pairwise_mcc']:.3f}",
        "",
        f"True sources: {metrics['n_true_sources']}",
        f"Pred clusters: {metrics['n_pred_clusters_excluding_noise']}",
        f"Noise share: {metrics['noise_share']:.3f}",
        f"Regions: {len(regions)}",
    ]

    ax_text.text(
        0.02,
        0.98,
        "\n".join(txt),
        transform=ax_text.transAxes,
        va="top",
        ha="left",
        fontsize=9,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.85),
    )

    plt.tight_layout()

    frame = fig_to_rgb_array(fig)
    plt.close(fig)

    return ensure_video_compatible_frame(frame)


# ============================================================
# VIDEO GENERATION
# ============================================================
def make_xai_video_for_file(model, file_row, preprocess_mode):
    file_path = Path(file_row["path"])
    file_id = str(file_row["file_id"])
    n_windows = int(file_row["n_windows"])

    win_indices = select_window_indices(
        n_windows=n_windows,
        max_windows=MAX_WINDOWS_PER_FILE_VIDEO,
    )

    mode_video_dir = VIDEOS_DIR / preprocess_mode
    mode_video_dir.mkdir(parents=True, exist_ok=True)

    out_video = mode_video_dir / f"xai_{preprocess_mode}_{safe_filename(file_id)}.mp4"

    print(f"[VIDEO] mode={preprocess_mode} | {file_id} | windows={len(win_indices)} -> {out_video}")

    writer = None
    window_rows = []
    region_rows = []

    try:
        writer = imageio.get_writer(
            str(out_video),
            fps=VIDEO_FPS,
            codec=VIDEO_CODEC,
            quality=VIDEO_QUALITY,
            macro_block_size=None,
            pixelformat="yuv420p",
        )

        for w in tqdm(win_indices, desc=f"video_{preprocess_mode}_{file_id}", leave=False):
            x_raw, x_proc, y_true, pulse_start, pulse_end = load_window(
                file_path,
                w,
                preprocess_mode=preprocess_mode,
            )

            emb, gate_initial = infer_embedding_and_gate(model, x_proc)
            y_pred = run_hdbscan(emb)

            metrics = deinterleaving_metrics(y_true, y_pred)

            xai = compute_cluster_compactness_saliency(model, x_proc, y_pred)
            saliency = xai["saliency"]
            gate = xai["gate"]
            xai_score = xai["xai_score"]

            regions, mask, thr = detect_salient_regions(saliency)

            feature_names = get_feature_names(preprocess_mode)

            for ridx, r in enumerate(regions):
                x1, y1, x2, y2 = r["bbox"]
                region_rows.append({
                    "preprocess_mode": preprocess_mode,
                    "file_id": file_id,
                    "window_index": int(w),
                    "region_id": int(ridx),
                    "x1_pulse": int(x1),
                    "x2_pulse": int(x2),
                    "y1_feature": int(y1),
                    "y2_feature": int(y2),
                    "feature_start": feature_names[y1] if 0 <= y1 < len(feature_names) else "",
                    "feature_end": feature_names[y2] if 0 <= y2 < len(feature_names) else "",
                    "area": int(r["area"]),
                    "mean_score": float(r["mean_score"]),
                    "max_score": float(r["max_score"]),
                    "sum_score": float(r["sum_score"]),
                    "threshold": float(thr),
                })

            row = {
                "preprocess_mode": preprocess_mode,
                "file_id": file_id,
                "file_path": str(file_path),
                "window_index": int(w),
                "pulse_start": int(pulse_start),
                "pulse_end": int(pulse_end),
                "xai_score": float(xai_score),
                "n_regions": int(len(regions)),
                "mean_gate_json": json.dumps(gate.mean(axis=0).tolist()),
            }
            row.update(metrics)
            window_rows.append(row)

            frame = render_xai_frame(
                x_proc=x_proc,
                y_true=y_true,
                y_pred=y_pred,
                saliency=saliency,
                gate=gate,
                regions=regions,
                metrics=metrics,
                file_id=file_id,
                window_index=w,
                pulse_start=pulse_start,
                pulse_end=pulse_end,
                xai_score=xai_score,
                preprocess_mode=preprocess_mode,
            )

            writer.append_data(frame)

    finally:
        if writer is not None:
            writer.close()

    return out_video, window_rows, region_rows


# ============================================================
# SUMMARY / COMPARISON
# ============================================================
def summarize_window_metrics(window_df):
    metrics = [
        "v_measure",
        "ari",
        "ami",
        "homogeneity",
        "completeness",
        "pairwise_mcc",
        "pairwise_f1",
        "n_true_sources",
        "n_pred_clusters_excluding_noise",
        "noise_share",
        "n_regions",
        "xai_score",
    ]

    rows = []

    for mode, g in window_df.groupby("preprocess_mode"):
        row = {
            "preprocess_mode": mode,
            "n_windows": int(len(g)),
        }

        for c in metrics:
            if c not in g.columns:
                continue

            vals = pd.to_numeric(g[c], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()

            if len(vals) == 0:
                row[f"{c}_mean"] = np.nan
                row[f"{c}_median"] = np.nan
                row[f"{c}_std"] = np.nan
            else:
                row[f"{c}_mean"] = float(vals.mean())
                row[f"{c}_median"] = float(vals.median())
                row[f"{c}_std"] = float(vals.std())

        rows.append(row)

    return pd.DataFrame(rows)


def build_raw_vs_engineered_window_comparison(window_df):
    """
    Compares raw and engineered on the same file_id/window_index pairs.
    """
    raw = window_df[window_df["preprocess_mode"] == "raw_pdw_moe"].copy()
    eng = window_df[window_df["preprocess_mode"] == "engineered_pdw_moe"].copy()

    if len(raw) == 0 or len(eng) == 0:
        return pd.DataFrame()

    key_cols = ["file_id", "window_index"]

    metric_cols = [
        "v_measure",
        "ari",
        "ami",
        "homogeneity",
        "completeness",
        "pairwise_mcc",
        "pairwise_f1",
        "n_true_sources",
        "n_pred_clusters_excluding_noise",
        "noise_share",
        "n_regions",
        "xai_score",
    ]

    raw_cols = key_cols + metric_cols
    eng_cols = key_cols + metric_cols

    raw = raw[raw_cols].rename(columns={c: f"{c}_raw" for c in metric_cols})
    eng = eng[eng_cols].rename(columns={c: f"{c}_engineered" for c in metric_cols})

    df = raw.merge(eng, on=key_cols, how="inner")

    for c in metric_cols:
        df[f"{c}_delta_engineered_minus_raw"] = df[f"{c}_engineered"] - df[f"{c}_raw"]

    return df


def summarize_raw_vs_engineered_deltas(compare_df):
    if len(compare_df) == 0:
        return pd.DataFrame()

    delta_cols = [c for c in compare_df.columns if c.endswith("_delta_engineered_minus_raw")]

    rows = []

    for c in delta_cols:
        vals = pd.to_numeric(compare_df[c], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()

        if len(vals) == 0:
            continue

        rows.append({
            "metric_delta": c,
            "mean_delta": float(vals.mean()),
            "median_delta": float(vals.median()),
            "std_delta": float(vals.std()),
            "share_positive": float(np.mean(vals > 0)),
            "share_negative": float(np.mean(vals < 0)),
        })

    return pd.DataFrame(rows)


# ============================================================
# SUMMARY PLOTS
# ============================================================
def plot_summary(window_df, compare_df):
    if len(window_df) == 0:
        return

    metrics = [
        "v_measure",
        "ari",
        "ami",
        "homogeneity",
        "completeness",
        "pairwise_mcc",
        "noise_share",
        "xai_score",
        "n_regions",
    ]

    for m in metrics:
        if m not in window_df.columns:
            continue

        plt.figure(figsize=(8, 5))

        for mode, g in window_df.groupby("preprocess_mode"):
            vals = pd.to_numeric(g[m], errors="coerce").dropna()
            if len(vals) == 0:
                continue
            plt.hist(vals.values, bins=40, alpha=0.45, label=mode)

        plt.title(f"XAI video windows - {m}")
        plt.xlabel(m)
        plt.ylabel("Windows")
        plt.legend()
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"hist_{m}_by_mode.png", bbox_inches="tight")
        plt.close()

    if len(compare_df) > 0:
        delta_cols = [
            "v_measure_delta_engineered_minus_raw",
            "ari_delta_engineered_minus_raw",
            "ami_delta_engineered_minus_raw",
            "pairwise_mcc_delta_engineered_minus_raw",
            "noise_share_delta_engineered_minus_raw",
            "xai_score_delta_engineered_minus_raw",
        ]

        for c in delta_cols:
            if c not in compare_df.columns:
                continue

            vals = pd.to_numeric(compare_df[c], errors="coerce").dropna()
            if len(vals) == 0:
                continue

            plt.figure(figsize=(8, 5))
            plt.hist(vals.values, bins=40)
            plt.axvline(0.0, linestyle="--", linewidth=1.2)
            plt.title(c)
            plt.xlabel("engineered - raw")
            plt.ylabel("Windows")
            plt.tight_layout()
            plt.savefig(PLOTS_DIR / f"hist_{c}.png", bbox_inches="tight")
            plt.close()


# ============================================================
# MAIN
# ============================================================
def main():
    print("=" * 80)
    print("TSRD MoE DEINTERLEAVING - XAI VIDEO EXPORT RAW VS ENGINEERED")
    print("=" * 80)
    print(f"Device          : {DEVICE}")
    print(f"HDBSCAN backend : {HDBSCAN_BACKEND}")
    print(f"Base run dir    : {BASE_RUN_DIR}")
    print(f"Test folder     : {TEST_DIR}")
    print(f"Output dir      : {XAI_COMPARE_OUT_DIR}")

    test_files = collect_test_files()

    selected_files = test_files.head(MAX_FILES_FOR_VIDEO).copy().reset_index(drop=True)

    print("=" * 80)
    print("SELECTED FILES FOR VIDEO")
    print("=" * 80)
    print(selected_files[["file_id", "n_pulses", "n_windows"]].to_string(index=False))

    all_window_rows = []
    all_region_rows = []
    video_rows = []

    loaded_models = {}

    for preprocess_mode in PREPROCESS_MODES:
        model, feature_groups, run_dir = load_trained_model(preprocess_mode)
        loaded_models[preprocess_mode] = {
            "model": model,
            "feature_groups": feature_groups,
            "run_dir": run_dir,
        }

        mode_table_dir = TABLES_DIR / preprocess_mode
        mode_table_dir.mkdir(parents=True, exist_ok=True)

        for _, row in selected_files.iterrows():
            video_path, window_rows, region_rows = make_xai_video_for_file(
                model=model,
                file_row=row,
                preprocess_mode=preprocess_mode,
            )

            video_rows.append({
                "preprocess_mode": preprocess_mode,
                "file_id": row["file_id"],
                "file_path": row["path"],
                "video_path": str(video_path),
                "n_windows_video": len(window_rows),
                "n_regions": len(region_rows),
            })

            all_window_rows.extend(window_rows)
            all_region_rows.extend(region_rows)

    video_df = pd.DataFrame(video_rows)
    window_df = pd.DataFrame(all_window_rows)
    region_df = pd.DataFrame(all_region_rows)

    video_path_csv = TABLES_DIR / "xai_video_files_raw_vs_engineered.csv"
    window_metrics_path = TABLES_DIR / "xai_window_metrics_raw_vs_engineered.csv"
    regions_path = TABLES_DIR / "xai_detected_regions_raw_vs_engineered.csv"

    video_df.to_csv(video_path_csv, index=False, encoding="utf-8-sig")
    window_df.to_csv(window_metrics_path, index=False, encoding="utf-8-sig")
    region_df.to_csv(regions_path, index=False, encoding="utf-8-sig")

    summary_df = summarize_window_metrics(window_df)
    summary_path = TABLES_DIR / "xai_metric_summary_by_mode.csv"
    summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

    compare_df = build_raw_vs_engineered_window_comparison(window_df)
    compare_path = TABLES_DIR / "xai_raw_vs_engineered_by_window.csv"
    compare_df.to_csv(compare_path, index=False, encoding="utf-8-sig")

    delta_summary_df = summarize_raw_vs_engineered_deltas(compare_df)
    delta_summary_path = TABLES_DIR / "xai_raw_vs_engineered_delta_summary.csv"
    delta_summary_df.to_csv(delta_summary_path, index=False, encoding="utf-8-sig")

    plot_summary(window_df, compare_df)

    summary = {
        "root": str(ROOT),
        "base_run_dir": str(BASE_RUN_DIR),
        "test_dir": str(TEST_DIR),
        "n_experts": int(N_EXPERTS_XAI),
        "preprocess_modes": PREPROCESS_MODES,
        "xai_method": XAI_METHOD,
        "max_files_for_video": MAX_FILES_FOR_VIDEO,
        "max_windows_per_file_video": MAX_WINDOWS_PER_FILE_VIDEO,
        "video_fps": VIDEO_FPS,
        "hdbscan_backend": HDBSCAN_BACKEND,
        "hdbscan_min_cluster_size": HDBSCAN_MIN_CLUSTER_SIZE,
        "hdbscan_min_samples": HDBSCAN_MIN_SAMPLES,
        "feature_groups": {
            mode: describe_feature_groups(mode, loaded_models[mode]["feature_groups"])
            for mode in loaded_models
        },
        "outputs": {
            "videos_dir": str(VIDEOS_DIR),
            "video_index": str(video_path_csv),
            "window_metrics": str(window_metrics_path),
            "regions": str(regions_path),
            "summary_by_mode": str(summary_path),
            "raw_vs_engineered_by_window": str(compare_path),
            "raw_vs_engineered_delta_summary": str(delta_summary_path),
            "plots_dir": str(PLOTS_DIR),
        },
    }

    config_path = XAI_COMPARE_OUT_DIR / "xai_video_raw_vs_engineered_config.json"
    save_json(summary, config_path)

    print("\n" + "=" * 80)
    print("XAI VIDEO SUMMARY BY MODE")
    print("=" * 80)

    if len(summary_df) > 0:
        cols = [
            "preprocess_mode",
            "n_windows",
            "v_measure_median",
            "ari_median",
            "ami_median",
            "homogeneity_median",
            "completeness_median",
            "pairwise_mcc_median",
            "pairwise_f1_median",
            "n_true_sources_median",
            "n_pred_clusters_excluding_noise_median",
            "noise_share_median",
            "n_regions_median",
            "xai_score_median",
        ]
        cols = [c for c in cols if c in summary_df.columns]
        print(summary_df[cols].to_string(index=False))

    print("\n" + "=" * 80)
    print("RAW VS ENGINEERED DELTA SUMMARY")
    print("=" * 80)

    if len(delta_summary_df) > 0:
        print(delta_summary_df.to_string(index=False))
    else:
        print("No raw-vs-engineered delta comparison generated.")

    print("\n" + "=" * 80)
    print("OUTPUT FILES")
    print("=" * 80)
    print(f"Videos dir                  : {VIDEOS_DIR}")
    print(f"Video index                 : {video_path_csv}")
    print(f"Window metrics              : {window_metrics_path}")
    print(f"Regions CSV                 : {regions_path}")
    print(f"Summary by mode             : {summary_path}")
    print(f"Raw vs engineered by window : {compare_path}")
    print(f"Delta summary               : {delta_summary_path}")
    print(f"Plots dir                   : {PLOTS_DIR}")
    print(f"Config JSON                 : {config_path}")

    print("\nDone.")


if __name__ == "__main__":
    main()

TSRD MoE DEINTERLEAVING - XAI VIDEO EXPORT RAW VS ENGINEERED
Device          : cpu
HDBSCAN backend : hdbscan
Base run dir    : D:\Fusion\turing_synthetic_radar_dataset\benchmark_moe_pdw_raw_vs_engineered_sweep
Test folder     : D:\Fusion\turing_synthetic_radar_dataset\archive\test
Output dir      : D:\Fusion\turing_synthetic_radar_dataset\benchmark_moe_pdw_raw_vs_engineered_sweep\xai_video_raw_vs_engineered_N2
SCANNING TEST FILES
Folder: D:\Fusion\turing_synthetic_radar_dataset\archive\test
H5 files found: 250


index_test: 100%|██████████| 250/250 [00:16<00:00, 15.07file/s]


SELECTED FILES FOR VIDEO
 file_id  n_pulses  n_windows
test_160   1324601       1293
test_248   1033601       1009
 test_97    945573        923
test_192    942433        920
test_126    912355        890
test_118    840930        821
test_174    820813        801
test_206    816434        797
 test_90    812943        793
 test_91    807539        788
LOADED MODEL
Mode           : raw_pdw_moe
Run dir        : D:\Fusion\turing_synthetic_radar_dataset\benchmark_moe_pdw_raw_vs_engineered_sweep\raw_pdw_moe_N2
Model path     : D:\Fusion\turing_synthetic_radar_dataset\benchmark_moe_pdw_raw_vs_engineered_sweep\raw_pdw_moe_N2\best_model.pt
N experts      : 2
Input dim      : 5
Feature groups : [['ToA_us', 'PulseWidth_us'], ['Frequency_MHz', 'AoA_deg', 'Amplitude_dB']]
emb_dim        : 64
hidden_dim     : 128
drop           : 0.15
[VIDEO] mode=raw_pdw_moe | test_160 | windows=24 -> D:\Fusion\turing_synthetic_radar_dataset\benchmark_moe_pdw_raw_vs_engineered_sweep\xai_video_raw_vs_engineered_N2

[VIDEO] mode=raw_pdw_moe | test_248 | windows=24 -> D:\Fusion\turing_synthetic_radar_dataset\benchmark_moe_pdw_raw_vs_engineered_sweep\xai_video_raw_vs_engineered_N2\videos\raw_pdw_moe\xai_raw_pdw_moe_test_248.mp4


[VIDEO] mode=raw_pdw_moe | test_97 | windows=24 -> D:\Fusion\turing_synthetic_radar_dataset\benchmark_moe_pdw_raw_vs_engineered_sweep\xai_video_raw_vs_engineered_N2\videos\raw_pdw_moe\xai_raw_pdw_moe_test_97.mp4


...

[VIDEO] mode=engineered_pdw_moe | test_174 | windows=24 -> D:\Fusion\turing_synthetic_radar_dataset\benchmark_moe_pdw_raw_vs_engineered_sweep\xai_video_raw_vs_engineered_N2\videos\engineered_pdw_moe\xai_engineered_pdw_moe_test_174.mp4


[VIDEO] mode=engineered_pdw_moe | test_206 | windows=24 -> D:\Fusion\turing_synthetic_radar_dataset\benchmark_moe_pdw_raw_vs_engineered_sweep\xai_video_raw_vs_engineered_N2\videos\engineered_pdw_moe\xai_engineered_pdw_moe_test_206.mp4


[VIDEO] mode=engineered_pdw_moe | test_90 | windows=24 -> D:\Fusion\turing_synthetic_radar_dataset\benchmark_moe_pdw_raw_vs_engineered_sweep\xai_video_raw_vs_engineered_N2\videos\engineered_pdw_moe\xai_engineered_pdw_moe_test_90.mp4


[VIDEO] mode=engineered_pdw_moe | test_91 | windows=24 -> D:\Fusion\turing_synthetic_radar_dataset\benchmark_moe_pdw_raw_vs_engineered_sweep\xai_video_raw_vs_engineered_N2\videos\engineered_pdw_moe\xai_engineered_pdw_moe_test_91.mp4



XAI VIDEO SUMMARY BY MODE
   preprocess_mode  n_windows  v_measure_median  ari_median  ami_median  homogeneity_median  completeness_median  pairwise_mcc_median  pairwise_f1_median  n_true_sources_median  n_pred_clusters_excluding_noise_median  noise_share_median  n_regions_median  xai_score_median
engineered_pdw_moe        240          0.950921    0.998407    0.950505            0.988060             0.997342             0.998408            0.999621                    4.0                                     3.0            0.005859               8.0          0.992305
       raw_pdw_moe        240          0.927575    0.985511    0.926606            0.981087             0.942962             0.985605            0.995280                    4.0                                     4.0            0.007812               7.0          0.993620

RAW VS ENGINEERED DELTA SUMMARY
                                              metric_delta  mean_delta  median_delta  std_delta  share_positive  share_ne